In [ ]:
# ============================================================
# CELL 1 — NEPSE ML PROJECT SETUP
# ============================================================

import os
import sys
import json
import gzip
import pandas as pd
import numpy as np
from datetime import datetime

PROJECT_ROOT = "/content/NEPSE_ML_PROJECT"

RAW_DIR = os.path.join(PROJECT_ROOT, "raw")
VALIDATION_DIR = os.path.join(PROJECT_ROOT, "validation")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "processed")
FEATURES_DIR = os.path.join(PROJECT_ROOT, "features")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")

for folder in [
    PROJECT_ROOT,
    RAW_DIR,
    VALIDATION_DIR,
    PROCESSED_DIR,
    FEATURES_DIR,
    MODELS_DIR,
    REPORTS_DIR,
]:
    os.makedirs(folder, exist_ok=True)

print("============================================================")
print("NEPSE ML PROJECT")
print("============================================================")
print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data     : {RAW_DIR}")
print(f"Validation   : {VALIDATION_DIR}")
print(f"Processed    : {PROCESSED_DIR}")
print(f"Features     : {FEATURES_DIR}")
print(f"Models       : {MODELS_DIR}")
print(f"Reports      : {REPORTS_DIR}")
print("============================================================")
print("Setup complete.")

NEPSE ML PROJECT
Project root : /content/NEPSE_ML_PROJECT
Raw data     : /content/NEPSE_ML_PROJECT/raw
Validation   : /content/NEPSE_ML_PROJECT/validation
Processed    : /content/NEPSE_ML_PROJECT/processed
Features     : /content/NEPSE_ML_PROJECT/features
Models       : /content/NEPSE_ML_PROJECT/models
Reports      : /content/NEPSE_ML_PROJECT/reports
Setup complete.


In [ ]:
# ============================================================
# CELL 2 — UPLOAD RAW DATA
# ============================================================

from google.colab import files
import os
import shutil

print("Select your raw NEPSE files.")
print("You can select all required files at once.")
print()

uploaded = files.upload()

for filename in uploaded.keys():
    destination = os.path.join(RAW_DIR, filename)

    # Do not overwrite an existing raw file accidentally
    if os.path.exists(destination):
        print(f"SKIPPED (already exists): {filename}")
    else:
        shutil.move(filename, destination)
        print(f"COPIED: {filename}")

print()
print("============================================================")
print("RAW DIRECTORY CONTENTS")
print("============================================================")

for filename in sorted(os.listdir(RAW_DIR)):
    path = os.path.join(RAW_DIR, filename)
    size_mb = os.path.getsize(path) / (1024 * 1024)

    print(f"{filename:<55} {size_mb:>8.2f} MB")

print("============================================================")

Select your raw NEPSE files.
You can select all required files at once.



Saving nepse_daily_ohlcv.json.gz to nepse_daily_ohlcv.json.gz
Saving nepse_daily_ohlcv_unadjusted.json.gz to nepse_daily_ohlcv_unadjusted.json.gz
Saving nepse_daily_ohlcv_unadjusted_errors.json to nepse_daily_ohlcv_unadjusted_errors.json
Saving nepse_dividends.json.gz to nepse_dividends.json.gz
Saving nepse_download_errors.json to nepse_download_errors.json
Saving nepse_marks.json.gz to nepse_marks.json.gz
Saving nepse_marks_errors.json to nepse_marks_errors.json
Saving nepse_rights_issues.json.gz to nepse_rights_issues.json.gz
Saving nepse_symbols.json to nepse_symbols.json
Saving nepsealpha_log.json to nepsealpha_log.json
Saving nepsealpha_symbols.csv to nepsealpha_symbols.csv
Saving nepsealpha_symbols.json to nepsealpha_symbols.json
COPIED: nepse_daily_ohlcv.json.gz
COPIED: nepse_daily_ohlcv_unadjusted.json.gz
COPIED: nepse_daily_ohlcv_unadjusted_errors.json
COPIED: nepse_dividends.json.gz
COPIED: nepse_download_errors.json
COPIED: nepse_marks.json.gz
COPIED: nepse_marks_errors.json

In [ ]:
# ============================================================
# CELL 3 — VERIFY RAW DATA FILES
# ============================================================

expected_files = [
    "nepse_symbols.json",
    "nepse_daily_ohlcv.json.gz",
    "nepse_daily_ohlcv_unadjusted.json.gz",
    "nepse_marks.json.gz",
    "nepse_rights_issues.json.gz",
    "nepse_dividends.json.gz",
]

optional_files = [
    "nepse_download_errors.json",
    "nepse_marks_errors.json",
    "nepse_daily_ohlcv_unadjusted_errors.json",
]

print("============================================================")
print("REQUIRED FILES")
print("============================================================")

missing = []

for filename in expected_files:
    path = os.path.join(RAW_DIR, filename)

    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"OK       {filename:<50} {size_mb:8.2f} MB")
    else:
        print(f"MISSING  {filename}")
        missing.append(filename)

print()
print("============================================================")
print("OPTIONAL / AUDIT FILES")
print("============================================================")

for filename in optional_files:
    path = os.path.join(RAW_DIR, filename)

    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"FOUND    {filename:<50} {size_kb:8.2f} KB")
    else:
        print(f"NOT FOUND {filename}")

print()
print("============================================================")

if missing:
    print("WARNING: Required files are missing.")
    print("Missing:")
    for x in missing:
        print(" -", x)
else:
    print("ALL REQUIRED RAW FILES FOUND.")

print("============================================================")

REQUIRED FILES
OK       nepse_symbols.json                                     0.23 MB
OK       nepse_daily_ohlcv.json.gz                             13.09 MB
OK       nepse_daily_ohlcv_unadjusted.json.gz                  12.47 MB
OK       nepse_marks.json.gz                                    0.29 MB
OK       nepse_rights_issues.json.gz                            0.01 MB
OK       nepse_dividends.json.gz                                0.08 MB

OPTIONAL / AUDIT FILES
FOUND    nepse_download_errors.json                             0.00 KB
FOUND    nepse_marks_errors.json                                0.00 KB
FOUND    nepse_daily_ohlcv_unadjusted_errors.json               0.00 KB

ALL REQUIRED RAW FILES FOUND.


In [ ]:
# ============================================================
# CELL 4 — RAW DATA SNAPSHOT
# ============================================================

from datetime import datetime, timezone
import hashlib

snapshot = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "files": {}
}

for filename in sorted(os.listdir(RAW_DIR)):
    path = os.path.join(RAW_DIR, filename)

    if not os.path.isfile(path):
        continue

    size = os.path.getsize(path)

    # SHA256 gives us a fingerprint of the original raw file
    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            sha256.update(chunk)

    snapshot["files"][filename] = {
        "size_bytes": size,
        "sha256": sha256.hexdigest()
    }

snapshot_path = os.path.join(
    VALIDATION_DIR,
    "raw_data_snapshot.json"
)

with open(snapshot_path, "w", encoding="utf-8") as f:
    json.dump(snapshot, f, indent=2)

print("============================================================")
print("RAW DATA SNAPSHOT CREATED")
print("============================================================")
print(f"File: {snapshot_path}")
print(f"Files recorded: {len(snapshot['files'])}")
print()
print("This snapshot lets us verify later that the raw files")
print("have not been accidentally modified.")
print("============================================================")

RAW DATA SNAPSHOT CREATED
File: /content/NEPSE_ML_PROJECT/validation/raw_data_snapshot.json
Files recorded: 12

This snapshot lets us verify later that the raw files
have not been accidentally modified.


In [ ]:
# ============================================================
# CELL 5 — LOAD RAW DATASETS
# ============================================================

import json
import gzip
import os
import pandas as pd
import numpy as np

print("============================================================")
print("LOADING RAW NEPSE DATA")
print("============================================================")


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_gzip_json(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return json.load(f)


# ------------------------------------------------------------
# 1. Securities universe
# ------------------------------------------------------------

symbols_path = os.path.join(RAW_DIR, "nepse_symbols.json")
symbols_raw = load_json(symbols_path)

print(f"Symbols loaded: {len(symbols_raw):,}")


# ------------------------------------------------------------
# 2. Adjusted OHLCV
# ------------------------------------------------------------

adjusted_path = os.path.join(
    RAW_DIR,
    "nepse_daily_ohlcv.json.gz"
)

adjusted_raw = load_gzip_json(adjusted_path)

print(f"Adjusted OHLCV securities: {len(adjusted_raw):,}")


# ------------------------------------------------------------
# 3. Unadjusted OHLCV
# ------------------------------------------------------------

unadjusted_path = os.path.join(
    RAW_DIR,
    "nepse_daily_ohlcv_unadjusted.json.gz"
)

unadjusted_raw = load_gzip_json(unadjusted_path)

print(f"Unadjusted OHLCV securities: {len(unadjusted_raw):,}")


# ------------------------------------------------------------
# 4. Marks / events
# ------------------------------------------------------------

marks_path = os.path.join(
    RAW_DIR,
    "nepse_marks.json.gz"
)

marks_raw = load_gzip_json(marks_path)

print(f"Marks securities: {len(marks_raw):,}")


# ------------------------------------------------------------
# 5. Rights issues
# ------------------------------------------------------------

rights_path = os.path.join(
    RAW_DIR,
    "nepse_rights_issues.json.gz"
)

rights_raw = load_gzip_json(rights_path)

print(f"Rights records: {len(rights_raw):,}")


# ------------------------------------------------------------
# 6. Dividends
# ------------------------------------------------------------

dividends_path = os.path.join(
    RAW_DIR,
    "nepse_dividends.json.gz"
)

dividends_raw = load_gzip_json(dividends_path)

print(f"Dividend records: {len(dividends_raw):,}")


print()
print("============================================================")
print("ALL RAW DATASETS LOADED SUCCESSFULLY")
print("============================================================")

LOADING RAW NEPSE DATA
Symbols loaded: 737
Adjusted OHLCV securities: 633,137
Unadjusted OHLCV securities: 7
Marks securities: 14,087
Rights records: 7
Dividend records: 7

ALL RAW DATASETS LOADED SUCCESSFULLY


In [ ]:
# ============================================================
# CELL 6 — INSPECT RAW DATA STRUCTURE
# ============================================================

def describe_object(name, obj):
    print()
    print("------------------------------------------------------------")
    print(name)
    print("------------------------------------------------------------")
    print("Python type:", type(obj).__name__)

    if isinstance(obj, dict):
        print("Dictionary entries:", len(obj))

        keys = list(obj.keys())[:10]
        print("First keys:", keys)

        if keys:
            first_value = obj[keys[0]]

            print(
                "First value type:",
                type(first_value).__name__
            )

            if isinstance(first_value, dict):
                print(
                    "First value keys:",
                    list(first_value.keys())[:20]
                )

            elif isinstance(first_value, list):
                print(
                    "First value list length:",
                    len(first_value)
                )

    elif isinstance(obj, list):
        print("List length:", len(obj))

        if len(obj) > 0:
            print(
                "First item type:",
                type(obj[0]).__name__
            )

            if isinstance(obj[0], dict):
                print(
                    "First item keys:",
                    list(obj[0].keys())[:20]
                )


describe_object("SYMBOLS", symbols_raw)
describe_object("ADJUSTED OHLCV", adjusted_raw)
describe_object("UNADJUSTED OHLCV", unadjusted_raw)
describe_object("MARKS", marks_raw)
describe_object("RIGHTS", rights_raw)
describe_object("DIVIDENDS", dividends_raw)

print()
print("============================================================")
print("STRUCTURE INSPECTION COMPLETE")
print("============================================================")


------------------------------------------------------------
SYMBOLS
------------------------------------------------------------
Python type: list
List length: 737
First item type: dict
First item keys: ['symbol', 'full_name', 'description', 'exchange', 'type', 'sector', 'logo_urls', 'exchange_logo', 'is_master']

------------------------------------------------------------
ADJUSTED OHLCV
------------------------------------------------------------
Python type: list
List length: 633137
First item type: dict
First item keys: ['date', 'timestamp', 'symbol', 'open', 'high', 'low', 'close', 'volume']

------------------------------------------------------------
UNADJUSTED OHLCV
------------------------------------------------------------
Python type: dict
Dictionary entries: 7
First keys: ['dataset', 'adjusted', 'endpoint', 'resolution', 'frame', 'downloaded_at', 'securities']
First value type: str

------------------------------------------------------------
MARKS
----------------------

In [ ]:
# ============================================================
# CELL 7 — VALIDATE SECURITIES UNIVERSE
# ============================================================

print("============================================================")
print("SECURITIES UNIVERSE VALIDATION")
print("============================================================")

# Convert to DataFrame
symbols_df = pd.DataFrame(symbols_raw)

print(f"Total securities: {len(symbols_df):,}")
print(f"Total columns:    {len(symbols_df.columns):,}")

print()
print("Columns:")
for col in symbols_df.columns:
    print(" -", col)


# ------------------------------------------------------------
# Find likely symbol column
# ------------------------------------------------------------

possible_symbol_columns = [
    "symbol",
    "Symbol",
    "SYMBOL",
    "ticker",
    "code",
    "security_symbol"
]

symbol_column = None

for col in possible_symbol_columns:
    if col in symbols_df.columns:
        symbol_column = col
        break

print()

if symbol_column:
    print(f"Detected symbol column: {symbol_column}")

    symbols = (
        symbols_df[symbol_column]
        .astype(str)
        .str.strip()
    )

    print(
        f"Unique symbols: {symbols.nunique():,}"
    )

    duplicate_symbols = symbols[
        symbols.duplicated(keep=False)
    ].sort_values()

    print(
        f"Duplicate symbol rows: {len(duplicate_symbols):,}"
    )

    if len(duplicate_symbols) > 0:
        print()
        print("Duplicate symbols:")
        print(
            duplicate_symbols
            .drop_duplicates()
            .head(30)
            .to_list()
        )

else:
    print(
        "WARNING: Could not automatically identify "
        "the symbol column."
    )


# ------------------------------------------------------------
# Save a validation summary
# ------------------------------------------------------------

universe_summary = {
    "total_rows": int(len(symbols_df)),
    "total_columns": int(len(symbols_df.columns)),
    "symbol_column": symbol_column,
    "unique_symbols": (
        int(symbols.nunique())
        if symbol_column
        else None
    ),
    "duplicate_symbol_rows": (
        int(len(duplicate_symbols))
        if symbol_column
        else None
    )
}

summary_path = os.path.join(
    VALIDATION_DIR,
    "securities_universe_summary.json"
)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(
        universe_summary,
        f,
        indent=2
    )

print()
print("============================================================")
print("UNIVERSE VALIDATION COMPLETE")
print(f"Summary saved: {summary_path}")
print("============================================================")

SECURITIES UNIVERSE VALIDATION
Total securities: 737
Total columns:    9

Columns:
 - symbol
 - full_name
 - description
 - exchange
 - type
 - sector
 - logo_urls
 - exchange_logo
 - is_master

Detected symbol column: symbol
Unique symbols: 737
Duplicate symbol rows: 0

UNIVERSE VALIDATION COMPLETE
Summary saved: /content/NEPSE_ML_PROJECT/validation/securities_universe_summary.json


In [ ]:
# ============================================================
# CELL 8 — NORMALIZE DATA STRUCTURES FOR ANALYSIS
# ============================================================

import pandas as pd
import numpy as np
import json
import os

# Adjusted OHLCV is already a list of row dictionaries
adjusted_df = pd.DataFrame(adjusted_raw)

# Unadjusted data is wrapped in a metadata dictionary
unadjusted_records = unadjusted_raw.get("securities", [])

# Marks is already a list of event dictionaries
marks_df = pd.DataFrame(marks_raw)

# Rights and dividends are wrapped in metadata dictionaries
rights_records = rights_raw.get("data", [])
dividend_records = dividends_raw.get("data", [])

unadjusted_df = pd.DataFrame(unadjusted_records)
rights_df = pd.DataFrame(rights_records)
dividends_df = pd.DataFrame(dividend_records)

print("=" * 65)
print("NORMALIZED DATASET SHAPES")
print("=" * 65)

for name, df in [
    ("Adjusted OHLCV", adjusted_df),
    ("Unadjusted OHLCV", unadjusted_df),
    ("Marks", marks_df),
    ("Rights", rights_df),
    ("Dividends", dividends_df),
]:
    print(f"{name:<22} rows={len(df):>9,} | columns={len(df.columns):>3}")

print("\nUNADJUSTED METADATA")
for key, value in unadjusted_raw.items():
    if key != "securities":
        print(f"{key}: {value}")

print("\nRIGHTS METADATA")
for key, value in rights_raw.items():
    if key != "data":
        print(f"{key}: {value}")

print("\nDIVIDENDS METADATA")
for key, value in dividends_raw.items():
    if key != "data":
        print(f"{key}: {value}")

print("\nCell 8 complete.")

NORMALIZED DATASET SHAPES
Adjusted OHLCV         rows=  633,137 | columns=  8
Unadjusted OHLCV       rows=      737 | columns= 14
Marks                  rows=   14,087 | columns=  9
Rights                 rows=      312 | columns= 16
Dividends              rows=    1,683 | columns= 17

UNADJUSTED METADATA
dataset: NEPSE Daily OHLCV — Unadjusted
adjusted: False
endpoint: /trading/0/history
resolution: 1D
frame: 1000
downloaded_at: 2026-10-06T13:04:39.739Z

RIGHTS METADATA
dataset: nepse_rights_issues
source: NepseAlpha
collected_at: 2026-10-06T12:34:03.358Z
description: Historical rights issue records collected from NepseAlpha investment calendar.
record_count: 312
fields: ['id', 'symbol', 'company_name', 'ratio', 'units_raw', 'units', 'opening_date', 'closing_date', 'book_closure_date', 'issue_manager', 'created_at', 'updated_at', 'status_raw', 'status']

DIVIDENDS METADATA
dataset: nepse_dividends
source: NepseAlpha
collected_at: 2026-10-06T12:44:13.965Z
fiscal_year_range: {'from': '2

In [ ]:
# ============================================================
# CELL 9 — OHLCV STRUCTURE CHECK
# ============================================================

for name, df in [
    ("ADJUSTED OHLCV", adjusted_df),
    ("UNADJUSTED OHLCV", unadjusted_df),
]:
    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

    print("Rows:", f"{len(df):,}")
    print("Columns:", list(df.columns))

    if len(df) > 0:
        print("\nData types:")
        print(df.dtypes.to_string())

        print("\nMissing values per column:")
        print(df.isna().sum().to_string())

        required = [
            "date", "timestamp", "symbol",
            "open", "high", "low", "close", "volume"
        ]

        missing_columns = [
            col for col in required if col not in df.columns
        ]

        print("\nMissing required columns:", missing_columns)

print("\nCell 9 complete.")


ADJUSTED OHLCV
Rows: 633,137
Columns: ['date', 'timestamp', 'symbol', 'open', 'high', 'low', 'close', 'volume']

Data types:
date          object
timestamp      int64
symbol        object
open         float64
high         float64
low          float64
close        float64
volume       float64

Missing values per column:
date         0
timestamp    0
symbol       0
open         0
high         0
low          0
close        0
volume       0

Missing required columns: []

UNADJUSTED OHLCV
Rows: 737
Columns: ['symbol', 'timestamps', 'opens', 'highs', 'lows', 'closes', 'volumes', 'status', 'resolution', 'frame', 'adjusted', 'endpoint', 'row_count', 'data']

Data types:
symbol         object
timestamps     object
opens          object
highs          object
lows           object
closes         object
volumes        object
status         object
resolution     object
frame         float64
adjusted       object
endpoint       object
row_count     float64
data           object

Missing values per 

In [ ]:
# ============================================================
# CELL 10D — DEFINITIVE UNADJUSTED DATA INTEGRITY CHECK
# ============================================================

print("=" * 65)
print("UNADJUSTED DATA INTEGRITY CHECK")
print("=" * 65)

total_records = len(unadjusted_raw["securities"])

status_counts = {}
records_with_data = 0
total_array_rows = 0

array_fields = [
    "timestamps",
    "opens",
    "highs",
    "lows",
    "closes",
    "volumes"
]

array_lengths = []

for record in unadjusted_raw["securities"]:

    symbol = record.get("symbol", "UNKNOWN")
    status = record.get("status", "UNKNOWN")

    status_counts[status] = status_counts.get(status, 0) + 1

    timestamps = record.get("timestamps")

    if isinstance(timestamps, list) and len(timestamps) > 0:

        records_with_data += 1

        length = len(timestamps)
        array_lengths.append(length)

        total_array_rows += length


print()
print("Total security records:", f"{total_records:,}")

print()
print("STATUS COUNTS")
print("-" * 65)

for status, count in sorted(
    status_counts.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{status:<20} {count:>8,}")


print()
print("RECORDS CONTAINING OHLCV")
print("-" * 65)

print(
    "Securities with timestamps:",
    f"{records_with_data:,}"
)

print(
    "Securities without timestamps:",
    f"{total_records - records_with_data:,}"
)

print(
    "Total daily rows represented:",
    f"{total_array_rows:,}"
)


print()
print("=" * 65)

if records_with_data == 0:
    print("CRITICAL FINDING:")
    print()
    print("The current unadjusted JSON file contains")
    print("ZERO actual OHLCV observations.")
    print()
    print("It contains 737 security metadata/status records,")
    print("but no timestamp/OHLCV arrays.")
    print()
    print("The earlier reported 642,059 rows therefore came")
    print("from the collection process/log, not from this")
    print("saved JSON file.")
else:
    print("Unadjusted OHLCV data is present.")
    print(
        f"Rows represented: {total_array_rows:,}"
    )

print("=" * 65)

UNADJUSTED DATA INTEGRITY CHECK

Total security records: 737

STATUS COUNTS
-----------------------------------------------------------------
UNKNOWN                   707
no_data                    30

RECORDS CONTAINING OHLCV
-----------------------------------------------------------------
Securities with timestamps: 0
Securities without timestamps: 737
Total daily rows represented: 0

CRITICAL FINDING:

The current unadjusted JSON file contains
ZERO actual OHLCV observations.

It contains 737 security metadata/status records,
but no timestamp/OHLCV arrays.

The earlier reported 642,059 rows therefore came
from the collection process/log, not from this
saved JSON file.


In [ ]:
# ============================================================
# CELL 11 — INSPECT UNADJUSTED RECORD TYPES
# ============================================================

print("=" * 65)
print("UNADJUSTED RECORD DIAGNOSTIC")
print("=" * 65)

unknown_records = []
no_data_records = []

for record in unadjusted_raw["securities"]:

    status = record.get("status")

    if status == "no_data":
        no_data_records.append(record)
    else:
        unknown_records.append(record)


print()
print("UNKNOWN records:", len(unknown_records))
print("no_data records:", len(no_data_records))


# ------------------------------------------------------------
# Inspect representative UNKNOWN records
# ------------------------------------------------------------

print()
print("-" * 65)
print("UNKNOWN RECORD STRUCTURE")
print("-" * 65)

for i, record in enumerate(unknown_records[:5]):

    print()
    print(f"Record #{i + 1}")
    print("Symbol:", record.get("symbol"))
    print("Keys:", list(record.keys()))

    for key in [
        "timestamps",
        "opens",
        "highs",
        "lows",
        "closes",
        "volumes",
        "status"
    ]:
        value = record.get(key)

        if isinstance(value, list):
            print(
                f"{key:<12}: list length = {len(value):,}"
            )
        else:
            print(
                f"{key:<12}: {repr(value)[:100]}"
            )


# ------------------------------------------------------------
# Inspect representative no_data records
# ------------------------------------------------------------

print()
print("-" * 65)
print("NO_DATA RECORD STRUCTURE")
print("-" * 65)

for i, record in enumerate(no_data_records[:3]):

    print()
    print(f"Record #{i + 1}")
    print("Symbol:", record.get("symbol"))
    print("Keys:", list(record.keys()))
    print("Status:", record.get("status"))

print()
print("=" * 65)
print("DIAGNOSTIC COMPLETE")
print("=" * 65)

UNADJUSTED RECORD DIAGNOSTIC

UNKNOWN records: 707
no_data records: 30

-----------------------------------------------------------------
UNKNOWN RECORD STRUCTURE
-----------------------------------------------------------------

Record #1
Symbol: ACLBSL
Keys: ['symbol', 'resolution', 'frame', 'adjusted', 'endpoint', 'row_count', 'data']
timestamps  : None
opens       : None
highs       : None
lows        : None
closes      : None
volumes     : None
status      : None

Record #2
Symbol: ADBL
Keys: ['symbol', 'resolution', 'frame', 'adjusted', 'endpoint', 'row_count', 'data']
timestamps  : None
opens       : None
highs       : None
lows        : None
closes      : None
volumes     : None
status      : None

Record #3
Symbol: ADBLB
Keys: ['symbol', 'resolution', 'frame', 'adjusted', 'endpoint', 'row_count', 'data']
timestamps  : None
opens       : None
highs       : None
lows        : None
closes      : None
volumes     : None
status      : None

Record #4
Symbol: ADBLB86
Keys: ['symbol'

In [ ]:
# ============================================================
# CELL 12 — INSPECT POPULATED UNADJUSTED DATA PAYLOAD
# ============================================================

print("=" * 65)
print("INSPECTING ACTUAL UNADJUSTED OHLCV PAYLOAD")
print("=" * 65)

populated_record = None

for record in unadjusted_raw["securities"]:

    data = record.get("data")

    if data is not None:
        populated_record = record
        break


if populated_record is None:

    print()
    print("NO RECORD WITH 'data' FOUND.")
    print("This would indicate a different problem.")

else:

    print()
    print("Symbol:", populated_record.get("symbol"))
    print("Endpoint:", populated_record.get("endpoint"))
    print("Adjusted:", populated_record.get("adjusted"))
    print("Resolution:", populated_record.get("resolution"))
    print("Frame:", populated_record.get("frame"))
    print("Row count:", populated_record.get("row_count"))

    data = populated_record.get("data")

    print()
    print("-" * 65)
    print("DATA FIELD")
    print("-" * 65)

    print("Type:", type(data).__name__)

    if isinstance(data, dict):

        print("Keys:")

        for key, value in data.items():

            if isinstance(value, list):
                print(
                    f"  {key:<15} -> list, "
                    f"length={len(value):,}"
                )

            elif isinstance(value, dict):
                print(
                    f"  {key:<15} -> dict, "
                    f"keys={list(value.keys())[:15]}"
                )

            else:
                print(
                    f"  {key:<15} -> "
                    f"{type(value).__name__}"
                )

    elif isinstance(data, list):

        print("List length:", len(data))

        if data:
            print(
                "First item type:",
                type(data[0]).__name__
            )

            if isinstance(data[0], dict):
                print(
                    "First item keys:",
                    list(data[0].keys())
                )

    else:

        print(
            "Data representation:",
            str(data)[:500]
        )


print()
print("=" * 65)
print("PAYLOAD INSPECTION COMPLETE")
print("=" * 65)

INSPECTING ACTUAL UNADJUSTED OHLCV PAYLOAD

Symbol: ACLBSL
Endpoint: /trading/0/history
Adjusted: False
Resolution: 1D
Frame: 1000
Row count: 1375

-----------------------------------------------------------------
DATA FIELD
-----------------------------------------------------------------
Type: list
List length: 1375
First item type: dict
First item keys: ['date', 'timestamp', 'open', 'high', 'low', 'close', 'volume']

PAYLOAD INSPECTION COMPLETE


In [ ]:
# ============================================================
# CELL 13 — FLATTEN UNADJUSTED OHLCV DATA
# ============================================================

print("=" * 65)
print("FLATTENING UNADJUSTED OHLCV DATA")
print("=" * 65)

unadjusted_rows = []

no_data_symbols = []
populated_symbols = []

expected_rows = 0
actual_rows = 0

for record in unadjusted_raw["securities"]:

    symbol = record.get("symbol")
    data = record.get("data")

    # --------------------------------------------------------
    # NO-DATA RECORD
    # --------------------------------------------------------
    if data is None:

        no_data_symbols.append(symbol)
        continue

    # --------------------------------------------------------
    # POPULATED RECORD
    # --------------------------------------------------------
    if isinstance(data, list):

        populated_symbols.append(symbol)

        expected_rows += record.get("row_count", len(data))

        for row in data:

            # Add symbol from parent record
            row_copy = row.copy()
            row_copy["symbol"] = symbol

            unadjusted_rows.append(row_copy)

            actual_rows += 1


# ------------------------------------------------------------
# CREATE DATAFRAME
# ------------------------------------------------------------

unadjusted_flat_df = pd.DataFrame(unadjusted_rows)

# Put columns in canonical order
required_columns = [
    "date",
    "timestamp",
    "symbol",
    "open",
    "high",
    "low",
    "close",
    "volume"
]

if all(col in unadjusted_flat_df.columns for col in required_columns):

    unadjusted_flat_df = unadjusted_flat_df[required_columns]


# ------------------------------------------------------------
# BASIC RESULTS
# ------------------------------------------------------------

print()
print("SECURITY COUNTS")
print("-" * 65)

print("Total security records :", len(unadjusted_raw["securities"]))
print("Populated securities   :", len(populated_symbols))
print("No-data securities     :", len(no_data_symbols))

print()
print("ROW COUNTS")
print("-" * 65)

print("Expected rows from row_count :", f"{expected_rows:,}")
print("Actual flattened rows        :", f"{actual_rows:,}")

print()
print("ROW COUNT MATCH")

if expected_rows == actual_rows:

    print("PASS — expected and actual row counts match.")

else:

    print("WARNING — row count mismatch.")
    print("Difference:", actual_rows - expected_rows)


# ------------------------------------------------------------
# DATAFRAME STRUCTURE
# ------------------------------------------------------------

print()
print("DATAFRAME")
print("-" * 65)

print("Rows :", f"{len(unadjusted_flat_df):,}")
print("Cols :", len(unadjusted_flat_df.columns))

print("Columns:")
print(list(unadjusted_flat_df.columns))


# ------------------------------------------------------------
# DUPLICATE CHECK
# ------------------------------------------------------------

print()
print("DUPLICATE CHECK")
print("-" * 65)

duplicate_rows = unadjusted_flat_df.duplicated(
    subset=["symbol", "date"]
).sum()

print(
    "Duplicate symbol/date rows:",
    f"{duplicate_rows:,}"
)


# ------------------------------------------------------------
# DATE RANGE
# ------------------------------------------------------------

if len(unadjusted_flat_df) > 0:

    print()
    print("DATE RANGE")
    print("-" * 65)

    print(
        "Earliest date:",
        unadjusted_flat_df["date"].min()
    )

    print(
        "Latest date  :",
        unadjusted_flat_df["date"].max()
    )


# ------------------------------------------------------------
# SAVE ONLY TO PROCESSED
# ------------------------------------------------------------

output_path = (
    "/content/NEPSE_ML_PROJECT/processed/"
    "nepse_daily_ohlcv_unadjusted_flat.csv.gz"
)

unadjusted_flat_df.to_csv(
    output_path,
    index=False,
    compression="gzip"
)

print()
print("Saved flattened dataset:")
print(output_path)

print()
print("=" * 65)
print("UNADJUSTED OHLCV FLATTENING COMPLETE")
print("=" * 65)

FLATTENING UNADJUSTED OHLCV DATA

SECURITY COUNTS
-----------------------------------------------------------------
Total security records : 737
Populated securities   : 707
No-data securities     : 30

ROW COUNTS
-----------------------------------------------------------------
Expected rows from row_count : 642,059
Actual flattened rows        : 642,059

ROW COUNT MATCH
PASS — expected and actual row counts match.

DATAFRAME
-----------------------------------------------------------------
Rows : 642,059
Cols : 8
Columns:
['date', 'timestamp', 'symbol', 'open', 'high', 'low', 'close', 'volume']

DUPLICATE CHECK
-----------------------------------------------------------------
Duplicate symbol/date rows: 0

DATE RANGE
-----------------------------------------------------------------
Earliest date: 1997-07-20
Latest date  : 2026-10-06

Saved flattened dataset:
/content/NEPSE_ML_PROJECT/processed/nepse_daily_ohlcv_unadjusted_flat.csv.gz

UNADJUSTED OHLCV FLATTENING COMPLETE


In [ ]:
# ============================================================
# CELL 14 — OHLCV INTEGRITY AUDIT
# ============================================================

print("=" * 70)
print("OHLCV INTEGRITY AUDIT")
print("=" * 70)

import pandas as pd
import numpy as np
import json
import os


# ============================================================
# HELPER FUNCTION
# ============================================================

def audit_ohlcv(df, dataset_name):

    print()
    print("=" * 70)
    print(dataset_name)
    print("=" * 70)

    required = [
        "date",
        "timestamp",
        "symbol",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]

    result = {
        "dataset": dataset_name,
        "rows": int(len(df)),
        "columns": list(df.columns),
        "missing_required_columns": [],
        "missing_values": {},
        "duplicate_symbol_date": 0,
        "duplicate_timestamps": 0,
        "invalid_dates": 0,
        "timestamp_date_mismatch": 0,
        "non_chronological_rows": 0,
        "zero_or_negative_open": 0,
        "zero_or_negative_high": 0,
        "zero_or_negative_low": 0,
        "zero_or_negative_close": 0,
        "negative_volume": 0,
        "zero_volume": 0,
        "low_greater_high": 0,
        "open_outside_range": 0,
        "close_outside_range": 0,
        "high_below_open": 0,
        "high_below_close": 0,
        "low_above_open": 0,
        "low_above_close": 0
    }


    # --------------------------------------------------------
    # REQUIRED COLUMNS
    # --------------------------------------------------------

    result["missing_required_columns"] = [
        col for col in required
        if col not in df.columns
    ]

    if result["missing_required_columns"]:

        print(
            "MISSING REQUIRED COLUMNS:",
            result["missing_required_columns"]
        )

        return result


    # --------------------------------------------------------
    # MISSING VALUES
    # --------------------------------------------------------

    missing = df[required].isna().sum()

    result["missing_values"] = {
        col: int(value)
        for col, value in missing.items()
    }

    print()
    print("MISSING VALUES")
    print("-" * 70)

    for col in required:

        print(
            f"{col:<12}: "
            f"{result['missing_values'][col]:,}"
        )


    # --------------------------------------------------------
    # DATE PARSING
    # --------------------------------------------------------

    dates = pd.to_datetime(
        df["date"],
        errors="coerce"
    )

    result["invalid_dates"] = int(
        dates.isna().sum()
    )

    print()
    print("DATE VALIDITY")
    print("-" * 70)

    print(
        "Invalid dates:",
        f"{result['invalid_dates']:,}"
    )


    # --------------------------------------------------------
    # TIMESTAMP CHECK
    # --------------------------------------------------------

    timestamps = pd.to_numeric(
        df["timestamp"],
        errors="coerce"
    )

    valid_timestamp = timestamps.notna() & dates.notna()

    if valid_timestamp.any():

        # NepseAlpha timestamps are Unix seconds.
        timestamp_dates = pd.to_datetime(
            timestamps[valid_timestamp],
            unit="s",
            errors="coerce"
        ).dt.strftime("%Y-%m-%d")

        date_strings = dates[valid_timestamp].dt.strftime(
            "%Y-%m-%d"
        )

        mismatch = (
            timestamp_dates.values !=
            date_strings.values
        )

        result["timestamp_date_mismatch"] = int(
            mismatch.sum()
        )

    print(
        "Timestamp/date mismatches:",
        f"{result['timestamp_date_mismatch']:,}"
    )


    # --------------------------------------------------------
    # DUPLICATES
    # --------------------------------------------------------

    result["duplicate_symbol_date"] = int(
        df.duplicated(
            subset=["symbol", "date"]
        ).sum()
    )

    result["duplicate_timestamps"] = int(
        df.duplicated(
            subset=["symbol", "timestamp"]
        ).sum()
    )

    print()
    print("DUPLICATES")
    print("-" * 70)

    print(
        "Duplicate symbol/date:",
        f"{result['duplicate_symbol_date']:,}"
    )

    print(
        "Duplicate symbol/timestamp:",
        f"{result['duplicate_timestamps']:,}"
    )


    # --------------------------------------------------------
    # NUMERIC CONVERSION
    # --------------------------------------------------------

    numeric_cols = [
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]

    numeric = df[numeric_cols].apply(
        pd.to_numeric,
        errors="coerce"
    )


    # --------------------------------------------------------
    # PRICE VALIDITY
    # --------------------------------------------------------

    result["zero_or_negative_open"] = int(
        (numeric["open"] <= 0).sum()
    )

    result["zero_or_negative_high"] = int(
        (numeric["high"] <= 0).sum()
    )

    result["zero_or_negative_low"] = int(
        (numeric["low"] <= 0).sum()
    )

    result["zero_or_negative_close"] = int(
        (numeric["close"] <= 0).sum()
    )

    print()
    print("PRICE VALIDITY")
    print("-" * 70)

    print(
        "Open <= 0 :",
        f"{result['zero_or_negative_open']:,}"
    )

    print(
        "High <= 0 :",
        f"{result['zero_or_negative_high']:,}"
    )

    print(
        "Low <= 0  :",
        f"{result['zero_or_negative_low']:,}"
    )

    print(
        "Close <= 0:",
        f"{result['zero_or_negative_close']:,}"
    )


    # --------------------------------------------------------
    # VOLUME VALIDITY
    # --------------------------------------------------------

    result["negative_volume"] = int(
        (numeric["volume"] < 0).sum()
    )

    result["zero_volume"] = int(
        (numeric["volume"] == 0).sum()
    )

    print()
    print("VOLUME")
    print("-" * 70)

    print(
        "Negative volume:",
        f"{result['negative_volume']:,}"
    )

    print(
        "Zero volume    :",
        f"{result['zero_volume']:,}"
    )


    # --------------------------------------------------------
    # OHLC LOGIC
    # --------------------------------------------------------

    o = numeric["open"]
    h = numeric["high"]
    l = numeric["low"]
    c = numeric["close"]


    result["low_greater_high"] = int(
        (l > h).sum()
    )

    result["open_outside_range"] = int(
        ((o < l) | (o > h)).sum()
    )

    result["close_outside_range"] = int(
        ((c < l) | (c > h)).sum()
    )

    result["high_below_open"] = int(
        (h < o).sum()
    )

    result["high_below_close"] = int(
        (h < c).sum()
    )

    result["low_above_open"] = int(
        (l > o).sum()
    )

    result["low_above_close"] = int(
        (l > c).sum()
    )


    print()
    print("OHLC LOGIC")
    print("-" * 70)

    print(
        "Low > High:",
        f"{result['low_greater_high']:,}"
    )

    print(
        "Open outside Low-High:",
        f"{result['open_outside_range']:,}"
    )

    print(
        "Close outside Low-High:",
        f"{result['close_outside_range']:,}"
    )

    print(
        "High < Open:",
        f"{result['high_below_open']:,}"
    )

    print(
        "High < Close:",
        f"{result['high_below_close']:,}"
    )

    print(
        "Low > Open:",
        f"{result['low_above_open']:,}"
    )

    print(
        "Low > Close:",
        f"{result['low_above_close']:,}"
    )


    # --------------------------------------------------------
    # CHRONOLOGICAL ORDER
    # --------------------------------------------------------

    temp = pd.DataFrame({
        "symbol": df["symbol"].values,
        "date": dates.values
    })

    temp = temp.sort_values(
        ["symbol", "date"]
    )

    chronological = (
        temp.groupby("symbol")["date"]
        .apply(lambda x: x.is_monotonic_increasing)
    )

    result["non_chronological_rows"] = int(
        (~chronological).sum()
    )

    print()
    print("CHRONOLOGICAL ORDER")
    print("-" * 70)

    print(
        "Securities not chronologically ordered:",
        f"{result['non_chronological_rows']:,}"
    )


    return result


# ============================================================
# RUN AUDIT — ADJUSTED
# ============================================================

adjusted_audit = audit_ohlcv(
    adjusted_df,
    "ADJUSTED OHLCV"
)


# ============================================================
# RUN AUDIT — UNADJUSTED
# ============================================================

unadjusted_audit = audit_ohlcv(
    unadjusted_flat_df,
    "UNADJUSTED OHLCV"
)


# ============================================================
# SAVE AUDIT REPORT
# ============================================================

audit_report = {
    "adjusted": adjusted_audit,
    "unadjusted": unadjusted_audit
}

audit_path = (
    "/content/NEPSE_ML_PROJECT/validation/"
    "ohlcv_integrity_audit.json"
)

with open(
    audit_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit_report,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 70)
print("OHLCV INTEGRITY AUDIT COMPLETE")
print("=" * 70)

print()
print("Audit saved to:")
print(audit_path)

OHLCV INTEGRITY AUDIT

ADJUSTED OHLCV

MISSING VALUES
----------------------------------------------------------------------
date        : 0
timestamp   : 0
symbol      : 0
open        : 0
high        : 0
low         : 0
close       : 0
volume      : 0

DATE VALIDITY
----------------------------------------------------------------------
Invalid dates: 0
Timestamp/date mismatches: 0

DUPLICATES
----------------------------------------------------------------------
Duplicate symbol/date: 0
Duplicate symbol/timestamp: 0

PRICE VALIDITY
----------------------------------------------------------------------
Open <= 0 : 267
High <= 0 : 234
Low <= 0  : 257
Close <= 0: 165

VOLUME
----------------------------------------------------------------------
Negative volume: 0
Zero volume    : 16,747

OHLC LOGIC
----------------------------------------------------------------------
Low > High: 269
Open outside Low-High: 33,525
Close outside Low-High: 2,431
High < Open: 16,240
High < Close: 1,259
Low >

In [ ]:
# ============================================================
# CELL 15 — INSPECT SUSPICIOUS OHLCV RECORDS
# ============================================================

print("=" * 70)
print("INSPECTING SUSPICIOUS OHLCV RECORDS")
print("=" * 70)


def inspect_anomalies(df, dataset_name):

    print()
    print("=" * 70)
    print(dataset_name)
    print("=" * 70)

    x = df.copy()

    numeric_cols = [
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]

    for col in numeric_cols:

        x[col] = pd.to_numeric(
            x[col],
            errors="coerce"
        )

    # --------------------------------------------------------
    # DEFINE ANOMALIES
    # --------------------------------------------------------

    conditions = (
        (x["open"] <= 0) |
        (x["high"] <= 0) |
        (x["low"] <= 0) |
        (x["close"] <= 0) |
        (x["low"] > x["high"]) |
        (x["open"] < x["low"]) |
        (x["open"] > x["high"]) |
        (x["close"] < x["low"]) |
        (x["close"] > x["high"])
    )

    anomalies = x.loc[
        conditions,
        [
            "date",
            "timestamp",
            "symbol",
            "open",
            "high",
            "low",
            "close",
            "volume"
        ]
    ].copy()

    print()
    print("Total anomalous rows:", f"{len(anomalies):,}")

    # --------------------------------------------------------
    # SAMPLE
    # --------------------------------------------------------

    print()
    print("-" * 70)
    print("FIRST 30 ANOMALOUS ROWS")
    print("-" * 70)

    if len(anomalies) > 0:

        print(
            anomalies
            .head(30)
            .to_string(index=False)
        )

    else:

        print("No anomalous rows found.")


    # --------------------------------------------------------
    # ZERO / NEGATIVE PRICE PATTERNS
    # --------------------------------------------------------

    print()
    print("-" * 70)
    print("ROWS CONTAINING NON-POSITIVE PRICES")
    print("-" * 70)

    nonpositive = x.loc[
        (
            (x["open"] <= 0) |
            (x["high"] <= 0) |
            (x["low"] <= 0) |
            (x["close"] <= 0)
        ),
        [
            "date",
            "symbol",
            "open",
            "high",
            "low",
            "close",
            "volume"
        ]
    ]

    print(
        "Count:",
        f"{len(nonpositive):,}"
    )

    if len(nonpositive) > 0:

        print(
            nonpositive
            .head(30)
            .to_string(index=False)
        )


    # --------------------------------------------------------
    # LOW > HIGH
    # --------------------------------------------------------

    print()
    print("-" * 70)
    print("ROWS WHERE LOW > HIGH")
    print("-" * 70)

    low_high = x.loc[
        x["low"] > x["high"],
        [
            "date",
            "symbol",
            "open",
            "high",
            "low",
            "close",
            "volume"
        ]
    ]

    print(
        "Count:",
        f"{len(low_high):,}"
    )

    if len(low_high) > 0:

        print(
            low_high
            .head(30)
            .to_string(index=False)
        )


    # --------------------------------------------------------
    # ZERO VOLUME EXAMPLES
    # --------------------------------------------------------

    print()
    print("-" * 70)
    print("ZERO-VOLUME EXAMPLES")
    print("-" * 70)

    zero_volume = x.loc[
        x["volume"] == 0,
        [
            "date",
            "symbol",
            "open",
            "high",
            "low",
            "close",
            "volume"
        ]
    ]

    print(
        "Total zero-volume rows:",
        f"{len(zero_volume):,}"
    )

    if len(zero_volume) > 0:

        print(
            zero_volume
            .head(20)
            .to_string(index=False)
        )


# ============================================================
# RUN FOR BOTH DATASETS
# ============================================================

inspect_anomalies(
    adjusted_df,
    "ADJUSTED OHLCV"
)

inspect_anomalies(
    unadjusted_flat_df,
    "UNADJUSTED OHLCV"
)


print()
print("=" * 70)
print("ANOMALY INSPECTION COMPLETE")
print("=" * 70)

INSPECTING SUSPICIOUS OHLCV RECORDS

ADJUSTED OHLCV

Total anomalous rows: 35,692

----------------------------------------------------------------------
FIRST 30 ANOMALOUS ROWS
----------------------------------------------------------------------
      date  timestamp symbol   open   high    low  close  volume
2012-01-01 1325376000  ALICL  61.63  61.42  61.42  61.42    50.0
2012-01-01 1325376000   BPCL 208.16 204.51 204.51 204.51    23.0
2012-01-01 1325376000    CIT  31.68  31.47  31.47  31.47    50.0
2012-01-01 1325376000   DDBL  26.13  25.95  25.95  25.95    89.0
2012-01-01 1325376000    EBL  72.92  72.83  71.58  71.93  6914.0
2012-01-01 1325376000  EBLCP 732.69 718.38 718.38 718.38    48.0
2012-01-01 1325376000   EDBL  35.02  33.90  33.90  33.90  2640.0
2012-01-01 1325376000    HBL  82.17  81.84  81.84  81.84    30.0
2012-01-01 1325376000    HFL  52.49  51.59  50.68  51.59  3460.0
2012-01-01 1325376000   ICFC  52.56  47.01  47.01  47.01  1000.0
2012-01-01 1325376000    KBL  62.69 

In [ ]:
# ============================================================
# CELL 16 — SYMBOL / SECURITY UNIVERSE DIAGNOSTIC
# ============================================================

print("=" * 70)
print("SECURITY UNIVERSE DIAGNOSTIC")
print("=" * 70)

import pandas as pd
import numpy as np
import json
import os
import re


# ============================================================
# COMBINE SYMBOL INFORMATION
# ============================================================

symbols_df = pd.DataFrame(symbols_raw)

print()
print("Symbol master rows:", len(symbols_df))

print()
print("Symbol master columns:")
print(list(symbols_df.columns))


# ============================================================
# CREATE TRADING HISTORY SUMMARY
# ============================================================

def history_summary(df, dataset_name):

    x = df.copy()

    x["date"] = pd.to_datetime(
        x["date"],
        errors="coerce"
    )

    summary = (
        x.groupby("symbol")
        .agg(
            first_date=("date", "min"),
            last_date=("date", "max"),
            trading_rows=("date", "size"),
            unique_dates=("date", "nunique")
        )
        .reset_index()
    )

    summary["dataset"] = dataset_name

    return summary


adjusted_history = history_summary(
    adjusted_df,
    "adjusted"
)

unadjusted_history = history_summary(
    unadjusted_flat_df,
    "unadjusted"
)


# ============================================================
# MERGE HISTORY INFORMATION
# ============================================================

universe = symbols_df[
    ["symbol", "full_name", "description", "exchange", "type", "sector"]
].copy()

universe = universe.merge(
    adjusted_history.drop(
        columns=["dataset"]
    ),
    on="symbol",
    how="left"
)

universe = universe.merge(
    unadjusted_history.drop(
        columns=["dataset"]
    ),
    on="symbol",
    how="left",
    suffixes=("_adjusted", "_unadjusted")
)


# ============================================================
# SIMPLE SYMBOL PATTERN CLASSIFICATION
# ============================================================

def classify_symbol(row):

    symbol = str(row["symbol"]).upper()

    name = str(
        row.get("full_name", "")
    ).upper()

    description = str(
        row.get("description", "")
    ).upper()

    combined = (
        symbol + " " +
        name + " " +
        description
    )

    # --------------------------------------------------------
    # INDEX / MARKET SERIES
    # --------------------------------------------------------

    index_names = [
        "NEPSE",
        "BANKING",
        "DEVBANK",
        "FINANCE",
        "HOTELS",
        "HYDROPOWER",
        "LIFEINSU",
        "NONLIFEINSU",
        "MICROFINANCE",
        "MANUFACTURE",
        "FLOAT",
        "SENFLOAT",
        "SENSITIVE",
        "OTHERS",
        "MUTUAL"
    ]

    if symbol in index_names:
        return "INDEX_OR_MARKET_SERIES"


    # --------------------------------------------------------
    # PREFERENCE / PROMOTER PATTERNS
    # --------------------------------------------------------

    if symbol.endswith("P"):
        return "PREFERENCE_OR_PROMOTER"


    # --------------------------------------------------------
    # BOND / DEBENTURE PATTERNS
    # --------------------------------------------------------

    if (
        "DEBENTURE" in combined or
        "BOND" in combined or
        "DEB" in symbol or
        re.search(r"D\d+$", symbol)
    ):
        return "BOND_OR_DEBENTURE"


    # --------------------------------------------------------
    # MUTUAL FUND
    # --------------------------------------------------------

    if (
        "MUTUAL FUND" in combined or
        symbol.startswith("NBF")
    ):
        return "MUTUAL_FUND"


    # --------------------------------------------------------
    # EVERYTHING ELSE
    # --------------------------------------------------------

    return "OTHER"


universe["initial_category"] = universe.apply(
    classify_symbol,
    axis=1
)


# ============================================================
# PRINT CATEGORY COUNTS
# ============================================================

print()
print("=" * 70)
print("INITIAL CATEGORY COUNTS")
print("=" * 70)

category_counts = (
    universe["initial_category"]
    .value_counts()
)

print(category_counts.to_string())


# ============================================================
# HISTORY COUNTS
# ============================================================

print()
print("=" * 70)
print("HISTORY COVERAGE")
print("=" * 70)

print(
    "Symbols with adjusted history:",
    universe["first_date_adjusted"].notna().sum()
)

print(
    "Symbols with unadjusted history:",
    universe["first_date_unadjusted"].notna().sum()
)

print(
    "Symbols with neither:",
    (
        universe["first_date_adjusted"].isna() &
        universe["first_date_unadjusted"].isna()
    ).sum()
)


# ============================================================
# SHOW INDEX / MARKET SERIES
# ============================================================

print()
print("=" * 70)
print("INDEX / MARKET SERIES")
print("=" * 70)

index_rows = universe[
    universe["initial_category"] ==
    "INDEX_OR_MARKET_SERIES"
].copy()

if len(index_rows):

    print(
        index_rows[
            [
                "symbol",
                "full_name",
                "sector",
                "first_date_adjusted",
                "last_date_adjusted",
                "trading_rows_adjusted"
            ]
        ].to_string(index=False)
    )


# ============================================================
# SHOW PREFERENCE / PROMOTER SYMBOLS
# ============================================================

print()
print("=" * 70)
print("PREFERENCE / PROMOTER CANDIDATES")
print("=" * 70)

preference_rows = universe[
    universe["initial_category"] ==
    "PREFERENCE_OR_PROMOTER"
].copy()

print(
    preference_rows[
        [
            "symbol",
            "full_name",
            "description"
        ]
    ].head(100).to_string(index=False)
)


# ============================================================
# SHOW BOND / DEBENTURE CANDIDATES
# ============================================================

print()
print("=" * 70)
print("BOND / DEBENTURE CANDIDATES")
print("=" * 70)

bond_rows = universe[
    universe["initial_category"] ==
    "BOND_OR_DEBENTURE"
].copy()

print(
    bond_rows[
        [
            "symbol",
            "full_name",
            "description"
        ]
    ].head(100).to_string(index=False)
)


# ============================================================
# SAVE DIAGNOSTIC
# ============================================================

diagnostic_path = (
    "/content/NEPSE_ML_PROJECT/validation/"
    "security_universe_diagnostic.csv"
)

universe.to_csv(
    diagnostic_path,
    index=False
)

print()
print("Diagnostic saved:")
print(diagnostic_path)

print()
print("=" * 70)
print("SECURITY UNIVERSE DIAGNOSTIC COMPLETE")
print("=" * 70)

SECURITY UNIVERSE DIAGNOSTIC

Symbol master rows: 737

Symbol master columns:
['symbol', 'full_name', 'description', 'exchange', 'type', 'sector', 'logo_urls', 'exchange_logo', 'is_master']

INITIAL CATEGORY COUNTS
initial_category
OTHER                     554
BOND_OR_DEBENTURE          91
PREFERENCE_OR_PROMOTER     66
INDEX_OR_MARKET_SERIES     15
MUTUAL_FUND                11

HISTORY COVERAGE
Symbols with adjusted history: 651
Symbols with unadjusted history: 659
Symbols with neither: 77

INDEX / MARKET SERIES
      symbol    full_name       sector first_date_adjusted last_date_adjusted  trading_rows_adjusted
     BANKING      BANKING      BANKING          2011-03-20         2026-10-06                 3555.0
     DEVBANK      DEVBANK      DEVBANK          2012-01-01         2026-10-06                 3380.0
     FINANCE      FINANCE      FINANCE          2011-03-20         2026-10-06                 3556.0
       FLOAT        FLOAT        FLOAT          2011-03-25         2026-10-0

In [ ]:
# ============================================================
# CELL 17 — NO-HISTORY SYMBOL DIAGNOSTIC
# ============================================================

print("=" * 70)
print("NO-HISTORY SYMBOL DIAGNOSTIC")
print("=" * 70)

no_history = universe[
    universe["first_date_adjusted"].isna() &
    universe["first_date_unadjusted"].isna()
].copy()

print()
print("Total symbols with NO adjusted or unadjusted history:")
print(len(no_history))

print()
print("=" * 70)
print("NO-HISTORY SYMBOLS")
print("=" * 70)

cols = [
    "symbol",
    "full_name",
    "description",
    "exchange",
    "type",
    "sector",
    "initial_category"
]

print(
    no_history[cols]
    .sort_values(["initial_category", "symbol"])
    .to_string(index=False)
)

print()
print("=" * 70)
print("NO-HISTORY CATEGORY COUNTS")
print("=" * 70)

print(
    no_history["initial_category"]
    .value_counts()
    .to_string()
)

print()
print("=" * 70)
print("MASTER TYPE COUNTS — NO HISTORY")
print("=" * 70)

print(
    no_history["type"]
    .fillna("NULL")
    .astype(str)
    .value_counts()
    .to_string()
)

print()
print("=" * 70)
print("MASTER SECTOR COUNTS — NO HISTORY")
print("=" * 70)

print(
    no_history["sector"]
    .fillna("NULL")
    .astype(str)
    .value_counts()
    .to_string()
)

print()
print("=" * 70)
print("CELL 17 COMPLETE")
print("=" * 70)

NO-HISTORY SYMBOL DIAGNOSTIC

Total symbols with NO adjusted or unadjusted history:
77

NO-HISTORY SYMBOLS
       symbol     full_name                                                       description                                        exchange  type      sector       initial_category
10.75NMB89/90 10.75NMB89/90                                    10.75% NMB Debenture 2089/2090                    CORPDEBEN src:Nepsealpha.com stock   CORPDEBEN      BOND_OR_DEBENTURE
     BOKD2079      BOKD2079                                             8% BOK Debenture 2079                    CORPDEBEN src:Nepsealpha.com stock   CORPDEBEN      BOND_OR_DEBENTURE
     EBLD2078      EBLD2078                                             8% EBL Debenture 2078                    CORPDEBEN src:Nepsealpha.com stock   CORPDEBEN      BOND_OR_DEBENTURE
   GBILD80|81    GBILD80|81                     10.25% GLOBAL IME BANK LTD. DEBENTURE 2080/81                    CORPDEBEN src:Nepsealpha.com stock   CORPDEBEN  

In [ ]:
# ============================================================
# CELL 18 — PROPER SECURITY CLASSIFICATION
# ============================================================

print("=" * 70)
print("PROPER SECURITY CLASSIFICATION")
print("=" * 70)

import pandas as pd
import numpy as np
import re


# ============================================================
# CLASSIFICATION FUNCTION
# ============================================================

def classify_security(row):

    symbol = str(row["symbol"]).upper().strip()
    name = str(row["full_name"]).upper().strip()
    desc = str(row["description"]).upper().strip()
    sector = str(row["sector"]).upper().strip()

    text = f"{symbol} {name} {desc} {sector}"


    # --------------------------------------------------------
    # 1. INDEX / MARKET SERIES
    # --------------------------------------------------------

    index_symbols = {
        "NEPSE",
        "BANKING",
        "DEVBANK",
        "FINANCE",
        "HOTELS",
        "HYDROPOWER",
        "LIFEINSU",
        "NONLIFEINSU",
        "MICROFINANCE",
        "MANUFACTURE",
        "FLOAT",
        "SENFLOAT",
        "SENSITIVE",
        "OTHERS",
        "MUTUAL"
    }

    if symbol in index_symbols:
        return "INDEX"


    # --------------------------------------------------------
    # 2. PROMOTER SHARES
    # --------------------------------------------------------

    if (
        sector == "PROMOTSHARE" or
        "PROMOTER SHARE" in text or
        "PROMOTOR SHARE" in text
    ):
        return "PROMOTER_SHARE"


    # --------------------------------------------------------
    # 3. PREFERENCE SHARES
    # --------------------------------------------------------

    if (
        sector == "PRESTOCK" or
        "PREFERENCE SHARE" in text or
        "PREFERENCE STOCK" in text or
        "PERPETUAL NON CUMULATIVE PREFERENCE" in text
    ):
        return "PREFERENCE_SHARE"


    # --------------------------------------------------------
    # 4. CORPORATE DEBENTURES / BONDS
    # --------------------------------------------------------

    if (
        sector == "CORPDEBEN" or
        "DEBENTURE" in text or
        "DEBENTURE" in symbol or
        "BOND" in text or
        "RINPATRA" in text
    ):
        return "BOND_DEBENTURE"


    # --------------------------------------------------------
    # 5. MUTUAL FUNDS
    # --------------------------------------------------------

    if (
        sector == "MUTUAL" or
        "MUTUAL FUND" in text
    ):
        return "MUTUAL_FUND"


    # --------------------------------------------------------
    # 6. ORDINARY / COMMON EQUITY
    # --------------------------------------------------------

    equity_sectors = {
        "BANKING",
        "DEVBANK",
        "FINANCE",
        "HOTELS",
        "HYDROPOWER",
        "LIFEINSU",
        "NONLIFEINSU",
        "MICROFINANCE",
        "MANUFACTURE",
        "TRADING",
        "INVESTMENT",
        "OTHERS"
    }

    if sector in equity_sectors:
        return "COMMON_EQUITY"


    # --------------------------------------------------------
    # 7. UNKNOWN
    # --------------------------------------------------------

    return "OTHER"


universe["security_class"] = universe.apply(
    classify_security,
    axis=1
)


# ============================================================
# CATEGORY COUNTS
# ============================================================

print()
print("=" * 70)
print("SECURITY CLASS COUNTS")
print("=" * 70)

print(
    universe["security_class"]
    .value_counts()
    .to_string()
)


# ============================================================
# HISTORY COVERAGE BY CLASS
# ============================================================

coverage = (
    universe
    .groupby("security_class")
    .agg(
        securities=("symbol", "size"),
        adjusted_history=(
            "first_date_adjusted",
            lambda x: x.notna().sum()
        ),
        unadjusted_history=(
            "first_date_unadjusted",
            lambda x: x.notna().sum()
        ),
        no_history=(
            "first_date_adjusted",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

print()
print("=" * 70)
print("HISTORY COVERAGE BY CLASS")
print("=" * 70)

print(
    coverage
    .sort_values("securities", ascending=False)
    .to_string(index=False)
)


# ============================================================
# SHOW COMMON EQUITY
# ============================================================

print()
print("=" * 70)
print("COMMON EQUITY SAMPLE")
print("=" * 70)

common = universe[
    universe["security_class"] == "COMMON_EQUITY"
].copy()

print(
    common[
        [
            "symbol",
            "full_name",
            "sector",
            "first_date_adjusted",
            "last_date_adjusted"
        ]
    ]
    .sort_values("symbol")
    .head(100)
    .to_string(index=False)
)


# ============================================================
# SHOW UNKNOWN / OTHER
# ============================================================

print()
print("=" * 70)
print("UNRESOLVED / OTHER SECURITIES")
print("=" * 70)

other = universe[
    universe["security_class"] == "OTHER"
].copy()

print("Count:", len(other))

if len(other):
    print(
        other[
            [
                "symbol",
                "full_name",
                "description",
                "sector",
                "type"
            ]
        ]
        .sort_values("symbol")
        .to_string(index=False)
    )


# ============================================================
# CLASSIFICATION CONFLICT CHECKS
# ============================================================

print()
print("=" * 70)
print("CLASSIFICATION SANITY CHECKS")
print("=" * 70)

print(
    "Total classified:",
    universe["security_class"].notna().sum()
)

print(
    "Total symbols:",
    len(universe)
)

print(
    "Unclassified:",
    universe["security_class"].isna().sum()
)


# ============================================================
# SAVE DERIVED SECURITY MASTER
# ============================================================

security_master_path = (
    "/content/NEPSE_ML_PROJECT/processed/"
    "security_master_classified.csv"
)

universe.to_csv(
    security_master_path,
    index=False
)

print()
print("Saved derived security master:")
print(security_master_path)

print()
print("=" * 70)
print("CELL 18 COMPLETE")
print("=" * 70)

PROPER SECURITY CLASSIFICATION

SECURITY CLASS COUNTS
security_class
COMMON_EQUITY       397
PROMOTER_SHARE      148
BOND_DEBENTURE       94
MUTUAL_FUND          75
INDEX                15
PREFERENCE_SHARE      8

HISTORY COVERAGE BY CLASS
  security_class  securities  adjusted_history  unadjusted_history  no_history
   COMMON_EQUITY         397               364                 364          33
  PROMOTER_SHARE         148               115                 122          33
  BOND_DEBENTURE          94                84                  85          10
     MUTUAL_FUND          75                72                  72           3
           INDEX          15                15                  15           0
PREFERENCE_SHARE           8                 1                   1           7

COMMON EQUITY SAMPLE
  symbol full_name       sector first_date_adjusted last_date_adjusted
  ACLBSL    ACLBSL MICROFINANCE          2020-07-29         2026-10-06
    ADBL      ADBL      BANKING          20

In [ ]:
# ============================================================
# CELL 19 — OHLC ANOMALIES BY SECURITY CLASS
# ============================================================

print("=" * 70)
print("OHLC ANOMALY ANALYSIS BY SECURITY CLASS")
print("=" * 70)

import pandas as pd
import numpy as np


# ============================================================
# FUNCTION
# ============================================================

def anomaly_summary(df, security_master):

    x = df.copy()

    x["date"] = pd.to_datetime(
        x["date"],
        errors="coerce"
    )

    # Ensure numeric
    price_cols = [
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]

    for col in price_cols:
        x[col] = pd.to_numeric(
            x[col],
            errors="coerce"
        )

    # Attach security classification
    x = x.merge(
        security_master[
            [
                "symbol",
                "security_class",
                "sector",
                "full_name"
            ]
        ],
        on="symbol",
        how="left"
    )

    # --------------------------------------------------------
    # Individual anomaly flags
    # --------------------------------------------------------

    x["nonpositive_price"] = (
        (x["open"] <= 0) |
        (x["high"] <= 0) |
        (x["low"] <= 0) |
        (x["close"] <= 0)
    )

    x["low_gt_high"] = (
        x["low"] > x["high"]
    )

    x["open_outside_range"] = (
        (x["open"] < x["low"]) |
        (x["open"] > x["high"])
    )

    x["close_outside_range"] = (
        (x["close"] < x["low"]) |
        (x["close"] > x["high"])
    )

    x["high_below_open_close"] = (
        (x["high"] < x["open"]) |
        (x["high"] < x["close"])
    )

    x["low_above_open_close"] = (
        (x["low"] > x["open"]) |
        (x["low"] > x["close"])
    )

    x["zero_volume"] = (
        x["volume"] == 0
    )

    x["negative_volume"] = (
        x["volume"] < 0
    )

    x["any_ohlc_anomaly"] = (
        x["nonpositive_price"] |
        x["low_gt_high"] |
        x["open_outside_range"] |
        x["close_outside_range"] |
        x["high_below_open_close"] |
        x["low_above_open_close"] |
        x["negative_volume"]
    )

    return x


# ============================================================
# ADJUSTED
# ============================================================

adj_anomaly_df = anomaly_summary(
    adjusted_df,
    universe
)

print()
print("=" * 70)
print("ADJUSTED DATASET — ANOMALIES BY CLASS")
print("=" * 70)

adj_class_summary = (
    adj_anomaly_df
    .groupby("security_class")
    .agg(
        rows=("symbol", "size"),
        anomaly_rows=("any_ohlc_anomaly", "sum"),
        nonpositive_price=("nonpositive_price", "sum"),
        low_gt_high=("low_gt_high", "sum"),
        open_outside_range=("open_outside_range", "sum"),
        close_outside_range=("close_outside_range", "sum"),
        zero_volume=("zero_volume", "sum"),
        negative_volume=("negative_volume", "sum")
    )
    .reset_index()
)

adj_class_summary["anomaly_pct"] = (
    100 *
    adj_class_summary["anomaly_rows"] /
    adj_class_summary["rows"]
)

print(
    adj_class_summary
    .sort_values("anomaly_rows", ascending=False)
    .to_string(index=False)
)


# ============================================================
# UNADJUSTED
# ============================================================

unadj_anomaly_df = anomaly_summary(
    unadjusted_flat_df,
    universe
)

print()
print("=" * 70)
print("UNADJUSTED DATASET — ANOMALIES BY CLASS")
print("=" * 70)

unadj_class_summary = (
    unadj_anomaly_df
    .groupby("security_class")
    .agg(
        rows=("symbol", "size"),
        anomaly_rows=("any_ohlc_anomaly", "sum"),
        nonpositive_price=("nonpositive_price", "sum"),
        low_gt_high=("low_gt_high", "sum"),
        open_outside_range=("open_outside_range", "sum"),
        close_outside_range=("close_outside_range", "sum"),
        zero_volume=("zero_volume", "sum"),
        negative_volume=("negative_volume", "sum")
    )
    .reset_index()
)

unadj_class_summary["anomaly_pct"] = (
    100 *
    unadj_class_summary["anomaly_rows"] /
    unadj_class_summary["rows"]
)

print(
    unadj_class_summary
    .sort_values("anomaly_rows", ascending=False)
    .to_string(index=False)
)


# ============================================================
# SAVE SUMMARIES
# ============================================================

adj_summary_path = (
    "/content/NEPSE_ML_PROJECT/validation/"
    "ohlc_anomaly_by_class_adjusted.csv"
)

unadj_summary_path = (
    "/content/NEPSE_ML_PROJECT/validation/"
    "ohlc_anomaly_by_class_unadjusted.csv"
)

adj_class_summary.to_csv(
    adj_summary_path,
    index=False
)

unadj_class_summary.to_csv(
    unadj_summary_path,
    index=False
)

print()
print("Saved:")
print(adj_summary_path)
print(unadj_summary_path)

print()
print("=" * 70)
print("CELL 19 COMPLETE")
print("=" * 70)

OHLC ANOMALY ANALYSIS BY SECURITY CLASS

ADJUSTED DATASET — ANOMALIES BY CLASS
  security_class   rows  anomaly_rows  nonpositive_price  low_gt_high  open_outside_range  close_outside_range  zero_volume  negative_volume  anomaly_pct
   COMMON_EQUITY 507350         29287                 24            1               28808                  536         1361                0     5.772544
     MUTUAL_FUND  50426          2441                177           28                2266                   40          397                0     4.840757
           INDEX  48481          1758                  0          233                1661                  412        14983                0     3.626163
  BOND_DEBENTURE  17159          1366                  0            7                  51                 1330            3                0     7.960837
  PROMOTER_SHARE   8930           562                 67            0                 482                   92            3                0     6.2933

In [ ]:
# ============================================================
# CELL 20 — TEMPORAL OHLC ANOMALY DISTRIBUTION
# ============================================================

print("=" * 70)
print("TEMPORAL OHLC ANOMALY DISTRIBUTION")
print("=" * 70)

import pandas as pd
import numpy as np


# ============================================================
# FUNCTION
# ============================================================

def temporal_anomaly_analysis(df, security_master, dataset_name):

    x = df.copy()

    x["date"] = pd.to_datetime(
        x["date"],
        errors="coerce"
    )

    for col in [
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]:
        x[col] = pd.to_numeric(
            x[col],
            errors="coerce"
        )

    x = x.merge(
        security_master[
            [
                "symbol",
                "security_class"
            ]
        ],
        on="symbol",
        how="left"
    )

    # --------------------------------------------------------
    # Anomaly flags
    # --------------------------------------------------------

    x["nonpositive_price"] = (
        (x["open"] <= 0) |
        (x["high"] <= 0) |
        (x["low"] <= 0) |
        (x["close"] <= 0)
    )

    x["low_gt_high"] = (
        x["low"] > x["high"]
    )

    x["open_outside_range"] = (
        (x["open"] < x["low"]) |
        (x["open"] > x["high"])
    )

    x["close_outside_range"] = (
        (x["close"] < x["low"]) |
        (x["close"] > x["high"])
    )

    x["any_anomaly"] = (
        x["nonpositive_price"] |
        x["low_gt_high"] |
        x["open_outside_range"] |
        x["close_outside_range"]
    )

    # --------------------------------------------------------
    # Year
    # --------------------------------------------------------

    x["year"] = x["date"].dt.year

    # --------------------------------------------------------
    # Annual summary
    # --------------------------------------------------------

    annual = (
        x.groupby("year")
        .agg(
            rows=("symbol", "size"),
            anomaly_rows=("any_anomaly", "sum"),
            nonpositive_price=("nonpositive_price", "sum"),
            low_gt_high=("low_gt_high", "sum"),
            open_outside_range=("open_outside_range", "sum"),
            close_outside_range=("close_outside_range", "sum")
        )
        .reset_index()
    )

    annual["anomaly_pct"] = (
        100 *
        annual["anomaly_rows"] /
        annual["rows"]
    )

    print()
    print("=" * 70)
    print(dataset_name.upper(), "— ANOMALIES BY YEAR")
    print("=" * 70)

    print(
        annual.to_string(index=False)
    )

    # --------------------------------------------------------
    # By class and year — only years with anomalies
    # --------------------------------------------------------

    class_year = (
        x.groupby(
            [
                "year",
                "security_class"
            ]
        )
        .agg(
            rows=("symbol", "size"),
            anomalies=("any_anomaly", "sum")
        )
        .reset_index()
    )

    class_year["anomaly_pct"] = (
        100 *
        class_year["anomalies"] /
        class_year["rows"]
    )

    print()
    print("=" * 70)
    print(dataset_name.upper(), "— CLASS/YEAR ANOMALIES")
    print("=" * 70)

    print(
        class_year[
            class_year["anomalies"] > 0
        ]
        .sort_values(
            "anomalies",
            ascending=False
        )
        .head(100)
        .to_string(index=False)
    )

    return x, annual, class_year


# ============================================================
# ADJUSTED
# ============================================================

adj_temporal_df, adj_annual, adj_class_year = (
    temporal_anomaly_analysis(
        adjusted_df,
        universe,
        "adjusted"
    )
)


# ============================================================
# UNADJUSTED
# ============================================================

unadj_temporal_df, unadj_annual, unadj_class_year = (
    temporal_anomaly_analysis(
        unadjusted_flat_df,
        universe,
        "unadjusted"
    )
)


# ============================================================
# SAVE
# ============================================================

adj_path = (
    "/content/NEPSE_ML_PROJECT/validation/"
    "ohlc_anomaly_by_year_adjusted.csv"
)

unadj_path = (
    "/content/NEPSE_ML_PROJECT/validation/"
    "ohlc_anomaly_by_year_unadjusted.csv"
)

adj_annual.to_csv(
    adj_path,
    index=False
)

unadj_annual.to_csv(
    unadj_path,
    index=False
)

print()
print("Saved:")
print(adj_path)
print(unadj_path)

print()
print("=" * 70)
print("CELL 20 COMPLETE")
print("=" * 70)

TEMPORAL OHLC ANOMALY DISTRIBUTION

ADJUSTED — ANOMALIES BY YEAR
 year  rows  anomaly_rows  nonpositive_price  low_gt_high  open_outside_range  close_outside_range  anomaly_pct
 1997   104             0                  0            0                   0                    0     0.000000
 1998   191             0                  0            0                   0                    0     0.000000
 1999   234             0                  0            0                   0                    0     0.000000
 2000   237             0                  0            0                   0                    0     0.000000
 2001   235             0                  0            0                   0                    0     0.000000
 2002   230             0                  0            0                   0                    0     0.000000
 2003   212             0                  0            0                   0                    0     0.000000
 2004   214             0              

In [ ]:
# ============================================================
# CELL 21 — INSPECT HISTORICAL OHLC ANOMALIES
# ============================================================

print("=" * 70)
print("HISTORICAL OHLC ANOMALY INSPECTION")
print("=" * 70)

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Select common-equity rows with the dominant historical issue
# ------------------------------------------------------------

x = adjusted_df.copy()

x["date"] = pd.to_datetime(x["date"], errors="coerce")

for col in ["open", "high", "low", "close", "volume"]:
    x[col] = pd.to_numeric(x[col], errors="coerce")

x = x.merge(
    universe[["symbol", "security_class", "full_name"]],
    on="symbol",
    how="left"
)

historical_anomalies = x[
    (x["security_class"] == "COMMON_EQUITY") &
    (x["date"].dt.year.between(2012, 2016)) &
    (
        (x["open"] < x["low"]) |
        (x["open"] > x["high"])
    )
].copy()

print()
print("Historical common-equity open-range anomalies:")
print(len(historical_anomalies))

# ------------------------------------------------------------
# Pick representative examples across years
# ------------------------------------------------------------

samples = (
    historical_anomalies
    .sort_values(["date", "symbol"])
    .groupby(historical_anomalies["date"].dt.year, group_keys=False)
    .head(5)
)

print()
print("=" * 70)
print("REPRESENTATIVE ANOMALIES")
print("=" * 70)

print(
    samples[
        [
            "date",
            "symbol",
            "full_name",
            "open",
            "high",
            "low",
            "close",
            "volume"
        ]
    ]
    .to_string(index=False)
)

# ------------------------------------------------------------
# Compare same rows against UNADJUSTED data
# ------------------------------------------------------------

u = unadjusted_flat_df.copy()

u["date"] = pd.to_datetime(u["date"], errors="coerce")

for col in ["open", "high", "low", "close", "volume"]:
    u[col] = pd.to_numeric(u[col], errors="coerce")

comparison = samples[
    [
        "date",
        "symbol",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]
].merge(
    u[
        [
            "date",
            "symbol",
            "open",
            "high",
            "low",
            "close",
            "volume"
        ]
    ],
    on=["date", "symbol"],
    how="left",
    suffixes=("_adjusted", "_unadjusted")
)

print()
print("=" * 70)
print("ADJUSTED vs UNADJUSTED")
print("=" * 70)

print(
    comparison.to_string(index=False)
)

# ------------------------------------------------------------
# Save inspection file
# ------------------------------------------------------------

path = (
    "/content/NEPSE_ML_PROJECT/validation/"
    "historical_ohlc_anomaly_samples.csv"
)

comparison.to_csv(path, index=False)

print()
print("Saved:")
print(path)

print()
print("=" * 70)
print("CELL 21 COMPLETE")
print("=" * 70)

HISTORICAL OHLC ANOMALY INSPECTION

Historical common-equity open-range anomalies:
25907

REPRESENTATIVE ANOMALIES
      date symbol full_name    open    high     low   close  volume
2012-01-01  ALICL     ALICL   61.63   61.42   61.42   61.42    50.0
2012-01-01   BPCL      BPCL  208.16  204.51  204.51  204.51    23.0
2012-01-01    CIT       CIT   31.68   31.47   31.47   31.47    50.0
2012-01-01   DDBL      DDBL   26.13   25.95   25.95   25.95    89.0
2012-01-01    EBL       EBL   72.92   72.83   71.58   71.93  6914.0
2013-01-01  ALICL     ALICL   70.36   71.06   70.62   71.06    51.0
2013-01-01    CIT       CIT   66.03   65.23   64.41   64.41   153.0
2013-01-01  CZBIL     CZBIL   82.09   81.70   80.75   80.75  7051.0
2013-01-01    EBL       EBL  171.28  171.16  168.48  168.83  5369.0
2013-01-01  GBIME     GBIME  104.20  102.24  100.28  100.53  1508.0
2014-01-01   ADBL      ADBL  116.66  115.54  110.40  111.96 15862.0
2014-01-01   BPCL      BPCL  438.34  436.51  430.31  432.86  1726.0
2

In [ ]:
# ============================================================
# CELL 22 — CREATE ML-READY OHLCV WITH OPEN > HIGH CORRECTION
# ============================================================

print("=" * 70)
print("CREATING CORRECTED OHLCV DATASETS")
print("=" * 70)

import pandas as pd
import numpy as np
import os

OUTPUT_DIR = "/content/NEPSE_ML_PROJECT/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def correct_open_high(df, dataset_name):

    x = df.copy()

    # Numeric conversion
    for col in ["open", "high", "low", "close", "volume"]:
        x[col] = pd.to_numeric(x[col], errors="coerce")

    # Preserve the original condition as an audit flag
    x["original_open_gt_high"] = (
        x["open"] > x["high"]
    )

    correction_count = int(
        x["original_open_gt_high"].sum()
    )

    # --------------------------------------------------------
    # Correction:
    # If Open > High, High becomes Open.
    # Raw source data remains untouched.
    # --------------------------------------------------------

    x.loc[
        x["original_open_gt_high"],
        "high"
    ] = x.loc[
        x["original_open_gt_high"],
        "open"
    ]

    # --------------------------------------------------------
    # Post-correction validation
    # --------------------------------------------------------

    remaining_open_gt_high = int(
        (x["open"] > x["high"]).sum()
    )

    remaining_open_outside_range = int(
        (
            (x["open"] < x["low"]) |
            (x["open"] > x["high"])
        ).sum()
    )

    remaining_close_outside_range = int(
        (
            (x["close"] < x["low"]) |
            (x["close"] > x["high"])
        ).sum()
    )

    print()
    print(dataset_name.upper())
    print("-" * 70)

    print("Rows:", len(x))
    print("Open > High corrections:", correction_count)
    print("Remaining Open > High:", remaining_open_gt_high)
    print(
        "Remaining Open outside Low-High:",
        remaining_open_outside_range
    )
    print(
        "Remaining Close outside Low-High:",
        remaining_close_outside_range
    )

    return x, correction_count


# ------------------------------------------------------------
# Adjusted dataset
# ------------------------------------------------------------

adjusted_corrected_df, adj_corrections = (
    correct_open_high(
        adjusted_df,
        "adjusted"
    )
)

# ------------------------------------------------------------
# Unadjusted dataset
# ------------------------------------------------------------

unadjusted_corrected_df, unadj_corrections = (
    correct_open_high(
        unadjusted_flat_df,
        "unadjusted"
    )
)

# ------------------------------------------------------------
# Save processed datasets
# ------------------------------------------------------------

adj_path = os.path.join(
    OUTPUT_DIR,
    "nepse_daily_ohlcv_adjusted_corrected.csv.gz"
)

unadj_path = os.path.join(
    OUTPUT_DIR,
    "nepse_daily_ohlcv_unadjusted_corrected.csv.gz"
)

adjusted_corrected_df.to_csv(
    adj_path,
    index=False,
    compression="gzip"
)

unadjusted_corrected_df.to_csv(
    unadj_path,
    index=False,
    compression="gzip"
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print()
print("=" * 70)
print("CORRECTION SUMMARY")
print("=" * 70)

print("Adjusted corrections :", adj_corrections)
print("Unadjusted corrections:", unadj_corrections)

print()
print("Saved:")
print(adj_path)
print(unadj_path)

print()
print("RAW DATA WAS NOT MODIFIED.")
print("Original anomaly flag retained as:")
print("  original_open_gt_high")

print()
print("=" * 70)
print("CELL 22 COMPLETE")
print("=" * 70)

CREATING CORRECTED OHLCV DATASETS

ADJUSTED
----------------------------------------------------------------------
Rows: 633137
Open > High corrections: 16240
Remaining Open > High: 0
Remaining Open outside Low-High: 17297
Remaining Close outside Low-High: 2412

UNADJUSTED
----------------------------------------------------------------------
Rows: 642059
Open > High corrections: 16396
Remaining Open > High: 0
Remaining Open outside Low-High: 17437
Remaining Close outside Low-High: 2312

CORRECTION SUMMARY
Adjusted corrections : 16240
Unadjusted corrections: 16396

Saved:
/content/NEPSE_ML_PROJECT/processed/nepse_daily_ohlcv_adjusted_corrected.csv.gz
/content/NEPSE_ML_PROJECT/processed/nepse_daily_ohlcv_unadjusted_corrected.csv.gz

RAW DATA WAS NOT MODIFIED.
Original anomaly flag retained as:
  original_open_gt_high

CELL 22 COMPLETE


In [ ]:
# ============================================================
# CELL 23 — ROBUST OHLC RANGE CORRECTION
# ============================================================

print("=" * 70)
print("ROBUST OHLC RANGE CORRECTION")
print("=" * 70)

import pandas as pd
import numpy as np
import os

OUTPUT_DIR = "/content/NEPSE_ML_PROJECT/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def correct_ohlc_range(df, dataset_name):

    x = df.copy()

    # --------------------------------------------------------
    # Numeric conversion
    # --------------------------------------------------------

    for col in ["open", "high", "low", "close", "volume"]:
        x[col] = pd.to_numeric(
            x[col],
            errors="coerce"
        )

    # --------------------------------------------------------
    # Preserve original OHLC values
    # --------------------------------------------------------

    x["source_open"] = x["open"]
    x["source_high"] = x["high"]
    x["source_low"] = x["low"]
    x["source_close"] = x["close"]

    # --------------------------------------------------------
    # Identify non-positive prices BEFORE correction
    # --------------------------------------------------------

    x["nonpositive_price"] = (
        (x["open"] <= 0) |
        (x["high"] <= 0) |
        (x["low"] <= 0) |
        (x["close"] <= 0)
    )

    # --------------------------------------------------------
    # Only rows where ALL OHLC prices are positive
    # are eligible for range reconstruction.
    # --------------------------------------------------------

    valid_prices = (
        (x["open"] > 0) &
        (x["high"] > 0) &
        (x["low"] > 0) &
        (x["close"] > 0)
    )

    # --------------------------------------------------------
    # Original structural anomalies
    # --------------------------------------------------------

    x["original_low_gt_high"] = (
        x["low"] > x["high"]
    )

    x["original_open_gt_high"] = (
        x["open"] > x["high"]
    )

    x["original_open_outside_range"] = (
        (x["open"] < x["low"]) |
        (x["open"] > x["high"])
    )

    x["original_close_outside_range"] = (
        (x["close"] < x["low"]) |
        (x["close"] > x["high"])
    )

    # --------------------------------------------------------
    # For positive OHLC rows:
    #
    # High = highest reported price
    # Low  = lowest reported price
    #
    # This resolves:
    #   Open > High
    #   Low > High
    #   Open < Low
    #   Close outside range
    # --------------------------------------------------------

    corrected_high = x[
        ["open", "high", "low", "close"]
    ].max(axis=1)

    corrected_low = x[
        ["open", "high", "low", "close"]
    ].min(axis=1)

    x.loc[valid_prices, "high"] = corrected_high[
        valid_prices
    ]

    x.loc[valid_prices, "low"] = corrected_low[
        valid_prices
    ]

    # --------------------------------------------------------
    # Count actual modifications
    # --------------------------------------------------------

    high_changed = (
        x["high"] != x["source_high"]
    )

    low_changed = (
        x["low"] != x["source_low"]
    )

    x["high_was_corrected"] = (
        high_changed & valid_prices
    )

    x["low_was_corrected"] = (
        low_changed & valid_prices
    )

    x["ohlc_range_corrected"] = (
        x["high_was_corrected"] |
        x["low_was_corrected"]
    )

    # --------------------------------------------------------
    # Post-correction validation
    # --------------------------------------------------------

    remaining_low_gt_high = int(
        (x["low"] > x["high"]).sum()
    )

    remaining_open_outside = int(
        (
            (x["open"] < x["low"]) |
            (x["open"] > x["high"])
        ).sum()
    )

    remaining_close_outside = int(
        (
            (x["close"] < x["low"]) |
            (x["close"] > x["high"])
        ).sum()
    )

    print()
    print(dataset_name.upper())
    print("-" * 70)

    print("Rows:", len(x))

    print(
        "Rows with non-positive OHLC:",
        int(x["nonpositive_price"].sum())
    )

    print(
        "Rows with Low > High originally:",
        int(x["original_low_gt_high"].sum())
    )

    print(
        "Rows with Open > High originally:",
        int(x["original_open_gt_high"].sum())
    )

    print(
        "Rows where High was corrected:",
        int(x["high_was_corrected"].sum())
    )

    print(
        "Rows where Low was corrected:",
        int(x["low_was_corrected"].sum())
    )

    print(
        "Rows with any range correction:",
        int(x["ohlc_range_corrected"].sum())
    )

    print()
    print("POST-CORRECTION")

    print(
        "Remaining Low > High:",
        remaining_low_gt_high
    )

    print(
        "Remaining Open outside range:",
        remaining_open_outside
    )

    print(
        "Remaining Close outside range:",
        remaining_close_outside
    )

    return x


# ============================================================
# ADJUSTED
# ============================================================

adjusted_corrected_df = correct_ohlc_range(
    adjusted_df,
    "adjusted"
)


# ============================================================
# UNADJUSTED
# ============================================================

unadjusted_corrected_df = correct_ohlc_range(
    unadjusted_flat_df,
    "unadjusted"
)


# ============================================================
# SAVE
# ============================================================

adj_path = os.path.join(
    OUTPUT_DIR,
    "nepse_daily_ohlcv_adjusted_corrected.csv.gz"
)

unadj_path = os.path.join(
    OUTPUT_DIR,
    "nepse_daily_ohlcv_unadjusted_corrected.csv.gz"
)

adjusted_corrected_df.to_csv(
    adj_path,
    index=False,
    compression="gzip"
)

unadjusted_corrected_df.to_csv(
    unadj_path,
    index=False,
    compression="gzip"
)


print()
print("=" * 70)
print("FILES SAVED")
print("=" * 70)

print(adj_path)
print(unadj_path)

print()
print("RAW DATA WAS NOT MODIFIED.")
print("Original OHLC values are preserved in:")
print("  source_open")
print("  source_high")
print("  source_low")
print("  source_close")

print()
print("=" * 70)
print("CELL 23 COMPLETE")
print("=" * 70)

ROBUST OHLC RANGE CORRECTION

ADJUSTED
----------------------------------------------------------------------
Rows: 633137
Rows with non-positive OHLC: 288
Rows with Low > High originally: 269
Rows with Open > High originally: 16240
Rows where High was corrected: 17461
Rows where Low was corrected: 18229
Rows with any range correction: 35404

POST-CORRECTION
Remaining Low > High: 0
Remaining Open outside range: 46
Remaining Close outside range: 80

UNADJUSTED
----------------------------------------------------------------------
Rows: 642059
Rows with non-positive OHLC: 183
Rows with Low > High originally: 271
Rows with Open > High originally: 16396
Rows where High was corrected: 17564
Rows where Low was corrected: 18283
Rows with any range correction: 35573

POST-CORRECTION
Remaining Low > High: 0
Remaining Open outside range: 70
Remaining Close outside range: 120

FILES SAVED
/content/NEPSE_ML_PROJECT/processed/nepse_daily_ohlcv_adjusted_corrected.csv.gz
/content/NEPSE_ML_PROJECT/pro

In [ ]:
# ============================================================
# CELL 24 — FINAL OHLC VALIDATION
# ============================================================

print("=" * 70)
print("FINAL OHLC VALIDATION")
print("=" * 70)

def final_ohlc_validation(df, name):

    x = df.copy()

    remaining_open = (
        (x["open"] < x["low"]) |
        (x["open"] > x["high"])
    )

    remaining_close = (
        (x["close"] < x["low"]) |
        (x["close"] > x["high"])
    )

    remaining_any = (
        remaining_open |
        remaining_close |
        (x["low"] > x["high"])
    )

    print()
    print(name.upper())
    print("-" * 70)

    print("Remaining Open outside range:", int(remaining_open.sum()))
    print("Remaining Close outside range:", int(remaining_close.sum()))
    print("Remaining Low > High:", int((x["low"] > x["high"]).sum()))
    print("Remaining any structural anomaly:", int(remaining_any.sum()))

    # Check whether remaining anomalies contain non-positive prices
    if remaining_any.sum() > 0:

        anomalous = x[remaining_any]

        nonpositive = (
            (anomalous["open"] <= 0) |
            (anomalous["high"] <= 0) |
            (anomalous["low"] <= 0) |
            (anomalous["close"] <= 0)
        )

        print(
            "Remaining anomalies with non-positive OHLC:",
            int(nonpositive.sum())
        )

        print(
            "Remaining anomalies with all-positive OHLC:",
            int((~nonpositive).sum())
        )

        if (~nonpositive).sum() > 0:
            print()
            print("ALL-POSITIVE REMAINING ANOMALIES:")
            print(
                anomalous.loc[
                    ~nonpositive,
                    [
                        "date",
                        "symbol",
                        "open",
                        "high",
                        "low",
                        "close"
                    ]
                ].to_string(index=False)
            )


final_ohlc_validation(
    adjusted_corrected_df,
    "adjusted"
)

final_ohlc_validation(
    unadjusted_corrected_df,
    "unadjusted"
)

print()
print("=" * 70)
print("CELL 24 COMPLETE")
print("=" * 70)

FINAL OHLC VALIDATION

ADJUSTED
----------------------------------------------------------------------
Remaining Open outside range: 46
Remaining Close outside range: 80
Remaining Low > High: 0
Remaining any structural anomaly: 117
Remaining anomalies with non-positive OHLC: 117
Remaining anomalies with all-positive OHLC: 0

UNADJUSTED
----------------------------------------------------------------------
Remaining Open outside range: 70
Remaining Close outside range: 120
Remaining Low > High: 0
Remaining any structural anomaly: 181
Remaining anomalies with non-positive OHLC: 181
Remaining anomalies with all-positive OHLC: 0

CELL 24 COMPLETE


In [ ]:
# ============================================================
# CELL 25 — CORPORATE ACTION / EVENT DATASET AUDIT
# ============================================================

print("=" * 70)
print("CORPORATE ACTION / EVENT DATASET AUDIT")
print("=" * 70)

import pandas as pd
import numpy as np

# ============================================================
# 1. DIVIDENDS
# ============================================================

print()
print("=" * 70)
print("DIVIDENDS")
print("=" * 70)

div = dividends_df.copy()

print("Rows:", len(div))
print("Columns:")
print(list(div.columns))

print()
print("Missing values:")
print(
    div.isna().sum().to_string()
)

print()
print("Data types:")
print(
    div.dtypes.to_string()
)

print()
print("Sample:")
print(
    div.head(10).to_string(index=False)
)


# ============================================================
# 2. RIGHTS ISSUES
# ============================================================

print()
print("=" * 70)
print("RIGHTS ISSUES")
print("=" * 70)

rights = rights_df.copy()

print("Rows:", len(rights))
print("Columns:")
print(list(rights.columns))

print()
print("Missing values:")
print(
    rights.isna().sum().to_string()
)

print()
print("Data types:")
print(
    rights.dtypes.to_string()
)

print()
print("Sample:")
print(
    rights.head(10).to_string(index=False)
)


# ============================================================
# 3. MARKS / EVENTS
# ============================================================

print()
print("=" * 70)
print("MARKS / EVENTS")
print("=" * 70)

marks = marks_df.copy()

print("Rows:", len(marks))
print("Columns:")
print(list(marks.columns))

print()
print("Missing values:")
print(
    marks.isna().sum().to_string()
)

print()
print("Data types:")
print(
    marks.dtypes.to_string()
)

print()
print("Sample:")
print(
    marks.head(10).to_string(index=False)
)


# ============================================================
# 4. SYMBOL COVERAGE
# ============================================================

print()
print("=" * 70)
print("SYMBOL COVERAGE")
print("=" * 70)

master_symbols = set(
    universe["symbol"].dropna().astype(str)
)

for name, df in [
    ("DIVIDENDS", div),
    ("RIGHTS", rights),
    ("MARKS", marks)
]:

    possible_symbol_cols = [
        c for c in df.columns
        if "symbol" in c.lower()
        or c.lower() in ["code", "scrip"]
    ]

    print()
    print(name)

    if possible_symbol_cols:

        col = possible_symbol_cols[0]

        symbols = set(
            df[col]
            .dropna()
            .astype(str)
        )

        print("Symbol column:", col)
        print("Unique symbols:", len(symbols))
        print(
            "Symbols in security master:",
            len(symbols & master_symbols)
        )
        print(
            "Symbols not in security master:",
            len(symbols - master_symbols)
        )

    else:
        print("No obvious symbol column detected.")


# ============================================================
# 5. DATE-LIKE COLUMNS
# ============================================================

print()
print("=" * 70)
print("DATE-LIKE COLUMNS")
print("=" * 70)

for name, df in [
    ("DIVIDENDS", div),
    ("RIGHTS", rights),
    ("MARKS", marks)
]:

    print()
    print(name)

    for col in df.columns:

        col_lower = col.lower()

        if any(
            word in col_lower
            for word in [
                "date",
                "time",
                "timestamp",
                "book",
                "close",
                "start",
                "end",
                "record",
                "announce"
            ]
        ):

            print(
                f"{col}:",
                df[col].head(5).tolist()
            )


print()
print("=" * 70)
print("CELL 25 COMPLETE")
print("=" * 70)

CORPORATE ACTION / EVENT DATASET AUDIT

DIVIDENDS
Rows: 1683
Columns:
['id', 'symbol', 'company_name', 'bonus_dividend', 'cash_dividend', 'book_closure_date', 'fiscal_year', 'created_at', 'updated_at', 'avg_3yr_dividend', 'dividend_growth_rate', 'total', 'status_raw', 'status', '_source', '_collected_at', '_key']

Missing values:
id                       0
symbol                   0
company_name             0
bonus_dividend           0
cash_dividend            0
book_closure_date       19
fiscal_year              0
created_at               0
updated_at               0
avg_3yr_dividend         0
dividend_growth_rate     0
total                    0
status_raw               0
status                   0
_source                  0
_collected_at            0
_key                     0

Data types:
id                       int64
symbol                  object
company_name            object
bonus_dividend          object
cash_dividend           object
book_closure_date       object
fiscal_yea

In [ ]:
# ============================================================
# CELL 26A — CONTINUE EVENT AUDIT AFTER MARK TEXT TYPE ISSUE
# ============================================================

print("=" * 70)
print("CELL 26A — MARK TEXT / EVENT STRUCTURE")
print("=" * 70)

import pandas as pd
import numpy as np
import re

# ============================================================
# 1. INSPECT MARK TEXT TYPES
# ============================================================

print()
print("=" * 70)
print("1. MARK TEXT DATA TYPES")
print("=" * 70)

print(
    marks_df["text"]
    .map(type)
    .value_counts()
    .to_string()
)


# ============================================================
# 2. NORMALIZE TEXT ONLY FOR DIAGNOSTIC PURPOSES
# ============================================================

def diagnostic_text(x):

    if isinstance(x, list):
        return " | ".join(map(str, x))

    if isinstance(x, dict):
        return str(x)

    if pd.isna(x):
        return ""

    return str(x)


mark_diag = marks_df[["label", "text"]].copy()

mark_diag["text_diag"] = (
    mark_diag["text"]
    .map(diagnostic_text)
    .str.replace("\n", " | ", regex=False)
)

# Remove duplicates AFTER converting list-like values to strings
mark_patterns = (
    mark_diag[["label", "text_diag"]]
    .drop_duplicates()
    .sort_values(["label", "text_diag"])
)

print()
print("Unique label/text combinations:",
      len(mark_patterns))

print()
print("Examples:")
print(
    mark_patterns
    .head(100)
    .to_string(index=False)
)


# ============================================================
# 3. EVENT LABEL + COMMON TEXT KEYWORDS
# ============================================================

print()
print("=" * 70)
print("3. EVENT TEXT KEYWORD COUNTS")
print("=" * 70)

text_all = (
    mark_diag["text_diag"]
    .str.lower()
    .fillna("")
)

keywords = [
    "dividend",
    "cash",
    "bonus",
    "right",
    "rights",
    "book closure",
    "book",
    "agm",
    "annual general",
    "close",
    "closure",
    "listing",
    "listed",
    "split",
    "auction",
    "ipo",
    "fpo",
    "debenture",
    "merger",
    "acquisition"
]

for kw in keywords:

    count = text_all.str.contains(
        kw,
        regex=False,
        na=False
    ).sum()

    if count:
        print(f"{kw:20s}: {count}")


# ============================================================
# 4. EACH LABEL — SAMPLE TEXT
# ============================================================

print()
print("=" * 70)
print("4. SAMPLE TEXT BY LABEL")
print("=" * 70)

for label in sorted(
    marks_df["label"]
    .dropna()
    .astype(str)
    .unique()
):

    print()
    print("-" * 50)
    print("LABEL:", label)

    subset = (
        mark_diag[
            mark_diag["label"].astype(str) == label
        ][["text_diag"]]
        .drop_duplicates()
        .head(20)
    )

    print(
        subset.to_string(index=False)
    )


# ============================================================
# 5. DATE / TIMESTAMP CONSISTENCY
# ============================================================

print()
print("=" * 70)
print("5. MARK DATE / TIMESTAMP CONSISTENCY")
print("=" * 70)

m = marks_df.copy()

m["parsed_date"] = pd.to_datetime(
    m["date"],
    errors="coerce"
)

m["timestamp_date"] = pd.to_datetime(
    m["timestamp"],
    unit="s",
    errors="coerce",
    utc=True
).dt.tz_convert(None).dt.normalize()

m["date_timestamp_mismatch"] = (
    m["parsed_date"].dt.normalize()
    != m["timestamp_date"]
)

print("Invalid dates:",
      m["parsed_date"].isna().sum())

print("Invalid timestamps:",
      m["timestamp_date"].isna().sum())

print("Date/timestamp mismatches:",
      m["date_timestamp_mismatch"].sum())

if m["date_timestamp_mismatch"].any():

    print()
    print(
        m.loc[
            m["date_timestamp_mismatch"],
            [
                "symbol",
                "timestamp",
                "date",
                "timestamp_date"
            ]
        ]
        .head(20)
        .to_string(index=False)
    )


# ============================================================
# 6. MARK DUPLICATES
# ============================================================

print()
print("=" * 70)
print("6. MARK DUPLICATES")
print("=" * 70)

print(
    "Duplicate symbol/date/label:",
    marks_df.duplicated(
        subset=["symbol", "date", "label"]
    ).sum()
)

print(
    "Duplicate symbol/timestamp/label:",
    marks_df.duplicated(
        subset=["symbol", "timestamp", "label"]
    ).sum()
)

print(
    "Duplicate event_id:",
    marks_df["event_id"].duplicated().sum()
)


# ============================================================
# 7. DIVIDEND DATE AUDIT
# ============================================================

print()
print("=" * 70)
print("7. DIVIDEND DATE AUDIT")
print("=" * 70)

d = dividends_df.copy()

for col in [
    "book_closure_date",
    "created_at",
    "updated_at"
]:

    d[col + "_parsed"] = pd.to_datetime(
        d[col],
        errors="coerce",
        utc=True
    )

    parsed = d[col + "_parsed"]

    print()
    print(col)
    print("  valid:", parsed.notna().sum())
    print("  missing:", parsed.isna().sum())

    if parsed.notna().any():
        print("  min:", parsed.min())
        print("  max:", parsed.max())


# ============================================================
# 8. DIVIDEND INTERNAL DATE ORDER
# ============================================================

print()
print("=" * 70)
print("8. DIVIDEND INTERNAL DATE ORDER")
print("=" * 70)

book = pd.to_datetime(
    d["book_closure_date"],
    errors="coerce"
)

created = pd.to_datetime(
    d["created_at"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

updated = pd.to_datetime(
    d["updated_at"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

print(
    "Book closure before created_at:",
    (
        book.notna()
        & (book < created.dt.normalize())
    ).sum()
)

print(
    "Book closure before updated_at:",
    (
        book.notna()
        & (book < updated.dt.normalize())
    ).sum()
)

print(
    "Created after updated:",
    (created > updated).sum()
)


# ============================================================
# 9. RIGHTS DATE AUDIT
# ============================================================

print()
print("=" * 70)
print("9. RIGHTS DATE AUDIT")
print("=" * 70)

r = rights_df.copy()

for col in [
    "opening_date",
    "closing_date",
    "book_closure_date"
]:

    r[col + "_parsed"] = pd.to_datetime(
        r[col],
        errors="coerce"
    )

    print()
    print(col)
    print("  valid:", r[col + "_parsed"].notna().sum())
    print("  missing:", r[col + "_parsed"].isna().sum())

    if r[col + "_parsed"].notna().any():
        print(
            "  min:", r[col + "_parsed"].min(),
            "max:", r[col + "_parsed"].max()
        )


r["created_date"] = pd.to_datetime(
    r["created_at"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

r["updated_date"] = pd.to_datetime(
    r["updated_at"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

print()
print(
    "Opening after closing:",
    (
        r["opening_date_parsed"]
        > r["closing_date_parsed"]
    ).sum()
)

print(
    "Book closure after opening:",
    (
        r["book_closure_date_parsed"]
        > r["opening_date_parsed"]
    ).sum()
)

print(
    "Book closure after closing:",
    (
        r["book_closure_date_parsed"]
        > r["closing_date_parsed"]
    ).sum()
)

print(
    "Book closure before created_at:",
    (
        r["book_closure_date_parsed"].notna()
        & (
            r["book_closure_date_parsed"]
            < r["created_date"].dt.normalize()
        )
    ).sum()
)


# ============================================================
# 10. RIGHTS SYMBOLS NOT IN CURRENT MASTER
# ============================================================

print()
print("=" * 70)
print("10. RIGHTS SYMBOLS NOT IN CURRENT MASTER")
print("=" * 70)

master_symbols = set(
    universe["symbol"]
    .dropna()
    .astype(str)
)

rights_symbols = set(
    rights_df["symbol"]
    .dropna()
    .astype(str)
)

missing_rights_symbols = sorted(
    rights_symbols - master_symbols
)

print(
    "Missing from current master:",
    len(missing_rights_symbols)
)

print(missing_rights_symbols)


# ============================================================
# 11. DIVIDEND / MARK OVERLAP
# ============================================================

print()
print("=" * 70)
print("11. DIVIDEND / MARK OVERLAP")
print("=" * 70)

div_dates = pd.to_datetime(
    dividends_df["book_closure_date"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

mark_dates = pd.to_datetime(
    marks_df["date"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

div_pairs = set(
    zip(
        dividends_df["symbol"].astype(str),
        div_dates
    )
)

mark_pairs = set(
    zip(
        marks_df["symbol"].astype(str),
        mark_dates
    )
)

div_pairs = {
    x for x in div_pairs
    if x[1] != "NaT"
}

overlap = div_pairs & mark_pairs

print(
    "Dividend symbol/date pairs:",
    len(div_pairs)
)

print(
    "Matching mark symbol/date pairs:",
    len(overlap)
)

if div_pairs:
    print(
        "Overlap rate:",
        round(
            100 * len(overlap) / len(div_pairs),
            2
        ),
        "%"
    )


# ============================================================
# 12. NUMERIC DIVIDEND PARSING
# ============================================================

print()
print("=" * 70)
print("12. NUMERIC DIVIDEND PARSING")
print("=" * 70)

for col in [
    "bonus_dividend",
    "cash_dividend",
    "avg_3yr_dividend",
    "dividend_growth_rate",
    "total"
]:

    parsed = (
        dividends_df[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    numeric = pd.to_numeric(
        parsed,
        errors="coerce"
    )

    print(
        f"{col:22s}",
        "| valid:", numeric.notna().sum(),
        "| failed:", numeric.isna().sum(),
        "| min:", numeric.min(),
        "| max:", numeric.max()
    )


print()
print("=" * 70)
print("CELL 26A COMPLETE")
print("=" * 70)

CELL 26A — MARK TEXT / EVENT STRUCTURE

1. MARK TEXT DATA TYPES
text
<class 'list'>    8637
<class 'str'>     5450

Unique label/text combinations: 12593

Examples:
label                                                                  text_diag
    D                   Cash Dividend:  %. | Bonus Dividend: 10 %. | Total: 10 %
    D             Cash Dividend:  %. | Bonus Dividend: 10.52 %. | Total: 10.52 %
    D             Cash Dividend:  %. | Bonus Dividend: 10.53 %. | Total: 10.53 %
    D         Cash Dividend:  %. | Bonus Dividend: 10.7539 %. | Total: 10.7539 %
    D                   Cash Dividend:  %. | Bonus Dividend: 11 %. | Total: 11 %
    D               Cash Dividend:  %. | Bonus Dividend: 11.5 %. | Total: 11.5 %
    D               Cash Dividend:  %. | Bonus Dividend: 11.7 %. | Total: 11.7 %
    D                   Cash Dividend:  %. | Bonus Dividend: 12 %. | Total: 12 %
    D                   Cash Dividend:  %. | Bonus Dividend: 13 %. | Total: 13 %
    D             Cash Di

In [ ]:
# ============================================================
# CELL 27 — CLASSIFY N EVENTS
# ============================================================

print("=" * 70)
print("CELL 27 — N EVENT CLASSIFICATION AUDIT")
print("=" * 70)

import pandas as pd
import numpy as np
import re

n = marks_df[
    marks_df["label"].astype(str) == "N"
].copy()

print("Total N events:", len(n))

# ------------------------------------------------------------
# Normalize text for inspection only
# ------------------------------------------------------------

def text_to_string(x):

    if isinstance(x, list):
        return " | ".join(map(str, x))

    if isinstance(x, dict):
        return str(x)

    if pd.isna(x):
        return ""

    return str(x)

n["text_clean"] = (
    n["text"]
    .map(text_to_string)
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# ------------------------------------------------------------
# Show unique N-event texts
# ------------------------------------------------------------

unique_n = (
    n[["text_clean"]]
    .drop_duplicates()
    .sort_values("text_clean")
)

print()
print("Unique N event texts:", len(unique_n))

print()
print("First 200 unique N events:")
print(
    unique_n.head(200).to_string(index=False)
)

# ------------------------------------------------------------
# Keyword classification counts
# ------------------------------------------------------------

print()
print("=" * 70)
print("KEYWORD CATEGORY COUNTS")
print("=" * 70)

categories = {

    "AGM / GENERAL MEETING": [
        "साधारण सभा",
        "general meeting",
        "agm",
        "सभा"
    ],

    "DIVIDEND": [
        "लाभांश",
        "dividend"
    ],

    "BONUS SHARE": [
        "बोनस",
        "bonus"
    ],

    "PRICE ADJUSTMENT": [
        "मुल्य समायोजन",
        "मूल्य समायोजन",
        "price adjustment"
    ],

    "LISTING": [
        "सुचिकृत",
        "सूचीकृत",
        "सूचीकृत",
        "listing",
        "listed"
    ],

    "TRADING SUSPENSION / CLOSURE": [
        "कारोबार बन्द",
        "कारोबार रोक्का",
        "trading halt",
        "trading suspended",
        "बन्द भयो"
    ],

    "PROMOTER / PUBLIC": [
        "प्रोमोटर",
        "promoter",
        "public"
    ],

    "MANAGEMENT": [
        "प्रमुख नियुक्त",
        "अध्यक्ष नियुक्त",
        "सिइओ",
        "ceo",
        "व्यवस्थापक",
        "management"
    ],

    "RIGHTS": [
        "हकप्रद",
        "rights"
    ],

    "IPO / FPO": [
        "ipo",
        "fpo",
        "आइपिओ",
        "एफपिओ"
    ],

    "MERGER / ACQUISITION": [
        "मर्जर",
        "merger",
        "acquisition",
        "एक्विजिसन"
    ],

    "BOOK CLOSURE": [
        "बुकक्लोज",
        "बुक क्लोज",
        "book closure"
    ]
}

for category, words in categories.items():

    pattern = "|".join(
        re.escape(w)
        for w in words
    )

    mask = n["text_clean"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )

    print(
        f"{category:32s}: {mask.sum()}"
    )

# ------------------------------------------------------------
# Multi-category events
# ------------------------------------------------------------

print()
print("=" * 70)
print("MULTI-CATEGORY EVENTS")
print("=" * 70)

category_flags = pd.DataFrame(index=n.index)

for category, words in categories.items():

    pattern = "|".join(
        re.escape(w)
        for w in words
    )

    category_flags[category] = (
        n["text_clean"]
        .str.contains(
            pattern,
            case=False,
            regex=True,
            na=False
        )
    )

category_flags["category_count"] = (
    category_flags.sum(axis=1)
)

print(
    category_flags["category_count"]
    .value_counts()
    .sort_index()
    .to_string()
)

print()
print("=" * 70)
print("CELL 27 COMPLETE")
print("=" * 70)

CELL 27 — N EVENT CLASSIFICATION AUDIT
Total N events: 3660

Unique N event texts: 3255

First 200 unique N events:
                                                                                                    text_clean
                                                       'सिटिजन्स सुपर ३० म्युचुअल फण्ड' को ५.२५% प्रतिफल घोषणा
                                            A Tale of Two Banks: Same Sector, Different Stories (SBL vs. ADBL)
                                              A Technical Look at Popular Heavyweight Stocks: SHIVM and SARBTM
             A Two-Time Multibagger Pauses at the Channel Top — Rest or Reversal? : Decoding RADHI’s Next Move
                                                        AKPL VS API :Different Fundamentals, Different Setups?
                                                                            Accumulation Phase in SPIL and HEI
                                                             Bishal Bazar (BBC): The Pricing of a Supply Sh

In [ ]:
# ============================================================
# CELL 28 — BUILD CLEAN N-EVENT CLASSIFICATIONS
# ============================================================

print("=" * 70)
print("CELL 28 — BUILDING CLEAN N-EVENT CLASSIFICATIONS")
print("=" * 70)

import pandas as pd
import numpy as np
import re

# ------------------------------------------------------------
# Start from original marks data
# ------------------------------------------------------------

n_events = marks_df[
    marks_df["label"].astype(str) == "N"
].copy()

print("N events:", len(n_events))

# ------------------------------------------------------------
# Normalize text
# ------------------------------------------------------------

def text_to_string(x):
    if isinstance(x, list):
        return " | ".join(map(str, x))
    if isinstance(x, dict):
        return str(x)
    if pd.isna(x):
        return ""
    return str(x)

n_events["event_text"] = (
    n_events["text"]
    .map(text_to_string)
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Lowercase English portion for matching
n_events["event_text_lower"] = n_events["event_text"].str.lower()

# ------------------------------------------------------------
# Category definitions
#
# Conservative classification:
# corporate actions + clearly identifiable fundamentals/news.
# Ambiguous/editorial articles remain OTHER_N.
# ------------------------------------------------------------

category_patterns = {

    "DIVIDEND": [
        "लाभांश",
        "dividend",
        "बोनस शेयर",
        "बोनस",
        "bonus share",
        "bonus"
    ],

    "RIGHTS": [
        "हकप्रद",
        "rights"
    ],

    "AGM": [
        "साधारण सभा",
        "वार्षिक साधारण सभा",
        "agm",
        "general meeting",
        "सभा आव्हान",
        "सभा आह्वान"
    ],

    "IPO_FPO": [
        "ipo",
        "fpo",
        "आइपिओ",
        "आइपीओ",
        "आइपिओ",
        "आइपीओ",
        "एफपिओ",
        "एफपीओ"
    ],

    "MERGER": [
        "मर्जर",
        "merger",
        "acquisition",
        "एक्विजिसन",
        "गाभ्ने",
        "गाभिए"
    ],

    "LISTING": [
        "सूचीकृत",
        "सुचिकृत",
        "listing",
        "listed"
    ],

    "TRADING_STATUS": [
        "कारोबार रोक्का",
        "कारोबार बन्द",
        "ट्रेडिङ रोक्का",
        "trading halt",
        "trading suspended",
        "share trading halted",
        "शेयर कारोबार फुकुवा",
        "कारोबार फुकुवा"
    ],

    "PRICE_ADJUSTMENT": [
        "मूल्य समायोजन",
        "मुल्य समायोजन",
        "price adjustment"
    ],

    "BOOK_CLOSURE": [
        "बुकक्लोज",
        "बुक क्लोज",
        "book closure",
        "बुक क्लोज मिति"
    ],

    "FUNDAMENTAL_NEWS": [
        "नाफा",
        "खुद नाफा",
        "मुनाफा",
        "आम्दानी",
        "वित्तिय अवस्था",
        "वित्तीय अवस्था",
        "व्यवसाय",
        "वित्तिय स्थिति",
        "वित्तीय स्थिति",
        "प्रति शेयर आम्दानी",
        "प्रति शेयर आय",
        "eps",
        "कमाइ",
        "आयमा वृद्धि",
        "नाफामा वृद्धि",
        "नाफामा गिरावट",
        "नोक्सानी",
        "घाटामा",
        "उत्पादन",
        "insider trading",
        "inside trading",
        "financial condition",
        "financial position",
        "profit",
        "earnings",
        "revenue"
    ]
}

# ------------------------------------------------------------
# Create boolean category flags
# ------------------------------------------------------------

for category, words in category_patterns.items():

    pattern = "|".join(
        re.escape(str(w).lower())
        for w in words
    )

    n_events[f"is_{category}"] = (
        n_events["event_text_lower"]
        .str.contains(
            pattern,
            regex=True,
            na=False
        )
    )

# ------------------------------------------------------------
# Assign primary category
#
# Priority matters.
# More specific corporate actions come first.
# ------------------------------------------------------------

priority = [
    "MERGER",
    "TRADING_STATUS",
    "PRICE_ADJUSTMENT",
    "BOOK_CLOSURE",
    "RIGHTS",
    "IPO_FPO",
    "LISTING",
    "AGM",
    "DIVIDEND",
    "FUNDAMENTAL_NEWS"
]

n_events["event_category"] = "OTHER_N"

for category in priority:

    mask = (
        n_events[f"is_{category}"]
        & (n_events["event_category"] == "OTHER_N")
    )

    n_events.loc[mask, "event_category"] = category

# ------------------------------------------------------------
# Count matched categories before primary-category assignment
# ------------------------------------------------------------

flag_columns = [
    f"is_{category}"
    for category in category_patterns
]

n_events["matched_category_count"] = (
    n_events[flag_columns].sum(axis=1)
)

# ------------------------------------------------------------
# Event date/timestamp normalization
# ------------------------------------------------------------

n_events["event_date"] = pd.to_datetime(
    n_events["date"],
    errors="coerce"
).dt.normalize()

n_events["event_timestamp"] = pd.to_datetime(
    n_events["timestamp"],
    errors="coerce",
    unit="s"
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print()
print("=" * 70)
print("PRIMARY EVENT CATEGORY DISTRIBUTION")
print("=" * 70)

category_counts = (
    n_events["event_category"]
    .value_counts()
)

print(category_counts.to_string())

print()
print("=" * 70)
print("PERCENTAGE")
print("=" * 70)

category_pct = (
    n_events["event_category"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(category_pct.to_string())

print()
print("=" * 70)
print("MULTI-MATCH EVENTS")
print("=" * 70)

print(
    n_events["matched_category_count"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# Show OTHER_N samples
# ------------------------------------------------------------

print()
print("=" * 70)
print("OTHER_N SAMPLE")
print("=" * 70)

other_sample = (
    n_events.loc[
        n_events["event_category"] == "OTHER_N",
        ["symbol", "event_date", "event_text"]
    ]
    .drop_duplicates()
    .head(50)
)

print(other_sample.to_string(index=False))

# ------------------------------------------------------------
# Basic integrity checks
# ------------------------------------------------------------

print()
print("=" * 70)
print("INTEGRITY")
print("=" * 70)

print("Original N events :", len(marks_df[marks_df["label"].astype(str) == "N"]))
print("Classified N events:", len(n_events))
print("Missing symbol     :", n_events["symbol"].isna().sum())
print("Missing event date :", n_events["event_date"].isna().sum())

print()
print("=" * 70)
print("CELL 28 COMPLETE")
print("=" * 70)

CELL 28 — BUILDING CLEAN N-EVENT CLASSIFICATIONS
N events: 3660

PRIMARY EVENT CATEGORY DISTRIBUTION
event_category
DIVIDEND            1006
OTHER_N              773
FUNDAMENTAL_NEWS     509
IPO_FPO              479
AGM                  268
RIGHTS               267
BOOK_CLOSURE         128
LISTING               89
MERGER                77
TRADING_STATUS        36
PRICE_ADJUSTMENT      28

PERCENTAGE
event_category
DIVIDEND            27.49
OTHER_N             21.12
FUNDAMENTAL_NEWS    13.91
IPO_FPO             13.09
AGM                  7.32
RIGHTS               7.30
BOOK_CLOSURE         3.50
LISTING              2.43
MERGER               2.10
TRADING_STATUS       0.98
PRICE_ADJUSTMENT     0.77

MULTI-MATCH EVENTS
matched_category_count
0     773
1    2564
2     274
3      47
4       2

OTHER_N SAMPLE
symbol event_date                                                                                                                                                              event_text
 

In [ ]:
# ============================================================
# CELL 29 — BUILD DAILY EVENT FEATURES
# ============================================================

print("=" * 70)
print("CELL 29 — BUILDING DAILY EVENT FEATURES")
print("=" * 70)

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Use classified N events
# ------------------------------------------------------------

events = n_events.copy()

events["event_date"] = pd.to_datetime(
    events["event_date"],
    errors="coerce"
).dt.normalize()

events = events.dropna(
    subset=["symbol", "event_date"]
).copy()

# ------------------------------------------------------------
# IMPORTANT:
# Keep only the information available on/after event date.
# No future event is allowed to contribute to an earlier date.
# ------------------------------------------------------------

events = events.sort_values(
    ["symbol", "event_date", "event_timestamp"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Build daily event counts by symbol/date
# ------------------------------------------------------------

event_daily = (
    events
    .groupby(
        ["symbol", "event_date", "event_category"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "event_count"})
)

print("Unique symbol/date/category rows:", len(event_daily))

# ------------------------------------------------------------
# Pivot categories
# ------------------------------------------------------------

category_daily = (
    event_daily
    .pivot_table(
        index=["symbol", "event_date"],
        columns="event_category",
        values="event_count",
        fill_value=0
    )
    .reset_index()
)

category_daily.columns.name = None

# ------------------------------------------------------------
# Ensure every expected category exists
# ------------------------------------------------------------

expected_categories = [
    "DIVIDEND",
    "RIGHTS",
    "AGM",
    "IPO_FPO",
    "MERGER",
    "LISTING",
    "TRADING_STATUS",
    "PRICE_ADJUSTMENT",
    "BOOK_CLOSURE",
    "FUNDAMENTAL_NEWS",
    "OTHER_N"
]

for category in expected_categories:
    if category not in category_daily.columns:
        category_daily[category] = 0

# ------------------------------------------------------------
# Total N event count
# ------------------------------------------------------------

category_daily["N_EVENT_TOTAL"] = (
    category_daily[expected_categories]
    .sum(axis=1)
)

# ------------------------------------------------------------
# Create a complete calendar for every symbol
#
# We use the OHLC data as the actual trading-day calendar.
# This means event features will be aligned to dates on which
# the security has market data.
# ------------------------------------------------------------

price_dates = adjusted_df[
    ["symbol", "date"]
].copy()

price_dates["date"] = pd.to_datetime(
    price_dates["date"],
    errors="coerce"
).dt.normalize()

price_dates = (
    price_dates
    .dropna()
    .drop_duplicates()
    .sort_values(["symbol", "date"])
    .reset_index(drop=True)
)

print("Price symbol/date rows:", len(price_dates))

# ------------------------------------------------------------
# Merge events onto trading calendar
# ------------------------------------------------------------

event_feature_daily = price_dates.merge(
    category_daily,
    left_on=["symbol", "date"],
    right_on=["symbol", "event_date"],
    how="left"
)

event_feature_daily = event_feature_daily.drop(
    columns=["event_date"],
    errors="ignore"
)

# ------------------------------------------------------------
# Fill missing event counts with zero
# ------------------------------------------------------------

for category in expected_categories + ["N_EVENT_TOTAL"]:

    if category not in event_feature_daily.columns:
        event_feature_daily[category] = 0

    event_feature_daily[category] = (
        pd.to_numeric(
            event_feature_daily[category],
            errors="coerce"
        )
        .fillna(0)
    )

# ------------------------------------------------------------
# Sort
# ------------------------------------------------------------

event_feature_daily = event_feature_daily.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Rolling EVENT FEATURES
#
# These are calendar-day windows.
# IMPORTANT:
# The current date IS included.
# Therefore an event occurring on date T can affect
# features for date T and later.
# ------------------------------------------------------------

rolling_windows = [1, 5, 20, 60]

base_categories = expected_categories + ["N_EVENT_TOTAL"]

print()
print("Creating rolling event features...")

for window in rolling_windows:

    print("  Window:", window, "days")

    grouped = event_feature_daily.groupby("symbol", sort=False)

    for category in base_categories:

        source = event_feature_daily[category]

        feature_name = (
            f"{category.lower()}_{window}d"
        )

        event_feature_daily[feature_name] = (
            source
            .groupby(event_feature_daily["symbol"])
            .rolling(
                window=window,
                min_periods=1
            )
            .sum()
            .reset_index(
                level=0,
                drop=True
            )
            .values
        )

# ------------------------------------------------------------
# Days since last event
#
# Based on actual event dates.
# ------------------------------------------------------------

print()
print("Creating days-since-event features...")

def days_since_event(group):

    dates = group["date"].values

    event_dates = group.loc[
        group["N_EVENT_TOTAL"] > 0,
        "date"
    ].values

    result = np.full(
        len(group),
        np.nan,
        dtype=float
    )

    if len(event_dates) == 0:
        return pd.Series(
            result,
            index=group.index
        )

    event_dates = pd.to_datetime(event_dates)

    last_event = None

    for i, current_date in enumerate(
        pd.to_datetime(dates)
    ):

        # Events on current date count as available
        todays_events = event_dates[
            event_dates <= current_date
        ]

        if len(todays_events) > 0:
            last_event = todays_events[-1]

        if last_event is not None:
            result[i] = (
                current_date - last_event
            ).days

    return pd.Series(
        result,
        index=group.index
    )

event_feature_daily["days_since_n_event"] = (
    event_feature_daily
    .groupby("symbol", group_keys=False)
    .apply(days_since_event)
    .reset_index(level=0, drop=True)
)

# ------------------------------------------------------------
# Days since each major category
# ------------------------------------------------------------

major_categories = [
    "DIVIDEND",
    "RIGHTS",
    "AGM",
    "IPO_FPO",
    "MERGER",
    "LISTING",
    "TRADING_STATUS",
    "PRICE_ADJUSTMENT",
    "BOOK_CLOSURE",
    "FUNDAMENTAL_NEWS"
]

for category in major_categories:

    event_col = category

    output_col = (
        f"days_since_{category.lower()}"
    )

    def calc_days(group, col=event_col):

        event_dates = group.loc[
            group[col] > 0,
            "date"
        ].values

        result = np.full(
            len(group),
            np.nan,
            dtype=float
        )

        if len(event_dates) == 0:
            return pd.Series(
                result,
                index=group.index
            )

        event_dates = pd.to_datetime(event_dates)

        last_event = None

        for i, current_date in enumerate(
            pd.to_datetime(group["date"])
        ):

            available = event_dates[
                event_dates <= current_date
            ]

            if len(available) > 0:
                last_event = available[-1]

            if last_event is not None:
                result[i] = (
                    current_date - last_event
                ).days

        return pd.Series(
            result,
            index=group.index
        )

    event_feature_daily[output_col] = (
        event_feature_daily
        .groupby("symbol", group_keys=False)
        .apply(calc_days)
        .reset_index(level=0, drop=True)
    )

# ------------------------------------------------------------
# Convert counts to compact integer types where possible
# ------------------------------------------------------------

count_columns = (
    expected_categories
    + ["N_EVENT_TOTAL"]
    + [
        f"{category.lower()}_{window}d"
        for window in rolling_windows
        for category in base_categories
    ]
)

for col in count_columns:

    if col in event_feature_daily.columns:

        event_feature_daily[col] = (
            event_feature_daily[col]
            .fillna(0)
            .astype(np.float32)
        )

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print()
print("=" * 70)
print("EVENT FEATURE DATASET SUMMARY")
print("=" * 70)

print("Rows:", len(event_feature_daily))
print("Columns:", len(event_feature_daily.columns))
print(
    "Symbols:",
    event_feature_daily["symbol"].nunique()
)
print(
    "Date range:",
    event_feature_daily["date"].min(),
    "to",
    event_feature_daily["date"].max()
)

print()
print("Event feature columns:")

feature_columns = [
    c for c in event_feature_daily.columns
    if c not in ["symbol", "date"]
]

print(
    "\n".join(feature_columns)
)

print()
print("=" * 70)
print("EVENT FEATURE NONZERO COUNTS")
print("=" * 70)

for col in [
    "N_EVENT_TOTAL",
    "n_event_total_1d",
    "n_event_total_5d",
    "n_event_total_20d",
    "n_event_total_60d",
    "dividend_5d",
    "rights_20d",
    "agm_20d",
    "fundamental_news_20d"
]:

    if col in event_feature_daily.columns:

        nonzero = (
            event_feature_daily[col] > 0
        ).sum()

        print(
            f"{col:30s}: {nonzero:,}"
        )

print()
print("=" * 70)
print("CELL 29 COMPLETE")
print("=" * 70)

CELL 29 — BUILDING DAILY EVENT FEATURES
Unique symbol/date/category rows: 3625
Price symbol/date rows: 633137

Creating rolling event features...
  Window: 1 days
  Window: 5 days
  Window: 20 days
  Window: 60 days

Creating days-since-event features...


/tmp/ipykernel_768/2172648255.py:274: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(days_since_event)
/tmp/ipykernel_768/2172648255.py:350: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_days)
/tmp/ipykernel_768/2172648255.py:350: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns 


EVENT FEATURE DATASET SUMMARY
Rows: 633137
Columns: 73
Symbols: 651
Date range: 1997-07-20 00:00:00 to 2026-10-06 00:00:00

Event feature columns:
AGM
BOOK_CLOSURE
DIVIDEND
FUNDAMENTAL_NEWS
IPO_FPO
LISTING
MERGER
OTHER_N
PRICE_ADJUSTMENT
RIGHTS
TRADING_STATUS
N_EVENT_TOTAL
dividend_1d
rights_1d
agm_1d
ipo_fpo_1d
merger_1d
listing_1d
trading_status_1d
price_adjustment_1d
book_closure_1d
fundamental_news_1d
other_n_1d
n_event_total_1d
dividend_5d
rights_5d
agm_5d
ipo_fpo_5d
merger_5d
listing_5d
trading_status_5d
price_adjustment_5d
book_closure_5d
fundamental_news_5d
other_n_5d
n_event_total_5d
dividend_20d
rights_20d
agm_20d
ipo_fpo_20d
merger_20d
listing_20d
trading_status_20d
price_adjustment_20d
book_closure_20d
fundamental_news_20d
other_n_20d
n_event_total_20d
dividend_60d
rights_60d
agm_60d
ipo_fpo_60d
merger_60d
listing_60d
trading_status_60d
price_adjustment_60d
book_closure_60d
fundamental_news_60d
other_n_60d
n_event_total_60d
days_since_n_event
days_since_dividend
days_since

In [ ]:
# ============================================================
# CELL 30 — DIVIDEND + RIGHTS ISSUE FEATURES
# ============================================================

print("=" * 70)
print("CELL 30 — DIVIDEND + RIGHTS FEATURES")
print("=" * 70)

import pandas as pd
import numpy as np
import re

# ============================================================
# PART 1 — DIVIDEND DATA
# ============================================================

print()
print("-" * 70)
print("DIVIDEND FEATURES")
print("-" * 70)

div = dividends_df.copy()

div["symbol"] = div["symbol"].astype(str).str.strip()

div["book_closure_date"] = pd.to_datetime(
    div["book_closure_date"],
    errors="coerce"
).dt.normalize()

div["cash_dividend_num"] = pd.to_numeric(
    div["cash_dividend"],
    errors="coerce"
).fillna(0)

div["bonus_dividend_num"] = pd.to_numeric(
    div["bonus_dividend"],
    errors="coerce"
).fillna(0)

div["total_num"] = pd.to_numeric(
    div["total"],
    errors="coerce"
).fillna(
    div["cash_dividend_num"] + div["bonus_dividend_num"]
)

# Only records with a usable event date
div = div[
    div["book_closure_date"].notna()
].copy()

print("Dividend records with valid book-closure date:", len(div))

# ------------------------------------------------------------
# Aggregate multiple records on same symbol/date
# ------------------------------------------------------------

div_daily = (
    div
    .groupby(
        ["symbol", "book_closure_date"],
        as_index=False
    )
    .agg(
        dividend_event_count=("symbol", "size"),
        cash_dividend=("cash_dividend_num", "sum"),
        bonus_dividend=("bonus_dividend_num", "sum"),
        total_dividend=("total_num", "sum")
    )
)

div_daily = div_daily.rename(
    columns={
        "book_closure_date": "event_date"
    }
)

# ------------------------------------------------------------
# Merge dividend events onto price calendar
# ------------------------------------------------------------

event_feature_daily = event_feature_daily.merge(
    div_daily,
    left_on=["symbol", "date"],
    right_on=["symbol", "event_date"],
    how="left"
)

event_feature_daily = event_feature_daily.drop(
    columns=["event_date"],
    errors="ignore"
)

dividend_value_columns = [
    "dividend_event_count",
    "cash_dividend",
    "bonus_dividend",
    "total_dividend"
]

for col in dividend_value_columns:

    event_feature_daily[col] = (
        pd.to_numeric(
            event_feature_daily[col],
            errors="coerce"
        )
        .fillna(0)
    )

# ------------------------------------------------------------
# Rolling dividend activity
# ------------------------------------------------------------

for window in [5, 20, 60]:

    event_feature_daily[
        f"dividend_events_{window}d"
    ] = (
        event_feature_daily[
            "dividend_event_count"
        ]
        .groupby(event_feature_daily["symbol"])
        .rolling(
            window=window,
            min_periods=1
        )
        .sum()
        .reset_index(
            level=0,
            drop=True
        )
        .values
    )

    event_feature_daily[
        f"dividend_cash_{window}d"
    ] = (
        event_feature_daily[
            "cash_dividend"
        ]
        .groupby(event_feature_daily["symbol"])
        .rolling(
            window=window,
            min_periods=1
        )
        .sum()
        .reset_index(
            level=0,
            drop=True
        )
        .values
    )

    event_feature_daily[
        f"dividend_bonus_{window}d"
    ] = (
        event_feature_daily[
            "bonus_dividend"
        ]
        .groupby(event_feature_daily["symbol"])
        .rolling(
            window=window,
            min_periods=1
        )
        .sum()
        .reset_index(
            level=0,
            drop=True
        )
        .values
    )

# ============================================================
# PART 2 — RIGHTS ISSUE DATA
# ============================================================

print()
print("-" * 70)
print("RIGHTS ISSUE FEATURES")
print("-" * 70)

rights = rights_df.copy()

rights["symbol"] = (
    rights["symbol"]
    .astype(str)
    .str.strip()
)

rights["opening_date"] = pd.to_datetime(
    rights["opening_date"],
    errors="coerce"
).dt.normalize()

rights["closing_date"] = pd.to_datetime(
    rights["closing_date"],
    errors="coerce"
).dt.normalize()

rights["book_closure_date"] = pd.to_datetime(
    rights["book_closure_date"],
    errors="coerce"
).dt.normalize()

# ------------------------------------------------------------
# Parse rights ratio
# ------------------------------------------------------------

def parse_rights_ratio(value):

    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    # Examples:
    # 1:1
    # 1 : 0.5
    # 10:3
    # etc.

    match = re.search(
        r"(\d+(?:\.\d+)?)\s*[:/]\s*(\d+(?:\.\d+)?)",
        text
    )

    if not match:
        return np.nan

    numerator = float(match.group(1))
    denominator = float(match.group(2))

    if denominator == 0:
        return np.nan

    return numerator / denominator


rights["rights_ratio_numeric"] = (
    rights["ratio"]
    .apply(parse_rights_ratio)
)

# ------------------------------------------------------------
# Use book closure as the primary rights event date.
# If unavailable, use opening date.
# ------------------------------------------------------------

rights["rights_event_date"] = (
    rights["book_closure_date"]
)

rights.loc[
    rights["rights_event_date"].isna(),
    "rights_event_date"
] = rights.loc[
    rights["rights_event_date"].isna(),
    "opening_date"
]

rights = rights[
    rights["rights_event_date"].notna()
].copy()

print("Rights records with usable event date:", len(rights))

# ------------------------------------------------------------
# Aggregate same symbol/date
# ------------------------------------------------------------

rights_daily = (
    rights
    .groupby(
        ["symbol", "rights_event_date"],
        as_index=False
    )
    .agg(
        rights_event_count=("symbol", "size"),
        rights_ratio=("rights_ratio_numeric", "max")
    )
)

rights_daily = rights_daily.rename(
    columns={
        "rights_event_date": "event_date"
    }
)

# ------------------------------------------------------------
# Merge onto trading calendar
# ------------------------------------------------------------

event_feature_daily = event_feature_daily.merge(
    rights_daily,
    left_on=["symbol", "date"],
    right_on=["symbol", "event_date"],
    how="left"
)

event_feature_daily = event_feature_daily.drop(
    columns=["event_date"],
    errors="ignore"
)

event_feature_daily["rights_event_count"] = (
    pd.to_numeric(
        event_feature_daily["rights_event_count"],
        errors="coerce"
    )
    .fillna(0)
)

event_feature_daily["rights_ratio"] = (
    pd.to_numeric(
        event_feature_daily["rights_ratio"],
        errors="coerce"
    )
)

# ------------------------------------------------------------
# Rolling rights activity
# ------------------------------------------------------------

for window in [5, 20, 60]:

    event_feature_daily[
        f"rights_events_{window}d"
    ] = (
        event_feature_daily[
            "rights_event_count"
        ]
        .groupby(event_feature_daily["symbol"])
        .rolling(
            window=window,
            min_periods=1
        )
        .sum()
        .reset_index(
            level=0,
            drop=True
        )
        .values
    )

# ============================================================
# PART 3 — DAYS SINCE DIVIDEND / RIGHTS
# ============================================================

print()
print("-" * 70)
print("DAYS SINCE CORPORATE ACTION")
print("-" * 70)

def add_days_since_feature(
    df,
    event_indicator,
    output_name
):

    def calc(group):

        dates = pd.to_datetime(
            group["date"]
        )

        event_dates = pd.to_datetime(
            group.loc[
                group[event_indicator] > 0,
                "date"
            ]
        ).values

        result = np.full(
            len(group),
            np.nan,
            dtype=float
        )

        if len(event_dates) == 0:
            return pd.Series(
                result,
                index=group.index
            )

        last_event = None

        for i, current_date in enumerate(dates):

            available = event_dates[
                event_dates <= current_date
            ]

            if len(available) > 0:
                last_event = available[-1]

            if last_event is not None:
                result[i] = (
                    current_date - last_event
                ).days

        return pd.Series(
            result,
            index=group.index
        )

    df[output_name] = (
        df
        .groupby("symbol", group_keys=False)
        .apply(calc)
        .reset_index(level=0, drop=True)
    )

    return df


event_feature_daily = add_days_since_feature(
    event_feature_daily,
    "dividend_event_count",
    "days_since_dividend_structured"
)

event_feature_daily = add_days_since_feature(
    event_feature_daily,
    "rights_event_count",
    "days_since_rights_structured"
)

# ============================================================
# PART 4 — VALIDATION
# ============================================================

print()
print("=" * 70)
print("CELL 30 VALIDATION")
print("=" * 70)

print(
    "Rows:",
    len(event_feature_daily)
)

print(
    "Symbols:",
    event_feature_daily["symbol"].nunique()
)

print(
    "Date range:",
    event_feature_daily["date"].min(),
    "to",
    event_feature_daily["date"].max()
)

print()

for col in [
    "dividend_event_count",
    "cash_dividend",
    "bonus_dividend",
    "total_dividend",
    "dividend_events_5d",
    "dividend_events_20d",
    "dividend_events_60d",
    "rights_event_count",
    "rights_events_5d",
    "rights_events_20d",
    "rights_events_60d",
    "rights_ratio"
]:

    if col in event_feature_daily.columns:

        nonzero = (
            event_feature_daily[col]
            .fillna(0)
            .ne(0)
            .sum()
        )

        print(
            f"{col:35s}: {nonzero:,}"
        )

print()
print(
    "Dividend structured event dates:",
    div_daily["event_date"].nunique()
)

print(
    "Rights structured event dates:",
    rights_daily["event_date"].nunique()
)

print()
print("=" * 70)
print("CELL 30 COMPLETE")
print("=" * 70)

CELL 30 — DIVIDEND + RIGHTS FEATURES

----------------------------------------------------------------------
DIVIDEND FEATURES
----------------------------------------------------------------------
Dividend records with valid book-closure date: 1664

----------------------------------------------------------------------
RIGHTS ISSUE FEATURES
----------------------------------------------------------------------
Rights records with usable event date: 312

----------------------------------------------------------------------
DAYS SINCE CORPORATE ACTION
----------------------------------------------------------------------


/tmp/ipykernel_768/2290588286.py:411: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc)



CELL 30 VALIDATION
Rows: 633137
Symbols: 651
Date range: 1997-07-20 00:00:00 to 2026-10-06 00:00:00

dividend_event_count               : 1,006
cash_dividend                      : 0
bonus_dividend                     : 0
total_dividend                     : 0
dividend_events_5d                 : 5,014
dividend_events_20d                : 19,718
dividend_events_60d                : 57,742
rights_event_count                 : 146
rights_events_5d                   : 726
rights_events_20d                  : 2,901
rights_events_60d                  : 8,701
rights_ratio                       : 146

Dividend structured event dates: 1056
Rights structured event dates: 284

CELL 30 COMPLETE


/tmp/ipykernel_768/2290588286.py:411: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc)


In [ ]:
# ============================================================
# DIAGNOSE CELL 31 INPUT
# ============================================================

print("=" * 70)
print("CELL 31 INPUT DIAGNOSTIC")
print("=" * 70)

print()
print("event_feature_daily exists:",
      "event_feature_daily" in globals())

if "event_feature_daily" in globals():

    print()
    print("Rows:", len(event_feature_daily))
    print("Columns:", len(event_feature_daily.columns))

    print()
    print("First 40 columns:")
    print(list(event_feature_daily.columns[:40]))

    print()
    print("Price columns present:")

    for col in [
        "open",
        "high",
        "low",
        "close",
        "volume",
        "source_open",
        "source_high",
        "source_low",
        "source_close"
    ]:
        print(
            f"  {col:20s}:",
            col in event_feature_daily.columns
        )

print()
print("=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

CELL 31 INPUT DIAGNOSTIC

event_feature_daily exists: True

Rows: 633137
Columns: 93

First 40 columns:
['symbol', 'date', 'AGM', 'BOOK_CLOSURE', 'DIVIDEND', 'FUNDAMENTAL_NEWS', 'IPO_FPO', 'LISTING', 'MERGER', 'OTHER_N', 'PRICE_ADJUSTMENT', 'RIGHTS', 'TRADING_STATUS', 'N_EVENT_TOTAL', 'dividend_1d', 'rights_1d', 'agm_1d', 'ipo_fpo_1d', 'merger_1d', 'listing_1d', 'trading_status_1d', 'price_adjustment_1d', 'book_closure_1d', 'fundamental_news_1d', 'other_n_1d', 'n_event_total_1d', 'dividend_5d', 'rights_5d', 'agm_5d', 'ipo_fpo_5d', 'merger_5d', 'listing_5d', 'trading_status_5d', 'price_adjustment_5d', 'book_closure_5d', 'fundamental_news_5d', 'other_n_5d', 'n_event_total_5d', 'dividend_20d', 'rights_20d']

Price columns present:
  open                : False
  high                : False
  low                 : False
  close               : False
  volume              : False
  source_open         : False
  source_high         : False
  source_low          : False
  source_close        

In [ ]:
# ============================================================
# CELL 30B — ROBUST PROJECT CHECKPOINT
# ============================================================

import os
import json
from datetime import datetime

PROJECT_DIR = "/content/NEPSE_ML_PROJECT"

RAW_DIR = os.path.join(PROJECT_DIR, "raw")
PROCESSED_DIR = os.path.join(PROJECT_DIR, "processed")
FEATURES_DIR = os.path.join(PROJECT_DIR, "features")
VALIDATION_DIR = os.path.join(PROJECT_DIR, "validation")
MODELS_DIR = os.path.join(PROJECT_DIR, "models")
REPORTS_DIR = os.path.join(PROJECT_DIR, "reports")

for d in [
    RAW_DIR,
    PROCESSED_DIR,
    FEATURES_DIR,
    VALIDATION_DIR,
    MODELS_DIR,
    REPORTS_DIR
]:
    os.makedirs(d, exist_ok=True)

print("=" * 70)
print("ROBUST PROJECT CHECKPOINT")
print("=" * 70)

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def save_dataframe_safe(df, path, name):

    print()
    print(f"Saving {name}...")

    # First try Parquet
    try:
        df.to_parquet(
            path,
            index=False
        )

        print(
            f"  OK: {len(df):,} rows × "
            f"{len(df.columns):,} columns"
        )
        print(f"  {path}")

        return True

    except Exception as e:

        print(
            f"  Parquet failed for {name}: "
            f"{type(e).__name__}"
        )

        # Safe fallback: CSV
        csv_path = os.path.splitext(path)[0] + ".csv.gz"

        try:

            df.to_csv(
                csv_path,
                index=False,
                compression="gzip"
            )

            print("  CSV fallback succeeded:")
            print(f"  {csv_path}")

            return True

        except Exception as e2:

            print(
                f"  CSV fallback also failed: "
                f"{type(e2).__name__}"
            )

            return False


# ------------------------------------------------------------
# Save current feature dataset
# ------------------------------------------------------------

saved_files = []

if "event_feature_daily" in globals():

    path = os.path.join(
        FEATURES_DIR,
        "event_feature_daily_checkpoint.parquet"
    )

    if save_dataframe_safe(
        event_feature_daily,
        path,
        "event_feature_daily"
    ):
        saved_files.append(path)


# ------------------------------------------------------------
# Supporting datasets
# ------------------------------------------------------------

datasets = {
    "adjusted_df": (
        PROCESSED_DIR,
        "adjusted_df_checkpoint.parquet"
    ),

    "unadjusted_flat_df": (
        PROCESSED_DIR,
        "unadjusted_flat_df_checkpoint.parquet"
    ),

    "marks_df": (
        PROCESSED_DIR,
        "marks_df_checkpoint.parquet"
    ),

    "rights_df": (
        PROCESSED_DIR,
        "rights_df_checkpoint.parquet"
    ),

    "dividends_df": (
        PROCESSED_DIR,
        "dividends_df_checkpoint.parquet"
    ),

    "universe": (
        PROCESSED_DIR,
        "security_master_checkpoint.parquet"
    ),
}

for variable_name, (directory, filename) in datasets.items():

    if variable_name not in globals():
        continue

    path = os.path.join(
        directory,
        filename
    )

    if save_dataframe_safe(
        globals()[variable_name],
        path,
        variable_name
    ):
        saved_files.append(path)


# ------------------------------------------------------------
# Save checkpoint manifest
# ------------------------------------------------------------

manifest = {
    "checkpoint_time": datetime.now().isoformat(),

    "stage": (
        "CELL 30 COMPLETE — "
        "CELL 31 NOT YET COMPLETE"
    ),

    "event_feature_daily": {
        "rows": (
            len(event_feature_daily)
            if "event_feature_daily" in globals()
            else None
        ),
        "columns": (
            len(event_feature_daily.columns)
            if "event_feature_daily" in globals()
            else None
        )
    },

    "saved_files": [
        os.path.relpath(
            p,
            PROJECT_DIR
        )
        for p in saved_files
    ]
}

manifest_path = os.path.join(
    VALIDATION_DIR,
    "project_checkpoint.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2
    )

print()
print("=" * 70)
print("CHECKPOINT COMPLETE")
print("=" * 70)

print()
print("Manifest:")
print(manifest_path)

print()
print("Files saved:", len(saved_files))

ROBUST PROJECT CHECKPOINT

Saving event_feature_daily...
  OK: 633,137 rows × 93 columns
  /content/NEPSE_ML_PROJECT/features/event_feature_daily_checkpoint.parquet

Saving adjusted_df...
  OK: 633,137 rows × 8 columns
  /content/NEPSE_ML_PROJECT/processed/adjusted_df_checkpoint.parquet

Saving unadjusted_flat_df...
  OK: 642,059 rows × 8 columns
  /content/NEPSE_ML_PROJECT/processed/unadjusted_flat_df_checkpoint.parquet

Saving marks_df...
  Parquet failed for marks_df: ArrowInvalid
  CSV fallback succeeded:
  /content/NEPSE_ML_PROJECT/processed/marks_df_checkpoint.csv.gz

Saving rights_df...
  OK: 312 rows × 16 columns
  /content/NEPSE_ML_PROJECT/processed/rights_df_checkpoint.parquet

Saving dividends_df...
  OK: 1,683 rows × 17 columns
  /content/NEPSE_ML_PROJECT/processed/dividends_df_checkpoint.parquet

Saving universe...
  OK: 737 rows × 16 columns
  /content/NEPSE_ML_PROJECT/processed/security_master_checkpoint.parquet

CHECKPOINT COMPLETE

Manifest:
/content/NEPSE_ML_PROJECT/v

In [ ]:
# ============================================================
# CELL 31 — PRICE + TECHNICAL FEATURES
# ============================================================

print("=" * 70)
print("CELL 31 — PRICE + TECHNICAL FEATURES")
print("=" * 70)

import pandas as pd
import numpy as np

# ============================================================
# PART 1 — LOAD CORRECTED ADJUSTED OHLCV
# ============================================================

print()
print("-" * 70)
print("LOADING CORRECTED OHLCV")
print("-" * 70)

OHLCV_PATH = (
    "/content/NEPSE_ML_PROJECT/processed/"
    "nepse_daily_ohlcv_adjusted_corrected.csv.gz"
)

price_df = pd.read_csv(
    OHLCV_PATH,
    compression="gzip"
)

price_df["date"] = pd.to_datetime(
    price_df["date"],
    errors="coerce"
).dt.normalize()

price_df["symbol"] = (
    price_df["symbol"]
    .astype(str)
    .str.strip()
)

print("Price rows:", len(price_df))
print("Price symbols:", price_df["symbol"].nunique())

# ============================================================
# PART 2 — PREPARE EVENT FEATURES
# ============================================================

df = event_feature_daily.copy()

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
).dt.normalize()

df["symbol"] = (
    df["symbol"]
    .astype(str)
    .str.strip()
)

print()
print("Event-feature rows:", len(df))
print("Event-feature symbols:", df["symbol"].nunique())

# ============================================================
# PART 3 — VERIFY PRICE KEY UNIQUENESS
# ============================================================

print()
print("-" * 70)
print("CHECKING MERGE KEYS")
print("-" * 70)

price_duplicates = price_df.duplicated(
    ["symbol", "date"]
).sum()

event_duplicates = df.duplicated(
    ["symbol", "date"]
).sum()

print(
    "Price duplicate symbol/date:",
    price_duplicates
)

print(
    "Event-feature duplicate symbol/date:",
    event_duplicates
)

if price_duplicates != 0:
    raise ValueError(
        "STOP: corrected OHLCV contains duplicate "
        "symbol/date rows."
    )

if event_duplicates != 0:
    raise ValueError(
        "STOP: event_feature_daily contains duplicate "
        "symbol/date rows."
    )

# ============================================================
# PART 4 — MERGE OHLCV
# ============================================================

print()
print("-" * 70)
print("MERGING PRICE DATA")
print("-" * 70)

price_columns = [
    "symbol",
    "date",
    "open",
    "high",
    "low",
    "close",
    "volume"
]

price_for_merge = price_df[
    price_columns
].copy()

before_rows = len(df)

df = df.merge(
    price_for_merge,
    on=["symbol", "date"],
    how="left",
    validate="one_to_one"
)

after_rows = len(df)

print("Rows before merge:", before_rows)
print("Rows after merge :", after_rows)

if before_rows != after_rows:
    raise ValueError(
        "STOP: OHLCV merge changed row count."
    )

# ============================================================
# PART 5 — BASIC PRICE CLEANING
# ============================================================

for col in [
    "open",
    "high",
    "low",
    "close",
    "volume"
]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# ============================================================
# PART 6 — RETURNS
# ============================================================

print()
print("-" * 70)
print("ADDING RETURNS")
print("-" * 70)

g = df.groupby(
    "symbol",
    group_keys=False
)

df["return_1d"] = (
    g["close"].pct_change(1)
)

df["return_5d"] = (
    g["close"].pct_change(5)
)

df["return_20d"] = (
    g["close"].pct_change(20)
)

df["return_60d"] = (
    g["close"].pct_change(60)
)

# ============================================================
# PART 7 — MOVING AVERAGES
# ============================================================

print("Adding SMA features...")

for window in [5, 20, 50, 100, 200]:

    df[f"sma_{window}"] = (
        g["close"]
        .rolling(
            window=window,
            min_periods=window
        )
        .mean()
        .reset_index(
            level=0,
            drop=True
        )
        .values
    )

    df[f"close_to_sma_{window}"] = (
        df["close"]
        / df[f"sma_{window}"]
        - 1
    )

# ============================================================
# PART 8 — EMA
# ============================================================

print("Adding EMA features...")

for span in [12, 26, 50]:

    df[f"ema_{span}"] = (
        g["close"]
        .ewm(
            span=span,
            adjust=False,
            min_periods=span
        )
        .mean()
        .reset_index(
            level=0,
            drop=True
        )
        .values
    )

# ============================================================
# PART 9 — PRICE RANGE
# ============================================================

print("Adding price-range features...")

df["intraday_range"] = (
    (df["high"] - df["low"])
    / df["close"].replace(0, np.nan)
)

df["open_close_return"] = (
    (df["close"] - df["open"])
    / df["open"].replace(0, np.nan)
)

df["high_close_distance"] = (
    (df["high"] - df["close"])
    / df["close"].replace(0, np.nan)
)

df["close_low_distance"] = (
    (df["close"] - df["low"])
    / df["close"].replace(0, np.nan)
)

# ============================================================
# PART 10 — VOLATILITY
# ============================================================

print("Adding volatility features...")

for window in [5, 10, 20, 60]:

    df[f"volatility_{window}d"] = (
        g["return_1d"]
        .rolling(
            window=window,
            min_periods=window
        )
        .std()
        .reset_index(
            level=0,
            drop=True
        )
        .values
    )

# ============================================================
# PART 11 — VOLUME FEATURES
# ============================================================

print("Adding volume features...")

df["volume_change_1d"] = (
    g["volume"].pct_change()
)

for window in [5, 20, 60]:

    df[f"volume_sma_{window}"] = (
        g["volume"]
        .rolling(
            window=window,
            min_periods=window
        )
        .mean()
        .reset_index(
            level=0,
            drop=True
        )
        .values
    )

    df[f"volume_ratio_{window}"] = (
        df["volume"]
        / df[f"volume_sma_{window}"]
        .replace(0, np.nan)
    )

# ============================================================
# PART 12 — RSI 14
# ============================================================

print("Adding RSI-14...")

delta = (
    df.groupby("symbol")["close"]
    .diff()
)

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = (
    gain
    .groupby(df["symbol"])
    .rolling(
        window=14,
        min_periods=14
    )
    .mean()
    .reset_index(
        level=0,
        drop=True
    )
)

avg_loss = (
    loss
    .groupby(df["symbol"])
    .rolling(
        window=14,
        min_periods=14
    )
    .mean()
    .reset_index(
        level=0,
        drop=True
    )
)

rs = (
    avg_gain
    / avg_loss.replace(0, np.nan)
)

df["rsi_14"] = (
    100
    - (100 / (1 + rs))
)

# ============================================================
# PART 13 — POSITION WITHIN RECENT RANGE
# ============================================================

print("Adding range-position features...")

for window in [5, 10, 20, 60]:

    rolling_high = (
        g["high"]
        .rolling(
            window=window,
            min_periods=window
        )
        .max()
        .reset_index(
            level=0,
            drop=True
        )
        .values
    )

    rolling_low = (
        g["low"]
        .rolling(
            window=window,
            min_periods=window
        )
        .min()
        .reset_index(
            level=0,
            drop=True
        )
        .values
    )

    denominator = (
        rolling_high - rolling_low
    )

    df[f"position_in_range_{window}d"] = (
        (
            df["close"].values
            - rolling_low
        )
        / np.where(
            denominator == 0,
            np.nan,
            denominator
        )
    )

# ============================================================
# PART 14 — CLEAN INFINITE VALUES
# ============================================================

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

# Keep sorted
df = df.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

event_feature_daily = df

# ============================================================
# VALIDATION
# ============================================================

print()
print("=" * 70)
print("CELL 31 VALIDATION")
print("=" * 70)

print(
    "Rows:",
    len(event_feature_daily)
)

print(
    "Columns:",
    len(event_feature_daily.columns)
)

print(
    "Symbols:",
    event_feature_daily["symbol"].nunique()
)

print(
    "Date range:",
    event_feature_daily["date"].min(),
    "to",
    event_feature_daily["date"].max()
)

print()
print("Price coverage:")

for col in [
    "open",
    "high",
    "low",
    "close",
    "volume"
]:

    valid = event_feature_daily[col].notna().sum()

    print(
        f"{col:10s}: "
        f"{valid:,} / {len(event_feature_daily):,}"
    )

print()
print("Technical features:")

technical_columns = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "sma_20",
    "sma_50",
    "sma_200",
    "ema_12",
    "ema_26",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "position_in_range_20d"
]

for col in technical_columns:

    valid = event_feature_daily[col].notna().sum()

    print(
        f"{col:30s}: {valid:,}"
    )

print()
print("=" * 70)
print("CELL 31 COMPLETE")
print("=" * 70)

CELL 31 — PRICE + TECHNICAL FEATURES

----------------------------------------------------------------------
LOADING CORRECTED OHLCV
----------------------------------------------------------------------
Price rows: 633137
Price symbols: 651

Event-feature rows: 633137
Event-feature symbols: 651

----------------------------------------------------------------------
CHECKING MERGE KEYS
----------------------------------------------------------------------
Price duplicate symbol/date: 0
Event-feature duplicate symbol/date: 0

----------------------------------------------------------------------
MERGING PRICE DATA
----------------------------------------------------------------------
Rows before merge: 633137
Rows after merge : 633137

----------------------------------------------------------------------
ADDING RETURNS
----------------------------------------------------------------------
Adding SMA features...
Adding EMA features...
Adding price-range features...
Adding volatility fea

In [ ]:
# ============================================================
# CELL 32 — NEPSE MARKET / MARKET-REGIME FEATURES
# RAM-SAFE VERSION
#
# IMPORTANT:
# - Does NOT calculate rolling market beta here.
# - Avoids the memory-heavy groupby().rolling().cov().
# - Raw data is untouched.
# ============================================================

print("=" * 70)
print("CELL 32 — NEPSE MARKET FEATURES (RAM-SAFE)")
print("=" * 70)

import pandas as pd
import numpy as np
import gc

# ============================================================
# PART 1 — START FROM CELL 31 DATA
# ============================================================

print("\nPreparing current feature dataset...")

df = event_feature_daily.copy()

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
).dt.normalize()

df["symbol"] = (
    df["symbol"]
    .astype(str)
    .str.strip()
)

df = df.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

gc.collect()

print("Rows:", len(df))
print("Columns:", len(df.columns))

# ============================================================
# PART 2 — EXTRACT NEPSE INDEX
# ============================================================

print()
print("-" * 70)
print("PREPARING NEPSE INDEX")
print("-" * 70)

nepse = df.loc[
    df["symbol"].eq("NEPSE"),
    ["date", "open", "high", "low", "close", "volume"]
].copy()

nepse = (
    nepse
    .sort_values("date")
    .drop_duplicates("date")
    .reset_index(drop=True)
)

print("NEPSE rows:", len(nepse))
print(
    "NEPSE date range:",
    nepse["date"].min(),
    "to",
    nepse["date"].max()
)

if len(nepse) == 0:
    raise ValueError("STOP: NEPSE index not found.")

# ============================================================
# PART 3 — MARKET RETURNS
# ============================================================

print("\nAdding market returns...")

nepse["market_return_1d"] = (
    nepse["close"].pct_change(1)
)

nepse["market_return_5d"] = (
    nepse["close"].pct_change(5)
)

nepse["market_return_20d"] = (
    nepse["close"].pct_change(20)
)

nepse["market_return_60d"] = (
    nepse["close"].pct_change(60)
)

# ============================================================
# PART 4 — MARKET MOVING AVERAGES
# ============================================================

print("Adding market moving averages...")

for window in [20, 50, 100, 200]:

    sma = (
        nepse["close"]
        .rolling(
            window=window,
            min_periods=window
        )
        .mean()
    )

    nepse[f"market_sma_{window}"] = sma

    nepse[f"market_close_to_sma_{window}"] = (
        nepse["close"] / sma - 1
    )

# ============================================================
# PART 5 — MARKET VOLATILITY
# ============================================================

print("Adding market volatility...")

for window in [5, 10, 20, 60]:

    nepse[f"market_volatility_{window}d"] = (
        nepse["market_return_1d"]
        .rolling(
            window=window,
            min_periods=window
        )
        .std()
    )

# ============================================================
# PART 6 — MARKET INTRADAY RANGE
# ============================================================

nepse["market_intraday_range"] = (
    (nepse["high"] - nepse["low"])
    / nepse["close"].replace(0, np.nan)
)

# ============================================================
# PART 7 — MARKET REGIME
# ============================================================

print("Adding market regime features...")

nepse["market_bull_trend"] = (
    (
        nepse["close"]
        > nepse["market_sma_20"]
    )
    &
    (
        nepse["market_sma_20"]
        > nepse["market_sma_50"]
    )
).astype("int8")

nepse["market_bear_trend"] = (
    (
        nepse["close"]
        < nepse["market_sma_20"]
    )
    &
    (
        nepse["market_sma_20"]
        < nepse["market_sma_50"]
    )
).astype("int8")

vol_median = (
    nepse["market_volatility_20d"]
    .rolling(
        window=120,
        min_periods=60
    )
    .median()
)

nepse["market_high_volatility"] = (
    nepse["market_volatility_20d"]
    > vol_median
).astype("int8")

# ============================================================
# PART 8 — MARKET FEATURE TABLE
# ============================================================

market_feature_columns = [
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_return_60d",

    "market_sma_20",
    "market_sma_50",
    "market_sma_100",
    "market_sma_200",

    "market_close_to_sma_20",
    "market_close_to_sma_50",
    "market_close_to_sma_100",
    "market_close_to_sma_200",

    "market_volatility_5d",
    "market_volatility_10d",
    "market_volatility_20d",
    "market_volatility_60d",

    "market_intraday_range",

    "market_bull_trend",
    "market_bear_trend",
    "market_high_volatility"
]

market_features = nepse[
    ["date"] + market_feature_columns
].copy()

if not market_features["date"].is_unique:
    raise ValueError(
        "STOP: duplicate NEPSE dates found."
    )

# Free unnecessary NEPSE columns before merge
nepse = None
vol_median = None

gc.collect()

# ============================================================
# PART 9 — MERGE MARKET FEATURES
# ============================================================

print()
print("-" * 70)
print("MERGING MARKET FEATURES")
print("-" * 70)

before_rows = len(df)

df = df.merge(
    market_features,
    on="date",
    how="left",
    validate="many_to_one",
    sort=False
)

after_rows = len(df)

print("Rows before merge:", before_rows)
print("Rows after merge :", after_rows)

if before_rows != after_rows:
    raise ValueError(
        "STOP: market merge changed row count."
    )

# ============================================================
# PART 10 — RELATIVE PERFORMANCE
# ============================================================

print()
print("Adding relative-performance features...")

for horizon in [1, 5, 20, 60]:

    df[f"relative_return_{horizon}d"] = (
        df[f"return_{horizon}d"]
        - df[f"market_return_{horizon}d"]
    )

# ============================================================
# PART 11 — MARKET BREADTH
# ============================================================

print("Calculating market breadth...")

# Only keep the two columns needed.
breadth_source = df[
    ["date", "symbol", "return_1d"]
].copy()

breadth_source = breadth_source[
    breadth_source["symbol"].ne("NEPSE")
]

breadth_source["up"] = (
    breadth_source["return_1d"] > 0
).astype("float32")

daily_breadth = (
    breadth_source
    .groupby("date", sort=False)["up"]
    .mean()
    .rename("market_breadth_up_ratio")
    .reset_index()
)

del breadth_source
gc.collect()

df = df.merge(
    daily_breadth,
    on="date",
    how="left",
    validate="many_to_one",
    sort=False
)

del daily_breadth
del market_features
gc.collect()

# ============================================================
# PART 12 — CLEANUP
# ============================================================

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

df = df.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

event_feature_daily = df

del df
gc.collect()

# ============================================================
# VALIDATION
# ============================================================

print()
print("=" * 70)
print("CELL 32 VALIDATION")
print("=" * 70)

print(
    "Rows:",
    len(event_feature_daily)
)

print(
    "Columns:",
    len(event_feature_daily.columns)
)

print(
    "Symbols:",
    event_feature_daily["symbol"].nunique()
)

print(
    "Date range:",
    event_feature_daily["date"].min(),
    "to",
    event_feature_daily["date"].max()
)

print()
print("Market feature coverage:")

test_columns = [
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_return_60d",

    "market_sma_20",
    "market_sma_50",
    "market_sma_200",

    "market_volatility_20d",

    "market_bull_trend",
    "market_bear_trend",
    "market_high_volatility",

    "relative_return_1d",
    "relative_return_20d",

    "market_breadth_up_ratio"
]

for col in test_columns:

    valid = (
        event_feature_daily[col]
        .notna()
        .sum()
    )

    print(
        f"{col:32s}: {valid:,}"
    )

# Confirm beta was intentionally not created.
print()
print("Market beta:")
print("  SKIPPED in Cell 32 to prevent RAM exhaustion.")
print("  Can be added later as a separate optimized feature.")

print()
print("=" * 70)
print("CELL 32 COMPLETE — RAM-SAFE")
print("=" * 70)

CELL 32 — NEPSE MARKET FEATURES (RAM-SAFE)

Preparing current feature dataset...


NameError: name 'event_feature_daily' is not defined

In [ ]:
# ============================================================
# EMERGENCY BACKUP — SAVE ENTIRE NEPSE PROJECT
# ============================================================

import os
import shutil
import zipfile
from datetime import datetime

PROJECT = "/content/NEPSE_ML_PROJECT"

if not os.path.exists(PROJECT):
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT}"
    )

print("=" * 70)
print("NEPSE PROJECT BACKUP")
print("=" * 70)

# ------------------------------------------------------------
# 1. SHOW WHAT IS ALREADY SAVED
# ------------------------------------------------------------

print("\nSaved project files:")

for root, dirs, files in os.walk(PROJECT):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / (1024**2)
        print(f"{size_mb:9.1f} MB  {os.path.relpath(path, PROJECT)}")

# ------------------------------------------------------------
# 2. CREATE ZIP
# ------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

zip_path = f"/content/NEPSE_ML_PROJECT_BACKUP_{timestamp}.zip"

print("\nCreating ZIP:")
print(zip_path)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1
) as z:

    for root, dirs, files in os.walk(PROJECT):

        for f in files:

            full_path = os.path.join(root, f)

            arcname = os.path.relpath(
                full_path,
                "/content"
            )

            z.write(
                full_path,
                arcname
            )

print("\nZIP created.")

zip_size_gb = os.path.getsize(zip_path) / (1024**3)

print(
    f"ZIP size: {zip_size_gb:.2f} GB"
)

# ------------------------------------------------------------
# 3. MOUNT GOOGLE DRIVE
# ------------------------------------------------------------

from google.colab import drive

drive.mount("/content/drive")

DRIVE_BACKUP = (
    "/content/drive/MyDrive/"
    "NEPSE_ML_PROJECT_BACKUP"
)

os.makedirs(
    DRIVE_BACKUP,
    exist_ok=True
)

print("\nCopying project to Google Drive...")
print(DRIVE_BACKUP)

# Remove previous backup only if it exists.
# This creates a clean, exact snapshot.
if os.path.exists(DRIVE_BACKUP):
    pass

for item in os.listdir(PROJECT):

    src = os.path.join(PROJECT, item)
    dst = os.path.join(DRIVE_BACKUP, item)

    if os.path.isdir(src):

        if os.path.exists(dst):
            shutil.rmtree(dst)

        shutil.copytree(src, dst)

    else:

        shutil.copy2(src, dst)

print("\nGoogle Drive backup complete.")

# ------------------------------------------------------------
# 4. COPY ZIP TO DRIVE TOO
# ------------------------------------------------------------

drive_zip = os.path.join(
    "/content/drive/MyDrive",
    os.path.basename(zip_path)
)

shutil.copy2(
    zip_path,
    drive_zip
)

print("\nZIP also copied to Google Drive:")
print(drive_zip)

# ------------------------------------------------------------
# 5. DOWNLOAD ZIP TO PC
# ------------------------------------------------------------

print("\nStarting PC download...")

from google.colab import files

files.download(zip_path)

print("\n" + "=" * 70)
print("BACKUP COMPLETE")
print("=" * 70)

NEPSE PROJECT BACKUP

Saved project files:
      3.1 MB  features/event_feature_daily_checkpoint.parquet
      0.0 MB  validation/ohlc_anomaly_by_class_unadjusted.csv
      0.0 MB  validation/securities_universe_summary.json
      0.0 MB  validation/ohlc_anomaly_by_year_adjusted.csv
      0.0 MB  validation/ohlc_anomaly_by_year_unadjusted.csv
      0.1 MB  validation/security_universe_diagnostic.csv
      0.0 MB  validation/raw_data_snapshot.json
      0.0 MB  validation/historical_ohlc_anomaly_samples.csv
      0.0 MB  validation/ohlc_anomaly_by_class_adjusted.csv
      0.0 MB  validation/project_checkpoint.json
      0.0 MB  validation/ohlcv_integrity_audit.json
      0.2 MB  raw/nepsealpha_symbols.json
      0.2 MB  raw/nepse_symbols.json
      0.0 MB  raw/nepse_download_errors.json
      0.3 MB  raw/nepse_marks.json.gz
      0.0 MB  raw/nepse_daily_ohlcv_unadjusted_errors.json
     13.1 MB  raw/nepse_daily_ohlcv.json.gz
      0.0 MB  raw/nepse_rights_issues.json.gz
      0.1 MB  ra

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


BACKUP COMPLETE


In [ ]:
# ============================================================
# CELL 30R — RESTORE SAVED PROJECT CHECKPOINT
# ============================================================

import os
import pandas as pd
import numpy as np
import gc

PROJECT = "/content/NEPSE_ML_PROJECT"

CHECKPOINT = (
    f"{PROJECT}/features/"
    "event_feature_daily_checkpoint.parquet"
)

print("=" * 70)
print("RESTORING CELL 30 CHECKPOINT")
print("=" * 70)

if not os.path.exists(CHECKPOINT):
    raise FileNotFoundError(
        f"Checkpoint not found:\n{CHECKPOINT}"
    )

event_feature_daily = pd.read_parquet(
    CHECKPOINT
)

event_feature_daily["date"] = pd.to_datetime(
    event_feature_daily["date"],
    errors="coerce"
).dt.normalize()

event_feature_daily["symbol"] = (
    event_feature_daily["symbol"]
    .astype(str)
    .str.strip()
)

print()
print("Restored successfully.")
print("Rows:", len(event_feature_daily))
print("Columns:", len(event_feature_daily.columns))
print("Symbols:", event_feature_daily["symbol"].nunique())
print(
    "Date range:",
    event_feature_daily["date"].min(),
    "to",
    event_feature_daily["date"].max()
)

print()
print("This is the SAVED CELL 30 state.")
print("Cell 31 technical features still need to be rebuilt.")

gc.collect()

print()
print("=" * 70)
print("CELL 30 RESTORE COMPLETE")
print("=" * 70)

RESTORING CELL 30 CHECKPOINT

Restored successfully.
Rows: 633137
Columns: 93
Symbols: 651
Date range: 1997-07-20 00:00:00 to 2026-10-06 00:00:00

This is the SAVED CELL 30 state.
Cell 31 technical features still need to be rebuilt.

CELL 30 RESTORE COMPLETE


In [ ]:
# ============================================================
# CELL 31R — TECHNICAL FEATURES (RAM-CONSCIOUS)
# ============================================================

import pandas as pd
import numpy as np
import gc
import os

print("=" * 70)
print("CELL 31 — TECHNICAL FEATURES")
print("=" * 70)

PROJECT = "/content/NEPSE_ML_PROJECT"

# ============================================================
# PART 1 — LOAD CORRECTED ADJUSTED OHLCV
# ============================================================

PRICE_FILE = (
    f"{PROJECT}/processed/"
    "nepse_daily_ohlcv_adjusted_corrected.csv.gz"
)

print("\nLoading corrected adjusted OHLCV...")

price = pd.read_csv(
    PRICE_FILE,
    compression="gzip"
)

price["date"] = pd.to_datetime(
    price["date"],
    errors="coerce"
).dt.normalize()

price["symbol"] = (
    price["symbol"]
    .astype(str)
    .str.strip()
)

price = price.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

print("Price rows:", len(price))
print("Price symbols:", price["symbol"].nunique())

# ============================================================
# PART 2 — MERGE WITH CELL 30 FEATURES
# ============================================================

print("\nMerging price data with Cell 30 features...")

df = event_feature_daily.merge(
    price,
    on=["symbol", "date"],
    how="left",
    validate="one_to_one",
    sort=False
)

print("Rows after merge:", len(df))

if len(df) != len(event_feature_daily):
    raise ValueError(
        "STOP: price merge changed row count."
    )

del price
gc.collect()

# ============================================================
# PART 3 — SORT
# ============================================================

df = df.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

# ============================================================
# PART 4 — BASIC RETURNS
# ============================================================

print("\nCalculating returns...")

grouped_close = df.groupby(
    "symbol",
    sort=False
)["close"]

df["return_1d"] = (
    grouped_close.pct_change(1)
)

df["return_5d"] = (
    grouped_close.pct_change(5)
)

df["return_20d"] = (
    grouped_close.pct_change(20)
)

df["return_60d"] = (
    grouped_close.pct_change(60)
)

gc.collect()

# ============================================================
# PART 5 — MOVING AVERAGES
# ============================================================

print("Calculating moving averages...")

for window in [20, 50, 200]:

    df[f"sma_{window}"] = (
        df.groupby("symbol", sort=False)["close"]
        .transform(
            lambda x: x.rolling(
                window=window,
                min_periods=window
            ).mean()
        )
    )

    df[f"close_to_sma_{window}"] = (
        df["close"] / df[f"sma_{window}"] - 1
    )

    gc.collect()

# ============================================================
# PART 6 — EMA
# ============================================================

print("Calculating EMAs...")

for window in [12, 26]:

    df[f"ema_{window}"] = (
        df.groupby("symbol", sort=False)["close"]
        .transform(
            lambda x: x.ewm(
                span=window,
                adjust=False,
                min_periods=window
            ).mean()
        )
    )

    gc.collect()

# ============================================================
# PART 7 — RSI
# ============================================================

print("Calculating RSI...")

delta = (
    df.groupby("symbol", sort=False)["close"]
    .diff()
)

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = (
    gain.groupby(df["symbol"], sort=False)
    .transform(
        lambda x: x.rolling(
            window=14,
            min_periods=14
        ).mean()
    )
)

avg_loss = (
    loss.groupby(df["symbol"], sort=False)
    .transform(
        lambda x: x.rolling(
            window=14,
            min_periods=14
        ).mean()
    )
)

rs = (
    avg_gain
    /
    avg_loss.replace(0, np.nan)
)

df["rsi_14"] = (
    100 - (100 / (1 + rs))
)

del delta, gain, loss, avg_gain, avg_loss, rs
gc.collect()

# ============================================================
# PART 8 — VOLATILITY
# ============================================================

print("Calculating volatility...")

df["volatility_20d"] = (
    df.groupby("symbol", sort=False)["return_1d"]
    .transform(
        lambda x: x.rolling(
            window=20,
            min_periods=20
        ).std()
    )
)

gc.collect()

# ============================================================
# PART 9 — VOLUME RATIO
# ============================================================

print("Calculating volume ratio...")

volume_ma20 = (
    df.groupby("symbol", sort=False)["volume"]
    .transform(
        lambda x: x.rolling(
            window=20,
            min_periods=20
        ).mean()
    )
)

df["volume_ratio_20"] = (
    df["volume"] / volume_ma20.replace(0, np.nan)
)

del volume_ma20
gc.collect()

# ============================================================
# PART 10 — 20-DAY PRICE POSITION
# ============================================================

print("Calculating 20-day price position...")

rolling_high_20 = (
    df.groupby("symbol", sort=False)["high"]
    .transform(
        lambda x: x.rolling(
            window=20,
            min_periods=20
        ).max()
    )
)

rolling_low_20 = (
    df.groupby("symbol", sort=False)["low"]
    .transform(
        lambda x: x.rolling(
            window=20,
            min_periods=20
        ).min()
    )
)

price_range = (
    rolling_high_20 - rolling_low_20
)

df["position_in_range_20d"] = (
    (df["close"] - rolling_low_20)
    /
    price_range.replace(0, np.nan)
)

del rolling_high_20, rolling_low_20, price_range
gc.collect()

# ============================================================
# PART 11 — CLEANUP
# ============================================================

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

df = df.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

event_feature_daily = df

del df
gc.collect()

# ============================================================
# VALIDATION
# ============================================================

print()
print("=" * 70)
print("CELL 31 VALIDATION")
print("=" * 70)

print(
    "Rows:",
    len(event_feature_daily)
)

print(
    "Columns:",
    len(event_feature_daily.columns)
)

print(
    "Symbols:",
    event_feature_daily["symbol"].nunique()
)

print(
    "Date range:",
    event_feature_daily["date"].min(),
    "to",
    event_feature_daily["date"].max()
)

technical_columns = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "sma_20",
    "sma_50",
    "sma_200",
    "ema_12",
    "ema_26",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "position_in_range_20d"
]

print()
print("Technical feature coverage:")

for col in technical_columns:

    valid = (
        event_feature_daily[col]
        .notna()
        .sum()
    )

    print(
        f"{col:28s}: {valid:,}"
    )

# ============================================================
# PART 12 — SAVE CHECKPOINT IMMEDIATELY
# ============================================================

print()
print("-" * 70)
print("SAVING CELL 31 CHECKPOINT")
print("-" * 70)

checkpoint_path = (
    f"{PROJECT}/features/"
    "event_feature_daily_cell31_checkpoint.parquet"
)

event_feature_daily.to_parquet(
    checkpoint_path,
    index=False
)

print(
    "Checkpoint saved:",
    checkpoint_path
)

print(
    "Checkpoint size:",
    f"{os.path.getsize(checkpoint_path)/(1024**2):.1f} MB"
)

print()
print("=" * 70)
print("CELL 31 COMPLETE + CHECKPOINTED")
print("=" * 70)

CELL 31 — TECHNICAL FEATURES

Loading corrected adjusted OHLCV...
Price rows: 633137
Price symbols: 651

Merging price data with Cell 30 features...
Rows after merge: 633137

Calculating returns...
Calculating moving averages...
Calculating EMAs...
Calculating RSI...
Calculating volatility...
Calculating volume ratio...
Calculating 20-day price position...

CELL 31 VALIDATION
Rows: 633137
Columns: 127
Symbols: 651
Date range: 1997-07-20 00:00:00 to 2026-10-06 00:00:00

Technical feature coverage:
return_1d                   : 632,484
return_5d                   : 629,956
return_20d                  : 621,010
return_60d                  : 599,066
sma_20                      : 621,591
sma_50                      : 604,906
sma_200                     : 531,919
ema_12                      : 626,316
ema_26                      : 618,136
rsi_14                      : 622,397
volatility_20d              : 620,991
volume_ratio_20             : 605,724
position_in_range_20d       : 621,355

---

In [ ]:
# ============================================================
# CELL 32 — NEPSE MARKET / MARKET-REGIME FEATURES
# RAM-SAFE + CHECKPOINTED
# ============================================================

import pandas as pd
import numpy as np
import gc
import os

print("=" * 70)
print("CELL 32 — NEPSE MARKET FEATURES (RAM-SAFE)")
print("=" * 70)

# ============================================================
# PART 1 — START FROM CELL 31
# ============================================================

print("\nPreparing Cell 31 dataset...")

df = event_feature_daily.copy()

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
).dt.normalize()

df["symbol"] = (
    df["symbol"]
    .astype(str)
    .str.strip()
)

df = df.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

print("Rows:", len(df))
print("Columns:", len(df.columns))

gc.collect()

# ============================================================
# PART 2 — EXTRACT NEPSE INDEX
# ============================================================

print()
print("-" * 70)
print("PREPARING NEPSE INDEX")
print("-" * 70)

nepse = df.loc[
    df["symbol"].eq("NEPSE"),
    ["date", "open", "high", "low", "close", "volume"]
].copy()

nepse = (
    nepse
    .sort_values("date")
    .drop_duplicates("date")
    .reset_index(drop=True)
)

print("NEPSE index rows:", len(nepse))
print(
    "NEPSE date range:",
    nepse["date"].min(),
    "to",
    nepse["date"].max()
)

if len(nepse) == 0:
    raise ValueError("STOP: NEPSE index not found.")

# ============================================================
# PART 3 — MARKET RETURNS
# ============================================================

print("\nAdding market returns...")

nepse["market_return_1d"] = nepse["close"].pct_change(1)
nepse["market_return_5d"] = nepse["close"].pct_change(5)
nepse["market_return_20d"] = nepse["close"].pct_change(20)
nepse["market_return_60d"] = nepse["close"].pct_change(60)

# ============================================================
# PART 4 — MARKET MOVING AVERAGES
# ============================================================

print("Adding market moving averages...")

for window in [20, 50, 100, 200]:

    sma = (
        nepse["close"]
        .rolling(
            window=window,
            min_periods=window
        )
        .mean()
    )

    nepse[f"market_sma_{window}"] = sma

    nepse[f"market_close_to_sma_{window}"] = (
        nepse["close"] / sma - 1
    )

# ============================================================
# PART 5 — MARKET VOLATILITY
# ============================================================

print("Adding market volatility...")

for window in [5, 10, 20, 60]:

    nepse[f"market_volatility_{window}d"] = (
        nepse["market_return_1d"]
        .rolling(
            window=window,
            min_periods=window
        )
        .std()
    )

# ============================================================
# PART 6 — MARKET INTRADAY RANGE
# ============================================================

nepse["market_intraday_range"] = (
    (nepse["high"] - nepse["low"])
    / nepse["close"].replace(0, np.nan)
)

# ============================================================
# PART 7 — MARKET TREND / REGIME
# ============================================================

print("Adding market regime features...")

nepse["market_bull_trend"] = (
    (nepse["close"] > nepse["market_sma_20"]) &
    (nepse["market_sma_20"] > nepse["market_sma_50"])
).astype("int8")

nepse["market_bear_trend"] = (
    (nepse["close"] < nepse["market_sma_20"]) &
    (nepse["market_sma_20"] < nepse["market_sma_50"])
).astype("int8")

vol_median = (
    nepse["market_volatility_20d"]
    .rolling(
        window=120,
        min_periods=60
    )
    .median()
)

nepse["market_high_volatility"] = (
    nepse["market_volatility_20d"] > vol_median
).astype("int8")

# ============================================================
# PART 8 — MARKET FEATURE TABLE
# ============================================================

market_feature_columns = [
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_return_60d",

    "market_sma_20",
    "market_sma_50",
    "market_sma_100",
    "market_sma_200",

    "market_close_to_sma_20",
    "market_close_to_sma_50",
    "market_close_to_sma_100",
    "market_close_to_sma_200",

    "market_volatility_5d",
    "market_volatility_10d",
    "market_volatility_20d",
    "market_volatility_60d",

    "market_intraday_range",

    "market_bull_trend",
    "market_bear_trend",
    "market_high_volatility"
]

market_features = nepse[
    ["date"] + market_feature_columns
].copy()

if not market_features["date"].is_unique:
    raise ValueError(
        "STOP: duplicate NEPSE dates found."
    )

del nepse
del vol_median
gc.collect()

# ============================================================
# PART 9 — MERGE MARKET FEATURES
# ============================================================

print()
print("-" * 70)
print("MERGING MARKET FEATURES")
print("-" * 70)

before_rows = len(df)

df = df.merge(
    market_features,
    on="date",
    how="left",
    validate="many_to_one",
    sort=False
)

after_rows = len(df)

print("Rows before merge:", before_rows)
print("Rows after merge :", after_rows)

if before_rows != after_rows:
    raise ValueError(
        "STOP: market merge changed row count."
    )

# ============================================================
# PART 10 — RELATIVE PERFORMANCE
# ============================================================

print("\nAdding relative-performance features...")

for horizon in [1, 5, 20, 60]:

    df[f"relative_return_{horizon}d"] = (
        df[f"return_{horizon}d"]
        - df[f"market_return_{horizon}d"]
    )

gc.collect()

# ============================================================
# PART 11 — MARKET BREADTH
# ============================================================

print("Calculating market breadth...")

breadth_source = df[
    ["date", "symbol", "return_1d"]
].copy()

breadth_source = breadth_source[
    breadth_source["symbol"].ne("NEPSE")
]

breadth_source["up"] = (
    breadth_source["return_1d"] > 0
).astype("float32")

daily_breadth = (
    breadth_source
    .groupby("date", sort=False)["up"]
    .mean()
    .rename("market_breadth_up_ratio")
    .reset_index()
)

del breadth_source
gc.collect()

df = df.merge(
    daily_breadth,
    on="date",
    how="left",
    validate="many_to_one",
    sort=False
)

del daily_breadth
del market_features
gc.collect()

# ============================================================
# PART 12 — CLEANUP
# ============================================================

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

df = df.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

event_feature_daily = df

del df
gc.collect()

# ============================================================
# VALIDATION
# ============================================================

print()
print("=" * 70)
print("CELL 32 VALIDATION")
print("=" * 70)

print("Rows:", len(event_feature_daily))
print("Columns:", len(event_feature_daily.columns))
print("Symbols:", event_feature_daily["symbol"].nunique())

print(
    "Date range:",
    event_feature_daily["date"].min(),
    "to",
    event_feature_daily["date"].max()
)

test_columns = [
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_return_60d",

    "market_sma_20",
    "market_sma_50",
    "market_sma_100",
    "market_sma_200",

    "market_volatility_5d",
    "market_volatility_10d",
    "market_volatility_20d",
    "market_volatility_60d",

    "market_bull_trend",
    "market_bear_trend",
    "market_high_volatility",

    "relative_return_1d",
    "relative_return_5d",
    "relative_return_20d",
    "relative_return_60d",

    "market_breadth_up_ratio"
]

print("\nMarket feature coverage:")

for col in test_columns:

    valid = event_feature_daily[col].notna().sum()

    print(
        f"{col:34s}: {valid:,}"
    )

print()
print("Market beta:")
print("  Not calculated in Cell 32.")
print("  Deferred to a separate optimized step if needed.")

# ============================================================
# PART 13 — CHECKPOINT
# ============================================================

print()
print("-" * 70)
print("SAVING CELL 32 CHECKPOINT")
print("-" * 70)

checkpoint_path = (
    "/content/NEPSE_ML_PROJECT/features/"
    "event_feature_daily_cell32_checkpoint.parquet"
)

event_feature_daily.to_parquet(
    checkpoint_path,
    index=False
)

checkpoint_size = (
    os.path.getsize(checkpoint_path)
    / (1024 ** 2)
)

print("Checkpoint saved:")
print(checkpoint_path)

print(
    f"Checkpoint size: {checkpoint_size:.1f} MB"
)

print()
print("=" * 70)
print("CELL 32 COMPLETE + CHECKPOINTED")
print("=" * 70)

CELL 32 — NEPSE MARKET FEATURES (RAM-SAFE)

Preparing Cell 31 dataset...
Rows: 633137
Columns: 127

----------------------------------------------------------------------
PREPARING NEPSE INDEX
----------------------------------------------------------------------
NEPSE index rows: 6607
NEPSE date range: 1997-07-20 00:00:00 to 2026-10-06 00:00:00

Adding market returns...
Adding market moving averages...
Adding market volatility...
Adding market regime features...

----------------------------------------------------------------------
MERGING MARKET FEATURES
----------------------------------------------------------------------
Rows before merge: 633137
Rows after merge : 633137

Adding relative-performance features...
Calculating market breadth...

CELL 32 VALIDATION
Rows: 633137
Columns: 152
Symbols: 651
Date range: 1997-07-20 00:00:00 to 2026-10-06 00:00:00

Market feature coverage:
market_return_1d                  : 632,154
market_return_5d                  : 632,150
market_return_

In [ ]:
# ============================================================
# CELL 32A — BACKUP CELL 32 CHECKPOINT TO GOOGLE DRIVE
# ============================================================

import os
import shutil

local_file = (
    "/content/NEPSE_ML_PROJECT/features/"
    "event_feature_daily_cell32_checkpoint.parquet"
)

drive_dir = (
    "/content/drive/MyDrive/"
    "NEPSE_ML_PROJECT_BACKUP/features"
)

if not os.path.exists(local_file):
    raise FileNotFoundError(
        "Cell 32 checkpoint not found."
    )

os.makedirs(
    drive_dir,
    exist_ok=True
)

drive_file = os.path.join(
    drive_dir,
    os.path.basename(local_file)
)

print("=" * 70)
print("BACKING UP CELL 32 CHECKPOINT")
print("=" * 70)

shutil.copy2(
    local_file,
    drive_file
)

print()
print("Local:")
print(local_file)

print()
print("Google Drive:")
print(drive_file)

print()
print(
    "Size:",
    f"{os.path.getsize(drive_file)/(1024**2):.1f} MB"
)

print()
print("=" * 70)
print("CELL 32 DRIVE BACKUP COMPLETE")
print("=" * 70)

BACKING UP CELL 32 CHECKPOINT

Local:
/content/NEPSE_ML_PROJECT/features/event_feature_daily_cell32_checkpoint.parquet

Google Drive:
/content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP/features/event_feature_daily_cell32_checkpoint.parquet

Size: 130.0 MB

CELL 32 DRIVE BACKUP COMPLETE


In [ ]:
# ==============================================================================
# CELL 33 — SECTOR-LEVEL FEATURES (RAM-SAFE)
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 33 — SECTOR-LEVEL FEATURES")
print("=" * 70)

# ----------------------------------------------------------------------
# LOAD CELL 32 CHECKPOINT
# ----------------------------------------------------------------------

checkpoint = "/content/NEPSE_ML_PROJECT/features/event_feature_daily_cell32_checkpoint.parquet"

print("\nLoading Cell 32 checkpoint...")
df = pd.read_parquet(checkpoint)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

# ----------------------------------------------------------------------
# LOAD SECURITY MASTER
# ----------------------------------------------------------------------

master_path = "/content/NEPSE_ML_PROJECT/processed/security_master_classified.csv"

print("\nLoading security master...")
master = pd.read_csv(master_path)

# Keep only what we need
sector_map = master[["symbol", "sector", "type"]].copy()

sector_map["sector"] = (
    sector_map["sector"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
)

sector_map.loc[
    sector_map["sector"].isin(["", "nan", "None", "NaN"]),
    "sector"
] = "UNKNOWN"

# ----------------------------------------------------------------------
# MERGE SECTOR INFORMATION
# ----------------------------------------------------------------------

print("\nMerging sector information...")

df = df.merge(
    sector_map,
    on="symbol",
    how="left",
    validate="many_to_one"
)

df["sector"] = df["sector"].fillna("UNKNOWN")

print(f"Sector coverage: {df['sector'].notna().sum():,} / {len(df):,}")
print(f"Unique sectors: {df['sector'].nunique()}")

# ----------------------------------------------------------------------
# PREPARE DAILY STOCK DATA
# ----------------------------------------------------------------------

print("\nPreparing sector daily data...")

# Use 1-day return already calculated in Cell 31.
# Exclude index securities from sector calculations.
stock_mask = df["type"].ne("INDEX")

sector_input = df.loc[
    stock_mask,
    ["date", "symbol", "sector", "return_1d"]
].copy()

sector_input["return_1d"] = pd.to_numeric(
    sector_input["return_1d"],
    errors="coerce"
)

# ----------------------------------------------------------------------
# SECTOR DAILY RETURN
# ----------------------------------------------------------------------

print("Calculating sector daily returns...")

sector_daily = (
    sector_input
    .groupby(["date", "sector"], sort=False)["return_1d"]
    .agg(
        sector_return_1d="mean",
        sector_stock_count="count",
        sector_up_count=lambda x: (x > 0).sum(),
        sector_down_count=lambda x: (x < 0).sum()
    )
    .reset_index()
)

sector_daily["sector_breadth_up_ratio"] = (
    sector_daily["sector_up_count"] /
    sector_daily["sector_stock_count"].replace(0, np.nan)
)

# ----------------------------------------------------------------------
# SORT FOR ROLLING FEATURES
# ----------------------------------------------------------------------

sector_daily = sector_daily.sort_values(
    ["sector", "date"]
).reset_index(drop=True)

# ----------------------------------------------------------------------
# SECTOR MOMENTUM / VOLATILITY
# ----------------------------------------------------------------------

print("Calculating sector momentum and volatility...")

g = sector_daily.groupby("sector", sort=False)

for window in [5, 20, 60]:
    sector_daily[f"sector_return_{window}d"] = (
        g["sector_return_1d"]
        .transform(lambda x: x.rolling(window, min_periods=window).sum())
    )

for window in [5, 20, 60]:
    sector_daily[f"sector_volatility_{window}d"] = (
        g["sector_return_1d"]
        .transform(lambda x: x.rolling(window, min_periods=window).std())
    )

# ----------------------------------------------------------------------
# SECTOR MOVING AVERAGE / TREND
# ----------------------------------------------------------------------

print("Calculating sector trend features...")

for window in [20, 60]:
    sector_daily[f"sector_breadth_{window}d"] = (
        g["sector_breadth_up_ratio"]
        .transform(lambda x: x.rolling(window, min_periods=window).mean())
    )

sector_daily["sector_bull_trend"] = (
    sector_daily["sector_return_20d"] > 0
).astype("int8")

sector_daily["sector_bear_trend"] = (
    sector_daily["sector_return_20d"] < 0
).astype("int8")

sector_daily["sector_strong_trend"] = (
    sector_daily["sector_return_20d"].abs() >
    sector_daily["sector_volatility_20d"] * 2
).astype("int8")

# ----------------------------------------------------------------------
# KEEP ONLY FEATURES TO MERGE
# ----------------------------------------------------------------------

sector_features = sector_daily[
    [
        "date",
        "sector",

        "sector_return_1d",
        "sector_return_5d",
        "sector_return_20d",
        "sector_return_60d",

        "sector_volatility_5d",
        "sector_volatility_20d",
        "sector_volatility_60d",

        "sector_breadth_up_ratio",
        "sector_breadth_20d",
        "sector_breadth_60d",

        "sector_stock_count",

        "sector_bull_trend",
        "sector_bear_trend",
        "sector_strong_trend"
    ]
].copy()

# ----------------------------------------------------------------------
# FREE TEMPORARY DATA
# ----------------------------------------------------------------------

del sector_input, sector_daily, g, master, sector_map
gc.collect()

# ----------------------------------------------------------------------
# MERGE BACK TO STOCK DATA
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("MERGING SECTOR FEATURES")
print("-" * 70)

rows_before = len(df)

df = df.merge(
    sector_features,
    on=["date", "sector"],
    how="left",
    validate="many_to_one"
)

rows_after = len(df)

print(f"Rows before merge: {rows_before:,}")
print(f"Rows after merge : {rows_after:,}")

del sector_features
gc.collect()

# ----------------------------------------------------------------------
# STOCK VS SECTOR RELATIVE PERFORMANCE
# ----------------------------------------------------------------------

print("\nCalculating stock vs sector relative performance...")

for window in [1, 5, 20, 60]:
    stock_col = f"return_{window}d"
    sector_col = f"sector_return_{window}d"

    if stock_col in df.columns and sector_col in df.columns:
        df[f"sector_relative_return_{window}d"] = (
            df[stock_col] - df[sector_col]
        )

# ----------------------------------------------------------------------
# SECTOR STRENGTH RANK
# ----------------------------------------------------------------------

print("Calculating sector strength...")

df["sector_strength_20d"] = df["sector_return_20d"]

# ----------------------------------------------------------------------
# CLEAN INF VALUES
# ----------------------------------------------------------------------

df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Restore canonical ordering
df.sort_values(["symbol", "date"], inplace=True)
df.reset_index(drop=True, inplace=True)

# ----------------------------------------------------------------------
# VALIDATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 33 VALIDATION")
print("=" * 70)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Symbols: {df['symbol'].nunique():,}")
print(f"Sectors: {df['sector'].nunique():,}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

print("\nSector feature coverage:")

check_cols = [
    "sector_return_1d",
    "sector_return_5d",
    "sector_return_20d",
    "sector_return_60d",
    "sector_volatility_5d",
    "sector_volatility_20d",
    "sector_volatility_60d",
    "sector_breadth_up_ratio",
    "sector_breadth_20d",
    "sector_breadth_60d",
    "sector_relative_return_1d",
    "sector_relative_return_5d",
    "sector_relative_return_20d",
    "sector_relative_return_60d",
]

for col in check_cols:
    if col in df.columns:
        print(f"{col:35s}: {df[col].notna().sum():,}")

# ----------------------------------------------------------------------
# SAVE CHECKPOINT
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("SAVING CELL 33 CHECKPOINT")
print("-" * 70)

output_path = (
    "/content/NEPSE_ML_PROJECT/features/"
    "event_feature_daily_cell33_checkpoint.parquet"
)

df.to_parquet(
    output_path,
    index=False,
    compression="snappy"
)

size_mb = os.path.getsize(output_path) / (1024 * 1024)

print(f"Checkpoint saved:")
print(output_path)
print(f"Checkpoint size: {size_mb:.1f} MB")

print("\n" + "=" * 70)
print("CELL 33 COMPLETE + CHECKPOINTED")
print("=" * 70)

CELL 33 — SECTOR-LEVEL FEATURES

Loading Cell 32 checkpoint...
Rows: 633,137
Columns: 152

Loading security master...

Merging sector information...
Sector coverage: 633,137 / 633,137
Unique sectors: 21

Preparing sector daily data...
Calculating sector daily returns...
Calculating sector momentum and volatility...
Calculating sector trend features...

----------------------------------------------------------------------
MERGING SECTOR FEATURES
----------------------------------------------------------------------
Rows before merge: 633,137
Rows after merge : 633,137

Calculating stock vs sector relative performance...
Calculating sector strength...

CELL 33 VALIDATION
Rows: 633,137
Columns: 173
Symbols: 651
Sectors: 21
Date range: 1997-07-20 00:00:00 to 2026-10-06 00:00:00

Sector feature coverage:
sector_return_1d                   : 633,079
sector_return_5d                   : 632,865
sector_return_20d                  : 632,193
sector_return_60d                  : 630,579
sector_v

In [ ]:
# ==============================================================================
# CELL 34 — LIQUIDITY & TRADING ACTIVITY FEATURES
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 34 — LIQUIDITY & TRADING ACTIVITY FEATURES")
print("=" * 70)

# ----------------------------------------------------------------------
# LOAD CELL 33 CHECKPOINT
# ----------------------------------------------------------------------

checkpoint = (
    "/content/NEPSE_ML_PROJECT/features/"
    "event_feature_daily_cell33_checkpoint.parquet"
)

print("\nLoading Cell 33 checkpoint...")
df = pd.read_parquet(checkpoint)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

# ----------------------------------------------------------------------
# ENSURE NUMERIC PRICE/VOLUME COLUMNS
# ----------------------------------------------------------------------

print("\nPreparing price and volume data...")

for col in ["open", "high", "low", "close", "volume"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

# ----------------------------------------------------------------------
# DAILY TURNOVER PROXY
# ----------------------------------------------------------------------

print("Calculating turnover and liquidity features...")

# NEPSE historical data does not always provide traded-value turnover.
# Close × volume is therefore used as a consistent traded-value proxy.
df["traded_value_proxy"] = (
    df["close"] * df["volume"]
)

# ----------------------------------------------------------------------
# ROLLING VOLUME FEATURES
# ----------------------------------------------------------------------

g = df.groupby("symbol", sort=False)

for window in [5, 20, 60]:
    df[f"volume_mean_{window}d"] = (
        g["volume"]
        .transform(lambda x: x.rolling(window, min_periods=window).mean())
    )

    df[f"volume_std_{window}d"] = (
        g["volume"]
        .transform(lambda x: x.rolling(window, min_periods=window).std())
    )

# ----------------------------------------------------------------------
# VOLUME CHANGE
# ----------------------------------------------------------------------

df["volume_change_1d"] = (
    g["volume"].pct_change()
)

df["volume_change_5d"] = (
    g["volume"].pct_change(5)
)

df["volume_change_20d"] = (
    g["volume"].pct_change(20)
)

# ----------------------------------------------------------------------
# VOLUME RELATIVE TO HISTORY
# ----------------------------------------------------------------------

df["volume_ratio_5d"] = (
    df["volume"] /
    df["volume_mean_5d"].replace(0, np.nan)
)

df["volume_ratio_60d"] = (
    df["volume"] /
    df["volume_mean_60d"].replace(0, np.nan)
)

# ----------------------------------------------------------------------
# TRADED VALUE FEATURES
# ----------------------------------------------------------------------

for window in [5, 20, 60]:
    df[f"traded_value_mean_{window}d"] = (
        g["traded_value_proxy"]
        .transform(
            lambda x: x.rolling(window, min_periods=window).mean()
        )
    )

# ----------------------------------------------------------------------
# PRICE × VOLUME ACTIVITY
# ----------------------------------------------------------------------

df["traded_value_ratio_20d"] = (
    df["traded_value_proxy"] /
    df["traded_value_mean_20d"].replace(0, np.nan)
)

df["traded_value_ratio_60d"] = (
    df["traded_value_proxy"] /
    df["traded_value_mean_60d"].replace(0, np.nan)
)

# ----------------------------------------------------------------------
# ZERO / LOW ACTIVITY FEATURES
# ----------------------------------------------------------------------

df["zero_volume"] = (
    df["volume"].fillna(0) <= 0
).astype("int8")

df["low_volume_vs_20d"] = (
    df["volume_ratio_20"].fillna(0) < 0.5
).astype("int8")

df["volume_spike_20d"] = (
    df["volume_ratio_20"].fillna(0) >= 2.0
).astype("int8")

df["volume_spike_5d"] = (
    df["volume_ratio_5d"].fillna(0) >= 2.0
).astype("int8")

# ----------------------------------------------------------------------
# PRICE RANGE / LIQUIDITY PROXIES
# ----------------------------------------------------------------------

df["intraday_range_pct"] = (
    (df["high"] - df["low"]) /
    df["close"].replace(0, np.nan)
)

df["open_close_range_pct"] = (
    (df["close"] - df["open"]).abs() /
    df["open"].replace(0, np.nan)
)

for window in [5, 20, 60]:
    df[f"intraday_range_mean_{window}d"] = (
        g["intraday_range_pct"]
        .transform(
            lambda x: x.rolling(window, min_periods=window).mean()
        )
    )

# ----------------------------------------------------------------------
# ILLIQUIDITY PROXY
# ----------------------------------------------------------------------

# Amihud-style proxy:
# absolute return / traded value.
#
# Larger value = less liquid.
df["amihud_illiquidity"] = (
    df["return_1d"].abs() /
    df["traded_value_proxy"].replace(0, np.nan)
)

for window in [20, 60]:
    df[f"amihud_illiquidity_{window}d"] = (
        g["amihud_illiquidity"]
        .transform(
            lambda x: x.rolling(window, min_periods=window).mean()
        )
    )

# ----------------------------------------------------------------------
# TRADING ACTIVITY SCORE
# ----------------------------------------------------------------------

df["trading_activity_score"] = (
    np.log1p(df["volume"].clip(lower=0)) *
    df["volume_ratio_20"].clip(lower=0, upper=10)
)

# ----------------------------------------------------------------------
# CLEAN EXTREME VALUES
# ----------------------------------------------------------------------

df.replace([np.inf, -np.inf], np.nan, inplace=True)

# ----------------------------------------------------------------------
# VALIDATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 34 VALIDATION")
print("=" * 70)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Symbols: {df['symbol'].nunique():,}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

print("\nLiquidity feature coverage:")

check_cols = [
    "traded_value_proxy",
    "volume_mean_5d",
    "volume_mean_20d",
    "volume_mean_60d",
    "volume_change_1d",
    "volume_change_5d",
    "volume_change_20d",
    "volume_ratio_5d",
    "volume_ratio_60d",
    "traded_value_mean_20d",
    "traded_value_mean_60d",
    "traded_value_ratio_20d",
    "traded_value_ratio_60d",
    "zero_volume",
    "volume_spike_5d",
    "volume_spike_20d",
    "intraday_range_mean_20d",
    "intraday_range_mean_60d",
    "amihud_illiquidity_20d",
    "amihud_illiquidity_60d",
    "trading_activity_score",
]

for col in check_cols:
    if col in df.columns:
        print(f"{col:35s}: {df[col].notna().sum():,}")

# ----------------------------------------------------------------------
# SAVE CHECKPOINT
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("SAVING CELL 34 CHECKPOINT")
print("-" * 70)

output_path = (
    "/content/NEPSE_ML_PROJECT/features/"
    "event_feature_daily_cell34_checkpoint.parquet"
)

df.to_parquet(
    output_path,
    index=False,
    compression="snappy"
)

size_mb = os.path.getsize(output_path) / (1024 * 1024)

print(f"Checkpoint saved:")
print(output_path)
print(f"Checkpoint size: {size_mb:.1f} MB")

print("\n" + "=" * 70)
print("CELL 34 COMPLETE + CHECKPOINTED")
print("=" * 70)

CELL 34 — LIQUIDITY & TRADING ACTIVITY FEATURES

Loading Cell 33 checkpoint...
Rows: 633,137
Columns: 173

Preparing price and volume data...
Calculating turnover and liquidity features...

CELL 34 VALIDATION
Rows: 633,137
Columns: 203
Symbols: 651
Date range: 1997-07-20 00:00:00 to 2026-10-06 00:00:00

Liquidity feature coverage:
traded_value_proxy                 : 633,137
volume_mean_5d                     : 630,579
volume_mean_20d                    : 621,591
volume_mean_60d                    : 599,596
volume_change_1d                   : 615,753
volume_change_5d                   : 613,281
volume_change_20d                  : 604,490
volume_ratio_5d                    : 614,205
volume_ratio_60d                   : 584,520
traded_value_mean_20d              : 621,591
traded_value_mean_60d              : 599,596
traded_value_ratio_20d             : 605,724
traded_value_ratio_60d             : 584,520
zero_volume                        : 633,137
volume_spike_5d                    : 

In [ ]:
# ==============================================================================
# CELL 35 — FUTURE RETURN & CLASSIFICATION TARGETS
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 35 — FUTURE RETURN & CLASSIFICATION TARGETS")
print("=" * 70)

# ----------------------------------------------------------------------
# LOAD CELL 34 CHECKPOINT
# ----------------------------------------------------------------------

checkpoint = (
    "/content/NEPSE_ML_PROJECT/features/"
    "event_feature_daily_cell34_checkpoint.parquet"
)

print("\nLoading Cell 34 checkpoint...")
df = pd.read_parquet(checkpoint)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

# ----------------------------------------------------------------------
# SORT BY SYMBOL AND DATE
# ----------------------------------------------------------------------

print("\nSorting chronologically by security...")

df = df.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

# ----------------------------------------------------------------------
# FUTURE CLOSE / RETURN TARGETS
# ----------------------------------------------------------------------

print("\nCalculating future returns...")

g = df.groupby("symbol", sort=False)

for horizon in [5, 20, 60]:

    print(f"  {horizon}-day targets...")

    future_close = (
        g["close"]
        .shift(-horizon)
    )

    # Future percentage return from today's close.
    df[f"future_return_{horizon}d"] = (
        future_close / df["close"] - 1.0
    )

    # Binary direction.
    df[f"target_up_{horizon}d"] = (
        df[f"future_return_{horizon}d"] > 0
    ).astype("float32")

    # Strong positive / negative movement.
    df[f"target_strong_up_{horizon}d"] = (
        df[f"future_return_{horizon}d"] >= 0.05
    ).astype("float32")

    df[f"target_strong_down_{horizon}d"] = (
        df[f"future_return_{horizon}d"] <= -0.05
    ).astype("float32")

# ----------------------------------------------------------------------
# FUTURE HIGH / LOW — RISK / OPPORTUNITY TARGETS
# ----------------------------------------------------------------------

print("\nCalculating future maximum/minimum movement...")

# We calculate these using shifted rolling windows.
# This gives the maximum/minimum close during the NEXT N trading sessions.

for horizon in [5, 20]:

    print(f"  Future {horizon}-day range...")

    future_max = (
        g["close"]
        .shift(-1)
        .groupby(df["symbol"], sort=False)
        .rolling(
            horizon,
            min_periods=horizon
        )
        .max()
        .reset_index(level=0, drop=True)
    )

    future_min = (
        g["close"]
        .shift(-1)
        .groupby(df["symbol"], sort=False)
        .rolling(
            horizon,
            min_periods=horizon
        )
        .min()
        .reset_index(level=0, drop=True)
    )

    df[f"future_max_return_{horizon}d"] = (
        future_max / df["close"] - 1.0
    )

    df[f"future_min_return_{horizon}d"] = (
        future_min / df["close"] - 1.0
    )

# ----------------------------------------------------------------------
# RANKING TARGET
# ----------------------------------------------------------------------

# Cross-sectional ranking is useful later for stock selection.
# Rank each trading date by future 20-day return.

print("\nCalculating cross-sectional future-performance rank...")

df["future_return_20d_rank"] = (
    df.groupby("date")["future_return_20d"]
    .rank(
        pct=True,
        method="average"
    )
)

# ----------------------------------------------------------------------
# CLEAN INF
# ----------------------------------------------------------------------

df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

# ----------------------------------------------------------------------
# VALIDATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 35 VALIDATION")
print("=" * 70)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Symbols: {df['symbol'].nunique():,}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

print("\nTarget coverage:")

target_cols = [
    "future_return_5d",
    "future_return_20d",
    "future_return_60d",
    "target_up_5d",
    "target_up_20d",
    "target_up_60d",
    "target_strong_up_5d",
    "target_strong_up_20d",
    "target_strong_up_60d",
    "target_strong_down_5d",
    "target_strong_down_20d",
    "target_strong_down_60d",
    "future_max_return_5d",
    "future_min_return_5d",
    "future_max_return_20d",
    "future_min_return_20d",
    "future_return_20d_rank",
]

for col in target_cols:
    if col in df.columns:
        print(
            f"{col:35s}: "
            f"{df[col].notna().sum():,}"
        )

# ----------------------------------------------------------------------
# TARGET DISTRIBUTION
# ----------------------------------------------------------------------

print("\nTarget direction distribution:")

for horizon in [5, 20, 60]:

    col = f"target_up_{horizon}d"

    valid = df[col].dropna()

    if len(valid):
        print(
            f"{horizon:2d}d -> "
            f"UP: {(valid == 1).mean()*100:.2f}% | "
            f"DOWN: {(valid == 0).mean()*100:.2f}%"
        )

# ----------------------------------------------------------------------
# SAVE CHECKPOINT
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("SAVING CELL 35 CHECKPOINT")
print("-" * 70)

output_path = (
    "/content/NEPSE_ML_PROJECT/features/"
    "event_feature_daily_cell35_targets_checkpoint.parquet"
)

df.to_parquet(
    output_path,
    index=False,
    compression="snappy"
)

size_mb = os.path.getsize(output_path) / (1024 * 1024)

print("Checkpoint saved:")
print(output_path)
print(f"Checkpoint size: {size_mb:.1f} MB")

print("\n" + "=" * 70)
print("CELL 35 COMPLETE + CHECKPOINTED")
print("=" * 70)

CELL 35 — FUTURE RETURN & CLASSIFICATION TARGETS

Loading Cell 34 checkpoint...
Rows: 633,137
Columns: 203

Sorting chronologically by security...

Calculating future returns...
  5-day targets...
  20-day targets...
  60-day targets...

Calculating future maximum/minimum movement...
  Future 5-day range...
  Future 20-day range...

Calculating cross-sectional future-performance rank...

CELL 35 VALIDATION
Rows: 633,137
Columns: 220
Symbols: 651
Date range: 1997-07-20 00:00:00 to 2026-10-06 00:00:00

Target coverage:
future_return_5d                   : 629,956
future_return_20d                  : 621,010
future_return_60d                  : 599,066
target_up_5d                       : 633,137
target_up_20d                      : 633,137
target_up_60d                      : 633,137
target_strong_up_5d                : 633,137
target_strong_up_20d               : 633,137
target_strong_up_60d               : 633,137
target_strong_down_5d              : 633,137
target_strong_down_20d     

In [ ]:
# ==============================================================================
# CELL 35A — FIX TARGET LABEL NaNs
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 35A — TARGET VALIDATION & NaN CORRECTION")
print("=" * 70)

checkpoint = (
    "/content/NEPSE_ML_PROJECT/features/"
    "event_feature_daily_cell35_targets_checkpoint.parquet"
)

print("\nLoading Cell 35 checkpoint...")
df = pd.read_parquet(checkpoint)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

# ----------------------------------------------------------------------
# CORRECT BINARY TARGETS
# ----------------------------------------------------------------------

print("\nCorrecting target labels...")

for horizon in [5, 20, 60]:

    future_col = f"future_return_{horizon}d"

    up_col = f"target_up_{horizon}d"
    strong_up_col = f"target_strong_up_{horizon}d"
    strong_down_col = f"target_strong_down_{horizon}d"

    valid = df[future_col].notna()

    # Binary direction
    df[up_col] = np.where(
        valid,
        (df[future_col] > 0).astype("float32"),
        np.nan
    )

    # Strong movement labels
    df[strong_up_col] = np.where(
        valid,
        (df[future_col] >= 0.05).astype("float32"),
        np.nan
    )

    df[strong_down_col] = np.where(
        valid,
        (df[future_col] <= -0.05).astype("float32"),
        np.nan
    )

# ----------------------------------------------------------------------
# CORRECT FUTURE RANGE TARGETS
# ----------------------------------------------------------------------

# The existing range targets are already NaN where insufficient
# future observations exist. Explicitly enforce validity using the
# corresponding future return target.

for horizon in [5, 20]:

    valid = df[f"future_return_{horizon}d"].notna()

    df.loc[
        ~valid,
        f"future_max_return_{horizon}d"
    ] = np.nan

    df.loc[
        ~valid,
        f"future_min_return_{horizon}d"
    ] = np.nan

# ----------------------------------------------------------------------
# CORRECT CROSS-SECTIONAL RANK
# ----------------------------------------------------------------------

# Rank is only meaningful where future 20-day return exists.

df.loc[
    df["future_return_20d"].isna(),
    "future_return_20d_rank"
] = np.nan

# ----------------------------------------------------------------------
# VALIDATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TARGET VALIDATION")
print("=" * 70)

for horizon in [5, 20, 60]:

    future_col = f"future_return_{horizon}d"
    up_col = f"target_up_{horizon}d"

    future_valid = df[future_col].notna()
    target_valid = df[up_col].notna()

    print(f"\n{horizon}-day target:")
    print(f"  Future return valid : {future_valid.sum():,}")
    print(f"  Direction valid     : {target_valid.sum():,}")
    print(
        f"  UP                  : "
        f"{(df.loc[target_valid, up_col] == 1).mean()*100:.2f}%"
    )
    print(
        f"  DOWN                : "
        f"{(df.loc[target_valid, up_col] == 0).mean()*100:.2f}%"
    )

    if future_valid.sum() != target_valid.sum():
        raise ValueError(
            f"Target validity mismatch for {horizon}d"
        )

# ----------------------------------------------------------------------
# FINAL TARGET COVERAGE
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("FINAL TARGET COVERAGE")
print("-" * 70)

target_cols = [
    "future_return_5d",
    "target_up_5d",
    "future_return_20d",
    "target_up_20d",
    "future_return_60d",
    "target_up_60d",
    "future_max_return_5d",
    "future_min_return_5d",
    "future_max_return_20d",
    "future_min_return_20d",
    "future_return_20d_rank",
]

for col in target_cols:
    print(
        f"{col:35s}: "
        f"{df[col].notna().sum():,}"
    )

# ----------------------------------------------------------------------
# SAVE CORRECTED CHECKPOINT
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("SAVING CORRECTED CELL 35 CHECKPOINT")
print("-" * 70)

output_path = (
    "/content/NEPSE_ML_PROJECT/features/"
    "event_feature_daily_cell35_targets_checkpoint.parquet"
)

df.to_parquet(
    output_path,
    index=False,
    compression="snappy"
)

size_mb = os.path.getsize(output_path) / (1024 * 1024)

print(f"Checkpoint saved:")
print(output_path)
print(f"Checkpoint size: {size_mb:.1f} MB")

print("\n" + "=" * 70)
print("CELL 35A COMPLETE")
print("=" * 70)

CELL 35A — TARGET VALIDATION & NaN CORRECTION

Loading Cell 35 checkpoint...
Rows: 633,137
Columns: 220

Correcting target labels...

TARGET VALIDATION

5-day target:
  Future return valid : 629,956
  Direction valid     : 629,956
  UP                  : 46.16%
  DOWN                : 53.84%

20-day target:
  Future return valid : 621,010
  Direction valid     : 621,010
  UP                  : 48.66%
  DOWN                : 51.34%

60-day target:
  Future return valid : 599,066
  Direction valid     : 599,066
  UP                  : 52.44%
  DOWN                : 47.56%

----------------------------------------------------------------------
FINAL TARGET COVERAGE
----------------------------------------------------------------------
future_return_5d                   : 629,956
target_up_5d                       : 629,956
future_return_20d                  : 621,010
target_up_20d                      : 621,010
future_return_60d                  : 599,066
target_up_60d                    

In [ ]:
# ==============================================================================
# CELL 36 — ML LEAKAGE AUDIT + FEATURE INVENTORY
# ==============================================================================

import os
import re
import gc
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 36 — ML LEAKAGE AUDIT + FEATURE INVENTORY")
print("=" * 70)

# ----------------------------------------------------------------------
# LOAD CELL 35A CHECKPOINT
# ----------------------------------------------------------------------

checkpoint = (
    "/content/NEPSE_ML_PROJECT/features/"
    "event_feature_daily_cell35_targets_checkpoint.parquet"
)

print("\nLoading dataset...")
df = pd.read_parquet(checkpoint)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

# ----------------------------------------------------------------------
# IDENTIFY TARGET COLUMNS
# ----------------------------------------------------------------------

target_patterns = [
    r"^future_",
    r"^target_",
]

target_cols = []

for col in df.columns:
    if any(re.search(pattern, col) for pattern in target_patterns):
        target_cols.append(col)

# Ranking target is also explicitly a target.
if "future_return_20d_rank" not in target_cols:
    target_cols.append("future_return_20d_rank")

target_cols = sorted(set(target_cols))

print("\n" + "-" * 70)
print("TARGET COLUMNS")
print("-" * 70)

print(f"Target columns: {len(target_cols)}")

for col in target_cols:
    print(f"  {col}")

# ----------------------------------------------------------------------
# IDENTIFY NON-FEATURE / METADATA COLUMNS
# ----------------------------------------------------------------------

metadata_cols = [
    "symbol",
    "date",
    "timestamp",
    "type",
    "sector",
    "full_name",
    "description",
    "exchange",
    "logo_urls",
    "exchange_logo",
    "is_master",
]

metadata_cols = [
    c for c in metadata_cols
    if c in df.columns
]

# ----------------------------------------------------------------------
# POTENTIAL LEAKAGE COLUMNS
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("POTENTIAL LEAKAGE AUDIT")
print("-" * 70)

leakage_cols = []

# Any feature with obvious future-looking names
future_keywords = [
    "future",
    "target",
    "forward",
    "lead",
    "next",
]

for col in df.columns:

    if col in target_cols or col in metadata_cols:
        continue

    name = col.lower()

    if any(keyword in name for keyword in future_keywords):
        leakage_cols.append(col)

# ----------------------------------------------------------------------
# CORPORATE-ACTION TIMING REVIEW
# ----------------------------------------------------------------------

corporate_timing_cols = []

for col in df.columns:

    name = col.lower()

    if any(
        keyword in name
        for keyword in [
            "dividend",
            "rights",
            "book_closure",
            "ipo",
            "fpo",
            "merger",
            "listing",
            "trading_status",
            "price_adjustment",
        ]
    ):
        if col not in target_cols and col not in metadata_cols:
            corporate_timing_cols.append(col)

print("\nPotential future-looking feature names:")
if leakage_cols:
    for col in sorted(leakage_cols):
        print(f"  WARNING: {col}")
else:
    print("  None")

print("\nCorporate-action/event features requiring timing review:")
print(f"  {len(corporate_timing_cols)} columns")

for col in sorted(corporate_timing_cols):
    print(f"  {col}")

# ----------------------------------------------------------------------
# FEATURE INVENTORY
# ----------------------------------------------------------------------

feature_cols = [
    c for c in df.columns
    if c not in target_cols
    and c not in metadata_cols
]

print("\n" + "-" * 70)
print("FEATURE INVENTORY")
print("-" * 70)

print(f"Metadata columns : {len(metadata_cols)}")
print(f"Target columns   : {len(target_cols)}")
print(f"Feature columns  : {len(feature_cols)}")

# ----------------------------------------------------------------------
# DATA TYPES
# ----------------------------------------------------------------------

numeric_features = [
    c for c in feature_cols
    if pd.api.types.is_numeric_dtype(df[c])
]

non_numeric_features = [
    c for c in feature_cols
    if not pd.api.types.is_numeric_dtype(df[c])
]

print(f"\nNumeric features    : {len(numeric_features)}")
print(f"Non-numeric features: {len(non_numeric_features)}")

if non_numeric_features:
    print("\nNon-numeric feature columns:")
    for col in non_numeric_features:
        print(f"  {col}")

# ----------------------------------------------------------------------
# CONSTANT / NEAR-CONSTANT FEATURES
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("CONSTANT FEATURE AUDIT")
print("-" * 70)

constant_features = []
low_variance_features = []

for col in numeric_features:

    nunique = df[col].nunique(dropna=True)

    if nunique <= 1:
        constant_features.append(col)

    elif nunique <= 2:
        low_variance_features.append(col)

print(f"Constant features      : {len(constant_features)}")
print(f"2-value features       : {len(low_variance_features)}")

if constant_features:
    print("\nConstant columns:")
    for col in constant_features:
        print(f"  {col}")

# ----------------------------------------------------------------------
# MISSINGNESS SUMMARY
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("FEATURE MISSINGNESS")
print("-" * 70)

missing_pct = (
    df[numeric_features]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

print("\nHighest missingness features:")

for col, pct in missing_pct.head(20).items():
    print(f"{col:40s}: {pct*100:6.2f}%")

# ----------------------------------------------------------------------
# DUPLICATE SYMBOL/DATE AUDIT
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("ROW INTEGRITY")
print("-" * 70)

duplicates = df.duplicated(
    ["symbol", "date"]
).sum()

print(f"Duplicate symbol/date rows: {duplicates:,}")

if duplicates != 0:
    raise ValueError(
        "Duplicate symbol/date rows detected."
    )

# ----------------------------------------------------------------------
# TARGET / FEATURE TEMPORAL SANITY
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("TEMPORAL SANITY")
print("-" * 70)

print(
    f"Minimum date: {df['date'].min()}"
)

print(
    f"Maximum date: {df['date'].max()}"
)

# ----------------------------------------------------------------------
# SAVE FEATURE INVENTORY
# ----------------------------------------------------------------------

inventory = pd.DataFrame({
    "column": df.columns,
    "is_target": [c in target_cols for c in df.columns],
    "is_metadata": [c in metadata_cols for c in df.columns],
    "is_feature": [c in feature_cols for c in df.columns],
    "is_numeric": [
        pd.api.types.is_numeric_dtype(df[c])
        for c in df.columns
    ],
    "missing_pct": [
        df[c].isna().mean() * 100
        for c in df.columns
    ],
    "n_unique": [
        df[c].nunique(dropna=True)
        for c in df.columns
    ],
})

inventory_path = (
    "/content/NEPSE_ML_PROJECT/validation/"
    "ml_feature_inventory.csv"
)

inventory.to_csv(
    inventory_path,
    index=False
)

# ----------------------------------------------------------------------
# SAVE CHECKPOINT
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("SAVING CELL 36 CHECKPOINT")
print("-" * 70)

output_path = (
    "/content/NEPSE_ML_PROJECT/features/"
    "event_feature_daily_cell36_audited_checkpoint.parquet"
)

df.to_parquet(
    output_path,
    index=False,
    compression="snappy"
)

size_mb = os.path.getsize(output_path) / (1024 * 1024)

print(f"Dataset checkpoint:")
print(output_path)
print(f"Size: {size_mb:.1f} MB")

print(f"\nFeature inventory:")
print(inventory_path)

print("\n" + "=" * 70)
print("CELL 36 COMPLETE")
print("=" * 70)

CELL 36 — ML LEAKAGE AUDIT + FEATURE INVENTORY

Loading dataset...
Rows: 633,137
Columns: 220

----------------------------------------------------------------------
TARGET COLUMNS
----------------------------------------------------------------------
Target columns: 17
  future_max_return_20d
  future_max_return_5d
  future_min_return_20d
  future_min_return_5d
  future_return_20d
  future_return_20d_rank
  future_return_5d
  future_return_60d
  target_strong_down_20d
  target_strong_down_5d
  target_strong_down_60d
  target_strong_up_20d
  target_strong_up_5d
  target_strong_up_60d
  target_up_20d
  target_up_5d
  target_up_60d

----------------------------------------------------------------------
POTENTIAL LEAKAGE AUDIT
----------------------------------------------------------------------

Potential future-looking feature names:
  None

Corporate-action/event features requiring timing review:
  68 columns
  BOOK_CLOSURE
  DIVIDEND
  IPO_FPO
  LISTING
  MERGER
  PRICE_ADJUSTMENT
  

In [ ]:
# ==============================================================================
# CELL 37A — BUILD CLEAN ML DATASET (FIXED)
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 37A — BUILD CLEAN ML DATASET")
print("=" * 70)

# ----------------------------------------------------------------------
# LOAD CELL 36 CHECKPOINT
# ----------------------------------------------------------------------

checkpoint = (
    "/content/NEPSE_ML_PROJECT/features/"
    "event_feature_daily_cell36_audited_checkpoint.parquet"
)

print("\nLoading audited dataset...")
df = pd.read_parquet(checkpoint)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

# ----------------------------------------------------------------------
# TARGET COLUMNS
# ----------------------------------------------------------------------

target_cols = [
    "future_return_5d",
    "future_return_20d",
    "future_return_60d",

    "future_max_return_5d",
    "future_min_return_5d",
    "future_max_return_20d",
    "future_min_return_20d",

    "future_return_20d_rank",

    "target_up_5d",
    "target_up_20d",
    "target_up_60d",

    "target_strong_up_5d",
    "target_strong_up_20d",
    "target_strong_up_60d",

    "target_strong_down_5d",
    "target_strong_down_20d",
    "target_strong_down_60d",
]

target_cols = [c for c in target_cols if c in df.columns]

# ----------------------------------------------------------------------
# IDENTIFIERS / METADATA THAT ACTUALLY EXIST
# ----------------------------------------------------------------------

id_cols = [
    c for c in ["symbol", "date"]
    if c in df.columns
]

metadata_cols = [
    c for c in ["type", "sector"]
    if c in df.columns
]

# ----------------------------------------------------------------------
# REMOVE CONSTANT FEATURES
# ----------------------------------------------------------------------

constant_cols = [
    "cash_dividend",
    "bonus_dividend",
    "total_dividend",
    "dividend_cash_5d",
    "dividend_bonus_5d",
    "dividend_cash_20d",
    "dividend_bonus_20d",
    "dividend_cash_60d",
    "dividend_bonus_60d",
]

constant_cols = [
    c for c in constant_cols
    if c in df.columns
]

print("\nRemoving constant features:")
for c in constant_cols:
    print(f"  {c}")

# ----------------------------------------------------------------------
# EXCLUDE CORPORATE-ACTION FEATURES WITH UNCERTAIN INFORMATION TIMING
# ----------------------------------------------------------------------

timing_exclusion = [
    "book_closure_1d",
    "book_closure_5d",
    "book_closure_20d",
    "book_closure_60d",

    "days_since_book_closure",

    "days_since_dividend_structured",
    "days_since_rights_structured",

    "dividend_event_count",
    "dividend_events_5d",
    "dividend_events_20d",
    "dividend_events_60d",

    "rights_event_count",
    "rights_events_5d",
    "rights_events_20d",
    "rights_events_60d",

    "rights_ratio",

    "cash_dividend",
    "bonus_dividend",
    "total_dividend",

    "dividend_cash_5d",
    "dividend_cash_20d",
    "dividend_cash_60d",

    "dividend_bonus_5d",
    "dividend_bonus_20d",
    "dividend_bonus_60d",
]

timing_exclusion = [
    c for c in timing_exclusion
    if c in df.columns
]

print("\nExcluding corporate-action timing features:")
print(f"Count: {len(timing_exclusion)}")

# ----------------------------------------------------------------------
# BUILD FEATURE LIST
# ----------------------------------------------------------------------

excluded = set(
    target_cols +
    id_cols +
    metadata_cols +
    constant_cols +
    timing_exclusion
)

feature_cols = [
    c for c in df.columns
    if c not in excluded
]

# Numeric features only
feature_cols = [
    c for c in feature_cols
    if pd.api.types.is_numeric_dtype(df[c])
]

print("\n" + "-" * 70)
print("CLEAN FEATURE SET")
print("-" * 70)

print(f"Original columns : {len(df.columns)}")
print(f"Target columns   : {len(target_cols)}")
print(f"Feature columns  : {len(feature_cols)}")
print(f"Excluded columns : {len(excluded)}")

# ----------------------------------------------------------------------
# CLEAN NUMERIC FEATURES
# ----------------------------------------------------------------------

df[feature_cols] = df[feature_cols].replace(
    [np.inf, -np.inf],
    np.nan
)

# ----------------------------------------------------------------------
# BUILD CLEAN DATASET
# ----------------------------------------------------------------------

keep_cols = (
    id_cols +
    metadata_cols +
    feature_cols +
    target_cols
)

# Remove accidental duplicates while preserving order
keep_cols = list(dict.fromkeys(keep_cols))

clean_df = df[keep_cols].copy()

# ----------------------------------------------------------------------
# VALIDATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 37A VALIDATION")
print("=" * 70)

print(f"Rows: {len(clean_df):,}")
print(f"Columns: {len(clean_df.columns):,}")
print(f"Features: {len(feature_cols):,}")
print(f"Targets: {len(target_cols):,}")
print(f"Symbols: {clean_df['symbol'].nunique():,}")

print(
    f"Date range: "
    f"{clean_df['date'].min()} to {clean_df['date'].max()}"
)

duplicates = clean_df.duplicated(
    ["symbol", "date"]
).sum()

print(f"\nDuplicate symbol/date rows: {duplicates:,}")

if duplicates != 0:
    raise ValueError("Duplicate symbol/date rows detected.")

# Check that all features are numeric
non_numeric = [
    c for c in feature_cols
    if not pd.api.types.is_numeric_dtype(clean_df[c])
]

print(f"Non-numeric ML features: {len(non_numeric)}")

if non_numeric:
    print(non_numeric)

# ----------------------------------------------------------------------
# FEATURE MISSINGNESS SUMMARY
# ----------------------------------------------------------------------

missing = (
    clean_df[feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

print("\nHighest feature missingness:")

for col, pct in missing.head(15).items():
    print(f"{col:40s}: {pct * 100:6.2f}%")

# ----------------------------------------------------------------------
# TARGET VALIDITY
# ----------------------------------------------------------------------

print("\nTarget validity:")

for horizon in [5, 20, 60]:

    future_col = f"future_return_{horizon}d"
    target_col = f"target_up_{horizon}d"

    valid = (
        clean_df[future_col].notna() &
        clean_df[target_col].notna()
    )

    print(
        f"{horizon:2d}d: "
        f"{valid.sum():,} usable rows"
    )

# ----------------------------------------------------------------------
# SAVE FEATURE LIST
# ----------------------------------------------------------------------

feature_list_path = (
    "/content/NEPSE_ML_PROJECT/validation/"
    "ml_feature_columns.csv"
)

pd.DataFrame({
    "feature": feature_cols
}).to_csv(
    feature_list_path,
    index=False
)

# ----------------------------------------------------------------------
# SAVE CLEAN DATASET
# ----------------------------------------------------------------------

output_path = (
    "/content/NEPSE_ML_PROJECT/features/"
    "nepse_ml_clean_dataset.parquet"
)

print("\n" + "-" * 70)
print("SAVING CLEAN ML DATASET")
print("-" * 70)

clean_df.to_parquet(
    output_path,
    index=False,
    compression="snappy"
)

size_mb = os.path.getsize(output_path) / (1024 * 1024)

print(f"Dataset saved:")
print(output_path)
print(f"Size: {size_mb:.1f} MB")

print(f"\nFeature list saved:")
print(feature_list_path)

# ----------------------------------------------------------------------
# CHECKPOINT SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 37A COMPLETE")
print("=" * 70)

print(f"Final ML rows       : {len(clean_df):,}")
print(f"Final ML features   : {len(feature_cols):,}")
print(f"Final targets       : {len(target_cols):,}")
print(f"Dataset size        : {size_mb:.1f} MB")

# Keep clean_df available in memory for the next cell
df = clean_df

del clean_df
gc.collect()

CELL 37A — BUILD CLEAN ML DATASET

Loading audited dataset...
Rows: 633,137
Columns: 220

Removing constant features:
  cash_dividend
  bonus_dividend
  total_dividend
  dividend_cash_5d
  dividend_bonus_5d
  dividend_cash_20d
  dividend_bonus_20d
  dividend_cash_60d
  dividend_bonus_60d

Excluding corporate-action timing features:
Count: 25

----------------------------------------------------------------------
CLEAN FEATURE SET
----------------------------------------------------------------------
Original columns : 220
Target columns   : 17
Feature columns  : 174
Excluded columns : 46

CELL 37A VALIDATION
Rows: 633,137
Columns: 195
Features: 174
Targets: 17
Symbols: 651
Date range: 1997-07-20 00:00:00 to 2026-10-06 00:00:00

Duplicate symbol/date rows: 0
Non-numeric ML features: 0

Highest feature missingness:
days_since_trading_status               :  98.93%
days_since_merger                       :  98.66%
days_since_ipo_fpo                      :  97.64%
days_since_price_adjustme

8

In [ ]:
# ============================================================================
# CELL 38 — CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================================
import os
import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 38 — CHRONOLOGICAL ML SPLIT")
print("=" * 70)

DATA_PATH = "/content/NEPSE_ML_PROJECT/features/nepse_ml_clean_dataset.parquet"
SPLIT_PATH = "/content/NEPSE_ML_PROJECT/validation/ml_split_manifest.csv"

print("\nLoading clean ML dataset...")
df = pd.read_parquet(DATA_PATH)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["date", "symbol"]).reset_index(drop=True)

# ---------------------------------------------------------------------------
# SPLIT DESIGN
# ---------------------------------------------------------------------------
# Historical data before 2011 is retained in the master dataset but is not
# used for model fitting because feature history/market structure is weaker.
#
# Train      : 2011-01-01 through 2021-12-31
# Purge      : 2022-01-01 through 2022-03-31
# Validation : 2022-04-01 through 2024-12-31
# Purge      : 2025-01-01 through 2025-03-31
# Test       : 2025-04-01 onward
#
# The purge prevents future-label overlap around split boundaries.

TRAIN_START = pd.Timestamp("2011-01-01")
TRAIN_END   = pd.Timestamp("2021-12-31")

VAL_START   = pd.Timestamp("2022-04-01")
VAL_END     = pd.Timestamp("2024-12-31")

TEST_START  = pd.Timestamp("2025-04-01")
TEST_END    = df["date"].max()

# ---------------------------------------------------------------------------
# TARGET VALIDITY
# ---------------------------------------------------------------------------
# Primary modeling target = 20-day direction.
# Rows without a valid 20-day future return cannot be used for the primary
# classifier.

PRIMARY_TARGET = "target_up_20d"

usable = df[PRIMARY_TARGET].notna()

train_mask = (
    usable &
    (df["date"] >= TRAIN_START) &
    (df["date"] <= TRAIN_END)
)

val_mask = (
    usable &
    (df["date"] >= VAL_START) &
    (df["date"] <= VAL_END)
)

test_mask = (
    usable &
    (df["date"] >= TEST_START) &
    (df["date"] <= TEST_END)
)

train_df = df.loc[train_mask].copy()
val_df   = df.loc[val_mask].copy()
test_df  = df.loc[test_mask].copy()

# ---------------------------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("SPLIT VALIDATION")
print("-" * 70)

print(f"Full dataset rows : {len(df):,}")
print(f"Full date range   : {df['date'].min().date()} to {df['date'].max().date()}")

print("\nTraining:")
print(f"  Date range : {train_df['date'].min().date()} to {train_df['date'].max().date()}")
print(f"  Rows       : {len(train_df):,}")
print(f"  Symbols    : {train_df['symbol'].nunique():,}")

print("\nValidation:")
print(f"  Date range : {val_df['date'].min().date()} to {val_df['date'].max().date()}")
print(f"  Rows       : {len(val_df):,}")
print(f"  Symbols    : {val_df['symbol'].nunique():,}")

print("\nTest:")
print(f"  Date range : {test_df['date'].min().date()} to {test_df['date'].max().date()}")
print(f"  Rows       : {len(test_df):,}")
print(f"  Symbols    : {test_df['symbol'].nunique():,}")

# ---------------------------------------------------------------------------
# CHECK CHRONOLOGICAL SEPARATION
# ---------------------------------------------------------------------------
assert train_df["date"].max() < val_df["date"].min()
assert val_df["date"].max() < test_df["date"].min()

print("\nChronological separation: PASS")

# ---------------------------------------------------------------------------
# CHECK PURGE GAPS
# ---------------------------------------------------------------------------
train_val_gap = (val_df["date"].min() - train_df["date"].max()).days
val_test_gap   = (test_df["date"].min() - val_df["date"].max()).days

print(f"Train → Validation calendar gap: {train_val_gap} days")
print(f"Validation → Test calendar gap: {val_test_gap} days")

# ---------------------------------------------------------------------------
# TARGET DISTRIBUTION
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("PRIMARY TARGET DISTRIBUTION — target_up_20d")
print("-" * 70)

for name, part in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df),
]:
    counts = part[PRIMARY_TARGET].value_counts(dropna=True).sort_index()
    total = counts.sum()

    print(f"\n{name}")
    for k, v in counts.items():
        label = "DOWN" if int(k) == 0 else "UP"
        print(f"  {label}: {int(v):,} ({100*v/total:.2f}%)")

# ---------------------------------------------------------------------------
# SAVE SPLIT MANIFEST
# ---------------------------------------------------------------------------
manifest = pd.DataFrame([
    {
        "split": "train",
        "start_date": train_df["date"].min(),
        "end_date": train_df["date"].max(),
        "rows": len(train_df),
        "symbols": train_df["symbol"].nunique(),
        "target": PRIMARY_TARGET,
    },
    {
        "split": "validation",
        "start_date": val_df["date"].min(),
        "end_date": val_df["date"].max(),
        "rows": len(val_df),
        "symbols": val_df["symbol"].nunique(),
        "target": PRIMARY_TARGET,
    },
    {
        "split": "test",
        "start_date": test_df["date"].min(),
        "end_date": test_df["date"].max(),
        "rows": len(test_df),
        "symbols": test_df["symbol"].nunique(),
        "target": PRIMARY_TARGET,
    },
])

manifest.to_csv(SPLIT_PATH, index=False)

print("\n" + "-" * 70)
print("SPLIT MANIFEST SAVED")
print("-" * 70)
print(SPLIT_PATH)

print("\n" + "=" * 70)
print("CELL 38 COMPLETE")
print("=" * 70)

CELL 38 — CHRONOLOGICAL ML SPLIT

Loading clean ML dataset...

----------------------------------------------------------------------
SPLIT VALIDATION
----------------------------------------------------------------------
Full dataset rows : 633,137
Full date range   : 1997-07-20 to 2026-10-06

Training:
  Date range : 2011-01-02 to 2021-12-29
  Rows       : 294,410
  Symbols    : 335

Validation:
  Date range : 2022-04-03 to 2024-12-31
  Rows       : 178,541
  Symbols    : 445

Test:
  Date range : 2025-04-01 to 2026-09-02
  Rows       : 112,127
  Symbols    : 443

Chronological separation: PASS
Train → Validation calendar gap: 95 days
Validation → Test calendar gap: 91 days

----------------------------------------------------------------------
PRIMARY TARGET DISTRIBUTION — target_up_20d
----------------------------------------------------------------------

TRAIN
  DOWN: 140,680 (47.78%)
  UP: 153,730 (52.22%)

VALIDATION
  DOWN: 92,369 (51.74%)
  UP: 86,172 (48.26%)

TEST
  DOWN: 6

In [ ]:
# ============================================================================
# CELL 39 — MODEL-READY DATA PREPARATION
# ============================================================================
import os
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 39 — MODEL-READY DATA PREPARATION")
print("=" * 70)

DATA_PATH = "/content/NEPSE_ML_PROJECT/features/nepse_ml_clean_dataset.parquet"
OUT_DIR = "/content/NEPSE_ML_PROJECT/features/model_ready"
os.makedirs(OUT_DIR, exist_ok=True)

print("\nLoading clean ML dataset...")
df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["date", "symbol"]).reset_index(drop=True)

PRIMARY_TARGET = "target_up_20d"

# ---------------------------------------------------------------------------
# FEATURE LIST
# ---------------------------------------------------------------------------
target_cols = [
    "future_return_5d",
    "future_return_20d",
    "future_return_60d",
    "target_up_5d",
    "target_up_20d",
    "target_up_60d",
    "target_strong_up_5d",
    "target_strong_up_20d",
    "target_strong_up_60d",
    "target_strong_down_5d",
    "target_strong_down_20d",
    "target_strong_down_60d",
    "future_max_return_5d",
    "future_min_return_5d",
    "future_max_return_20d",
    "future_min_return_20d",
    "future_return_20d_rank",
]

id_cols = ["symbol", "date"]

# Use the feature inventory already produced by Cell 37A.
feature_file = "/content/NEPSE_ML_PROJECT/validation/ml_feature_columns.csv"

if os.path.exists(feature_file):
    feature_list_df = pd.read_csv(feature_file)
    if "feature" in feature_list_df.columns:
        feature_cols = feature_list_df["feature"].tolist()
    elif "column" in feature_list_df.columns:
        feature_cols = feature_list_df["column"].tolist()
    else:
        feature_cols = [
            c for c in df.columns
            if c not in target_cols + id_cols
        ]
else:
    feature_cols = [
        c for c in df.columns
        if c not in target_cols + id_cols
    ]

feature_cols = [c for c in feature_cols if c in df.columns]

# Keep numeric features only.
numeric_features = [
    c for c in feature_cols
    if pd.api.types.is_numeric_dtype(df[c])
]

print("\nFeature inventory:")
print(f"  Candidate features : {len(feature_cols)}")
print(f"  Numeric features   : {len(numeric_features)}")

assert len(numeric_features) > 0

# ---------------------------------------------------------------------------
# CHRONOLOGICAL SPLITS
# ---------------------------------------------------------------------------
TRAIN_START = pd.Timestamp("2011-01-01")
TRAIN_END   = pd.Timestamp("2021-12-31")

VAL_START   = pd.Timestamp("2022-04-01")
VAL_END     = pd.Timestamp("2024-12-31")

TEST_START  = pd.Timestamp("2025-04-01")
TEST_END    = df["date"].max()

train_mask = (
    df[PRIMARY_TARGET].notna() &
    (df["date"] >= TRAIN_START) &
    (df["date"] <= TRAIN_END)
)

val_mask = (
    df[PRIMARY_TARGET].notna() &
    (df["date"] >= VAL_START) &
    (df["date"] <= VAL_END)
)

test_mask = (
    df[PRIMARY_TARGET].notna() &
    (df["date"] >= TEST_START) &
    (df["date"] <= TEST_END)
)

train = df.loc[train_mask].copy()
val   = df.loc[val_mask].copy()
test  = df.loc[test_mask].copy()

print("\nSplit sizes:")
print(f"  Train      : {len(train):,}")
print(f"  Validation : {len(val):,}")
print(f"  Test       : {len(test):,}")

# ---------------------------------------------------------------------------
# TRAINING-ONLY MEDIAN IMPUTATION
# ---------------------------------------------------------------------------
print("\nCalculating training-only feature medians...")

X_train = train[numeric_features].copy()
X_val   = val[numeric_features].copy()
X_test  = test[numeric_features].copy()

# Replace infinities before calculating medians.
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val   = X_val.replace([np.inf, -np.inf], np.nan)
X_test  = X_test.replace([np.inf, -np.inf], np.nan)

train_medians = X_train.median()

# Features completely missing in training receive 0.
train_medians = train_medians.fillna(0)

X_train = X_train.fillna(train_medians)
X_val   = X_val.fillna(train_medians)
X_test  = X_test.fillna(train_medians)

# ---------------------------------------------------------------------------
# TARGETS
# ---------------------------------------------------------------------------
y_train = train[PRIMARY_TARGET].astype("int8")
y_val   = val[PRIMARY_TARGET].astype("int8")
y_test  = test[PRIMARY_TARGET].astype("int8")

# ---------------------------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("MODEL MATRIX VALIDATION")
print("-" * 70)

print(f"X_train shape : {X_train.shape}")
print(f"X_val shape   : {X_val.shape}")
print(f"X_test shape  : {X_test.shape}")

print("\nRemaining NaNs:")
print(f"  Train      : {int(X_train.isna().sum().sum()):,}")
print(f"  Validation : {int(X_val.isna().sum().sum()):,}")
print(f"  Test       : {int(X_test.isna().sum().sum()):,}")

print("\nRemaining infinities:")
print(f"  Train      : {int(np.isinf(X_train.to_numpy()).sum()):,}")
print(f"  Validation : {int(np.isinf(X_val.to_numpy()).sum()):,}")
print(f"  Test       : {int(np.isinf(X_test.to_numpy()).sum()):,}")

assert X_train.isna().sum().sum() == 0
assert X_val.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

assert np.isfinite(X_train.to_numpy()).all()
assert np.isfinite(X_val.to_numpy()).all()
assert np.isfinite(X_test.to_numpy()).all()

# ---------------------------------------------------------------------------
# SAVE MATRICES
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("SAVING MODEL-READY DATA")
print("-" * 70)

X_train.to_parquet(
    f"{OUT_DIR}/X_train.parquet",
    index=False
)

X_val.to_parquet(
    f"{OUT_DIR}/X_validation.parquet",
    index=False
)

X_test.to_parquet(
    f"{OUT_DIR}/X_test.parquet",
    index=False
)

pd.DataFrame({
    PRIMARY_TARGET: y_train.values
}).to_parquet(
    f"{OUT_DIR}/y_train.parquet",
    index=False
)

pd.DataFrame({
    PRIMARY_TARGET: y_val.values
}).to_parquet(
    f"{OUT_DIR}/y_validation.parquet",
    index=False
)

pd.DataFrame({
    PRIMARY_TARGET: y_test.values
}).to_parquet(
    f"{OUT_DIR}/y_test.parquet",
    index=False
)

# Preserve IDs/dates for later evaluation and backtesting.
train[["symbol", "date"]].to_parquet(
    f"{OUT_DIR}/train_metadata.parquet",
    index=False
)

val[["symbol", "date"]].to_parquet(
    f"{OUT_DIR}/validation_metadata.parquet",
    index=False
)

test[["symbol", "date"]].to_parquet(
    f"{OUT_DIR}/test_metadata.parquet",
    index=False
)

# Save the exact feature ordering.
pd.DataFrame({
    "feature": numeric_features,
    "train_median": [train_medians[c] for c in numeric_features]
}).to_csv(
    f"{OUT_DIR}/feature_medians.csv",
    index=False
)

# Manifest.
manifest = {
    "primary_target": PRIMARY_TARGET,
    "feature_count": len(numeric_features),
    "train_rows": len(X_train),
    "validation_rows": len(X_val),
    "test_rows": len(X_test),
    "train_start": str(train["date"].min().date()),
    "train_end": str(train["date"].max().date()),
    "validation_start": str(val["date"].min().date()),
    "validation_end": str(val["date"].max().date()),
    "test_start": str(test["date"].min().date()),
    "test_end": str(test["date"].max().date()),
    "imputation": "training-only median",
    "scaling": "none",
}

with open(f"{OUT_DIR}/model_ready_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved:")
print(f"  {OUT_DIR}/X_train.parquet")
print(f"  {OUT_DIR}/X_validation.parquet")
print(f"  {OUT_DIR}/X_test.parquet")
print(f"  {OUT_DIR}/y_train.parquet")
print(f"  {OUT_DIR}/y_validation.parquet")
print(f"  {OUT_DIR}/y_test.parquet")
print(f"  {OUT_DIR}/feature_medians.csv")
print(f"  {OUT_DIR}/model_ready_manifest.json")

print("\n" + "=" * 70)
print("CELL 39 COMPLETE")
print("=" * 70)

CELL 39 — MODEL-READY DATA PREPARATION

Loading clean ML dataset...

Feature inventory:
  Candidate features : 174
  Numeric features   : 174

Split sizes:
  Train      : 294,410
  Validation : 178,541
  Test       : 112,127

Calculating training-only feature medians...

----------------------------------------------------------------------
MODEL MATRIX VALIDATION
----------------------------------------------------------------------
X_train shape : (294410, 174)
X_val shape   : (178541, 174)
X_test shape  : (112127, 174)

Remaining NaNs:
  Train      : 0
  Validation : 0
  Test       : 0

Remaining infinities:


TypeError: ufunc 'isinf' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [ ]:
# ============================================================================
# CELL 39A — FINISH MODEL-READY DATA PREPARATION
# ============================================================================
import os
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 39A — FINISH MODEL-READY DATA PREPARATION")
print("=" * 70)

DATA_PATH = "/content/NEPSE_ML_PROJECT/features/nepse_ml_clean_dataset.parquet"
OUT_DIR = "/content/NEPSE_ML_PROJECT/features/model_ready"
os.makedirs(OUT_DIR, exist_ok=True)

PRIMARY_TARGET = "target_up_20d"

# ---------------------------------------------------------------------------
# LOAD CLEAN DATA
# ---------------------------------------------------------------------------
print("\nLoading clean ML dataset...")
df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["date", "symbol"]).reset_index(drop=True)

target_cols = [
    "future_return_5d",
    "future_return_20d",
    "future_return_60d",
    "target_up_5d",
    "target_up_20d",
    "target_up_60d",
    "target_strong_up_5d",
    "target_strong_up_20d",
    "target_strong_up_60d",
    "target_strong_down_5d",
    "target_strong_down_20d",
    "target_strong_down_60d",
    "future_max_return_5d",
    "future_min_return_5d",
    "future_max_return_20d",
    "future_min_return_20d",
    "future_return_20d_rank",
]

id_cols = ["symbol", "date"]

feature_file = "/content/NEPSE_ML_PROJECT/validation/ml_feature_columns.csv"

if os.path.exists(feature_file):
    feature_inventory = pd.read_csv(feature_file)

    if "feature" in feature_inventory.columns:
        feature_cols = feature_inventory["feature"].tolist()
    elif "column" in feature_inventory.columns:
        feature_cols = feature_inventory["column"].tolist()
    else:
        feature_cols = [
            c for c in df.columns
            if c not in target_cols + id_cols
        ]
else:
    feature_cols = [
        c for c in df.columns
        if c not in target_cols + id_cols
    ]

feature_cols = [c for c in feature_cols if c in df.columns]

numeric_features = [
    c for c in feature_cols
    if pd.api.types.is_numeric_dtype(df[c])
]

print(f"Rows              : {len(df):,}")
print(f"Numeric features  : {len(numeric_features)}")

# ---------------------------------------------------------------------------
# CHRONOLOGICAL SPLITS
# ---------------------------------------------------------------------------
TRAIN_START = pd.Timestamp("2011-01-01")
TRAIN_END   = pd.Timestamp("2021-12-31")

VAL_START   = pd.Timestamp("2022-04-01")
VAL_END     = pd.Timestamp("2024-12-31")

TEST_START  = pd.Timestamp("2025-04-01")
TEST_END    = df["date"].max()

usable = df[PRIMARY_TARGET].notna()

train_mask = (
    usable &
    (df["date"] >= TRAIN_START) &
    (df["date"] <= TRAIN_END)
)

val_mask = (
    usable &
    (df["date"] >= VAL_START) &
    (df["date"] <= VAL_END)
)

test_mask = (
    usable &
    (df["date"] >= TEST_START) &
    (df["date"] <= TEST_END)
)

train = df.loc[train_mask].copy()
val   = df.loc[val_mask].copy()
test  = df.loc[test_mask].copy()

# ---------------------------------------------------------------------------
# FEATURE MATRICES
# ---------------------------------------------------------------------------
X_train = train[numeric_features].copy()
X_val   = val[numeric_features].copy()
X_test  = test[numeric_features].copy()

# Force every ML feature to numeric.
# Any unexpected conversion failure becomes NaN and is handled by median.
for c in numeric_features:
    X_train[c] = pd.to_numeric(X_train[c], errors="coerce")
    X_val[c]   = pd.to_numeric(X_val[c], errors="coerce")
    X_test[c]  = pd.to_numeric(X_test[c], errors="coerce")

# Replace infinities safely using DataFrame operations.
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val   = X_val.replace([np.inf, -np.inf], np.nan)
X_test  = X_test.replace([np.inf, -np.inf], np.nan)

# Training-only medians.
train_medians = X_train.median().fillna(0)

X_train = X_train.fillna(train_medians)
X_val   = X_val.fillna(train_medians)
X_test  = X_test.fillna(train_medians)

# ---------------------------------------------------------------------------
# TARGETS
# ---------------------------------------------------------------------------
y_train = train[PRIMARY_TARGET].astype("int8")
y_val   = val[PRIMARY_TARGET].astype("int8")
y_test  = test[PRIMARY_TARGET].astype("int8")

# ---------------------------------------------------------------------------
# SAFE VALIDATION
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("MODEL MATRIX VALIDATION")
print("-" * 70)

print(f"X_train shape : {X_train.shape}")
print(f"X_val shape   : {X_val.shape}")
print(f"X_test shape  : {X_test.shape}")

print("\nRemaining NaNs:")
print(f"  Train      : {int(X_train.isna().sum().sum()):,}")
print(f"  Validation : {int(X_val.isna().sum().sum()):,}")
print(f"  Test       : {int(X_test.isna().sum().sum()):,}")

# Safe infinity check through pandas.
print("\nRemaining infinities:")
print(f"  Train      : {int(np.isinf(X_train.select_dtypes(include=[np.number])).to_numpy().sum()):,}")
print(f"  Validation : {int(np.isinf(X_val.select_dtypes(include=[np.number])).to_numpy().sum()):,}")
print(f"  Test       : {int(np.isinf(X_test.select_dtypes(include=[np.number])).to_numpy().sum()):,}")

assert X_train.isna().sum().sum() == 0
assert X_val.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

assert np.isinf(X_train.select_dtypes(include=[np.number])).to_numpy().sum() == 0
assert np.isinf(X_val.select_dtypes(include=[np.number])).to_numpy().sum() == 0
assert np.isinf(X_test.select_dtypes(include=[np.number])).to_numpy().sum() == 0

# ---------------------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("SAVING MODEL-READY DATA")
print("-" * 70)

X_train.to_parquet(f"{OUT_DIR}/X_train.parquet", index=False)
X_val.to_parquet(f"{OUT_DIR}/X_validation.parquet", index=False)
X_test.to_parquet(f"{OUT_DIR}/X_test.parquet", index=False)

pd.DataFrame({PRIMARY_TARGET: y_train.values}).to_parquet(
    f"{OUT_DIR}/y_train.parquet", index=False
)

pd.DataFrame({PRIMARY_TARGET: y_val.values}).to_parquet(
    f"{OUT_DIR}/y_validation.parquet", index=False
)

pd.DataFrame({PRIMARY_TARGET: y_test.values}).to_parquet(
    f"{OUT_DIR}/y_test.parquet", index=False
)

train[["symbol", "date"]].to_parquet(
    f"{OUT_DIR}/train_metadata.parquet", index=False
)

val[["symbol", "date"]].to_parquet(
    f"{OUT_DIR}/validation_metadata.parquet", index=False
)

test[["symbol", "date"]].to_parquet(
    f"{OUT_DIR}/test_metadata.parquet", index=False
)

pd.DataFrame({
    "feature": numeric_features,
    "train_median": [train_medians[c] for c in numeric_features]
}).to_csv(
    f"{OUT_DIR}/feature_medians.csv",
    index=False
)

manifest = {
    "primary_target": PRIMARY_TARGET,
    "feature_count": len(numeric_features),
    "train_rows": len(X_train),
    "validation_rows": len(X_val),
    "test_rows": len(X_test),
    "train_start": str(train["date"].min().date()),
    "train_end": str(train["date"].max().date()),
    "validation_start": str(val["date"].min().date()),
    "validation_end": str(val["date"].max().date()),
    "test_start": str(test["date"].min().date()),
    "test_end": str(test["date"].max().date()),
    "imputation": "training-only median",
    "scaling": "none",
}

with open(f"{OUT_DIR}/model_ready_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

# Keep these available for the next cell.
print("\nModel matrices saved successfully.")

print("\n" + "=" * 70)
print("CELL 39A COMPLETE")
print("=" * 70)

CELL 39A — FINISH MODEL-READY DATA PREPARATION

Loading clean ML dataset...
Rows              : 633,137
Numeric features  : 174

----------------------------------------------------------------------
MODEL MATRIX VALIDATION
----------------------------------------------------------------------
X_train shape : (294410, 174)
X_val shape   : (178541, 174)
X_test shape  : (112127, 174)

Remaining NaNs:
  Train      : 0
  Validation : 0
  Test       : 0

Remaining infinities:
  Train      : 0
  Validation : 0
  Test       : 0

----------------------------------------------------------------------
SAVING MODEL-READY DATA
----------------------------------------------------------------------

Model matrices saved successfully.

CELL 39A COMPLETE


In [ ]:
# ============================================================================
# BACKUP CURRENT PROJECT TO GOOGLE DRIVE
# ============================================================================
import os
import shutil
from datetime import datetime

PROJECT = "/content/NEPSE_ML_PROJECT"
DRIVE_ROOT = "/content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP"

print("=" * 70)
print("BACKING UP CURRENT NEPSE ML PROJECT")
print("=" * 70)

# Mount Drive if necessary
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

os.makedirs(DRIVE_ROOT, exist_ok=True)

# Copy/update the project recursively.
# Existing files are replaced with the current versions.
for root, dirs, files in os.walk(PROJECT):
    rel = os.path.relpath(root, PROJECT)
    dest_dir = DRIVE_ROOT if rel == "." else os.path.join(DRIVE_ROOT, rel)
    os.makedirs(dest_dir, exist_ok=True)

    for filename in files:
        src = os.path.join(root, filename)
        dst = os.path.join(dest_dir, filename)
        shutil.copy2(src, dst)

print("\nBackup complete.")

# Show important newest files
important = [
    "features/nepse_ml_clean_dataset.parquet",
    "features/model_ready/X_train.parquet",
    "features/model_ready/X_validation.parquet",
    "features/model_ready/X_test.parquet",
    "features/model_ready/y_train.parquet",
    "features/model_ready/y_validation.parquet",
    "features/model_ready/y_test.parquet",
    "validation/ml_split_manifest.csv",
]

print("\nImportant files in Drive:")
for rel in important:
    path = os.path.join(DRIVE_ROOT, rel)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024**2)
        print(f"  OK  {rel}  ({size_mb:.1f} MB)")
    else:
        print(f"  MISSING  {rel}")

print("\n" + "=" * 70)
print("BACKUP COMPLETE")
print("=" * 70)

BACKING UP CURRENT NEPSE ML PROJECT
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Backup complete.

Important files in Drive:
  OK  features/nepse_ml_clean_dataset.parquet  (318.0 MB)
  OK  features/model_ready/X_train.parquet  (131.4 MB)
  OK  features/model_ready/X_validation.parquet  (86.5 MB)
  OK  features/model_ready/X_test.parquet  (56.1 MB)
  OK  features/model_ready/y_train.parquet  (0.0 MB)
  OK  features/model_ready/y_validation.parquet  (0.0 MB)
  OK  features/model_ready/y_test.parquet  (0.0 MB)
  OK  validation/ml_split_manifest.csv  (0.0 MB)

BACKUP COMPLETE


In [ ]:
# ============================================================================
# RECOVER NEPSE ML PROJECT AFTER COLAB RUNTIME RESET
# ============================================================================

import os
import shutil

print("=" * 70)
print("RESTORING NEPSE ML PROJECT FROM GOOGLE DRIVE")
print("=" * 70)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_PROJECT = "/content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP"
LOCAL_PROJECT = "/content/NEPSE_ML_PROJECT"

if not os.path.exists(DRIVE_PROJECT):
    raise FileNotFoundError(
        f"Drive backup not found:\n{DRIVE_PROJECT}"
    )

print("\nDrive backup found.")

# Remove any incomplete local copy.
if os.path.exists(LOCAL_PROJECT):
    shutil.rmtree(LOCAL_PROJECT)

print("Copying project from Drive...")
shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT)

print("\nChecking required model-ready files...")

required = [
    "features/nepse_ml_clean_dataset.parquet",
    "features/model_ready/X_train.parquet",
    "features/model_ready/X_validation.parquet",
    "features/model_ready/X_test.parquet",
    "features/model_ready/y_train.parquet",
    "features/model_ready/y_validation.parquet",
    "features/model_ready/y_test.parquet",
    "features/model_ready/feature_medians.csv",
    "features/model_ready/model_ready_manifest.json",
    "validation/ml_split_manifest.csv",
]

all_ok = True

for rel in required:
    path = os.path.join(LOCAL_PROJECT, rel)

    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024**2)
        print(f"  OK       {rel}  ({size_mb:.1f} MB)")
    else:
        print(f"  MISSING  {rel}")
        all_ok = False

print("\n" + "=" * 70)

if all_ok:
    print("PROJECT RESTORE COMPLETE")
    print("All required model-ready files are available.")
else:
    print("WARNING: SOME REQUIRED FILES ARE MISSING")

print("=" * 70)

RESTORING NEPSE ML PROJECT FROM GOOGLE DRIVE
Mounted at /content/drive

Drive backup found.
Copying project from Drive...

Checking required model-ready files...
  OK       features/nepse_ml_clean_dataset.parquet  (318.0 MB)
  OK       features/model_ready/X_train.parquet  (131.4 MB)
  OK       features/model_ready/X_validation.parquet  (86.5 MB)
  OK       features/model_ready/X_test.parquet  (56.1 MB)
  OK       features/model_ready/y_train.parquet  (0.0 MB)
  OK       features/model_ready/y_validation.parquet  (0.0 MB)
  OK       features/model_ready/y_test.parquet  (0.0 MB)
  OK       features/model_ready/feature_medians.csv  (0.0 MB)
  OK       features/model_ready/model_ready_manifest.json  (0.0 MB)
  OK       validation/ml_split_manifest.csv  (0.0 MB)

PROJECT RESTORE COMPLETE
All required model-ready files are available.


In [ ]:
# ============================================================================
# CELL 40 — BASELINE XGBOOST MODEL
# ============================================================================
import os
import time
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 40 — BASELINE XGBOOST MODEL")
print("=" * 70)

MODEL_DIR = "/content/NEPSE_ML_PROJECT/models"
REPORT_DIR = "/content/NEPSE_ML_PROJECT/reports"
DATA_DIR = "/content/NEPSE_ML_PROJECT/features/model_ready"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

PRIMARY_TARGET = "target_up_20d"

# ---------------------------------------------------------------------------
# LOAD XGBOOST
# ---------------------------------------------------------------------------
print("\nChecking XGBoost...")

try:
    import xgboost as xgb
    print(f"XGBoost version: {xgb.__version__}")
except Exception as e:
    print("XGBoost is not available.")
    print("Error:", e)
    raise

# ---------------------------------------------------------------------------
# LOAD MODEL MATRICES
# ---------------------------------------------------------------------------
print("\nLoading model-ready matrices...")

X_train = pd.read_parquet(f"{DATA_DIR}/X_train.parquet")
X_val   = pd.read_parquet(f"{DATA_DIR}/X_validation.parquet")

y_train = pd.read_parquet(
    f"{DATA_DIR}/y_train.parquet"
)[PRIMARY_TARGET]

y_val = pd.read_parquet(
    f"{DATA_DIR}/y_validation.parquet"
)[PRIMARY_TARGET]

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_val   : {y_val.shape}")

# ---------------------------------------------------------------------------
# GPU DETECTION
# ---------------------------------------------------------------------------
print("\nChecking GPU support...")

gpu_available = False

try:
    import subprocess

    result = subprocess.run(
        ["nvidia-smi"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode == 0:
        gpu_available = True
        print("NVIDIA GPU detected.")
        print(result.stdout[:1000])
    else:
        print("NVIDIA GPU not detected.")

except Exception:
    print("nvidia-smi unavailable.")

# ---------------------------------------------------------------------------
# MODEL CONFIGURATION
# ---------------------------------------------------------------------------
# Hist-style tree boosting.
# Moderate depth keeps the first baseline fast and reduces overfitting.

params = {
    "n_estimators": 700,
    "learning_rate": 0.04,
    "max_depth": 7,
    "min_child_weight": 10,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.10,
    "reg_lambda": 2.0,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
}

# Modern XGBoost uses device="cuda".
if gpu_available:
    params["device"] = "cuda"
    print("\nTraining device: GPU (CUDA)")
else:
    print("\nTraining device: CPU")

print("\nModel parameters:")
for k, v in params.items():
    print(f"  {k}: {v}")

# ---------------------------------------------------------------------------
# TRAIN
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("TRAINING BASELINE MODEL")
print("-" * 70)

model = xgb.XGBClassifier(**params)

start_time = time.time()

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

elapsed = time.time() - start_time

print(f"\nTraining completed in {elapsed/60:.2f} minutes")

# ---------------------------------------------------------------------------
# VALIDATION PREDICTIONS
# ---------------------------------------------------------------------------
print("\nGenerating validation predictions...")

val_prob = model.predict_proba(X_val)[:, 1]
val_pred = (val_prob >= 0.50).astype("int8")

# ---------------------------------------------------------------------------
# METRICS
# ---------------------------------------------------------------------------
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix
)

accuracy = accuracy_score(y_val, val_pred)
balanced_acc = balanced_accuracy_score(y_val, val_pred)
precision = precision_score(y_val, val_pred, zero_division=0)
recall = recall_score(y_val, val_pred, zero_division=0)
f1 = f1_score(y_val, val_pred, zero_division=0)
auc = roc_auc_score(y_val, val_prob)
ll = log_loss(y_val, val_prob)

cm = confusion_matrix(y_val, val_pred)

print("\n" + "-" * 70)
print("VALIDATION RESULTS")
print("-" * 70)

print(f"Accuracy           : {accuracy:.4f}")
print(f"Balanced Accuracy  : {balanced_acc:.4f}")
print(f"Precision          : {precision:.4f}")
print(f"Recall             : {recall:.4f}")
print(f"F1 Score           : {f1:.4f}")
print(f"ROC-AUC            : {auc:.4f}")
print(f"Log Loss           : {ll:.4f}")

print("\nConfusion matrix:")
print(cm)

# ---------------------------------------------------------------------------
# SAVE MODEL
# ---------------------------------------------------------------------------
model_path = f"{MODEL_DIR}/baseline_xgboost_20d.json"
model.save_model(model_path)

# Save validation predictions for later analysis.
val_predictions = pd.DataFrame({
    "prediction_probability": val_prob,
    "prediction": val_pred,
    "actual": y_val.to_numpy(),
})

val_predictions.to_parquet(
    f"{REPORT_DIR}/baseline_xgboost_validation_predictions.parquet",
    index=False
)

metrics = {
    "model": "XGBoost baseline",
    "target": PRIMARY_TARGET,
    "features": X_train.shape[1],
    "train_rows": len(X_train),
    "validation_rows": len(X_val),
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "roc_auc": float(auc),
    "log_loss": float(ll),
    "training_minutes": float(elapsed / 60),
    "gpu_used": bool(gpu_available),
}

with open(
    f"{REPORT_DIR}/baseline_xgboost_20d_metrics.json",
    "w"
) as f:
    json.dump(metrics, f, indent=2)

# ---------------------------------------------------------------------------
# FEATURE IMPORTANCE
# ---------------------------------------------------------------------------
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

importance.to_csv(
    f"{REPORT_DIR}/baseline_xgboost_feature_importance.csv",
    index=False
)

print("\nSaved:")
print(f"  Model      : {model_path}")
print(f"  Predictions: {REPORT_DIR}/baseline_xgboost_validation_predictions.parquet")
print(f"  Metrics    : {REPORT_DIR}/baseline_xgboost_20d_metrics.json")
print(f"  Importance : {REPORT_DIR}/baseline_xgboost_feature_importance.csv")

print("\nTop 20 features:")
print(importance.head(20).to_string(index=False))

print("\n" + "=" * 70)
print("CELL 40 COMPLETE")
print("=" * 70)

CELL 40 — BASELINE XGBOOST MODEL

Checking XGBoost...
XGBoost version: 3.4.1

Loading model-ready matrices...
X_train : (294410, 174)
X_val   : (178541, 174)
y_train : (294410,)
y_val   : (178541,)

Checking GPU support...
NVIDIA GPU detected.
Tue Oct  6 17:41:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |            

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [17:41:17] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



----------------------------------------------------------------------
VALIDATION RESULTS
----------------------------------------------------------------------
Accuracy           : 0.4866
Balanced Accuracy  : 0.4979
Precision          : 0.4814
Recall             : 0.8230
F1 Score           : 0.6074
ROC-AUC            : 0.4885
Log Loss           : 1.2142

Confusion matrix:
[[15958 76411]
 [15255 70917]]

Saved:
  Model      : /content/NEPSE_ML_PROJECT/models/baseline_xgboost_20d.json
  Predictions: /content/NEPSE_ML_PROJECT/reports/baseline_xgboost_validation_predictions.parquet
  Metrics    : /content/NEPSE_ML_PROJECT/reports/baseline_xgboost_20d_metrics.json
  Importance : /content/NEPSE_ML_PROJECT/reports/baseline_xgboost_feature_importance.csv

Top 20 features:
                feature  importance
              timestamp    0.051183
          market_sma_20    0.045342
         market_sma_100    0.036663
          market_sma_50    0.031374
market_close_to_sma_200    0.030769
    sec

In [ ]:
# ============================================================================
# CELL 41 — CLEAN BASELINE: REMOVE RAW TIMESTAMP
# ============================================================================
import os
import time
import json
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix
)

print("=" * 70)
print("CELL 41 — CLEAN BASELINE WITHOUT RAW TIMESTAMP")
print("=" * 70)

MODEL_DIR = "/content/NEPSE_ML_PROJECT/models"
REPORT_DIR = "/content/NEPSE_ML_PROJECT/reports"
DATA_DIR = "/content/NEPSE_ML_PROJECT/features/model_ready"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

TARGET = "target_up_20d"

# ---------------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------------
print("\nLoading model-ready matrices...")

X_train = pd.read_parquet(f"{DATA_DIR}/X_train.parquet")
X_val   = pd.read_parquet(f"{DATA_DIR}/X_validation.parquet")

y_train = pd.read_parquet(
    f"{DATA_DIR}/y_train.parquet"
)[TARGET]

y_val = pd.read_parquet(
    f"{DATA_DIR}/y_validation.parquet"
)[TARGET]

print(f"Original features: {X_train.shape[1]}")

# ---------------------------------------------------------------------------
# REMOVE RAW TIMESTAMP
# ---------------------------------------------------------------------------
remove_features = [
    "timestamp",
]

removed = [c for c in remove_features if c in X_train.columns]

X_train = X_train.drop(columns=removed)
X_val   = X_val.drop(columns=removed)

print("\nRemoved features:")
for c in removed:
    print(f"  {c}")

print(f"\nFinal features: {X_train.shape[1]}")

# ---------------------------------------------------------------------------
# GPU CHECK
# ---------------------------------------------------------------------------
gpu_available = False

try:
    import subprocess

    result = subprocess.run(
        ["nvidia-smi"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    gpu_available = result.returncode == 0
except Exception:
    gpu_available = False

print(f"\nGPU available: {gpu_available}")

# ---------------------------------------------------------------------------
# MODEL
# ---------------------------------------------------------------------------
params = {
    "n_estimators": 700,
    "learning_rate": 0.04,
    "max_depth": 7,
    "min_child_weight": 10,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.10,
    "reg_lambda": 2.0,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
}

if gpu_available:
    params["device"] = "cuda"

print("\nTraining baseline without timestamp...")

model = xgb.XGBClassifier(**params)

start = time.time()

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

elapsed = time.time() - start

print(f"Training time: {elapsed / 60:.2f} minutes")

# ---------------------------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------------------------
print("\nGenerating validation predictions...")

val_prob = model.predict_proba(X_val)[:, 1]
val_pred = (val_prob >= 0.50).astype("int8")

accuracy = accuracy_score(y_val, val_pred)
balanced_acc = balanced_accuracy_score(y_val, val_pred)
precision = precision_score(y_val, val_pred, zero_division=0)
recall = recall_score(y_val, val_pred, zero_division=0)
f1 = f1_score(y_val, val_pred, zero_division=0)
auc = roc_auc_score(y_val, val_prob)
ll = log_loss(y_val, val_prob)
cm = confusion_matrix(y_val, val_pred)

print("\n" + "-" * 70)
print("VALIDATION RESULTS")
print("-" * 70)

print(f"Accuracy           : {accuracy:.4f}")
print(f"Balanced Accuracy  : {balanced_acc:.4f}")
print(f"Precision          : {precision:.4f}")
print(f"Recall             : {recall:.4f}")
print(f"F1 Score           : {f1:.4f}")
print(f"ROC-AUC            : {auc:.4f}")
print(f"Log Loss           : {ll:.4f}")

print("\nConfusion matrix:")
print(cm)

# ---------------------------------------------------------------------------
# FEATURE IMPORTANCE
# ---------------------------------------------------------------------------
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\nTop 20 features:")
print(importance.head(20).to_string(index=False))

# ---------------------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------------------
model_path = f"{MODEL_DIR}/baseline_xgboost_20d_no_timestamp.json"

model.save_model(model_path)

pd.DataFrame({
    "prediction_probability": val_prob,
    "prediction": val_pred,
    "actual": y_val.to_numpy()
}).to_parquet(
    f"{REPORT_DIR}/baseline_xgboost_no_timestamp_validation_predictions.parquet",
    index=False
)

importance.to_csv(
    f"{REPORT_DIR}/baseline_xgboost_no_timestamp_feature_importance.csv",
    index=False
)

metrics = {
    "model": "XGBoost baseline without raw timestamp",
    "target": TARGET,
    "features": int(X_train.shape[1]),
    "removed_features": removed,
    "train_rows": int(len(X_train)),
    "validation_rows": int(len(X_val)),
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "roc_auc": float(auc),
    "log_loss": float(ll),
    "training_minutes": float(elapsed / 60),
    "gpu_used": bool(gpu_available)
}

with open(
    f"{REPORT_DIR}/baseline_xgboost_no_timestamp_metrics.json",
    "w"
) as f:
    json.dump(metrics, f, indent=2)

print("\nSaved:")
print(f"  Model: {model_path}")
print(f"  Metrics: {REPORT_DIR}/baseline_xgboost_no_timestamp_metrics.json")
print(f"  Importance: {REPORT_DIR}/baseline_xgboost_no_timestamp_feature_importance.csv")

print("\n" + "=" * 70)
print("CELL 41 COMPLETE")
print("=" * 70)

CELL 41 — CLEAN BASELINE WITHOUT RAW TIMESTAMP

Loading model-ready matrices...
Original features: 174

Removed features:
  timestamp

Final features: 173

GPU available: True

Training baseline without timestamp...
Training time: 0.41 minutes

Generating validation predictions...

----------------------------------------------------------------------
VALIDATION RESULTS
----------------------------------------------------------------------
Accuracy           : 0.4627
Balanced Accuracy  : 0.4584
Precision          : 0.4280
Recall             : 0.3366
F1 Score           : 0.3768
ROC-AUC            : 0.4408
Log Loss           : 0.9023

Confusion matrix:
[[53598 38771]
 [57164 29008]]

Top 20 features:
                feature  importance
market_close_to_sma_200    0.037412
          market_sma_50    0.036943
          market_sma_20    0.036831
         market_sma_100    0.034931
market_close_to_sma_100    0.032519
    sector_strength_20d    0.028955
  market_volatility_60d    0.028254
    

In [ ]:
# ============================================================================
# CELL 42 — TARGET / FEATURE SANITY CHECK
# ============================================================================
import os
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 42 — TARGET / FEATURE SANITY CHECK")
print("=" * 70)

DATA_PATH = "/content/NEPSE_ML_PROJECT/features/nepse_ml_clean_dataset.parquet"

print("\nLoading clean dataset...")
df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])

TARGET = "future_return_20d"

# ---------------------------------------------------------------------------
# BASIC TARGET CHECK
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("TARGET CHECK")
print("-" * 70)

valid = df[TARGET].notna()

print(f"Usable target rows: {valid.sum():,}")
print(f"Mean future 20d return: {df.loc[valid, TARGET].mean():.6f}")
print(f"Median future 20d return: {df.loc[valid, TARGET].median():.6f}")
print(f"Minimum: {df.loc[valid, TARGET].min():.6f}")
print(f"Maximum: {df.loc[valid, TARGET].max():.6f}")

# Check relationship between future return and target_up_20d.
check = df.loc[valid, [TARGET, "target_up_20d"]].copy()

print("\nTarget direction:")
print(
    check.groupby("target_up_20d")[TARGET]
    .agg(["count", "mean", "median", "min", "max"])
)

# ---------------------------------------------------------------------------
# FEATURE SANITY
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("KNOWN FEATURE SANITY CHECK")
print("-" * 70)

candidate_features = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "relative_return_20d",
    "sector_relative_return_20d",
    "market_return_20d",
]

available = [c for c in candidate_features if c in df.columns]

print(f"Testing {len(available)} features...")

rows = []

for feature in available:
    temp = df[[feature, TARGET]].replace([np.inf, -np.inf], np.nan).dropna()

    if len(temp) < 1000:
        continue

    corr = temp[feature].corr(temp[TARGET])

    # Compare future returns when feature is above/below zero.
    positive = temp.loc[temp[feature] > 0, TARGET]
    negative = temp.loc[temp[feature] <= 0, TARGET]

    rows.append({
        "feature": feature,
        "rows": len(temp),
        "correlation": corr,
        "future_return_when_positive": positive.mean(),
        "future_return_when_nonpositive": negative.mean(),
        "difference": positive.mean() - negative.mean()
    })

sanity = pd.DataFrame(rows).sort_values(
    "correlation",
    key=lambda x: x.abs(),
    ascending=False
)

print("\nFeature → future 20-day return:")
print(sanity.to_string(index=False))

# ---------------------------------------------------------------------------
# TARGET ALIGNMENT CHECK
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("TARGET ALIGNMENT CHECK")
print("-" * 70)

# Pick several symbols with substantial history.
symbol_counts = (
    df.groupby("symbol")
    .size()
    .sort_values(ascending=False)
)

test_symbols = symbol_counts.head(5).index.tolist()

for symbol in test_symbols:
    s = df[df["symbol"] == symbol].sort_values("date").copy()

    if len(s) < 100:
        continue

    # Manually calculate future 20-trading-row return.
    manual = (
        s["close"].shift(-20) / s["close"] - 1
    )

    stored = s[TARGET]

    comparison = pd.DataFrame({
        "manual": manual,
        "stored": stored
    }).dropna()

    if len(comparison) == 0:
        continue

    diff = (comparison["manual"] - comparison["stored"]).abs()

    print(
        f"\n{symbol}: "
        f"rows={len(s):,}, "
        f"max_abs_difference={diff.max():.12f}, "
        f"mean_abs_difference={diff.mean():.12f}"
    )

# ---------------------------------------------------------------------------
# FEATURE/TARGET SAME-DAY LEAKAGE CHECK
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("FEATURE DATE / TARGET DATE LOGIC")
print("-" * 70)

print("Features represent information available on date T.")
print("future_return_20d should represent movement after date T.")
print("Manual alignment above should therefore match:")
print("    close[T+20] / close[T] - 1")

print("\n" + "=" * 70)
print("CELL 42 COMPLETE")
print("=" * 70)

CELL 42 — TARGET / FEATURE SANITY CHECK

Loading clean dataset...

----------------------------------------------------------------------
TARGET CHECK
----------------------------------------------------------------------
Usable target rows: 621,010
Mean future 20d return: 0.021968
Median future 20d return: -0.001142
Minimum: -12.250000
Maximum: 34.000000

Target direction:
                count      mean    median        min   max
target_up_20d                                             
0.0            318838 -0.064221 -0.048473 -12.250000   0.0
1.0            302172  0.112911  0.063718   0.000003  34.0

----------------------------------------------------------------------
KNOWN FEATURE SANITY CHECK
----------------------------------------------------------------------
Testing 13 features...

Feature → future 20-day return:
                   feature   rows  correlation  future_return_when_positive  future_return_when_nonpositive  difference
          close_to_sma_200 523243     0.0

In [ ]:
# ============================================================================
# CELL 43A — MODEL DIAGNOSTIC: OVERFITTING / SIGNAL DIRECTION
# ============================================================================
import os
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import roc_auc_score, accuracy_score

print("=" * 70)
print("CELL 43A — MODEL DIAGNOSTIC")
print("=" * 70)

DATA_DIR = "/content/NEPSE_ML_PROJECT/features/model_ready"

TARGET = "target_up_20d"

# ---------------------------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------------------------
print("\nLoading model matrices...")

X_train_full = pd.read_parquet(f"{DATA_DIR}/X_train.parquet")
X_val_full   = pd.read_parquet(f"{DATA_DIR}/X_validation.parquet")

y_train = pd.read_parquet(
    f"{DATA_DIR}/y_train.parquet"
)[TARGET]

y_val = pd.read_parquet(
    f"{DATA_DIR}/y_validation.parquet"
)[TARGET]

print(f"Train matrix: {X_train_full.shape}")
print(f"Validation matrix: {X_val_full.shape}")

# ---------------------------------------------------------------------------
# LOAD MODEL
# ---------------------------------------------------------------------------
MODEL_PATH = (
    "/content/NEPSE_ML_PROJECT/models/"
    "baseline_xgboost_20d_no_timestamp.json"
)

print("\nLoading Cell 41 model...")

model = xgb.XGBClassifier()
model.load_model(MODEL_PATH)

# Get the exact feature list used when the model was trained.
model_features = model.get_booster().feature_names

print(f"Model expects features: {len(model_features)}")

# ---------------------------------------------------------------------------
# ALIGN MATRICES EXACTLY TO MODEL
# ---------------------------------------------------------------------------
missing_train = [
    c for c in model_features
    if c not in X_train_full.columns
]

missing_val = [
    c for c in model_features
    if c not in X_val_full.columns
]

extra_train = [
    c for c in X_train_full.columns
    if c not in model_features
]

extra_val = [
    c for c in X_val_full.columns
    if c not in model_features
]

print(f"Missing train features: {len(missing_train)}")
print(f"Missing validation features: {len(missing_val)}")
print(f"Extra train features: {len(extra_train)}")
print(f"Extra validation features: {len(extra_val)}")

if missing_train or missing_val:
    raise RuntimeError(
        "Model-required features are missing from model-ready matrices."
    )

# Exact order required by XGBoost.
X_train = X_train_full[model_features]
X_val   = X_val_full[model_features]

print(f"\nAligned train: {X_train.shape}")
print(f"Aligned validation: {X_val.shape}")

# ---------------------------------------------------------------------------
# TRAIN VS VALIDATION
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("TRAIN VS VALIDATION PERFORMANCE")
print("-" * 70)

train_prob = model.predict_proba(
    X_train,
    validate_features=True
)[:, 1]

val_prob = model.predict_proba(
    X_val,
    validate_features=True
)[:, 1]

train_auc = roc_auc_score(y_train, train_prob)
val_auc = roc_auc_score(y_val, val_prob)

train_acc = accuracy_score(
    y_train,
    (train_prob >= 0.5).astype("int8")
)

val_acc = accuracy_score(
    y_val,
    (val_prob >= 0.5).astype("int8")
)

print(f"Train ROC-AUC      : {train_auc:.4f}")
print(f"Validation ROC-AUC : {val_auc:.4f}")

print(f"\nTrain accuracy      : {train_acc:.4f}")
print(f"Validation accuracy : {val_acc:.4f}")

# ---------------------------------------------------------------------------
# PREDICTION DISTRIBUTION
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("PREDICTION PROBABILITY")
print("-" * 70)

print(f"Train mean: {train_prob.mean():.6f}")
print(f"Val mean  : {val_prob.mean():.6f}")

print(f"Train min : {train_prob.min():.6f}")
print(f"Train max : {train_prob.max():.6f}")

print(f"Val min   : {val_prob.min():.6f}")
print(f"Val max   : {val_prob.max():.6f}")

# ---------------------------------------------------------------------------
# DIRECTION CHECK
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("PREDICTION DIRECTION CHECK")
print("-" * 70)

normal_auc = roc_auc_score(y_val, val_prob)
inverted_auc = roc_auc_score(y_val, 1.0 - val_prob)

print(f"Normal prediction AUC   : {normal_auc:.4f}")
print(f"Inverted prediction AUC : {inverted_auc:.4f}")

if inverted_auc > normal_auc:
    print("\nWARNING: Inverted predictions perform better.")
    print("The model may be learning the wrong direction.")
else:
    print("\nPrediction is NOT simply inverted.")

# ---------------------------------------------------------------------------
# SIMPLE FEATURE-ONLY SIGNAL TEST
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("SIMPLE FEATURE SIGNAL TEST")
print("-" * 70)

simple_features = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volume_ratio_20",
    "volatility_20d",
    "relative_return_20d",
    "sector_relative_return_20d",
    "market_return_20d",
]

rows = []

for feature in simple_features:

    if feature not in X_train.columns:
        continue

    train_values = X_train[feature].to_numpy()
    val_values = X_val[feature].to_numpy()

    train_auc_feature = roc_auc_score(
        y_train,
        train_values
    )

    val_auc_feature = roc_auc_score(
        y_val,
        val_values
    )

    best_direction = max(
        val_auc_feature,
        1.0 - val_auc_feature
    )

    rows.append({
        "feature": feature,
        "train_auc": train_auc_feature,
        "validation_auc": val_auc_feature,
        "validation_best_direction_auc": best_direction
    })

simple_results = pd.DataFrame(rows).sort_values(
    "validation_best_direction_auc",
    ascending=False
)

print(
    simple_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ---------------------------------------------------------------------------
# VALIDATION DECILES
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("VALIDATION PREDICTION DECILES")
print("-" * 70)

bucket = pd.DataFrame({
    "prob": val_prob,
    "actual": y_val.to_numpy()
})

bucket["decile"] = pd.qcut(
    bucket["prob"],
    10,
    labels=False,
    duplicates="drop"
)

bucket_summary = bucket.groupby("decile").agg(
    observations=("actual", "size"),
    actual_up_rate=("actual", "mean"),
    mean_probability=("prob", "mean")
)

print(
    bucket_summary.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)

# ---------------------------------------------------------------------------
# FINAL
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("CELL 43A COMPLETE")
print("=" * 70)

print(f"\nTrain AUC      : {train_auc:.4f}")
print(f"Validation AUC : {val_auc:.4f}")
print(f"Inverted AUC   : {inverted_auc:.4f}")

print("=" * 70)

CELL 43A — MODEL DIAGNOSTIC

Loading model matrices...
Train matrix: (294410, 174)
Validation matrix: (178541, 174)

Loading Cell 41 model...
Model expects features: 173
Missing train features: 0
Missing validation features: 0
Extra train features: 1
Extra validation features: 1

Aligned train: (294410, 173)
Aligned validation: (178541, 173)

----------------------------------------------------------------------
TRAIN VS VALIDATION PERFORMANCE
----------------------------------------------------------------------
Train ROC-AUC      : 0.9520
Validation ROC-AUC : 0.4408

Train accuracy      : 0.8770
Validation accuracy : 0.4627

----------------------------------------------------------------------
PREDICTION PROBABILITY
----------------------------------------------------------------------
Train mean: 0.522146
Val mean  : 0.450350
Train min : 0.001489
Train max : 0.999582
Val min   : 0.006965
Val max   : 0.995471

----------------------------------------------------------------------
PR

In [ ]:
# ============================================================================
# CELL 44 — ROBUST FEATURE BASELINE
# ============================================================================
import os
import json
import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    log_loss
)

print("=" * 70)
print("CELL 44 — ROBUST FEATURE BASELINE")
print("=" * 70)

DATA_DIR = "/content/NEPSE_ML_PROJECT/features/model_ready"

TARGET = "target_up_20d"

# ---------------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------------
print("\nLoading matrices...")

X_train_full = pd.read_parquet(
    f"{DATA_DIR}/X_train.parquet"
)

X_val_full = pd.read_parquet(
    f"{DATA_DIR}/X_validation.parquet"
)

y_train = pd.read_parquet(
    f"{DATA_DIR}/y_train.parquet"
)[TARGET].astype("int8")

y_val = pd.read_parquet(
    f"{DATA_DIR}/y_validation.parquet"
)[TARGET].astype("int8")

# ---------------------------------------------------------------------------
# ROBUST FEATURE SET
# ---------------------------------------------------------------------------
# Deliberately exclude:
# - timestamp
# - raw OHLC
# - source/correction fields
# - corporate event-count explosion
# - highly sparse days-since features
# - security-specific metadata
#
# Keep features that represent broad, economically interpretable signals.
# ---------------------------------------------------------------------------

ROBUST_FEATURES = [
    # Momentum / trend
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",

    # Mean-reversion / momentum oscillator
    "rsi_14",

    # Volatility
    "volatility_20d",

    # Volume / liquidity
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",

    # Relative strength
    "relative_return_1d",
    "relative_return_5d",
    "relative_return_20d",
    "relative_return_60d",

    # Market regime
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_return_60d",
    "market_volatility_20d",
    "market_volatility_60d",
    "market_bull_trend",
    "market_bear_trend",
    "market_high_volatility",

    # Sector regime
    "sector_return_1d",
    "sector_return_5d",
    "sector_return_20d",
    "sector_return_60d",
    "sector_volatility_20d",
    "sector_volatility_60d",
    "sector_breadth_up_ratio",
    "sector_breadth_20d",
    "sector_relative_return_1d",
    "sector_relative_return_5d",
    "sector_relative_return_20d",
    "sector_relative_return_60d",
]

ROBUST_FEATURES = [
    c for c in ROBUST_FEATURES
    if c in X_train_full.columns
    and c in X_val_full.columns
]

print(f"\nFull features : {X_train_full.shape[1]}")
print(f"Robust features: {len(ROBUST_FEATURES)}")

print("\nFeatures used:")
for c in ROBUST_FEATURES:
    print(" ", c)

X_train = X_train_full[ROBUST_FEATURES]
X_val = X_val_full[ROBUST_FEATURES]

# ---------------------------------------------------------------------------
# MODEL
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("TRAINING SHALLOW XGBOOST")
print("-" * 70)

model = xgb.XGBClassifier(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=5.0,

    objective="binary:logistic",
    eval_metric="logloss",

    tree_method="hist",
    device="cuda",

    random_state=42,
    n_jobs=-1
)

start = time.time()

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

elapsed = time.time() - start

print(f"Training time: {elapsed / 60:.2f} minutes")

# ---------------------------------------------------------------------------
# PREDICTIONS
# ---------------------------------------------------------------------------
train_prob = model.predict_proba(X_train)[:, 1]
val_prob = model.predict_proba(X_val)[:, 1]

train_pred = (train_prob >= 0.5).astype("int8")
val_pred = (val_prob >= 0.5).astype("int8")

# ---------------------------------------------------------------------------
# METRICS
# ---------------------------------------------------------------------------
train_auc = roc_auc_score(y_train, train_prob)
val_auc = roc_auc_score(y_val, val_prob)

train_acc = accuracy_score(y_train, train_pred)
val_acc = accuracy_score(y_val, val_pred)

train_bal = balanced_accuracy_score(y_train, train_pred)
val_bal = balanced_accuracy_score(y_val, val_pred)

val_logloss = log_loss(y_val, val_prob)

print("\n" + "-" * 70)
print("RESULTS")
print("-" * 70)

print(f"Train ROC-AUC       : {train_auc:.4f}")
print(f"Validation ROC-AUC  : {val_auc:.4f}")

print(f"\nTrain accuracy      : {train_acc:.4f}")
print(f"Validation accuracy : {val_acc:.4f}")

print(f"\nTrain balanced acc  : {train_bal:.4f}")
print(f"Validation balanced : {val_bal:.4f}")

print(f"\nValidation log loss : {val_logloss:.4f}")

# ---------------------------------------------------------------------------
# DIRECTION CHECK
# ---------------------------------------------------------------------------
inverse_auc = roc_auc_score(
    y_val,
    1.0 - val_prob
)

print("\n" + "-" * 70)
print("DIRECTION CHECK")
print("-" * 70)

print(f"Normal AUC   : {val_auc:.4f}")
print(f"Inverted AUC : {inverse_auc:.4f}")

# ---------------------------------------------------------------------------
# DECILES
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("VALIDATION DECILES")
print("-" * 70)

dec = pd.DataFrame({
    "prob": val_prob,
    "actual": y_val.to_numpy()
})

dec["decile"] = pd.qcut(
    dec["prob"],
    10,
    labels=False,
    duplicates="drop"
)

summary = dec.groupby("decile").agg(
    observations=("actual", "size"),
    actual_up_rate=("actual", "mean"),
    mean_probability=("prob", "mean")
)

print(
    summary.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)

# ---------------------------------------------------------------------------
# SAVE EXPERIMENT
# ---------------------------------------------------------------------------
MODEL_DIR = "/content/NEPSE_ML_PROJECT/models"
REPORT_DIR = "/content/NEPSE_ML_PROJECT/reports"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

model_path = (
    f"{MODEL_DIR}/robust_xgboost_20d_baseline.json"
)

metrics_path = (
    f"{REPORT_DIR}/robust_xgboost_20d_baseline_metrics.json"
)

importance_path = (
    f"{REPORT_DIR}/robust_xgboost_20d_baseline_feature_importance.csv"
)

model.save_model(model_path)

metrics = {
    "train_auc": float(train_auc),
    "validation_auc": float(val_auc),
    "inverted_validation_auc": float(inverse_auc),
    "train_accuracy": float(train_acc),
    "validation_accuracy": float(val_acc),
    "train_balanced_accuracy": float(train_bal),
    "validation_balanced_accuracy": float(val_bal),
    "validation_logloss": float(val_logloss),
    "feature_count": len(ROBUST_FEATURES),
    "features": ROBUST_FEATURES,
}

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

importance = pd.DataFrame({
    "feature": model.get_booster().feature_names,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

importance.to_csv(
    importance_path,
    index=False
)

print("\nSaved:")
print(model_path)
print(metrics_path)
print(importance_path)

print("\n" + "=" * 70)
print("CELL 44 COMPLETE")
print("=" * 70)

CELL 44 — ROBUST FEATURE BASELINE

Loading matrices...

Full features : 174
Robust features: 38

Features used:
  return_1d
  return_5d
  return_20d
  return_60d
  close_to_sma_20
  close_to_sma_50
  close_to_sma_200
  rsi_14
  volatility_20d
  volume_ratio_20
  volume_change_1d
  volume_spike_20d
  trading_activity_score
  relative_return_1d
  relative_return_5d
  relative_return_20d
  relative_return_60d
  market_return_1d
  market_return_5d
  market_return_20d
  market_return_60d
  market_volatility_20d
  market_volatility_60d
  market_bull_trend
  market_bear_trend
  market_high_volatility
  sector_return_1d
  sector_return_5d
  sector_return_20d
  sector_return_60d
  sector_volatility_20d
  sector_volatility_60d
  sector_breadth_up_ratio
  sector_breadth_20d
  sector_relative_return_1d
  sector_relative_return_5d
  sector_relative_return_20d
  sector_relative_return_60d

----------------------------------------------------------------------
TRAINING SHALLOW XGBOOST
---------------

In [ ]:
# ============================================================================
# CELL 45 — FEATURE FAMILY ABLATION
# ============================================================================
import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL 45 — FEATURE FAMILY ABLATION")
print("=" * 70)

DATA_DIR = "/content/NEPSE_ML_PROJECT/features/model_ready"

TARGET = "target_up_20d"

# ---------------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------------
print("\nLoading matrices...")

X_train_full = pd.read_parquet(
    f"{DATA_DIR}/X_train.parquet"
)

X_val_full = pd.read_parquet(
    f"{DATA_DIR}/X_validation.parquet"
)

y_train = pd.read_parquet(
    f"{DATA_DIR}/y_train.parquet"
)[TARGET].astype("int8")

y_val = pd.read_parquet(
    f"{DATA_DIR}/y_validation.parquet"
)[TARGET].astype("int8")

# ---------------------------------------------------------------------------
# FEATURE FAMILIES
# ---------------------------------------------------------------------------

TECHNICAL = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

RELATIVE = [
    "relative_return_1d",
    "relative_return_5d",
    "relative_return_20d",
    "relative_return_60d",
]

MARKET = [
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_return_60d",
    "market_volatility_20d",
    "market_volatility_60d",
    "market_bull_trend",
    "market_bear_trend",
    "market_high_volatility",
]

SECTOR = [
    "sector_return_1d",
    "sector_return_5d",
    "sector_return_20d",
    "sector_return_60d",
    "sector_volatility_20d",
    "sector_volatility_60d",
    "sector_breadth_up_ratio",
    "sector_breadth_20d",
]

SECTOR_RELATIVE = [
    "sector_relative_return_1d",
    "sector_relative_return_5d",
    "sector_relative_return_20d",
    "sector_relative_return_60d",
]

# Only use features that actually exist.
def existing(features):
    return [
        f for f in features
        if f in X_train_full.columns
        and f in X_val_full.columns
    ]

TECHNICAL = existing(TECHNICAL)
RELATIVE = existing(RELATIVE)
MARKET = existing(MARKET)
SECTOR = existing(SECTOR)
SECTOR_RELATIVE = existing(SECTOR_RELATIVE)

FEATURE_SETS = {
    "Technical": TECHNICAL,

    "Market_Sector": MARKET + SECTOR,

    "Technical_Relative": TECHNICAL + RELATIVE + SECTOR_RELATIVE,

    "All_Robust": (
        TECHNICAL
        + RELATIVE
        + MARKET
        + SECTOR
        + SECTOR_RELATIVE
    ),
}

print("\nFeature groups:")
for name, features in FEATURE_SETS.items():
    print(f"  {name:22s}: {len(features)} features")

# ---------------------------------------------------------------------------
# MODEL FUNCTION
# ---------------------------------------------------------------------------

def train_and_evaluate(name, features):

    Xtr = X_train_full[features]
    Xv = X_val_full[features]

    model = xgb.XGBClassifier(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=30,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=5.0,

        objective="binary:logistic",
        eval_metric="logloss",

        tree_method="hist",
        device="cuda",

        random_state=42,
        n_jobs=-1
    )

    start = time.time()

    model.fit(
        Xtr,
        y_train,
        eval_set=[(Xv, y_val)],
        verbose=False
    )

    elapsed = time.time() - start

    train_prob = model.predict_proba(Xtr)[:, 1]
    val_prob = model.predict_proba(Xv)[:, 1]

    train_auc = roc_auc_score(y_train, train_prob)
    val_auc = roc_auc_score(y_val, val_prob)

    return {
        "feature_family": name,
        "features": len(features),
        "train_auc": train_auc,
        "validation_auc": val_auc,
        "auc_gap": train_auc - val_auc,
        "training_minutes": elapsed / 60
    }

# ---------------------------------------------------------------------------
# RUN EXPERIMENTS
# ---------------------------------------------------------------------------

results = []

for name, features in FEATURE_SETS.items():

    print("\n" + "-" * 70)
    print(f"Training: {name}")
    print("-" * 70)

    result = train_and_evaluate(name, features)

    results.append(result)

    print(
        f"Features       : {result['features']}"
    )
    print(
        f"Train AUC      : {result['train_auc']:.4f}"
    )
    print(
        f"Validation AUC : {result['validation_auc']:.4f}"
    )
    print(
        f"AUC gap        : {result['auc_gap']:.4f}"
    )

# ---------------------------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------------------------

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("FEATURE FAMILY COMPARISON")
print("=" * 70)

print(
    results_df[
        [
            "feature_family",
            "features",
            "train_auc",
            "validation_auc",
            "auc_gap",
            "training_minutes"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

best = results_df.loc[
    results_df["validation_auc"].idxmax()
]

print("\n" + "-" * 70)
print("BEST FEATURE FAMILY")
print("-" * 70)

print(f"Family          : {best['feature_family']}")
print(f"Features        : {int(best['features'])}")
print(f"Validation AUC  : {best['validation_auc']:.4f}")
print(f"Train AUC       : {best['train_auc']:.4f}")
print(f"AUC gap         : {best['auc_gap']:.4f}")

print("\n" + "=" * 70)
print("CELL 45 COMPLETE")
print("=" * 70)

CELL 45 — FEATURE FAMILY ABLATION

Loading matrices...

Feature groups:
  Technical             : 13 features
  Market_Sector         : 17 features
  Technical_Relative    : 21 features
  All_Robust            : 38 features

----------------------------------------------------------------------
Training: Technical
----------------------------------------------------------------------
Features       : 13
Train AUC      : 0.6306
Validation AUC : 0.5388
AUC gap        : 0.0918

----------------------------------------------------------------------
Training: Market_Sector
----------------------------------------------------------------------
Features       : 17
Train AUC      : 0.7959
Validation AUC : 0.5142
AUC gap        : 0.2816

----------------------------------------------------------------------
Training: Technical_Relative
----------------------------------------------------------------------
Features       : 21
Train AUC      : 0.6462
Validation AUC : 0.5391
AUC gap        : 0.107

In [ ]:
# ============================================================================
# CELL 46 — WALK-FORWARD STABILITY TEST
# ============================================================================
import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL 46 — WALK-FORWARD STABILITY TEST")
print("=" * 70)

DATA_DIR = "/content/NEPSE_ML_PROJECT/features/model_ready"

TARGET = "target_up_20d"

# ---------------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------------
print("\nLoading model-ready data...")

X_train_full = pd.read_parquet(
    f"{DATA_DIR}/X_train.parquet"
)

y_train = pd.read_parquet(
    f"{DATA_DIR}/y_train.parquet"
)[TARGET].astype("int8")

metadata = pd.read_parquet(
    f"{DATA_DIR}/train_metadata.parquet"
)

print(f"Training rows: {len(X_train_full):,}")

# ---------------------------------------------------------------------------
# VERIFY DATE
# ---------------------------------------------------------------------------
metadata["date"] = pd.to_datetime(metadata["date"])

print(
    f"Training date range: "
    f"{metadata['date'].min().date()} → "
    f"{metadata['date'].max().date()}"
)

# ---------------------------------------------------------------------------
# 21-FEATURE SET
# ---------------------------------------------------------------------------
FEATURES = [
    # Technical
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",

    # Relative strength
    "relative_return_1d",
    "relative_return_5d",
    "relative_return_20d",
    "relative_return_60d",

    # Sector-relative
    "sector_relative_return_1d",
    "sector_relative_return_5d",
    "sector_relative_return_20d",
    "sector_relative_return_60d",
]

FEATURES = [
    f for f in FEATURES
    if f in X_train_full.columns
]

X = X_train_full[FEATURES].copy()

# ---------------------------------------------------------------------------
# CHRONOLOGICAL FOLDS
# ---------------------------------------------------------------------------
# We deliberately use later periods as validation.
#
# Fold 1:
#   Train: 2011–2016
#   Test : 2017–2018
#
# Fold 2:
#   Train: 2011–2018
#   Test : 2019–2020
#
# Fold 3:
#   Train: 2011–2020
#   Test : 2021
#
# Fold 4:
#   Train: 2011–2021
#   Test : 2022
#
# These are stability checks only.
# The official validation/test split remains untouched.
# ---------------------------------------------------------------------------

folds = [
    ("2017-2018", "2011-01-01", "2016-12-31", "2017-01-01", "2018-12-31"),
    ("2019-2020", "2011-01-01", "2018-12-31", "2019-01-01", "2020-12-31"),
    ("2021",       "2011-01-01", "2020-12-31", "2021-01-01", "2021-12-31"),
]

results = []

# ---------------------------------------------------------------------------
# TRAIN EACH FOLD
# ---------------------------------------------------------------------------
for fold_name, train_start, train_end, val_start, val_end in folds:

    print("\n" + "-" * 70)
    print(f"FOLD: {fold_name}")
    print("-" * 70)

    train_mask = (
        (metadata["date"] >= train_start) &
        (metadata["date"] <= train_end)
    )

    val_mask = (
        (metadata["date"] >= val_start) &
        (metadata["date"] <= val_end)
    )

    train_idx = np.flatnonzero(train_mask.to_numpy())
    val_idx = np.flatnonzero(val_mask.to_numpy())

    if len(train_idx) == 0 or len(val_idx) == 0:
        print("Skipping fold — insufficient rows.")
        continue

    Xtr = X.iloc[train_idx]
    Xv = X.iloc[val_idx]

    ytr = y_train.iloc[train_idx]
    yv = y_train.iloc[val_idx]

    print(
        f"Train: {len(train_idx):,} rows | "
        f"{metadata.iloc[train_idx]['date'].min().date()} → "
        f"{metadata.iloc[train_idx]['date'].max().date()}"
    )

    print(
        f"Valid: {len(val_idx):,} rows | "
        f"{metadata.iloc[val_idx]['date'].min().date()} → "
        f"{metadata.iloc[val_idx]['date'].max().date()}"
    )

    model = xgb.XGBClassifier(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=30,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=5.0,

        objective="binary:logistic",
        eval_metric="logloss",

        tree_method="hist",
        device="cuda",

        random_state=42,
        n_jobs=-1
    )

    start = time.time()

    model.fit(
        Xtr,
        ytr,
        eval_set=[(Xv, yv)],
        verbose=False
    )

    train_prob = model.predict_proba(Xtr)[:, 1]
    val_prob = model.predict_proba(Xv)[:, 1]

    train_auc = roc_auc_score(ytr, train_prob)
    val_auc = roc_auc_score(yv, val_prob)

    elapsed = time.time() - start

    print(f"\nTrain AUC      : {train_auc:.4f}")
    print(f"Validation AUC : {val_auc:.4f}")
    print(f"AUC gap        : {train_auc - val_auc:.4f}")

    results.append({
        "fold": fold_name,
        "train_rows": len(train_idx),
        "validation_rows": len(val_idx),
        "train_auc": train_auc,
        "validation_auc": val_auc,
        "auc_gap": train_auc - val_auc,
        "minutes": elapsed / 60
    })

# ---------------------------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------------------------
results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("WALK-FORWARD RESULTS")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

if len(results_df) > 0:

    print("\n" + "-" * 70)
    print("STABILITY SUMMARY")
    print("-" * 70)

    print(
        f"Mean validation AUC : "
        f"{results_df['validation_auc'].mean():.4f}"
    )

    print(
        f"Median validation AUC : "
        f"{results_df['validation_auc'].median():.4f}"
    )

    print(
        f"Minimum validation AUC : "
        f"{results_df['validation_auc'].min():.4f}"
    )

    print(
        f"Maximum validation AUC : "
        f"{results_df['validation_auc'].max():.4f}"
    )

    positive_folds = (
        results_df["validation_auc"] > 0.50
    ).sum()

    print(
        f"\nFolds above random (AUC > 0.50): "
        f"{positive_folds}/{len(results_df)}"
    )

print("\n" + "=" * 70)
print("CELL 46 COMPLETE")
print("=" * 70)

CELL 46 — WALK-FORWARD STABILITY TEST

Loading model-ready data...
Training rows: 294,410
Training date range: 2011-01-02 → 2021-12-29

----------------------------------------------------------------------
FOLD: 2017-2018
----------------------------------------------------------------------
Train: 92,917 rows | 2011-01-02 → 2016-12-29
Valid: 69,761 rows | 2017-01-01 → 2018-12-31

Train AUC      : 0.6876
Validation AUC : 0.4732
AUC gap        : 0.2144

----------------------------------------------------------------------
FOLD: 2019-2020
----------------------------------------------------------------------
Train: 162,678 rows | 2011-01-02 → 2018-12-31
Valid: 78,047 rows | 2019-01-01 → 2020-12-31

Train AUC      : 0.6771
Validation AUC : 0.5502
AUC gap        : 0.1269

----------------------------------------------------------------------
FOLD: 2021
----------------------------------------------------------------------
Train: 240,725 rows | 2011-01-02 → 2020-12-31
Valid: 53,685 rows |

In [ ]:
# ============================================================================
# CELL 47 — CONSERVATIVE STOCK MODEL COMPARISON
# ============================================================================
import os
import time
import json
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    log_loss
)

print("=" * 70)
print("CELL 47 — CONSERVATIVE STOCK MODEL COMPARISON")
print("=" * 70)

DATA_DIR = "/content/NEPSE_ML_PROJECT/features/model_ready"

TARGET = "target_up_20d"

# ---------------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------------
print("\nLoading model-ready matrices...")

X_train_full = pd.read_parquet(
    f"{DATA_DIR}/X_train.parquet"
)

X_val_full = pd.read_parquet(
    f"{DATA_DIR}/X_validation.parquet"
)

y_train = pd.read_parquet(
    f"{DATA_DIR}/y_train.parquet"
)[TARGET].astype("int8")

y_val = pd.read_parquet(
    f"{DATA_DIR}/y_validation.parquet"
)[TARGET].astype("int8")

# ---------------------------------------------------------------------------
# STOCK-ONLY FEATURES
# ---------------------------------------------------------------------------
FEATURES = [
    # Technical
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",

    # Market-relative stock strength
    "relative_return_1d",
    "relative_return_5d",
    "relative_return_20d",
    "relative_return_60d",

    # Sector-relative stock strength
    "sector_relative_return_1d",
    "sector_relative_return_5d",
    "sector_relative_return_20d",
    "sector_relative_return_60d",
]

FEATURES = [
    f for f in FEATURES
    if f in X_train_full.columns
    and f in X_val_full.columns
]

X_train = X_train_full[FEATURES]
X_val = X_val_full[FEATURES]

print(f"\nStock features: {len(FEATURES)}")
print(f"Train rows    : {len(X_train):,}")
print(f"Validation    : {len(X_val):,}")

# ---------------------------------------------------------------------------
# MODEL DEFINITIONS
# ---------------------------------------------------------------------------
models = {}

# 1. Regularized linear model.
#
# Scaling is performed using TRAINING data only inside the pipeline.
models["Logistic_Regularized"] = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            C=0.05,
            max_iter=1000,
            solver="lbfgs",
            random_state=42
        )
    )
])

# 2. Very conservative XGBoost.
models["XGB_Very_Conservative"] = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.025,
    max_depth=2,
    min_child_weight=50,
    subsample=0.70,
    colsample_bytree=0.70,
    reg_alpha=1.0,
    reg_lambda=10.0,
    gamma=0.10,

    objective="binary:logistic",
    eval_metric="logloss",

    tree_method="hist",
    device="cuda",

    random_state=42,
    n_jobs=-1
)

# 3. Conservative XGBoost.
models["XGB_Conservative"] = xgb.XGBClassifier(
    n_estimators=400,
    learning_rate=0.025,
    max_depth=2,
    min_child_weight=30,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.5,
    reg_lambda=5.0,
    gamma=0.05,

    objective="binary:logistic",
    eval_metric="logloss",

    tree_method="hist",
    device="cuda",

    random_state=42,
    n_jobs=-1
)

# ---------------------------------------------------------------------------
# TRAIN + EVALUATE
# ---------------------------------------------------------------------------
results = []
trained_models = {}

for name, model in models.items():

    print("\n" + "-" * 70)
    print(f"MODEL: {name}")
    print("-" * 70)

    start = time.time()

    if name.startswith("XGB"):

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

    else:

        model.fit(
            X_train,
            y_train
        )

    elapsed = time.time() - start

    train_prob = model.predict_proba(X_train)[:, 1]
    val_prob = model.predict_proba(X_val)[:, 1]

    train_pred = (train_prob >= 0.5).astype("int8")
    val_pred = (val_prob >= 0.5).astype("int8")

    train_auc = roc_auc_score(
        y_train,
        train_prob
    )

    val_auc = roc_auc_score(
        y_val,
        val_prob
    )

    train_acc = accuracy_score(
        y_train,
        train_pred
    )

    val_acc = accuracy_score(
        y_val,
        val_pred
    )

    val_balanced = balanced_accuracy_score(
        y_val,
        val_pred
    )

    val_logloss = log_loss(
        y_val,
        val_prob
    )

    print(f"Train AUC       : {train_auc:.4f}")
    print(f"Validation AUC  : {val_auc:.4f}")
    print(f"AUC gap         : {train_auc - val_auc:.4f}")

    print(f"Validation acc  : {val_acc:.4f}")
    print(f"Validation bal. : {val_balanced:.4f}")
    print(f"Validation loss : {val_logloss:.4f}")

    results.append({
        "model": name,
        "train_auc": train_auc,
        "validation_auc": val_auc,
        "auc_gap": train_auc - val_auc,
        "validation_accuracy": val_acc,
        "validation_balanced_accuracy": val_balanced,
        "validation_logloss": val_logloss,
        "minutes": elapsed / 60
    })

    trained_models[name] = model

# ---------------------------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------------------------
results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ---------------------------------------------------------------------------
# BEST MODEL
# ---------------------------------------------------------------------------
best_idx = results_df["validation_auc"].idxmax()
best = results_df.loc[best_idx]

print("\n" + "-" * 70)
print("CURRENT BEST")
print("-" * 70)

print(f"Model          : {best['model']}")
print(f"Validation AUC : {best['validation_auc']:.4f}")
print(f"Train AUC      : {best['train_auc']:.4f}")
print(f"AUC gap        : {best['auc_gap']:.4f}")

# ---------------------------------------------------------------------------
# SAVE EXPERIMENT RESULTS
# ---------------------------------------------------------------------------
REPORT_DIR = "/content/NEPSE_ML_PROJECT/reports"
os.makedirs(REPORT_DIR, exist_ok=True)

results_path = (
    f"{REPORT_DIR}/cell47_stock_model_comparison.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

print(f"\nSaved comparison:")
print(results_path)

print("\n" + "=" * 70)
print("CELL 47 COMPLETE")
print("=" * 70)

CELL 47 — CONSERVATIVE STOCK MODEL COMPARISON

Loading model-ready matrices...

Stock features: 21
Train rows    : 294,410
Validation    : 178,541

----------------------------------------------------------------------
MODEL: Logistic_Regularized
----------------------------------------------------------------------
Train AUC       : 0.5663
Validation AUC  : 0.5187
AUC gap         : 0.0475
Validation acc  : 0.5050
Validation bal. : 0.5065
Validation loss : 0.6941

----------------------------------------------------------------------
MODEL: XGB_Very_Conservative
----------------------------------------------------------------------
Train AUC       : 0.6149
Validation AUC  : 0.5222
AUC gap         : 0.0927
Validation acc  : 0.5194
Validation bal. : 0.5193
Validation loss : 0.6986

----------------------------------------------------------------------
MODEL: XGB_Conservative
----------------------------------------------------------------------
Train AUC       : 0.6198
Validation AUC  : 

In [ ]:
# ============================================================================
# CELL 48 — TECHNICAL-ONLY MODEL COMPARISON
# ============================================================================
import os
import time
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    log_loss
)

import xgboost as xgb

print("=" * 70)
print("CELL 48 — TECHNICAL-ONLY MODEL COMPARISON")
print("=" * 70)

PROJECT = "/content/NEPSE_ML_PROJECT"
MODEL_DIR = os.path.join(PROJECT, "models")
REPORT_DIR = os.path.join(PROJECT, "reports")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# LOAD MODEL-READY DATA
# ---------------------------------------------------------------------------
print("\nLoading model-ready matrices...")

X_train = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_train.parquet"
)
X_val = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_validation.parquet"
)

y_train = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_train.parquet"
).iloc[:, 0]

y_val = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_validation.parquet"
).iloc[:, 0]

# ---------------------------------------------------------------------------
# STRICT STOCK-ONLY TECHNICAL FEATURES
# ---------------------------------------------------------------------------
technical_features = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

missing = [c for c in technical_features if c not in X_train.columns]

if missing:
    raise ValueError(f"Missing technical features: {missing}")

Xtr = X_train[technical_features].copy()
Xv = X_val[technical_features].copy()

print(f"\nTechnical features : {len(technical_features)}")
print(f"Train rows         : {len(Xtr):,}")
print(f"Validation rows    : {len(Xv):,}")

print("\nFeatures:")
for f in technical_features:
    print("  -", f)

# ---------------------------------------------------------------------------
# GPU CHECK
# ---------------------------------------------------------------------------
gpu_available = False

try:
    test_model = xgb.XGBClassifier(
        n_estimators=1,
        max_depth=1,
        tree_method="hist",
        device="cuda"
    )
    test_model.fit(
        Xtr.iloc[:1000],
        y_train.iloc[:1000]
    )
    gpu_available = True
    del test_model
except Exception as e:
    print("\nGPU test failed; falling back to CPU.")
    print("Reason:", str(e)[:200])

device = "cuda" if gpu_available else "cpu"

print(f"\nXGBoost device : {device}")

# ---------------------------------------------------------------------------
# MODELS
# ---------------------------------------------------------------------------
models = {
    "XGB_Technical_Conservative": xgb.XGBClassifier(
        n_estimators=400,
        learning_rate=0.025,
        max_depth=2,
        min_child_weight=30,
        subsample=0.80,
        colsample_bytree=0.90,
        reg_alpha=0.5,
        reg_lambda=5.0,
        gamma=0.05,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        device=device,
        random_state=42,
        n_jobs=-1
    ),

    "XGB_Technical_Stronger": xgb.XGBClassifier(
        n_estimators=500,
        learning_rate=0.025,
        max_depth=3,
        min_child_weight=30,
        subsample=0.80,
        colsample_bytree=0.90,
        reg_alpha=1.0,
        reg_lambda=10.0,
        gamma=0.05,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        device=device,
        random_state=42,
        n_jobs=-1
    ),

    "XGB_Technical_Regularized": xgb.XGBClassifier(
        n_estimators=500,
        learning_rate=0.02,
        max_depth=3,
        min_child_weight=50,
        subsample=0.75,
        colsample_bytree=0.80,
        reg_alpha=2.0,
        reg_lambda=15.0,
        gamma=0.10,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        device=device,
        random_state=42,
        n_jobs=-1
    ),
}

# ---------------------------------------------------------------------------
# TRAIN + EVALUATE
# ---------------------------------------------------------------------------
results = []
trained_models = {}

for name, model in models.items():

    print("\n" + "-" * 70)
    print(f"MODEL: {name}")
    print("-" * 70)

    start = time.time()

    model.fit(Xtr, y_train)

    train_prob = model.predict_proba(Xtr)[:, 1]
    val_prob = model.predict_proba(Xv)[:, 1]

    train_pred = (train_prob >= 0.5).astype(int)
    val_pred = (val_prob >= 0.5).astype(int)

    train_auc = roc_auc_score(y_train, train_prob)
    val_auc = roc_auc_score(y_val, val_prob)

    val_acc = accuracy_score(y_val, val_pred)
    val_bal = balanced_accuracy_score(y_val, val_pred)
    val_loss = log_loss(y_val, val_prob)

    gap = train_auc - val_auc
    minutes = (time.time() - start) / 60

    print(f"Train AUC       : {train_auc:.4f}")
    print(f"Validation AUC  : {val_auc:.4f}")
    print(f"AUC gap         : {gap:.4f}")
    print(f"Validation acc  : {val_acc:.4f}")
    print(f"Validation bal. : {val_bal:.4f}")
    print(f"Validation loss : {val_loss:.4f}")
    print(f"Time            : {minutes:.3f} min")

    results.append({
        "model": name,
        "train_auc": train_auc,
        "validation_auc": val_auc,
        "auc_gap": gap,
        "validation_accuracy": val_acc,
        "validation_balanced_accuracy": val_bal,
        "validation_logloss": val_loss,
        "minutes": minutes,
    })

    trained_models[name] = model

# ---------------------------------------------------------------------------
# COMPARISON
# ---------------------------------------------------------------------------
results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("TECHNICAL-ONLY MODEL COMPARISON")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ---------------------------------------------------------------------------
# SELECT BEST VALIDATION AUC
# ---------------------------------------------------------------------------
best_row = results_df.loc[
    results_df["validation_auc"].idxmax()
]

best_name = best_row["model"]

print("\n" + "-" * 70)
print("CURRENT TECHNICAL-ONLY BEST")
print("-" * 70)

print(f"Model          : {best_name}")
print(f"Validation AUC : {best_row['validation_auc']:.4f}")
print(f"Train AUC      : {best_row['train_auc']:.4f}")
print(f"AUC gap        : {best_row['auc_gap']:.4f}")

# ---------------------------------------------------------------------------
# SAVE COMPARISON
# ---------------------------------------------------------------------------
comparison_path = (
    f"{REPORT_DIR}/cell48_technical_only_model_comparison.csv"
)

results_df.to_csv(comparison_path, index=False)

# Save the best model for later walk-forward evaluation.
best_model_path = (
    f"{MODEL_DIR}/cell48_best_technical_only_xgboost.json"
)

trained_models[best_name].save_model(best_model_path)

print("\nSaved comparison:")
print(comparison_path)

print("\nSaved best technical-only model:")
print(best_model_path)

print("\n" + "=" * 70)
print("CELL 48 COMPLETE")
print("=" * 70)

CELL 48 — TECHNICAL-ONLY MODEL COMPARISON

Loading model-ready matrices...

Technical features : 13
Train rows         : 294,410
Validation rows    : 178,541

Features:
  - return_1d
  - return_5d
  - return_20d
  - return_60d
  - close_to_sma_20
  - close_to_sma_50
  - close_to_sma_200
  - rsi_14
  - volatility_20d
  - volume_ratio_20
  - volume_change_1d
  - volume_spike_20d
  - trading_activity_score

XGBoost device : cuda

----------------------------------------------------------------------
MODEL: XGB_Technical_Conservative
----------------------------------------------------------------------
Train AUC       : 0.6121
Validation AUC  : 0.5269
AUC gap         : 0.0852
Validation acc  : 0.5209
Validation bal. : 0.5203
Validation loss : 0.6984
Time            : 0.038 min

----------------------------------------------------------------------
MODEL: XGB_Technical_Stronger
----------------------------------------------------------------------
Train AUC       : 0.6314
Validation AUC  :

In [ ]:
# ============================================================================
# CELL 49 — WALK-FORWARD STABILITY TEST
# ============================================================================
import os
import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score

print("=" * 70)
print("CELL 49 — WALK-FORWARD STABILITY TEST")
print("=" * 70)

PROJECT = "/content/NEPSE_ML_PROJECT"
REPORT_DIR = os.path.join(PROJECT, "reports")
os.makedirs(REPORT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------------------------
print("\nLoading model-ready data...")

X_train = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_train.parquet"
)
X_val = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_validation.parquet"
)

y_train = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_train.parquet"
).iloc[:, 0]

y_val = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_validation.parquet"
).iloc[:, 0]

train_meta = pd.read_parquet(
    f"{PROJECT}/features/model_ready/train_metadata.parquet"
)

val_meta = pd.read_parquet(
    f"{PROJECT}/features/model_ready/validation_metadata.parquet"
)

train_meta["date"] = pd.to_datetime(train_meta["date"])
val_meta["date"] = pd.to_datetime(val_meta["date"])

# ---------------------------------------------------------------------------
# STRICT TECHNICAL-ONLY FEATURES
# ---------------------------------------------------------------------------
features = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

X_train = X_train[features].copy()
X_val = X_val[features].copy()

print(f"\nFeatures : {len(features)}")
print(f"Train   : {len(X_train):,}")
print(f"Val     : {len(X_val):,}")

# ---------------------------------------------------------------------------
# COMBINE TRAIN + VALIDATION FOR WALK-FORWARD ANALYSIS
# ---------------------------------------------------------------------------
X_all = pd.concat(
    [X_train, X_val],
    axis=0,
    ignore_index=True
)

y_all = pd.concat(
    [y_train, y_val],
    axis=0,
    ignore_index=True
)

meta_all = pd.concat(
    [train_meta[["date"]], val_meta[["date"]]],
    axis=0,
    ignore_index=True
)

# ---------------------------------------------------------------------------
# DEFINE HISTORICAL WALK-FORWARD PERIODS
#
# Each validation period is strictly AFTER its training period.
# We deliberately leave a 20-trading-day gap because the target itself
# looks 20 trading days into the future.
# ---------------------------------------------------------------------------
folds = [
    {
        "name": "2017-2018",
        "train_end": "2016-12-31",
        "val_start": "2017-01-01",
        "val_end": "2018-12-31",
    },
    {
        "name": "2019-2020",
        "train_end": "2018-12-31",
        "val_start": "2019-01-01",
        "val_end": "2020-12-31",
    },
    {
        "name": "2021",
        "train_end": "2020-12-31",
        "val_start": "2021-01-01",
        "val_end": "2021-12-31",
    },
    {
        "name": "2022-2023",
        "train_end": "2021-12-31",
        "val_start": "2022-04-01",
        "val_end": "2023-12-31",
    },
]

results = []

# ---------------------------------------------------------------------------
# TRAIN EACH FOLD
# ---------------------------------------------------------------------------
for fold in folds:

    print("\n" + "-" * 70)
    print(f"FOLD: {fold['name']}")
    print("-" * 70)

    train_end = pd.Timestamp(fold["train_end"])
    val_start = pd.Timestamp(fold["val_start"])
    val_end = pd.Timestamp(fold["val_end"])

    # 20 trading-day-ish calendar buffer.
    # We use date separation here rather than random row removal.
    gap_days = 30
    effective_val_start = val_start + pd.Timedelta(days=gap_days)

    train_mask = meta_all["date"] <= train_end
    val_mask = (
        (meta_all["date"] >= effective_val_start) &
        (meta_all["date"] <= val_end)
    )

    Xtr = X_all.loc[train_mask]
    ytr = y_all.loc[train_mask]

    Xv = X_all.loc[val_mask]
    yv = y_all.loc[val_mask]

    if len(Xtr) == 0 or len(Xv) == 0:
        print("SKIPPED — insufficient data")
        continue

    print(f"Training rows   : {len(Xtr):,}")
    print(f"Validation rows : {len(Xv):,}")
    print(f"Train end       : {meta_all.loc[train_mask, 'date'].max().date()}")
    print(f"Val start       : {meta_all.loc[val_mask, 'date'].min().date()}")
    print(f"Val end         : {meta_all.loc[val_mask, 'date'].max().date()}")

    model = xgb.XGBClassifier(
        n_estimators=500,
        learning_rate=0.025,
        max_depth=3,
        min_child_weight=30,
        subsample=0.80,
        colsample_bytree=0.90,
        reg_alpha=1.0,
        reg_lambda=10.0,
        gamma=0.05,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        device="cuda",
        random_state=42,
        n_jobs=-1
    )

    start = time.time()

    model.fit(Xtr, ytr)

    train_prob = model.predict_proba(Xtr)[:, 1]
    val_prob = model.predict_proba(Xv)[:, 1]

    train_pred = (train_prob >= 0.5).astype(int)
    val_pred = (val_prob >= 0.5).astype(int)

    train_auc = roc_auc_score(ytr, train_prob)
    val_auc = roc_auc_score(yv, val_prob)

    val_acc = accuracy_score(yv, val_pred)
    val_bal = balanced_accuracy_score(yv, val_pred)

    elapsed = (time.time() - start) / 60

    print(f"\nTrain AUC       : {train_auc:.4f}")
    print(f"Validation AUC  : {val_auc:.4f}")
    print(f"AUC gap         : {train_auc - val_auc:.4f}")
    print(f"Validation acc  : {val_acc:.4f}")
    print(f"Validation bal. : {val_bal:.4f}")
    print(f"Time            : {elapsed:.3f} min")

    results.append({
        "fold": fold["name"],
        "train_rows": len(Xtr),
        "validation_rows": len(Xv),
        "train_auc": train_auc,
        "validation_auc": val_auc,
        "auc_gap": train_auc - val_auc,
        "validation_accuracy": val_acc,
        "validation_balanced_accuracy": val_bal,
        "minutes": elapsed,
    })

# ---------------------------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------------------------
results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("WALK-FORWARD SUMMARY")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

if len(results_df):

    print("\n" + "-" * 70)
    print("STABILITY STATISTICS")
    print("-" * 70)

    print(
        f"Mean validation AUC   : "
        f"{results_df['validation_auc'].mean():.4f}"
    )

    print(
        f"Median validation AUC : "
        f"{results_df['validation_auc'].median():.4f}"
    )

    print(
        f"Minimum validation AUC: "
        f"{results_df['validation_auc'].min():.4f}"
    )

    print(
        f"Maximum validation AUC: "
        f"{results_df['validation_auc'].max():.4f}"
    )

    above_random = (
        results_df["validation_auc"] > 0.5000
    ).sum()

    print(
        f"Folds above random    : "
        f"{above_random}/{len(results_df)}"
    )

    mean_auc = results_df["validation_auc"].mean()

    if mean_auc >= 0.53 and above_random >= len(results_df) * 0.75:
        verdict = "PROMISING — signal is reasonably stable."
    elif mean_auc >= 0.515 and above_random >= len(results_df) * 0.50:
        verdict = "WEAK BUT POSSIBLY USEFUL — signal is regime-dependent."
    else:
        verdict = "UNSTABLE — do not rely on this model yet."

    print(f"\nVERDICT: {verdict}")

# ---------------------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------------------
path = f"{REPORT_DIR}/cell49_technical_walk_forward_results.csv"

results_df.to_csv(path, index=False)

print("\nSaved:")
print(path)

print("\n" + "=" * 70)
print("CELL 49 COMPLETE")
print("=" * 70)

CELL 49 — WALK-FORWARD STABILITY TEST

Loading model-ready data...

Features : 13
Train   : 294,410
Val     : 178,541

----------------------------------------------------------------------
FOLD: 2017-2018
----------------------------------------------------------------------
Training rows   : 92,917
Validation rows : 67,326
Train end       : 2016-12-29
Val start       : 2017-01-31
Val end         : 2018-12-31

Train AUC       : 0.6467
Validation AUC  : 0.4550
AUC gap         : 0.1917
Validation acc  : 0.4196
Validation bal. : 0.4591
Time            : 0.022 min

----------------------------------------------------------------------
FOLD: 2019-2020
----------------------------------------------------------------------
Training rows   : 162,678
Validation rows : 74,509
Train end       : 2018-12-31
Val start       : 2019-01-31
Val end         : 2020-12-31

Train AUC       : 0.6559
Validation AUC  : 0.5282
AUC gap         : 0.1278
Validation acc  : 0.4983
Validation bal. : 0.5190
Time     

In [ ]:
# ============================================================================
# CELL 50 — REGIME SIGNAL DIAGNOSTIC
# ============================================================================
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL 50 — REGIME SIGNAL DIAGNOSTIC")
print("=" * 70)

PROJECT = "/content/NEPSE_ML_PROJECT"
REPORT_DIR = os.path.join(PROJECT, "reports")
os.makedirs(REPORT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------------------------
print("\nLoading data...")

X_train = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_train.parquet"
)

X_val = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_validation.parquet"
)

y_train = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_train.parquet"
).iloc[:, 0]

y_val = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_validation.parquet"
).iloc[:, 0]

train_meta = pd.read_parquet(
    f"{PROJECT}/features/model_ready/train_metadata.parquet"
)

val_meta = pd.read_parquet(
    f"{PROJECT}/features/model_ready/validation_metadata.parquet"
)

train_meta["date"] = pd.to_datetime(train_meta["date"])
val_meta["date"] = pd.to_datetime(val_meta["date"])

# ---------------------------------------------------------------------------
# FEATURES
# ---------------------------------------------------------------------------
features = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

# ---------------------------------------------------------------------------
# COMBINE
# ---------------------------------------------------------------------------
X = pd.concat(
    [
        X_train[features],
        X_val[features]
    ],
    ignore_index=True
)

y = pd.concat(
    [y_train, y_val],
    ignore_index=True
)

meta = pd.concat(
    [
        train_meta[["date"]],
        val_meta[["date"]]
    ],
    ignore_index=True
)

# ---------------------------------------------------------------------------
# DEFINE REGIMES
# ---------------------------------------------------------------------------
regimes = {
    "2017-2018": ("2017-01-01", "2018-12-31"),
    "2019-2020": ("2019-01-01", "2020-12-31"),
    "2021":      ("2021-01-01", "2021-12-31"),
    "2022-2023": ("2022-01-01", "2023-12-31"),
    "2024":      ("2024-01-01", "2024-12-31"),
}

# ---------------------------------------------------------------------------
# CALCULATE FEATURE AUC BY REGIME
# ---------------------------------------------------------------------------
rows = []

for regime_name, (start, end) in regimes.items():

    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    mask = (
        (meta["date"] >= start) &
        (meta["date"] <= end)
    )

    Xr = X.loc[mask]
    yr = y.loc[mask]

    print("\n" + "-" * 70)
    print(f"REGIME: {regime_name}")
    print("-" * 70)

    print(f"Rows : {len(Xr):,}")

    if len(Xr) == 0:
        continue

    up_rate = yr.mean()

    print(f"UP rate : {up_rate:.4f}")

    for feature in features:

        values = Xr[feature]

        valid = (
            values.notna() &
            np.isfinite(values.to_numpy())
        )

        if valid.sum() < 1000:
            continue

        y_valid = yr.loc[valid]
        x_valid = values.loc[valid]

        if y_valid.nunique() < 2:
            continue

        auc = roc_auc_score(y_valid, x_valid)

        # Direction-adjusted AUC.
        # This tells us whether the feature has any relationship
        # regardless of whether high values predict UP or DOWN.
        directional_auc = max(auc, 1.0 - auc)

        direction = "positive" if auc >= 0.5 else "negative"

        rows.append({
            "regime": regime_name,
            "feature": feature,
            "auc": auc,
            "directional_auc": directional_auc,
            "direction": direction,
            "valid_rows": int(valid.sum()),
        })

# ---------------------------------------------------------------------------
# RESULTS
# ---------------------------------------------------------------------------
results = pd.DataFrame(rows)

print("\n" + "=" * 70)
print("FEATURE SIGNAL BY REGIME")
print("=" * 70)

# Show strongest signals in each regime.
for regime in regimes.keys():

    subset = results[
        results["regime"] == regime
    ].sort_values(
        "directional_auc",
        ascending=False
    )

    print("\n" + "-" * 70)
    print(regime)
    print("-" * 70)

    if len(subset):

        display_cols = [
            "feature",
            "auc",
            "directional_auc",
            "direction"
        ]

        print(
            subset[display_cols]
            .head(13)
            .to_string(
                index=False,
                float_format=lambda x: f"{x:.4f}"
            )
        )

# ---------------------------------------------------------------------------
# BUILD REGIME x FEATURE MATRIX
# ---------------------------------------------------------------------------
auc_matrix = results.pivot(
    index="feature",
    columns="regime",
    values="auc"
)

directional_matrix = results.pivot(
    index="feature",
    columns="regime",
    values="directional_auc"
)

print("\n" + "=" * 70)
print("RAW AUC MATRIX")
print("=" * 70)

print(
    auc_matrix.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n" + "=" * 70)
print("DIRECTION-ADJUSTED AUC MATRIX")
print("=" * 70)

print(
    directional_matrix.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)

# ---------------------------------------------------------------------------
# STABILITY SCORE
# ---------------------------------------------------------------------------
summary = []

for feature in features:

    vals = directional_matrix.loc[
        feature
    ].dropna()

    if len(vals) == 0:
        continue

    summary.append({
        "feature": feature,
        "mean_directional_auc": vals.mean(),
        "median_directional_auc": vals.median(),
        "min_directional_auc": vals.min(),
        "max_directional_auc": vals.max(),
        "regimes_above_0.52": int((vals >= 0.52).sum()),
        "regimes_above_0.53": int((vals >= 0.53).sum()),
    })

summary_df = pd.DataFrame(summary).sort_values(
    "mean_directional_auc",
    ascending=False
)

print("\n" + "=" * 70)
print("FEATURE STABILITY SUMMARY")
print("=" * 70)

print(
    summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ---------------------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------------------
results_path = (
    f"{REPORT_DIR}/cell50_regime_feature_auc.csv"
)

summary_path = (
    f"{REPORT_DIR}/cell50_feature_stability_summary.csv"
)

matrix_path = (
    f"{REPORT_DIR}/cell50_regime_auc_matrix.csv"
)

results.to_csv(results_path, index=False)
summary_df.to_csv(summary_path, index=False)
auc_matrix.to_csv(matrix_path)

print("\nSaved:")
print(results_path)
print(summary_path)
print(matrix_path)

print("\n" + "=" * 70)
print("CELL 50 COMPLETE")
print("=" * 70)

CELL 50 — REGIME SIGNAL DIAGNOSTIC

Loading data...

----------------------------------------------------------------------
REGIME: 2017-2018
----------------------------------------------------------------------
Rows : 69,761
UP rate : 0.3684

----------------------------------------------------------------------
REGIME: 2019-2020
----------------------------------------------------------------------
Rows : 78,047
UP rate : 0.5633

----------------------------------------------------------------------
REGIME: 2021
----------------------------------------------------------------------
Rows : 53,685
UP rate : 0.5627

----------------------------------------------------------------------
REGIME: 2022-2023
----------------------------------------------------------------------
Rows : 105,159
UP rate : 0.4288

----------------------------------------------------------------------
REGIME: 2024
----------------------------------------------------------------------
Rows : 73,382
UP rate : 0.55

In [ ]:
# ============================================================================
# CELL 51 — MARKET + SECTOR REGIME SIGNAL DIAGNOSTIC
# ============================================================================
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL 51 — MARKET + SECTOR REGIME SIGNAL DIAGNOSTIC")
print("=" * 70)

PROJECT = "/content/NEPSE_ML_PROJECT"
REPORT_DIR = os.path.join(PROJECT, "reports")
os.makedirs(REPORT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------------------------
print("\nLoading model-ready data...")

X_train = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_train.parquet"
)

X_val = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_validation.parquet"
)

y_train = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_train.parquet"
).iloc[:, 0]

y_val = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_validation.parquet"
).iloc[:, 0]

train_meta = pd.read_parquet(
    f"{PROJECT}/features/model_ready/train_metadata.parquet"
)

val_meta = pd.read_parquet(
    f"{PROJECT}/features/model_ready/validation_metadata.parquet"
)

train_meta["date"] = pd.to_datetime(train_meta["date"])
val_meta["date"] = pd.to_datetime(val_meta["date"])

# ---------------------------------------------------------------------------
# COMBINE
# ---------------------------------------------------------------------------
X = pd.concat(
    [X_train, X_val],
    ignore_index=True
)

y = pd.concat(
    [y_train, y_val],
    ignore_index=True
)

meta = pd.concat(
    [
        train_meta[["date"]],
        val_meta[["date"]]
    ],
    ignore_index=True
)

# ---------------------------------------------------------------------------
# MARKET + SECTOR FEATURES
#
# We intentionally exclude raw timestamp and event/corporate-action fields.
# These are contextual regime variables only.
# ---------------------------------------------------------------------------
context_features = [
    # Market returns
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_return_60d",

    # Market trend / position
    "market_close_to_sma_20",
    "market_close_to_sma_50",
    "market_close_to_sma_100",
    "market_close_to_sma_200",
    "market_bull_trend",
    "market_bear_trend",

    # Market volatility
    "market_volatility_5d",
    "market_volatility_10d",
    "market_volatility_20d",
    "market_volatility_60d",
    "market_high_volatility",

    # Market breadth
    "market_breadth_up_ratio",

    # Sector returns
    "sector_return_1d",
    "sector_return_5d",
    "sector_return_20d",
    "sector_return_60d",

    # Sector volatility
    "sector_volatility_5d",
    "sector_volatility_20d",
    "sector_volatility_60d",

    # Sector breadth
    "sector_breadth_up_ratio",
    "sector_breadth_20d",
    "sector_breadth_60d",

    # Sector strength
    "sector_strength_20d",

    # Stock relative to market
    "relative_return_1d",
    "relative_return_5d",
    "relative_return_20d",
    "relative_return_60d",

    # Stock relative to sector
    "sector_relative_return_1d",
    "sector_relative_return_5d",
    "sector_relative_return_20d",
    "sector_relative_return_60d",
]

missing = [
    c for c in context_features
    if c not in X.columns
]

if missing:
    raise ValueError(
        "Missing context features:\n" +
        "\n".join(missing)
    )

print(f"\nContext features : {len(context_features)}")
print(f"Rows             : {len(X):,}")

# ---------------------------------------------------------------------------
# REGIMES
# ---------------------------------------------------------------------------
regimes = {
    "2017-2018": ("2017-01-01", "2018-12-31"),
    "2019-2020": ("2019-01-01", "2020-12-31"),
    "2021":      ("2021-01-01", "2021-12-31"),
    "2022-2023": ("2022-01-01", "2023-12-31"),
    "2024":      ("2024-01-01", "2024-12-31"),
}

# ---------------------------------------------------------------------------
# AUC BY REGIME
# ---------------------------------------------------------------------------
rows = []

for regime_name, (start, end) in regimes.items():

    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    mask = (
        (meta["date"] >= start) &
        (meta["date"] <= end)
    )

    Xr = X.loc[mask]
    yr = y.loc[mask]

    print("\n" + "-" * 70)
    print(f"REGIME: {regime_name}")
    print("-" * 70)

    print(f"Rows   : {len(Xr):,}")
    print(f"UP rate: {yr.mean():.4f}")

    for feature in context_features:

        values = Xr[feature]

        valid = (
            values.notna() &
            np.isfinite(values.to_numpy())
        )

        if valid.sum() < 1000:
            continue

        xv = values.loc[valid]
        yv = yr.loc[valid]

        if yv.nunique() < 2:
            continue

        auc = roc_auc_score(yv, xv)

        directional_auc = max(
            auc,
            1.0 - auc
        )

        direction = (
            "positive"
            if auc >= 0.5
            else "negative"
        )

        rows.append({
            "regime": regime_name,
            "feature": feature,
            "auc": auc,
            "directional_auc": directional_auc,
            "direction": direction,
            "valid_rows": int(valid.sum()),
        })

# ---------------------------------------------------------------------------
# RESULTS
# ---------------------------------------------------------------------------
results = pd.DataFrame(rows)

print("\n" + "=" * 70)
print("STRONGEST CONTEXT FEATURES BY REGIME")
print("=" * 70)

for regime in regimes.keys():

    subset = results[
        results["regime"] == regime
    ].sort_values(
        "directional_auc",
        ascending=False
    )

    print("\n" + "-" * 70)
    print(regime)
    print("-" * 70)

    print(
        subset[
            [
                "feature",
                "auc",
                "directional_auc",
                "direction"
            ]
        ]
        .head(15)
        .to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

# ---------------------------------------------------------------------------
# MATRICES
# ---------------------------------------------------------------------------
auc_matrix = results.pivot(
    index="feature",
    columns="regime",
    values="auc"
)

directional_matrix = results.pivot(
    index="feature",
    columns="regime",
    values="directional_auc"
)

# ---------------------------------------------------------------------------
# STABILITY SUMMARY
# ---------------------------------------------------------------------------
summary = []

for feature in context_features:

    if feature not in directional_matrix.index:
        continue

    vals = directional_matrix.loc[
        feature
    ].dropna()

    if len(vals) == 0:
        continue

    summary.append({
        "feature": feature,
        "mean_directional_auc": vals.mean(),
        "median_directional_auc": vals.median(),
        "min_directional_auc": vals.min(),
        "max_directional_auc": vals.max(),
        "regimes_above_0.52": int(
            (vals >= 0.52).sum()
        ),
        "regimes_above_0.53": int(
            (vals >= 0.53).sum()
        ),
    })

summary_df = pd.DataFrame(summary).sort_values(
    "mean_directional_auc",
    ascending=False
)

print("\n" + "=" * 70)
print("CONTEXT FEATURE STABILITY")
print("=" * 70)

print(
    summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ---------------------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------------------
results_path = (
    f"{REPORT_DIR}/cell51_context_regime_feature_auc.csv"
)

summary_path = (
    f"{REPORT_DIR}/cell51_context_feature_stability_summary.csv"
)

matrix_path = (
    f"{REPORT_DIR}/cell51_context_regime_auc_matrix.csv"
)

results.to_csv(results_path, index=False)
summary_df.to_csv(summary_path, index=False)
auc_matrix.to_csv(matrix_path)

print("\nSaved:")
print(results_path)
print(summary_path)
print(matrix_path)

print("\n" + "=" * 70)
print("CELL 51 COMPLETE")
print("=" * 70)

CELL 51 — MARKET + SECTOR REGIME SIGNAL DIAGNOSTIC

Loading model-ready data...

Context features : 35
Rows             : 472,951

----------------------------------------------------------------------
REGIME: 2017-2018
----------------------------------------------------------------------
Rows   : 69,761
UP rate: 0.3684

----------------------------------------------------------------------
REGIME: 2019-2020
----------------------------------------------------------------------
Rows   : 78,047
UP rate: 0.5633

----------------------------------------------------------------------
REGIME: 2021
----------------------------------------------------------------------
Rows   : 53,685
UP rate: 0.5627

----------------------------------------------------------------------
REGIME: 2022-2023
----------------------------------------------------------------------
Rows   : 105,159
UP rate: 0.4288

----------------------------------------------------------------------
REGIME: 2024
-----------------

In [ ]:
# ============================================================================
# CELL 52 — STOCK + MARKET + SECTOR CONTEXT MODEL COMPARISON
# ============================================================================
import os
import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    log_loss
)

print("=" * 70)
print("CELL 52 — STOCK + MARKET + SECTOR CONTEXT MODEL COMPARISON")
print("=" * 70)

PROJECT = "/content/NEPSE_ML_PROJECT"
MODEL_DIR = os.path.join(PROJECT, "models")
REPORT_DIR = os.path.join(PROJECT, "reports")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------------
print("\nLoading model-ready matrices...")

X_train = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_train.parquet"
)

X_val = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_validation.parquet"
)

y_train = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_train.parquet"
).iloc[:, 0]

y_val = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_validation.parquet"
).iloc[:, 0]

print(f"Train rows : {len(X_train):,}")
print(f"Val rows   : {len(X_val):,}")

# ---------------------------------------------------------------------------
# FEATURE GROUPS
# ---------------------------------------------------------------------------

stock_features = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

# Strongest and most stable MARKET features from Cell 51.
market_features = [
    "market_volatility_60d",
    "market_volatility_20d",
    "market_volatility_10d",
    "market_volatility_5d",

    "market_close_to_sma_200",
    "market_close_to_sma_100",
    "market_close_to_sma_50",
    "market_close_to_sma_20",

    "market_return_60d",
    "market_return_20d",

    "market_bear_trend",
    "market_bull_trend",

    "market_high_volatility",
    "market_breadth_up_ratio",
]

# Strongest SECTOR features from Cell 51.
sector_features = [
    "sector_return_60d",
    "sector_return_20d",

    "sector_volatility_60d",
    "sector_volatility_20d",

    "sector_breadth_60d",
    "sector_breadth_20d",

    "sector_strength_20d",
]

# Relative features are kept as a separate experimental group.
relative_features = [
    "relative_return_1d",
    "relative_return_5d",
    "relative_return_20d",
    "relative_return_60d",

    "sector_relative_return_1d",
    "sector_relative_return_5d",
    "sector_relative_return_20d",
    "sector_relative_return_60d",
]

# ---------------------------------------------------------------------------
# VERIFY
# ---------------------------------------------------------------------------
all_features = (
    stock_features +
    market_features +
    sector_features +
    relative_features
)

missing = [
    f for f in all_features
    if f not in X_train.columns
]

if missing:
    raise ValueError(
        "Missing features:\n" +
        "\n".join(missing)
    )

print("\nFeature groups:")
print(f"  Stock     : {len(stock_features)}")
print(f"  Market    : {len(market_features)}")
print(f"  Sector    : {len(sector_features)}")
print(f"  Relative  : {len(relative_features)}")

# ---------------------------------------------------------------------------
# MODEL SETS
# ---------------------------------------------------------------------------
feature_sets = {
    "Stock_Only": stock_features,

    "Stock_Market": (
        stock_features +
        market_features
    ),

    "Stock_Market_Sector": (
        stock_features +
        market_features +
        sector_features
    ),

    "Stock_Market_Sector_Relative": (
        stock_features +
        market_features +
        sector_features +
        relative_features
    ),
}

# ---------------------------------------------------------------------------
# TRAINING FUNCTION
# ---------------------------------------------------------------------------
def train_and_evaluate(name, features):

    print("\n" + "-" * 70)
    print(f"MODEL: {name}")
    print("-" * 70)

    print(f"Features: {len(features)}")

    Xtr = X_train[features]
    Xv = X_val[features]

    model = xgb.XGBClassifier(
        n_estimators=500,
        learning_rate=0.025,
        max_depth=3,
        min_child_weight=30,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_alpha=1.0,
        reg_lambda=10.0,
        gamma=0.05,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        device="cuda",
        random_state=42,
        n_jobs=-1
    )

    start = time.time()

    model.fit(
        Xtr,
        y_train
    )

    train_prob = model.predict_proba(Xtr)[:, 1]
    val_prob = model.predict_proba(Xv)[:, 1]

    train_pred = (
        train_prob >= 0.5
    ).astype(int)

    val_pred = (
        val_prob >= 0.5
    ).astype(int)

    train_auc = roc_auc_score(
        y_train,
        train_prob
    )

    val_auc = roc_auc_score(
        y_val,
        val_prob
    )

    val_acc = accuracy_score(
        y_val,
        val_pred
    )

    val_bal = balanced_accuracy_score(
        y_val,
        val_pred
    )

    val_loss = log_loss(
        y_val,
        val_prob
    )

    gap = train_auc - val_auc

    minutes = (
        time.time() - start
    ) / 60

    print(f"Train AUC       : {train_auc:.4f}")
    print(f"Validation AUC  : {val_auc:.4f}")
    print(f"AUC gap         : {gap:.4f}")
    print(f"Validation acc  : {val_acc:.4f}")
    print(f"Validation bal. : {val_bal:.4f}")
    print(f"Validation loss : {val_loss:.4f}")
    print(f"Time            : {minutes:.3f} min")

    return (
        model,
        {
            "model": name,
            "feature_count": len(features),
            "train_auc": train_auc,
            "validation_auc": val_auc,
            "auc_gap": gap,
            "validation_accuracy": val_acc,
            "validation_balanced_accuracy": val_bal,
            "validation_logloss": val_loss,
            "minutes": minutes,
        }
    )

# ---------------------------------------------------------------------------
# RUN MODELS
# ---------------------------------------------------------------------------
results = []
trained = {}

for name, features in feature_sets.items():

    model, result = train_and_evaluate(
        name,
        features
    )

    trained[name] = model
    results.append(result)

# ---------------------------------------------------------------------------
# RESULTS
# ---------------------------------------------------------------------------
results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("CONTEXT MODEL COMPARISON")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ---------------------------------------------------------------------------
# IMPROVEMENT OVER STOCK ONLY
# ---------------------------------------------------------------------------
base_auc = results_df.loc[
    results_df["model"] == "Stock_Only",
    "validation_auc"
].iloc[0]

print("\n" + "=" * 70)
print("VALIDATION AUC IMPROVEMENT")
print("=" * 70)

for _, row in results_df.iterrows():

    improvement = (
        row["validation_auc"] -
        base_auc
    )

    print(
        f"{row['model']:<35} "
        f"{row['validation_auc']:.4f}  "
        f"change: {improvement:+.4f}"
    )

# ---------------------------------------------------------------------------
# BEST MODEL
# ---------------------------------------------------------------------------
best_row = results_df.loc[
    results_df["validation_auc"].idxmax()
]

best_name = best_row["model"]

print("\n" + "-" * 70)
print("CURRENT BEST CONTEXT MODEL")
print("-" * 70)

print(f"Model          : {best_name}")
print(f"Validation AUC : {best_row['validation_auc']:.4f}")
print(f"Train AUC      : {best_row['train_auc']:.4f}")
print(f"AUC gap        : {best_row['auc_gap']:.4f}")

# ---------------------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------------------
comparison_path = (
    f"{REPORT_DIR}/cell52_context_model_comparison.csv"
)

results_df.to_csv(
    comparison_path,
    index=False
)

best_model_path = (
    f"{MODEL_DIR}/cell52_best_context_xgboost.json"
)

trained[best_name].save_model(
    best_model_path
)

print("\nSaved comparison:")
print(comparison_path)

print("\nSaved best model:")
print(best_model_path)

print("\n" + "=" * 70)
print("CELL 52 COMPLETE")
print("=" * 70)

CELL 52 — STOCK + MARKET + SECTOR CONTEXT MODEL COMPARISON

Loading model-ready matrices...
Train rows : 294,410
Val rows   : 178,541

Feature groups:
  Stock     : 13
  Market    : 14
  Sector    : 7
  Relative  : 8

----------------------------------------------------------------------
MODEL: Stock_Only
----------------------------------------------------------------------
Features: 13
Train AUC       : 0.6306
Validation AUC  : 0.5384
AUC gap         : 0.0922
Validation acc  : 0.5269
Validation bal. : 0.5267
Validation loss : 0.6976
Time            : 0.040 min

----------------------------------------------------------------------
MODEL: Stock_Market
----------------------------------------------------------------------
Features: 27
Train AUC       : 0.8127
Validation AUC  : 0.5097
AUC gap         : 0.3030
Validation acc  : 0.4973
Validation bal. : 0.4948
Validation loss : 0.7362
Time            : 0.076 min

----------------------------------------------------------------------
MODEL

In [ ]:
# ============================================================================
# CELL 53 — REGIME-CONDITIONED STOCK SIGNAL DIAGNOSTIC
# ============================================================================
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL 53 — REGIME-CONDITIONED STOCK SIGNAL DIAGNOSTIC")
print("=" * 70)

PROJECT = "/content/NEPSE_ML_PROJECT"
REPORT_DIR = os.path.join(PROJECT, "reports")
os.makedirs(REPORT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------------
print("\nLoading data...")

X_train = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_train.parquet"
)

X_val = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_validation.parquet"
)

y_train = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_train.parquet"
).iloc[:, 0]

y_val = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_validation.parquet"
).iloc[:, 0]

train_meta = pd.read_parquet(
    f"{PROJECT}/features/model_ready/train_metadata.parquet"
)

val_meta = pd.read_parquet(
    f"{PROJECT}/features/model_ready/validation_metadata.parquet"
)

train_meta["date"] = pd.to_datetime(train_meta["date"])
val_meta["date"] = pd.to_datetime(val_meta["date"])

X = pd.concat(
    [X_train, X_val],
    ignore_index=True
)

y = pd.concat(
    [y_train, y_val],
    ignore_index=True
)

meta = pd.concat(
    [
        train_meta[["date"]],
        val_meta[["date"]]
    ],
    ignore_index=True
)

print(f"Rows: {len(X):,}")

# ---------------------------------------------------------------------------
# STOCK SIGNALS
# ---------------------------------------------------------------------------
stock_features = [
    "close_to_sma_50",
    "close_to_sma_200",
    "return_20d",
    "return_60d",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "trading_activity_score",
]

# ---------------------------------------------------------------------------
# REGIME VARIABLES
# ---------------------------------------------------------------------------
regime_features = [
    "market_volatility_60d",
    "market_volatility_20d",
    "market_close_to_sma_200",
    "market_return_60d",
    "sector_return_60d",
    "sector_breadth_20d",
    "sector_volatility_60d",
]

missing = [
    c for c in stock_features + regime_features
    if c not in X.columns
]

if missing:
    raise ValueError(
        "Missing features:\n" +
        "\n".join(missing)
    )

# ---------------------------------------------------------------------------
# VALIDATION ONLY
#
# We deliberately evaluate the regime-conditioned relationship directly.
# No model is fitted here, so we avoid another train-period overfit.
# ---------------------------------------------------------------------------
val_start = pd.Timestamp("2022-01-01")
val_end = pd.Timestamp("2024-12-31")

mask = (
    (meta["date"] >= val_start) &
    (meta["date"] <= val_end)
)

Xv = X.loc[mask].copy()
yv = y.loc[mask].copy()

print("\nValidation period:")
print(f"Rows   : {len(Xv):,}")
print(f"UP rate: {yv.mean():.4f}")

# ---------------------------------------------------------------------------
# CREATE MARKET/SECTOR REGIMES
#
# Quantile thresholds are calculated ONLY from the validation period.
# This is a diagnostic, not yet a deployable model.
# ---------------------------------------------------------------------------
regime_masks = {}

# Market volatility
mv60 = Xv["market_volatility_60d"]

q25 = mv60.quantile(0.25)
q75 = mv60.quantile(0.75)

regime_masks["Low_Market_Volatility"] = (
    mv60 <= q25
)

regime_masks["High_Market_Volatility"] = (
    mv60 >= q75
)

# Market long-term trend
m200 = Xv["market_close_to_sma_200"]

regime_masks["Market_Below_SMA200"] = (
    m200 < 0
)

regime_masks["Market_Above_SMA200"] = (
    m200 >= 0
)

# Sector strength
sr60 = Xv["sector_return_60d"]

regime_masks["Weak_Sector"] = (
    sr60 <= sr60.quantile(0.25)
)

regime_masks["Strong_Sector"] = (
    sr60 >= sr60.quantile(0.75)
)

# ---------------------------------------------------------------------------
# AUC FUNCTION
# ---------------------------------------------------------------------------
def directional_auc(feature_values, target):

    valid = (
        feature_values.notna() &
        np.isfinite(
            feature_values.to_numpy()
        )
    )

    xv = feature_values.loc[valid]
    yv = target.loc[valid]

    if len(xv) < 1000:
        return np.nan, np.nan, 0

    if yv.nunique() < 2:
        return np.nan, np.nan, len(xv)

    auc = roc_auc_score(
        yv,
        xv
    )

    return (
        auc,
        max(auc, 1.0 - auc),
        len(xv)
    )

# ---------------------------------------------------------------------------
# BASELINE
# ---------------------------------------------------------------------------
results = []

print("\n" + "=" * 70)
print("BASELINE STOCK SIGNALS")
print("=" * 70)

for feature in stock_features:

    auc, dauc, n = directional_auc(
        Xv[feature],
        yv
    )

    print(
        f"{feature:<28} "
        f"AUC={auc:.4f} "
        f"Directional={dauc:.4f}"
    )

    results.append({
        "regime": "ALL",
        "feature": feature,
        "auc": auc,
        "directional_auc": dauc,
        "valid_rows": n,
    })

# ---------------------------------------------------------------------------
# REGIME-CONDITIONED RESULTS
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("REGIME-CONDITIONED STOCK SIGNALS")
print("=" * 70)

for regime_name, rmask in regime_masks.items():

    print("\n" + "-" * 70)
    print(regime_name)
    print("-" * 70)

    print(
        f"Rows: {int(rmask.sum()):,}"
    )

    if rmask.sum() < 1000:
        continue

    for feature in stock_features:

        auc, dauc, n = directional_auc(
            Xv.loc[rmask, feature],
            yv.loc[rmask]
        )

        print(
            f"{feature:<28} "
            f"AUC={auc:.4f} "
            f"Directional={dauc:.4f}"
        )

        results.append({
            "regime": regime_name,
            "feature": feature,
            "auc": auc,
            "directional_auc": dauc,
            "valid_rows": n,
        })

# ---------------------------------------------------------------------------
# BUILD TABLE
# ---------------------------------------------------------------------------
results_df = pd.DataFrame(results)

matrix = results_df.pivot(
    index="feature",
    columns="regime",
    values="directional_auc"
)

print("\n" + "=" * 70)
print("REGIME-CONDITIONED SIGNAL MATRIX")
print("=" * 70)

print(
    matrix.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)

# ---------------------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------------------
results_path = (
    f"{REPORT_DIR}/cell53_regime_conditioned_stock_signals.csv"
)

matrix_path = (
    f"{REPORT_DIR}/cell53_regime_conditioned_signal_matrix.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

matrix.to_csv(
    matrix_path
)

print("\nSaved:")
print(results_path)
print(matrix_path)

print("\n" + "=" * 70)
print("CELL 53 COMPLETE")
print("=" * 70)

CELL 53 — REGIME-CONDITIONED STOCK SIGNAL DIAGNOSTIC

Loading data...
Rows: 472,951

Validation period:
Rows   : 178,541
UP rate: 0.4826

BASELINE STOCK SIGNALS
close_to_sma_50              AUC=0.4893 Directional=0.5107
close_to_sma_200             AUC=0.5035 Directional=0.5035
return_20d                   AUC=0.5031 Directional=0.5031
return_60d                   AUC=0.4764 Directional=0.5236
rsi_14                       AUC=0.5237 Directional=0.5237
volatility_20d               AUC=0.4672 Directional=0.5328
volume_ratio_20              AUC=0.5301 Directional=0.5301
trading_activity_score       AUC=0.5165 Directional=0.5165

REGIME-CONDITIONED STOCK SIGNALS

----------------------------------------------------------------------
Low_Market_Volatility
----------------------------------------------------------------------
Rows: 44,702
close_to_sma_50              AUC=0.5174 Directional=0.5174
close_to_sma_200             AUC=0.5256 Directional=0.5256
return_20d                   AUC=0.43

In [ ]:
# ============================================================================
# CELL 54 — REGIME-AWARE STOCK MODEL
# ============================================================================
import os
import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    log_loss
)

print("=" * 70)
print("CELL 54 — REGIME-AWARE STOCK MODEL")
print("=" * 70)

PROJECT = "/content/NEPSE_ML_PROJECT"
MODEL_DIR = os.path.join(PROJECT, "models")
REPORT_DIR = os.path.join(PROJECT, "reports")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------------
print("\nLoading data...")

X_train = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_train.parquet"
)

X_val = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_validation.parquet"
)

y_train = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_train.parquet"
).iloc[:, 0]

y_val = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_validation.parquet"
).iloc[:, 0]

print(f"Train rows : {len(X_train):,}")
print(f"Val rows   : {len(X_val):,}")

# ---------------------------------------------------------------------------
# BASE STOCK FEATURES
# ---------------------------------------------------------------------------
stock_features = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

# ---------------------------------------------------------------------------
# MARKET / SECTOR REGIME VARIABLES
#
# These are deliberately limited to the strongest stable variables from
# Cell 51 rather than feeding all 35 context features into XGBoost.
# ---------------------------------------------------------------------------
context_features = [
    "market_volatility_60d",
    "market_volatility_20d",
    "market_close_to_sma_200",
    "market_return_60d",
    "sector_return_60d",
    "sector_breadth_20d",
    "sector_volatility_60d",
]

missing = [
    c for c in stock_features + context_features
    if c not in X_train.columns
]

if missing:
    raise ValueError(
        "Missing features:\n" +
        "\n".join(missing)
    )

# ---------------------------------------------------------------------------
# COPY ONLY REQUIRED COLUMNS
# ---------------------------------------------------------------------------
Xtr = X_train[
    stock_features + context_features
].copy()

Xv = X_val[
    stock_features + context_features
].copy()

# ---------------------------------------------------------------------------
# REGIME INTERACTIONS
#
# Continuous interactions are preferable to hard thresholds.
#
# Each interaction means:
#
#   stock signal × market/sector regime
#
# This gives the model information about WHEN a stock signal is stronger.
# ---------------------------------------------------------------------------

def add_regime_interactions(df):

    df = df.copy()

    # ---------------------------------------------------------------
    # Market volatility interactions
    # ---------------------------------------------------------------
    df["regime_vol60_x_sma50"] = (
        df["market_volatility_60d"] *
        df["close_to_sma_50"]
    )

    df["regime_vol60_x_sma200"] = (
        df["market_volatility_60d"] *
        df["close_to_sma_200"]
    )

    df["regime_vol60_x_return20"] = (
        df["market_volatility_60d"] *
        df["return_20d"]
    )

    df["regime_vol60_x_volatility20"] = (
        df["market_volatility_60d"] *
        df["volatility_20d"]
    )

    # ---------------------------------------------------------------
    # Market long-term trend interactions
    # ---------------------------------------------------------------
    df["regime_markettrend_x_return60"] = (
        df["market_close_to_sma_200"] *
        df["return_60d"]
    )

    df["regime_markettrend_x_rsi"] = (
        df["market_close_to_sma_200"] *
        df["rsi_14"]
    )

    df["regime_markettrend_x_volume"] = (
        df["market_close_to_sma_200"] *
        df["volume_ratio_20"]
    )

    # ---------------------------------------------------------------
    # Sector regime interactions
    # ---------------------------------------------------------------
    df["regime_sectorreturn_x_return60"] = (
        df["sector_return_60d"] *
        df["return_60d"]
    )

    df["regime_sectorreturn_x_sma50"] = (
        df["sector_return_60d"] *
        df["close_to_sma_50"]
    )

    df["regime_sectorbreadth_x_volume"] = (
        df["sector_breadth_20d"] *
        df["volume_ratio_20"]
    )

    df["regime_sectorvol_x_volatility"] = (
        df["sector_volatility_60d"] *
        df["volatility_20d"]
    )

    return df

Xtr = add_regime_interactions(Xtr)
Xv = add_regime_interactions(Xv)

# ---------------------------------------------------------------------------
# CLEAN NUMERIC VALUES
# ---------------------------------------------------------------------------
for df in [Xtr, Xv]:

    df.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

# Training-only medians
medians = Xtr.median()

Xtr = Xtr.fillna(medians)
Xv = Xv.fillna(medians)

print("\nFeature counts:")
print(f"Stock features       : {len(stock_features)}")
print(f"Context features     : {len(context_features)}")
print(f"Regime interactions  : {len(Xtr.columns) - len(stock_features) - len(context_features)}")
print(f"TOTAL                 : {len(Xtr.columns)}")

# ---------------------------------------------------------------------------
# MODEL
# ---------------------------------------------------------------------------
print("\nTraining regime-aware XGBoost...")

model = xgb.XGBClassifier(
    n_estimators=600,
    learning_rate=0.025,
    max_depth=3,
    min_child_weight=40,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=1.0,
    reg_lambda=12.0,
    gamma=0.10,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    device="cuda",
    random_state=42,
    n_jobs=-1
)

start = time.time()

model.fit(
    Xtr,
    y_train
)

train_prob = model.predict_proba(Xtr)[:, 1]
val_prob = model.predict_proba(Xv)[:, 1]

train_pred = (
    train_prob >= 0.5
).astype(int)

val_pred = (
    val_prob >= 0.5
).astype(int)

# ---------------------------------------------------------------------------
# METRICS
# ---------------------------------------------------------------------------
train_auc = roc_auc_score(
    y_train,
    train_prob
)

val_auc = roc_auc_score(
    y_val,
    val_prob
)

val_acc = accuracy_score(
    y_val,
    val_pred
)

val_bal = balanced_accuracy_score(
    y_val,
    val_pred
)

val_loss = log_loss(
    y_val,
    val_prob
)

gap = train_auc - val_auc

minutes = (
    time.time() - start
) / 60

print("\n" + "=" * 70)
print("RESULT")
print("=" * 70)

print(f"Train AUC       : {train_auc:.4f}")
print(f"Validation AUC  : {val_auc:.4f}")
print(f"AUC gap         : {gap:.4f}")
print(f"Validation acc  : {val_acc:.4f}")
print(f"Validation bal. : {val_bal:.4f}")
print(f"Validation loss : {val_loss:.4f}")
print(f"Training time   : {minutes:.3f} min")

# ---------------------------------------------------------------------------
# COMPARE WITH PREVIOUS BEST
# ---------------------------------------------------------------------------
previous_auc = 0.5384

print("\n" + "-" * 70)
print("COMPARISON WITH CELL 52 STOCK-ONLY")
print("-" * 70)

print(
    f"Previous Stock-only AUC : {previous_auc:.4f}"
)

print(
    f"Regime-aware AUC        : {val_auc:.4f}"
)

print(
    f"Change                  : "
    f"{val_auc - previous_auc:+.4f}"
)

# ---------------------------------------------------------------------------
# FEATURE IMPORTANCE
# ---------------------------------------------------------------------------
importance = pd.DataFrame({
    "feature": Xtr.columns,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\n" + "=" * 70)
print("TOP REGIME-AWARE FEATURES")
print("=" * 70)

print(
    importance.head(25).to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ---------------------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------------------
model_path = (
    f"{MODEL_DIR}/cell54_regime_aware_xgboost.json"
)

metrics_path = (
    f"{REPORT_DIR}/cell54_regime_aware_metrics.json"
)

importance_path = (
    f"{REPORT_DIR}/cell54_regime_aware_feature_importance.csv"
)

import json

model.save_model(
    model_path
)

with open(
    metrics_path,
    "w"
) as f:

    json.dump(
        {
            "model": "cell54_regime_aware_xgboost",
            "train_auc": float(train_auc),
            "validation_auc": float(val_auc),
            "auc_gap": float(gap),
            "validation_accuracy": float(val_acc),
            "validation_balanced_accuracy": float(val_bal),
            "validation_logloss": float(val_loss),
            "feature_count": int(len(Xtr.columns)),
            "training_minutes": float(minutes),
        },
        f,
        indent=2
    )

importance.to_csv(
    importance_path,
    index=False
)

print("\nSaved:")
print(model_path)
print(metrics_path)
print(importance_path)

print("\n" + "=" * 70)
print("CELL 54 COMPLETE")
print("=" * 70)

CELL 54 — REGIME-AWARE STOCK MODEL

Loading data...
Train rows : 294,410
Val rows   : 178,541

Feature counts:
Stock features       : 13
Context features     : 7
Regime interactions  : 11
TOTAL                 : 31

Training regime-aware XGBoost...

RESULT
Train AUC       : 0.8093
Validation AUC  : 0.5106
AUC gap         : 0.2987
Validation acc  : 0.4982
Validation bal. : 0.4940
Validation loss : 0.7407
Training time   : 0.081 min

----------------------------------------------------------------------
COMPARISON WITH CELL 52 STOCK-ONLY
----------------------------------------------------------------------
Previous Stock-only AUC : 0.5384
Regime-aware AUC        : 0.5106
Change                  : -0.0278

TOP REGIME-AWARE FEATURES
                       feature  importance
      regime_markettrend_x_rsi    0.129349
       regime_vol60_x_return20    0.091627
         market_volatility_60d    0.077768
         market_volatility_20d    0.071813
       market_close_to_sma_200    0.069037
  

In [ ]:
# ============================================================================
# CELL 55 — 20-DAY FUTURE RETURN REGRESSION
# ============================================================================
import os
import time
import json
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from scipy.stats import spearmanr

print("=" * 70)
print("CELL 55 — 20-DAY FUTURE RETURN REGRESSION")
print("=" * 70)

PROJECT = "/content/NEPSE_ML_PROJECT"
MODEL_DIR = os.path.join(PROJECT, "models")
REPORT_DIR = os.path.join(PROJECT, "reports")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------------
print("\nLoading data...")

X_train = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_train.parquet"
)

X_val = pd.read_parquet(
    f"{PROJECT}/features/model_ready/X_validation.parquet"
)

y_train_all = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_train.parquet"
).iloc[:, 0]

y_val_all = pd.read_parquet(
    f"{PROJECT}/features/model_ready/y_validation.parquet"
).iloc[:, 0]

# ---------------------------------------------------------------------------
# LOAD FUTURE RETURN TARGET
#
# The model-ready y file contains the classification target.
# Load the clean dataset only for the continuous target.
# ---------------------------------------------------------------------------
clean_path = (
    f"{PROJECT}/features/nepse_ml_clean_dataset.parquet"
)

clean = pd.read_parquet(
    clean_path,
    columns=[
        "symbol",
        "date",
        "future_return_20d"
    ]
)

clean["date"] = pd.to_datetime(
    clean["date"]
)

train_meta = pd.read_parquet(
    f"{PROJECT}/features/model_ready/train_metadata.parquet"
)

val_meta = pd.read_parquet(
    f"{PROJECT}/features/model_ready/validation_metadata.parquet"
)

train_meta["date"] = pd.to_datetime(
    train_meta["date"]
)

val_meta["date"] = pd.to_datetime(
    val_meta["date"]
)

# ---------------------------------------------------------------------------
# ALIGN TARGET USING SYMBOL + DATE
# ---------------------------------------------------------------------------
train_keys = train_meta[
    ["symbol", "date"]
].copy()

val_keys = val_meta[
    ["symbol", "date"]
].copy()

target_lookup = clean.set_index(
    ["symbol", "date"]
)["future_return_20d"]

y_train = pd.Series(
    [
        target_lookup.get(
            (s, d),
            np.nan
        )
        for s, d in zip(
            train_keys["symbol"],
            train_keys["date"]
        )
    ],
    index=X_train.index
)

y_val = pd.Series(
    [
        target_lookup.get(
            (s, d),
            np.nan
        )
        for s, d in zip(
            val_keys["symbol"],
            val_keys["date"]
        )
    ],
    index=X_val.index
)

# ---------------------------------------------------------------------------
# VALID TARGET ROWS
# ---------------------------------------------------------------------------
train_valid = (
    y_train.notna() &
    np.isfinite(y_train.to_numpy())
)

val_valid = (
    y_val.notna() &
    np.isfinite(y_val.to_numpy())
)

Xtr_base = X_train.loc[
    train_valid
].copy()

Xv_base = X_val.loc[
    val_valid
].copy()

ytr = y_train.loc[
    train_valid
].astype(float)

yv = y_val.loc[
    val_valid
].astype(float)

print(f"\nTrain usable rows : {len(ytr):,}")
print(f"Val usable rows   : {len(yv):,}")

print(
    f"Train mean return : {ytr.mean():.6f}"
)

print(
    f"Val mean return   : {yv.mean():.6f}"
)

print(
    f"Train median      : {ytr.median():.6f}"
)

print(
    f"Val median        : {yv.median():.6f}"
)

# ---------------------------------------------------------------------------
# FEATURE GROUPS
# ---------------------------------------------------------------------------
stock_features = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

market_features = [
    "market_volatility_60d",
    "market_volatility_20d",
    "market_volatility_10d",
    "market_close_to_sma_200",
    "market_return_60d",
    "market_return_20d",
    "market_bear_trend",
]

sector_features = [
    "sector_return_60d",
    "sector_return_20d",
    "sector_volatility_60d",
    "sector_breadth_20d",
    "sector_strength_20d",
]

# Three controlled experiments.
feature_sets = {
    "Stock_Only": stock_features,

    "Stock_Market": (
        stock_features +
        market_features
    ),

    "Stock_Market_Sector": (
        stock_features +
        market_features +
        sector_features
    ),
}

# ---------------------------------------------------------------------------
# METRIC FUNCTION
# ---------------------------------------------------------------------------
def evaluate_regression(
    name,
    model,
    Xtr,
    ytr,
    Xv,
    yv
):

    train_pred = model.predict(Xtr)
    val_pred = model.predict(Xv)

    train_mae = mean_absolute_error(
        ytr,
        train_pred
    )

    val_mae = mean_absolute_error(
        yv,
        val_pred
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            ytr,
            train_pred
        )
    )

    val_rmse = np.sqrt(
        mean_squared_error(
            yv,
            val_pred
        )
    )

    train_r2 = r2_score(
        ytr,
        train_pred
    )

    val_r2 = r2_score(
        yv,
        val_pred
    )

    train_spear = spearmanr(
        ytr,
        train_pred
    ).statistic

    val_spear = spearmanr(
        yv,
        val_pred
    ).statistic

    # Directional AUC from predicted return.
    from sklearn.metrics import roc_auc_score

    yv_direction = (
        yv > 0
    ).astype(int)

    val_auc = roc_auc_score(
        yv_direction,
        val_pred
    )

    return {
        "model": name,
        "features": len(Xtr.columns),
        "train_mae": train_mae,
        "validation_mae": val_mae,
        "train_rmse": train_rmse,
        "validation_rmse": val_rmse,
        "train_r2": train_r2,
        "validation_r2": val_r2,
        "train_spearman": train_spear,
        "validation_spearman": val_spear,
        "validation_direction_auc": val_auc,
    }

# ---------------------------------------------------------------------------
# TRAIN
# ---------------------------------------------------------------------------
results = []
models = {}

for name, features in feature_sets.items():

    print("\n" + "-" * 70)
    print(f"MODEL: {name}")
    print("-" * 70)

    Xtr = Xtr_base[
        features
    ].copy()

    Xv = Xv_base[
        features
    ].copy()

    # Training-only medians.
    medians = Xtr.median()

    Xtr = Xtr.fillna(
        medians
    )

    Xv = Xv.fillna(
        medians
    )

    model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.025,
        max_depth=3,
        min_child_weight=40,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_alpha=1.0,
        reg_lambda=15.0,
        gamma=0.10,
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        device="cuda",
        random_state=42,
        n_jobs=-1
    )

    start = time.time()

    model.fit(
        Xtr,
        ytr
    )

    elapsed = (
        time.time() - start
    ) / 60

    metrics = evaluate_regression(
        name,
        model,
        Xtr,
        ytr,
        Xv,
        yv
    )

    metrics["minutes"] = elapsed

    results.append(
        metrics
    )

    models[name] = model

    print(
        f"Train Spearman       : "
        f"{metrics['train_spearman']:.4f}"
    )

    print(
        f"Validation Spearman  : "
        f"{metrics['validation_spearman']:.4f}"
    )

    print(
        f"Validation R²        : "
        f"{metrics['validation_r2']:.4f}"
    )

    print(
        f"Validation Dir. AUC  : "
        f"{metrics['validation_direction_auc']:.4f}"
    )

    print(
        f"Validation MAE       : "
        f"{metrics['validation_mae']:.6f}"
    )

    print(
        f"Time                 : "
        f"{elapsed:.3f} min"
    )

# ---------------------------------------------------------------------------
# RESULTS
# ---------------------------------------------------------------------------
results_df = pd.DataFrame(
    results
)

print("\n" + "=" * 70)
print("RETURN REGRESSION COMPARISON")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ---------------------------------------------------------------------------
# BEST BY VALIDATION SPEARMAN
# ---------------------------------------------------------------------------
best_idx = results_df[
    "validation_spearman"
].idxmax()

best_name = results_df.loc[
    best_idx,
    "model"
]

best_model = models[
    best_name
]

print("\n" + "-" * 70)
print("BEST RETURN MODEL")
print("-" * 70)

print(
    f"Model: {best_name}"
)

print(
    f"Validation Spearman: "
    f"{results_df.loc[best_idx, 'validation_spearman']:.4f}"
)

print(
    f"Validation Direction AUC: "
    f"{results_df.loc[best_idx, 'validation_direction_auc']:.4f}"
)

# ---------------------------------------------------------------------------
# FEATURE IMPORTANCE
# ---------------------------------------------------------------------------
best_features = feature_sets[
    best_name
]

importance = pd.DataFrame({
    "feature": best_features,
    "importance": best_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\n" + "=" * 70)
print("TOP RETURN MODEL FEATURES")
print("=" * 70)

print(
    importance.head(25).to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ---------------------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------------------
results_path = (
    f"{REPORT_DIR}/cell55_return_regression_comparison.csv"
)

importance_path = (
    f"{REPORT_DIR}/cell55_return_regression_feature_importance.csv"
)

model_path = (
    f"{MODEL_DIR}/cell55_best_return_xgboost.json"
)

metrics_path = (
    f"{REPORT_DIR}/cell55_best_return_model_metrics.json"
)

results_df.to_csv(
    results_path,
    index=False
)

importance.to_csv(
    importance_path,
    index=False
)

best_model.save_model(
    model_path
)

with open(
    metrics_path,
    "w"
) as f:

    json.dump(
        {
            "model": best_name,
            "validation_spearman": float(
                results_df.loc[
                    best_idx,
                    "validation_spearman"
                ]
            ),
            "validation_direction_auc": float(
                results_df.loc[
                    best_idx,
                    "validation_direction_auc"
                ]
            ),
            "validation_r2": float(
                results_df.loc[
                    best_idx,
                    "validation_r2"
                ]
            ),
            "validation_mae": float(
                results_df.loc[
                    best_idx,
                    "validation_mae"
                ]
            ),
            "validation_rmse": float(
                results_df.loc[
                    best_idx,
                    "validation_rmse"
                ]
            ),
        },
        f,
        indent=2
    )

print("\nSaved:")
print(results_path)
print(importance_path)
print(model_path)
print(metrics_path)

print("\n" + "=" * 70)
print("CELL 55 COMPLETE")
print("=" * 70)

CELL 55 — 20-DAY FUTURE RETURN REGRESSION

Loading data...

Train usable rows : 294,410
Val usable rows   : 178,541
Train mean return : 0.036578
Val mean return   : 0.019728
Train median      : 0.005861
Val median        : -0.002918

----------------------------------------------------------------------
MODEL: Stock_Only
----------------------------------------------------------------------
Train Spearman       : 0.2113
Validation Spearman  : 0.0342
Validation R²        : 0.0539
Validation Dir. AUC  : 0.5256
Validation MAE       : 0.095243
Time                 : 0.034 min

----------------------------------------------------------------------
MODEL: Stock_Market
----------------------------------------------------------------------
Train Spearman       : 0.5313
Validation Spearman  : -0.1382
Validation R²        : -0.1264
Validation Dir. AUC  : 0.4487
Validation MAE       : 0.107295
Time                 : 0.042 min

----------------------------------------------------------------------

In [ ]:
# ============================================================================
# CELL 56 — PREPARE HUGGING FACE DATASET PACKAGE
# ============================================================================
# Creates a clean, self-contained package for the NEPSE ML project.
# Does NOT upload anything yet.
# ============================================================================

import os
import shutil
import json
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/NEPSE_ML_PROJECT")
HF_PACKAGE = PROJECT / "huggingface_dataset"

print("=" * 78)
print("CELL 56 — PREPARING HUGGING FACE DATASET PACKAGE")
print("=" * 78)

# ---------------------------------------------------------------------------
# 1. Clean/recreate package directory
# ---------------------------------------------------------------------------

if HF_PACKAGE.exists():
    shutil.rmtree(HF_PACKAGE)

HF_PACKAGE.mkdir(parents=True, exist_ok=True)

for folder in [
    "dataset",
    "model_ready",
    "metadata",
    "reports"
]:
    (HF_PACKAGE / folder).mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# 2. Verify canonical clean dataset
# ---------------------------------------------------------------------------

clean_dataset = PROJECT / "features/nepse_ml_clean_dataset.parquet"

if not clean_dataset.exists():
    raise FileNotFoundError(
        f"Canonical clean dataset not found:\n{clean_dataset}"
    )

print("\nCanonical dataset:")
print(clean_dataset)
print(f"Size: {clean_dataset.stat().st_size / (1024**2):.2f} MB")

# ---------------------------------------------------------------------------
# 3. Copy canonical dataset
# ---------------------------------------------------------------------------

shutil.copy2(
    clean_dataset,
    HF_PACKAGE / "dataset/nepse_ml_clean_dataset.parquet"
)

# ---------------------------------------------------------------------------
# 4. Copy model-ready matrices
# ---------------------------------------------------------------------------

MODEL_READY = PROJECT / "features/model_ready"

required_model_files = [
    "X_train.parquet",
    "X_validation.parquet",
    "X_test.parquet",
    "y_train.parquet",
    "y_validation.parquet",
    "y_test.parquet",
    "train_metadata.parquet",
    "validation_metadata.parquet",
    "test_metadata.parquet",
    "feature_medians.csv",
    "model_ready_manifest.json",
]

print("\nChecking model-ready files...")

for filename in required_model_files:
    source = MODEL_READY / filename

    if not source.exists():
        raise FileNotFoundError(
            f"Missing model-ready file:\n{source}"
        )

    shutil.copy2(
        source,
        HF_PACKAGE / "model_ready" / filename
    )

    print("  OK:", filename)

# ---------------------------------------------------------------------------
# 5. Copy important feature documentation
# ---------------------------------------------------------------------------

metadata_files = [
    "ml_feature_columns.csv",
    "ml_feature_inventory.csv",
]

for filename in metadata_files:

    source = PROJECT / "validation" / filename

    if source.exists():
        shutil.copy2(
            source,
            HF_PACKAGE / "metadata" / filename
        )
        print("  Copied:", filename)

# ---------------------------------------------------------------------------
# 6. Copy important modeling reports
# ---------------------------------------------------------------------------

report_files = [
    "cell47_stock_model_comparison.csv",
    "cell48_technical_only_model_comparison.csv",
    "cell49_technical_walk_forward_results.csv",
    "cell55_return_regression_comparison.csv",
    "cell55_return_regression_feature_importance.csv",
    "cell55_best_return_model_metrics.json",
]

for filename in report_files:

    source = PROJECT / "reports" / filename

    if source.exists():
        shutil.copy2(
            source,
            HF_PACKAGE / "reports" / filename
        )
        print("  Copied:", filename)

# ---------------------------------------------------------------------------
# 7. Create package manifest
# ---------------------------------------------------------------------------

clean_df = pd.read_parquet(clean_dataset)

manifest = {
    "project": "NEPSE ML Research Dataset",
    "dataset_version": "v1.0",
    "created_from": "NepseAlpha historical data",
    "canonical_dataset": {
        "file": "dataset/nepse_ml_clean_dataset.parquet",
        "rows": int(len(clean_df)),
        "columns": int(len(clean_df.columns)),
        "symbols": int(clean_df["symbol"].nunique()),
        "min_date": str(clean_df["date"].min()),
        "max_date": str(clean_df["date"].max()),
    },
    "primary_prediction_target": "future_return_20d",
    "primary_classification_target": "target_up_20d",
    "split": {
        "train": "2011-2021",
        "validation": "2022-2024",
        "test": "2025-2026",
    },
    "model_policy": {
        "stock_model_uses_market_features": False,
        "stock_model_uses_sector_features": False,
        "market_sector_data_retained_separately": True,
    },
    "important_note": (
        "The 2025-2026 test period is intentionally preserved as an "
        "untouched final evaluation period."
    ),
}

with open(
    HF_PACKAGE / "metadata/dataset_manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

# ---------------------------------------------------------------------------
# 8. Generate a README
# ---------------------------------------------------------------------------

readme = """# NEPSE ML Research Dataset

Research-ready historical NEPSE dataset prepared for machine-learning
experiments.

## Canonical dataset

`dataset/nepse_ml_clean_dataset.parquet`

Contains the cleaned stock-level feature and target dataset.

## Primary targets

- `future_return_20d`
- `target_up_20d`

## Chronological split

- Train: 2011–2021
- Validation: 2022–2024
- Final test: 2025–2026

The final test period must remain untouched until the model is finalized.

## Current modeling policy

The stock prediction model uses stock-specific technical, volume and
liquidity information.

Market and sector datasets were investigated separately but are NOT used
as direct features in the final stock model.

## Important

This dataset is a research dataset derived from historical NEPSE/NepseAlpha
data. Historical data contains limitations and should not be interpreted as
investment advice.
"""

with open(
    HF_PACKAGE / "README.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(readme)

# ---------------------------------------------------------------------------
# 9. Final package inventory
# ---------------------------------------------------------------------------

print("\n" + "=" * 78)
print("PACKAGE INVENTORY")
print("=" * 78)

total_size = 0
file_count = 0

for path in sorted(HF_PACKAGE.rglob("*")):

    if path.is_file():

        size_mb = path.stat().st_size / (1024**2)
        total_size += path.stat().st_size
        file_count += 1

        print(
            f"{size_mb:9.2f} MB  "
            f"{path.relative_to(HF_PACKAGE)}"
        )

print("-" * 78)
print(f"Files       : {file_count}")
print(f"Total size  : {total_size / (1024**2):.2f} MB")

print("\n" + "=" * 78)
print("CELL 56 COMPLETE")
print("=" * 78)
print("\nPackage ready:")
print(HF_PACKAGE)
print("\nNothing has been uploaded yet.")

CELL 56 — PREPARING HUGGING FACE DATASET PACKAGE

Canonical dataset:
/content/NEPSE_ML_PROJECT/features/nepse_ml_clean_dataset.parquet
Size: 318.00 MB

Checking model-ready files...
  OK: X_train.parquet
  OK: X_validation.parquet
  OK: X_test.parquet
  OK: y_train.parquet
  OK: y_validation.parquet
  OK: y_test.parquet
  OK: train_metadata.parquet
  OK: validation_metadata.parquet
  OK: test_metadata.parquet
  OK: feature_medians.csv
  OK: model_ready_manifest.json
  Copied: ml_feature_columns.csv
  Copied: ml_feature_inventory.csv
  Copied: cell47_stock_model_comparison.csv
  Copied: cell48_technical_only_model_comparison.csv
  Copied: cell49_technical_walk_forward_results.csv
  Copied: cell55_return_regression_comparison.csv
  Copied: cell55_return_regression_feature_importance.csv
  Copied: cell55_best_return_model_metrics.json

PACKAGE INVENTORY
     0.00 MB  README.md
   318.00 MB  dataset/nepse_ml_clean_dataset.parquet
     0.00 MB  metadata/dataset_manifest.json
     0.00 MB  m

In [ ]:
# ============================================================================
# CELL 57 — UPLOAD NEPSE ML DATASET TO HUGGING FACE
# ============================================================================
# Public Hugging Face Dataset
# Repository:
# https://huggingface.co/datasets/mukeshpokharel/nepse-ml-research-dataset
# ============================================================================

!pip -q install -U huggingface_hub

from huggingface_hub import login, HfApi
from pathlib import Path

# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------

REPO_ID = "mukeshpokharel/nepse-ml-research-dataset"

HF_PACKAGE = Path(
    "/content/NEPSE_ML_PROJECT/huggingface_dataset"
)

# ---------------------------------------------------------------------------
# VERIFY LOCAL PACKAGE
# ---------------------------------------------------------------------------

print("=" * 78)
print("CELL 57 — HUGGING FACE DATASET UPLOAD")
print("=" * 78)

if not HF_PACKAGE.exists():
    raise FileNotFoundError(
        f"Hugging Face package not found:\n{HF_PACKAGE}"
    )

files = [
    p for p in HF_PACKAGE.rglob("*")
    if p.is_file()
]

total_bytes = sum(p.stat().st_size for p in files)

print("\nLocal package:")
print(HF_PACKAGE)

print(f"\nFiles      : {len(files)}")
print(f"Total size : {total_bytes / (1024**2):.2f} MB")

# ---------------------------------------------------------------------------
# LOGIN
# ---------------------------------------------------------------------------

print("\n" + "-" * 78)
print("HUGGING FACE LOGIN")
print("-" * 78)

print("\nA Hugging Face login window/widget will appear.")
print("Use your Hugging Face account: mukeshpokharel")

login()

# ---------------------------------------------------------------------------
# VERIFY AUTHENTICATION
# ---------------------------------------------------------------------------

api = HfApi()

me = api.whoami()

print("\nAuthenticated successfully.")

username = me.get("name", "")

print("Account:", username)

if username.lower() != "mukeshpokharel":
    raise RuntimeError(
        f"Logged into '{username}', but expected 'mukeshpokharel'. "
        "Please log into the correct Hugging Face account."
    )

# ---------------------------------------------------------------------------
# CREATE PUBLIC DATASET REPOSITORY
# ---------------------------------------------------------------------------

print("\n" + "-" * 78)
print("CREATING / VERIFYING PUBLIC DATASET REPOSITORY")
print("-" * 78)

api.create_repo(
    repo_id=REPO_ID,
    repo_type="dataset",
    private=False,
    exist_ok=True,
)

print("\nPublic dataset repository ready:")
print(
    f"https://huggingface.co/datasets/{REPO_ID}"
)

# ---------------------------------------------------------------------------
# UPLOAD
# ---------------------------------------------------------------------------

print("\n" + "-" * 78)
print("STARTING UPLOAD")
print("-" * 78)

print("""
Approximately 592 MB will be uploaded.

This can take some time.

If the upload is interrupted:
    simply rerun this same cell.

Hugging Face will detect files/chunks that are already present
and avoid unnecessarily uploading them again.
""")

result = api.upload_folder(
    folder_path=str(HF_PACKAGE),
    repo_id=REPO_ID,
    repo_type="dataset",
    path_in_repo=".",
    commit_message="Add NEPSE ML research dataset v1.0",
)

# ---------------------------------------------------------------------------
# COMPLETE
# ---------------------------------------------------------------------------

print("\n" + "=" * 78)
print("CELL 57 COMPLETE")
print("=" * 78)

print("\nPUBLIC DATASET:")
print(
    f"https://huggingface.co/datasets/{REPO_ID}"
)

if hasattr(result, "commit_url"):
    print("\nCommit:")
    print(result.commit_url)

print("\nUploaded files:")
for p in files:
    print("  ", p.relative_to(HF_PACKAGE))

print("\nYou can now use this dataset as the permanent remote copy.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.0/838.0 kB 13.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.1.1 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.1.1 which is incompatible.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which is incompatible.
gradio-client 2.7.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.1.1 which is incompatible.
CELL 57 — HUGGING FACE DATASET UPLOAD

Local package:
/content/NEPSE_ML_PROJECT/huggingface_dataset

Files      : 22
Total size : 592.25 MB

------------------------------------------------------------------------------
HUGGING FACE LOGIN
-----------------------------------------------


Authenticated successfully.
Account: mukeshpokharel

------------------------------------------------------------------------------
CREATING / VERIFYING PUBLIC DATASET REPOSITORY
------------------------------------------------------------------------------

Public dataset repository ready:
https://huggingface.co/datasets/mukeshpokharel/nepse-ml-research-dataset

------------------------------------------------------------------------------
STARTING UPLOAD
------------------------------------------------------------------------------

Approximately 592 MB will be uploaded.

This can take some time.

If the upload is interrupted:
    simply rerun this same cell.

Hugging Face will detect files/chunks that are already present
and avoid unnecessarily uploading them again.



/usr/local/lib/python3.13/dist-packages/huggingface_hub/hf_api.py:11464: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")



CELL 57 COMPLETE

PUBLIC DATASET:
https://huggingface.co/datasets/mukeshpokharel/nepse-ml-research-dataset

Commit:
https://huggingface.co/datasets/mukeshpokharel/nepse-ml-research-dataset/commit/d950b3f02092d660d8fd4be97b859d17c37577f1

Uploaded files:
   README.md
   dataset/nepse_ml_clean_dataset.parquet
   model_ready/X_train.parquet
   model_ready/feature_medians.csv
   model_ready/y_validation.parquet
   model_ready/y_test.parquet
   model_ready/X_test.parquet
   model_ready/model_ready_manifest.json
   model_ready/y_train.parquet
   model_ready/validation_metadata.parquet
   model_ready/train_metadata.parquet
   model_ready/test_metadata.parquet
   model_ready/X_validation.parquet
   metadata/ml_feature_inventory.csv
   metadata/dataset_manifest.json
   metadata/ml_feature_columns.csv
   reports/cell55_return_regression_feature_importance.csv
   reports/cell48_technical_only_model_comparison.csv
   reports/cell55_return_regression_comparison.csv
   reports/cell55_best_return_mo

In [ ]:
# ============================================================================
# CELL 57B — UPDATE PUBLIC HUGGING FACE README
# ============================================================================

!pip -q install "huggingface_hub<2.0"

from huggingface_hub import HfApi, login
from pathlib import Path

REPO_ID = "mukeshpokharel/nepse-ml-research-dataset"

README_PATH = Path(
    "/content/NEPSE_ML_PROJECT/huggingface_dataset/README.md"
)

# ---------------------------------------------------------------------------
# WRITE NEUTRAL README
# ---------------------------------------------------------------------------

README_TEXT = """# NEPSE ML Research Dataset

A historical dataset prepared for machine-learning research and quantitative
analysis of Nepal's stock market.

## Contents

- Historical stock-level features
- Technical and trading features
- Prediction targets
- Chronological train/validation/test splits
- Model-ready datasets
- Research reports and feature metadata

## Primary targets

- `future_return_20d`
- `target_up_20d`

## Chronological split

- Train: 2011–2021
- Validation: 2022–2024
- Test: 2025–2026

The final test period is reserved for out-of-sample evaluation.

## Usage

This dataset is provided for research, educational, and quantitative
analysis purposes.

Historical market data may contain limitations and should be independently
validated before use.

This dataset does not constitute financial or investment advice.
"""

README_PATH.write_text(
    README_TEXT,
    encoding="utf-8"
)

print("README updated locally.")

# ---------------------------------------------------------------------------
# LOGIN
# ---------------------------------------------------------------------------

login()

api = HfApi()

# ---------------------------------------------------------------------------
# PUSH ONLY README
# ---------------------------------------------------------------------------

result = api.upload_file(
    path_or_fileobj=str(README_PATH),
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="dataset",
    commit_message="Update dataset description",
)

print("\n" + "=" * 78)
print("README UPDATE COMPLETE")
print("=" * 78)

print("\nDataset:")
print(
    "https://huggingface.co/datasets/"
    + REPO_ID
)

if hasattr(result, "commit_url"):
    print("\nCommit:")
    print(result.commit_url)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 14.0 MB/s eta 0:00:00
README updated locally.

README UPDATE COMPLETE

Dataset:
https://huggingface.co/datasets/mukeshpokharel/nepse-ml-research-dataset

Commit:
https://huggingface.co/datasets/mukeshpokharel/nepse-ml-research-dataset/commit/e85b135d8fb772083d6292c4cce0e6c7f756f4c3


In [ ]:
# ============================================================================
# CELL 58 — STOCK-ONLY CROSS-SECTIONAL RANKING BASELINE
# ============================================================================

import os
import time
import json
import numpy as np
import pandas as pd
import xgboost as xgb

from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

PROJECT = "/content/NEPSE_ML_PROJECT"

DATASET = os.path.join(
    PROJECT,
    "features/nepse_ml_clean_dataset.parquet"
)

REPORTS = os.path.join(PROJECT, "reports")
MODELS = os.path.join(PROJECT, "models")

os.makedirs(REPORTS, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

print("=" * 78)
print("CELL 58 — STOCK-ONLY CROSS-SECTIONAL RANKING")
print("=" * 78)

# ============================================================================
# 1. LOAD DATA
# ============================================================================

print("\nLoading clean dataset...")

df = pd.read_parquet(DATASET)
df["date"] = pd.to_datetime(df["date"])

print("Rows:", len(df))
print("Symbols:", df["symbol"].nunique())
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

# ============================================================================
# 2. STOCK-ONLY FEATURES
# ============================================================================

stock_features = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

# Explicit protection against accidentally using context features.
bad_context = [
    c for c in stock_features
    if c.startswith("market_")
    or c.startswith("sector_")
    or c.startswith("relative_")
]

if bad_context:
    raise RuntimeError(
        f"Unexpected market/sector/relative features: {bad_context}"
    )

missing = [c for c in stock_features if c not in df.columns]

if missing:
    raise RuntimeError(f"Missing features: {missing}")

print("\nStock features:", len(stock_features))

# ============================================================================
# 3. CHRONOLOGICAL TRAIN / VALIDATION SPLIT
# ============================================================================

train = df[
    (df["date"] >= "2011-01-01") &
    (df["date"] <= "2021-12-31")
].copy()

val = df[
    (df["date"] >= "2022-01-01") &
    (df["date"] <= "2024-12-31")
].copy()

train = train[train["future_return_20d"].notna()].copy()
val = val[val["future_return_20d"].notna()].copy()

print("\nTraining rows   :", len(train))
print("Validation rows :", len(val))

# ============================================================================
# 4. TRAINING-ONLY IMPUTATION
# ============================================================================

train_medians = train[stock_features].median()

X_train = (
    train[stock_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(train_medians)
)

X_val = (
    val[stock_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(train_medians)
)

y_train = train["future_return_20d"].astype(float).values
y_val = val["future_return_20d"].astype(float).values

print("\nX_train:", X_train.shape)
print("X_val  :", X_val.shape)

# ============================================================================
# 5. TRAIN MODEL
# ============================================================================

print("\n" + "-" * 78)
print("TRAINING")
print("-" * 78)

model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.025,
    max_depth=3,
    min_child_weight=30,
    subsample=0.80,
    colsample_bytree=0.85,
    reg_alpha=1.0,
    reg_lambda=10.0,
    gamma=0.10,
    objective="reg:squarederror",
    eval_metric="rmse",
    tree_method="hist",
    device="cuda",
    random_state=42,
    n_jobs=-1
)

start = time.time()

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

minutes = (time.time() - start) / 60

# ============================================================================
# 6. PREDICTIONS
# ============================================================================

train_pred = model.predict(X_train)
val_pred = model.predict(X_val)

train_spearman = spearmanr(
    train_pred, y_train
).statistic

val_spearman = spearmanr(
    val_pred, y_val
).statistic

val_auc = roc_auc_score(
    (y_val > 0).astype(int),
    val_pred
)

print("\nTraining time       :", round(minutes, 3), "minutes")
print("Train Spearman      :", round(train_spearman, 4))
print("Validation Spearman :", round(val_spearman, 4))
print("Validation Dir. AUC :", round(val_auc, 4))

# ============================================================================
# 7. DAILY CROSS-SECTIONAL RANKING
# ============================================================================

ranking = val[
    ["symbol", "date", "future_return_20d"]
].copy()

ranking["predicted_return"] = val_pred

ranking["daily_rank"] = (
    ranking
    .groupby("date")["predicted_return"]
    .rank(method="first", ascending=False)
)

ranking["daily_count"] = (
    ranking
    .groupby("date")["symbol"]
    .transform("count")
)

ranking["daily_percentile"] = (
    1.0 -
    (
        (ranking["daily_rank"] - 1) /
        (ranking["daily_count"] - 1).replace(0, np.nan)
    )
)

# ============================================================================
# 8. TOP-N PERCENTILE PERFORMANCE
# ============================================================================

results = []

for top_pct in [0.05, 0.10, 0.20, 0.30]:

    selected = ranking[
        ranking["daily_percentile"] >= (1.0 - top_pct)
    ]

    daily_returns = (
        selected
        .groupby("date")["future_return_20d"]
        .mean()
    )

    results.append({
        "top_percent": top_pct,
        "days": len(daily_returns),
        "stocks_selected_total": len(selected),
        "mean_20d_return": daily_returns.mean(),
        "median_20d_return": daily_returns.median(),
        "std_20d_return": daily_returns.std(),
        "positive_day_fraction": (
            daily_returns > 0
        ).mean(),
        "mean_predicted_return": (
            selected["predicted_return"].mean()
        ),
    })

ranking_results = pd.DataFrame(results)

print("\n" + "-" * 78)
print("TOP-RANKED STOCK PERFORMANCE")
print("-" * 78)

print(
    ranking_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ============================================================================
# 9. DECILE ANALYSIS
# ============================================================================

ranking["decile"] = pd.qcut(
    ranking["predicted_return"],
    10,
    labels=False,
    duplicates="drop"
)

decile_results = (
    ranking
    .groupby("decile")
    .agg(
        stocks=("symbol", "size"),
        mean_predicted_return=("predicted_return", "mean"),
        actual_mean_return=("future_return_20d", "mean"),
        actual_median_return=("future_return_20d", "median"),
        positive_fraction=(
            "future_return_20d",
            lambda x: (x > 0).mean()
        )
    )
    .reset_index()
)

print("\n" + "-" * 78)
print("VALIDATION DECILES")
print("-" * 78)

print(
    decile_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ============================================================================
# 10. FEATURE IMPORTANCE
# ============================================================================

importance = (
    pd.DataFrame({
        "feature": stock_features,
        "importance": model.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

print("\n" + "-" * 78)
print("FEATURE IMPORTANCE")
print("-" * 78)

print(
    importance.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ============================================================================
# 11. SAVE
# ============================================================================

ranking_results.to_csv(
    os.path.join(
        REPORTS,
        "cell58_cross_sectional_topn_results.csv"
    ),
    index=False
)

decile_results.to_csv(
    os.path.join(
        REPORTS,
        "cell58_cross_sectional_deciles.csv"
    ),
    index=False
)

importance.to_csv(
    os.path.join(
        REPORTS,
        "cell58_cross_sectional_feature_importance.csv"
    ),
    index=False
)

ranking.to_parquet(
    os.path.join(
        REPORTS,
        "cell58_validation_stock_rankings.parquet"
    ),
    index=False
)

model.save_model(
    os.path.join(
        MODELS,
        "cell58_stock_only_cross_sectional_xgboost.json"
    )
)

metrics = {
    "model": "Stock-only cross-sectional ranking",
    "features": stock_features,
    "train_rows": int(len(train)),
    "validation_rows": int(len(val)),
    "train_spearman": float(train_spearman),
    "validation_spearman": float(val_spearman),
    "validation_direction_auc": float(val_auc),
    "training_minutes": float(minutes)
}

with open(
    os.path.join(
        REPORTS,
        "cell58_cross_sectional_metrics.json"
    ),
    "w"
) as f:
    json.dump(metrics, f, indent=2)

print("\n" + "=" * 78)
print("CELL 58 COMPLETE")
print("=" * 78)

print("\nTEST SET WAS NOT USED.")
print("Market/sector features were NOT used.")

print("\nSaved:")
print("  cell58_cross_sectional_topn_results.csv")
print("  cell58_cross_sectional_deciles.csv")
print("  cell58_cross_sectional_feature_importance.csv")
print("  cell58_validation_stock_rankings.parquet")
print("  cell58_cross_sectional_metrics.json")
print("  cell58_stock_only_cross_sectional_xgboost.json")

CELL 58 — STOCK-ONLY CROSS-SECTIONAL RANKING

Loading clean dataset...
Rows: 633137
Symbols: 651
Date range: 1997-07-20 to 2026-10-06

Stock features: 13

Training rows   : 294410
Validation rows : 192775

X_train: (294410, 13)
X_val  : (192775, 13)

------------------------------------------------------------------------------
TRAINING
------------------------------------------------------------------------------

Training time       : 0.044 minutes
Train Spearman      : 0.2103
Validation Spearman : 0.0466
Validation Dir. AUC : 0.5351

------------------------------------------------------------------------------
TOP-RANKED STOCK PERFORMANCE
------------------------------------------------------------------------------
 top_percent  days  stocks_selected_total  mean_20d_return  median_20d_return  std_20d_return  positive_day_fraction  mean_predicted_return
    0.050000   701                   9971         0.097131           0.053647        0.169675               0.696148              

In [ ]:
# ============================================================================
# CELL 59 — CROSS-SECTIONAL RANKING STABILITY / WALK-FORWARD TEST
# ============================================================================
# Stock-only.
# Market/sector features excluded.
# Final 2025-2026 test set remains untouched.
# ============================================================================

import os
import time
import json
import numpy as np
import pandas as pd
import xgboost as xgb

from scipy.stats import spearmanr

PROJECT = "/content/NEPSE_ML_PROJECT"

DATASET = os.path.join(
    PROJECT,
    "features/nepse_ml_clean_dataset.parquet"
)

REPORTS = os.path.join(PROJECT, "reports")
MODELS = os.path.join(PROJECT, "models")

os.makedirs(REPORTS, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

print("=" * 78)
print("CELL 59 — CROSS-SECTIONAL RANKING STABILITY")
print("=" * 78)

# ============================================================================
# 1. LOAD DATA
# ============================================================================

df = pd.read_parquet(DATASET)

df["date"] = pd.to_datetime(df["date"])

stock_features = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

print("\nRows:", len(df))
print("Symbols:", df["symbol"].nunique())

# ============================================================================
# 2. DEFINE WALK-FORWARD FOLDS
# ============================================================================
# Each validation period is strictly after its training period.
#
# We deliberately stop at 2024.
# 2025-2026 remains untouched.

folds = [
    {
        "name": "2017-2018",
        "train_end": "2016-12-31",
        "val_start": "2017-01-01",
        "val_end": "2018-12-31",
    },
    {
        "name": "2019-2020",
        "train_end": "2018-12-31",
        "val_start": "2019-01-01",
        "val_end": "2020-12-31",
    },
    {
        "name": "2021",
        "train_end": "2020-12-31",
        "val_start": "2021-01-01",
        "val_end": "2021-12-31",
    },
    {
        "name": "2022-2023",
        "train_end": "2021-12-31",
        "val_start": "2022-01-01",
        "val_end": "2023-12-31",
    },
    {
        "name": "2024",
        "train_end": "2023-12-31",
        "val_start": "2024-01-01",
        "val_end": "2024-12-31",
    },
]

all_results = []
all_rankings = []

# ============================================================================
# 3. RUN FOLDS
# ============================================================================

for fold in folds:

    print("\n" + "=" * 78)
    print("FOLD:", fold["name"])
    print("=" * 78)

    train = df[
        df["date"] <= fold["train_end"]
    ].copy()

    val = df[
        (df["date"] >= fold["val_start"]) &
        (df["date"] <= fold["val_end"])
    ].copy()

    # Require future target.
    train = train[
        train["future_return_20d"].notna()
    ].copy()

    val = val[
        val["future_return_20d"].notna()
    ].copy()

    print("Train rows:", len(train))
    print("Val rows  :", len(val))

    # ------------------------------------------------------------
    # Training-only imputation
    # ------------------------------------------------------------

    medians = train[stock_features].median()

    X_train = (
        train[stock_features]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(medians)
    )

    X_val = (
        val[stock_features]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(medians)
    )

    y_train = train["future_return_20d"].values
    y_val = val["future_return_20d"].values

    # ------------------------------------------------------------
    # Conservative model
    # ------------------------------------------------------------

    model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.025,
        max_depth=3,
        min_child_weight=30,
        subsample=0.80,
        colsample_bytree=0.85,
        reg_alpha=1.0,
        reg_lambda=10.0,
        gamma=0.10,
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        device="cuda",
        random_state=42,
        n_jobs=-1
    )

    start = time.time()

    model.fit(
        X_train,
        y_train,
        verbose=False
    )

    minutes = (time.time() - start) / 60

    # ------------------------------------------------------------
    # Predictions
    # ------------------------------------------------------------

    pred = model.predict(X_val)

    spearman = spearmanr(
        pred,
        y_val
    ).statistic

    # ------------------------------------------------------------
    # Daily cross-sectional ranking
    # ------------------------------------------------------------

    r = val[
        ["symbol", "date", "future_return_20d"]
    ].copy()

    r["predicted_return"] = pred

    r["rank"] = (
        r.groupby("date")["predicted_return"]
        .rank(
            method="first",
            ascending=False
        )
    )

    r["count"] = (
        r.groupby("date")["symbol"]
        .transform("count")
    )

    r["percentile"] = (
        1.0 -
        (
            (r["rank"] - 1) /
            (r["count"] - 1).replace(0, np.nan)
        )
    )

    # ------------------------------------------------------------
    # Top/bottom portfolio performance
    # ------------------------------------------------------------

    top5 = r[
        r["percentile"] >= 0.95
    ]

    top10 = r[
        r["percentile"] >= 0.90
    ]

    bottom10 = r[
        r["percentile"] <= 0.10
    ]

    top20 = r[
        r["percentile"] >= 0.80
    ]

    daily_top5 = (
        top5.groupby("date")["future_return_20d"]
        .mean()
    )

    daily_top10 = (
        top10.groupby("date")["future_return_20d"]
        .mean()
    )

    daily_bottom10 = (
        bottom10.groupby("date")["future_return_20d"]
        .mean()
    )

    daily_top20 = (
        top20.groupby("date")["future_return_20d"]
        .mean()
    )

    # Align dates for spread calculations.
    common_tb = daily_top10.index.intersection(
        daily_bottom10.index
    )

    long_short = (
        daily_top10.loc[common_tb]
        - daily_bottom10.loc[common_tb]
    )

    # ------------------------------------------------------------
    # Daily rank correlation
    # ------------------------------------------------------------

    daily_ic = (
        r.groupby("date")
        .apply(
            lambda x: spearmanr(
                x["predicted_return"],
                x["future_return_20d"]
            ).statistic
            if len(x) >= 5
            else np.nan
        )
    )

    # ------------------------------------------------------------
    # Save fold rankings
    # ------------------------------------------------------------

    r["fold"] = fold["name"]

    all_rankings.append(r)

    # ------------------------------------------------------------
    # Fold summary
    # ------------------------------------------------------------

    result = {
        "fold": fold["name"],
        "train_rows": len(train),
        "validation_rows": len(val),
        "validation_dates": r["date"].nunique(),

        "global_spearman": spearman,

        "mean_daily_ic": daily_ic.mean(),
        "median_daily_ic": daily_ic.median(),
        "positive_ic_fraction": (
            daily_ic > 0
        ).mean(),

        "top5_mean_return": daily_top5.mean(),
        "top10_mean_return": daily_top10.mean(),
        "top20_mean_return": daily_top20.mean(),
        "bottom10_mean_return": daily_bottom10.mean(),

        "top10_minus_bottom10": long_short.mean(),

        "top5_positive_fraction": (
            daily_top5 > 0
        ).mean(),

        "top10_positive_fraction": (
            daily_top10 > 0
        ).mean(),

        "bottom10_positive_fraction": (
            daily_bottom10 > 0
        ).mean(),

        "training_minutes": minutes,
    }

    all_results.append(result)

    print("\nGlobal Spearman:", round(spearman, 4))
    print("Mean daily IC :", round(daily_ic.mean(), 4))
    print("Top 5% return :", round(daily_top5.mean(), 4))
    print("Top 10% return:", round(daily_top10.mean(), 4))
    print("Top 20% return:", round(daily_top20.mean(), 4))
    print("Bottom 10%    :", round(daily_bottom10.mean(), 4))
    print(
        "Top10 - Bottom10:",
        round(long_short.mean(), 4)
    )

# ============================================================================
# 4. SUMMARY
# ============================================================================

results_df = pd.DataFrame(all_results)

print("\n" + "=" * 78)
print("WALK-FORWARD SUMMARY")
print("=" * 78)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ============================================================================
# 5. AGGREGATE STABILITY
# ============================================================================

print("\n" + "-" * 78)
print("STABILITY SUMMARY")
print("-" * 78)

for column in [
    "global_spearman",
    "mean_daily_ic",
    "top5_mean_return",
    "top10_mean_return",
    "top20_mean_return",
    "bottom10_mean_return",
    "top10_minus_bottom10",
]:

    values = results_df[column]

    print(
        f"{column:30s}"
        f" mean={values.mean():.6f}"
        f" median={values.median():.6f}"
        f" min={values.min():.6f}"
        f" max={values.max():.6f}"
    )

# ============================================================================
# 6. SAVE
# ============================================================================

results_df.to_csv(
    os.path.join(
        REPORTS,
        "cell59_cross_sectional_walk_forward.csv"
    ),
    index=False
)

rankings_df = pd.concat(
    all_rankings,
    ignore_index=True
)

rankings_df.to_parquet(
    os.path.join(
        REPORTS,
        "cell59_walk_forward_rankings.parquet"
    ),
    index=False
)

summary = {
    "folds": len(results_df),
    "mean_global_spearman": float(
        results_df["global_spearman"].mean()
    ),
    "mean_daily_ic": float(
        results_df["mean_daily_ic"].mean()
    ),
    "mean_top5_return": float(
        results_df["top5_mean_return"].mean()
    ),
    "mean_top10_return": float(
        results_df["top10_mean_return"].mean()
    ),
    "mean_top20_return": float(
        results_df["top20_mean_return"].mean()
    ),
    "mean_bottom10_return": float(
        results_df["bottom10_mean_return"].mean()
    ),
    "mean_top10_minus_bottom10": float(
        results_df["top10_minus_bottom10"].mean()
    ),
    "test_set_used": False,
    "market_features_used": False,
    "sector_features_used": False,
}

with open(
    os.path.join(
        REPORTS,
        "cell59_cross_sectional_walk_forward_summary.json"
    ),
    "w"
) as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 78)
print("CELL 59 COMPLETE")
print("=" * 78)

print("\nSaved:")
print("  cell59_cross_sectional_walk_forward.csv")
print("  cell59_walk_forward_rankings.parquet")
print("  cell59_cross_sectional_walk_forward_summary.json")

print("\nTEST SET WAS NOT USED.")

CELL 59 — CROSS-SECTIONAL RANKING STABILITY

Rows: 633137
Symbols: 651

FOLD: 2017-2018
Train rows: 95916
Val rows  : 69761


/tmp/ipykernel_582/2295966023.py:288: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(



Global Spearman: -0.035
Mean daily IC : 0.0006
Top 5% return : 0.1062
Top 10% return: 0.0572
Top 20% return: 0.0319
Bottom 10%    : -0.0013
Top10 - Bottom10: 0.0585

FOLD: 2019-2020
Train rows: 165677
Val rows  : 78047


/tmp/ipykernel_582/2295966023.py:288: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(



Global Spearman: 0.0801
Mean daily IC : 0.0408
Top 5% return : 0.0759
Top 10% return: 0.0664
Top 20% return: 0.0587
Bottom 10%    : 0.0271
Top10 - Bottom10: 0.0394

FOLD: 2021
Train rows: 243724
Val rows  : 53685


...


Global Spearman: 0.1061
Mean daily IC : 0.0501
Top 5% return : 0.1948
Top 10% return: 0.1272
Top 20% return: 0.0916
Bottom 10%    : 0.0289
Top10 - Bottom10: 0.0983

FOLD: 2022-2023
Train rows: 297409
Val rows  : 119393


/tmp/ipykernel_582/2295966023.py:288: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(



Global Spearman: 0.0102
Mean daily IC : -0.0028
Top 5% return : 0.1122
Top 10% return: 0.0579
Top 20% return: 0.0287
Bottom 10%    : -0.0114
Top10 - Bottom10: 0.0693

FOLD: 2024
Train rows: 416802
Val rows  : 73382


/tmp/ipykernel_582/2295966023.py:288: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(



Global Spearman: -0.0054
Mean daily IC : -0.0019
Top 5% return : 0.0649
Top 10% return: 0.0555
Top 20% return: 0.0484
Bottom 10%    : 0.0386
Top10 - Bottom10: 0.0169

WALK-FORWARD SUMMARY
     fold  train_rows  validation_rows  validation_dates  global_spearman  mean_daily_ic  median_daily_ic  positive_ic_fraction  top5_mean_return  top10_mean_return  top20_mean_return  bottom10_mean_return  top10_minus_bottom10  top5_positive_fraction  top10_positive_fraction  bottom10_positive_fraction  training_minutes
2017-2018       95916            69761               470        -0.034978       0.000579         0.004260              0.506383          0.106157           0.057190           0.031937             -0.001325              0.058515                0.685106                 0.625532                    0.359574          0.022386
2019-2020      165677            78047               430         0.080056       0.040751         0.036602              0.588372          0.075920           0.066436 

In [ ]:
# ============================================================================
# CELL 60 — REALISTIC CROSS-SECTIONAL PORTFOLIO BACKTEST
# ============================================================================
# Stock-only model.
# Market/sector features excluded.
# Test set (2025-2026) remains completely untouched.
#
# Strategy:
#   - Train model using information available before each validation period
#   - Rank stocks cross-sectionally each trading day
#   - Select top 5%, 10%, or 20%
#   - Equal-weight positions
#   - Hold for 20 trading sessions
#   - Rebalance every 20 trading sessions
#   - Apply transaction costs
#
# This is a research backtest, not the final test-set evaluation.
# ============================================================================

import os
import json
import time
import numpy as np
import pandas as pd
import xgboost as xgb

PROJECT = "/content/NEPSE_ML_PROJECT"

DATASET = os.path.join(
    PROJECT,
    "features/nepse_ml_clean_dataset.parquet"
)

REPORTS = os.path.join(PROJECT, "reports")
MODELS = os.path.join(PROJECT, "models")

os.makedirs(REPORTS, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

print("=" * 78)
print("CELL 60 — REALISTIC CROSS-SECTIONAL PORTFOLIO BACKTEST")
print("=" * 78)

# ============================================================================
# CONFIGURATION
# ============================================================================

FEATURES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

TARGET = "future_return_20d"

TOP_PCTS = [0.05, 0.10, 0.20]

# Approximate one-way transaction cost.
# This is intentionally conservative for a research backtest.
ONE_WAY_COST = 0.003

# Rebalance every 20 trading sessions.
HOLDING_PERIOD = 20

# Validation-only period.
# 2025-2026 remains untouched.
BACKTEST_START = pd.Timestamp("2017-01-01")
BACKTEST_END = pd.Timestamp("2024-12-31")

# ============================================================================
# LOAD DATA
# ============================================================================

print("\nLoading clean dataset...")

df = pd.read_parquet(DATASET)

df["date"] = pd.to_datetime(df["date"])

print("Rows   :", len(df))
print("Symbols:", df["symbol"].nunique())
print(
    "Range  :",
    df["date"].min().date(),
    "to",
    df["date"].max().date()
)

# ============================================================================
# IMPORTANT:
# We need actual forward prices, not only the precomputed overlapping
# future_return_20d target, for the portfolio simulation.
#
# Reconstruct the 20-trading-session forward return directly from close.
# ============================================================================

print("\nPreparing forward returns...")

df = df.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

df["close"] = pd.to_numeric(
    df["close"],
    errors="coerce"
)

df["future_close_20d"] = (
    df.groupby("symbol")["close"]
      .shift(-HOLDING_PERIOD)
)

df["realized_return_20d"] = (
    df["future_close_20d"] /
    df["close"] - 1.0
)

# ============================================================================
# LIMIT TO BACKTEST PERIOD
# ============================================================================

bt = df[
    (df["date"] >= BACKTEST_START) &
    (df["date"] <= BACKTEST_END) &
    (df["realized_return_20d"].notna())
].copy()

print("\nBacktest rows:", len(bt))
print(
    "Backtest dates:",
    bt["date"].min().date(),
    "to",
    bt["date"].max().date()
)

# ============================================================================
# MODEL TRAINING WINDOWS
# ============================================================================
#
# To prevent look-ahead:
#
# For each rebalance date:
#   training data = everything strictly before that date
#   prediction = stocks available on rebalance date
#
# We use a rolling expanding training set.
#
# To keep computation reasonable, the model is retrained every 20 trading
# sessions, exactly when the portfolio is rebalanced.
# ============================================================================

all_dates = np.sort(
    bt["date"].unique()
)

# Only rebalance dates separated by HOLDING_PERIOD trading sessions.
rebalance_dates = all_dates[
    ::HOLDING_PERIOD
]

print(
    "\nRebalance dates:",
    len(rebalance_dates)
)

# ============================================================================
# PORTFOLIO STORAGE
# ============================================================================

portfolio_records = []

# Current portfolio state.
previous_positions = {}

# ============================================================================
# WALK FORWARD
# ============================================================================

for i, rebalance_date in enumerate(rebalance_dates):

    print(
        f"\rRebalance {i+1}/{len(rebalance_dates)} "
        f"({pd.Timestamp(rebalance_date).date()})",
        end=""
    )

    rebalance_date = pd.Timestamp(rebalance_date)

    # ------------------------------------------------------------
    # Training data strictly before rebalance date.
    # ------------------------------------------------------------

    train = df[
        (df["date"] < rebalance_date) &
        (df[TARGET].notna())
    ].copy()

    # Keep training period beginning 2011.
    train = train[
        train["date"] >= pd.Timestamp("2011-01-01")
    ]

    # ------------------------------------------------------------
    # Current universe.
    # ------------------------------------------------------------

    current = bt[
        bt["date"] == rebalance_date
    ].copy()

    if len(current) < 20:
        continue

    # ------------------------------------------------------------
    # Remove rows with unusable features.
    # ------------------------------------------------------------

    medians = train[FEATURES].median()

    X_train = (
        train[FEATURES]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(medians)
    )

    X_current = (
        current[FEATURES]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(medians)
    )

    y_train = train[TARGET].values

    # ------------------------------------------------------------
    # Model.
    # ------------------------------------------------------------

    model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.025,
        max_depth=3,
        min_child_weight=30,
        subsample=0.80,
        colsample_bytree=0.85,
        reg_alpha=1.0,
        reg_lambda=10.0,
        gamma=0.10,
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        device="cuda",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train,
        verbose=False
    )

    # ------------------------------------------------------------
    # Predict today's cross-sectional ranking.
    # ------------------------------------------------------------

    current["predicted_return"] = model.predict(
        X_current
    )

    current = current.sort_values(
        "predicted_return",
        ascending=False
    )

    # ------------------------------------------------------------
    # Evaluate each portfolio size.
    # ------------------------------------------------------------

    for top_pct in TOP_PCTS:

        n_select = max(
            1,
            int(np.floor(
                len(current) * top_pct
            ))
        )

        selected = current.head(
            n_select
        ).copy()

        selected_symbols = set(
            selected["symbol"]
        )

        # Equal weight.
        weight = 1.0 / n_select

        # --------------------------------------------------------
        # Realized return.
        # --------------------------------------------------------

        gross_return = (
            selected["realized_return_20d"]
            .mean()
        )

        # --------------------------------------------------------
        # Turnover.
        #
        # Current portfolio has equal weights.
        # Existing position retained = weight.
        # New position = weight.
        #
        # Approximate turnover as sum(abs(new-old)).
        # --------------------------------------------------------

        new_positions = {
            symbol: weight
            for symbol in selected_symbols
        }

        all_symbols = set(
            previous_positions.keys()
        ).union(
            new_positions.keys()
        )

        turnover = sum(
            abs(
                new_positions.get(symbol, 0.0)
                -
                previous_positions.get(symbol, 0.0)
            )
            for symbol in all_symbols
        )

        # Transaction cost applied to portfolio turnover.
        transaction_cost = (
            turnover * ONE_WAY_COST
        )

        net_return = (
            gross_return -
            transaction_cost
        )

        # --------------------------------------------------------
        # Save.
        # --------------------------------------------------------

        portfolio_records.append({
            "date": rebalance_date,
            "top_pct": top_pct,
            "n_stocks": n_select,
            "gross_return": gross_return,
            "turnover": turnover,
            "transaction_cost": transaction_cost,
            "net_return": net_return,
        })

        # Keep previous portfolio for turnover calculation.
        previous_positions = new_positions

# ============================================================================
# FINISH PROGRESS LINE
# ============================================================================

print()

# ============================================================================
# RESULTS DATAFRAME
# ============================================================================

results = pd.DataFrame(
    portfolio_records
)

print("\nPortfolio observations:", len(results))

# ============================================================================
# PORTFOLIO PERFORMANCE FUNCTION
# ============================================================================

def calculate_metrics(group):

    group = group.sort_values(
        "date"
    ).copy()

    returns = group["net_return"].values

    equity = np.cumprod(
        1.0 + returns
    )

    group["equity"] = equity

    total_return = (
        equity[-1] - 1.0
        if len(equity)
        else np.nan
    )

    years = (
        len(group) *
        HOLDING_PERIOD /
        252.0
    )

    if years > 0:
        cagr = (
            equity[-1] ** (1.0 / years)
            - 1.0
        )
    else:
        cagr = np.nan

    running_max = np.maximum.accumulate(
        equity
    )

    drawdown = (
        equity /
        running_max - 1.0
    )

    max_drawdown = drawdown.min()

    mean_return = returns.mean()
    std_return = returns.std(ddof=1)

    if std_return > 0:
        sharpe = (
            mean_return /
            std_return
        ) * np.sqrt(
            252.0 / HOLDING_PERIOD
        )
    else:
        sharpe = np.nan

    positive_fraction = (
        returns > 0
    ).mean()

    return {
        "observations": len(group),
        "total_return": total_return,
        "CAGR": cagr,
        "max_drawdown": max_drawdown,
        "annualized_sharpe": sharpe,
        "mean_20d_return": mean_return,
        "median_20d_return": np.median(returns),
        "positive_period_fraction": positive_fraction,
        "mean_turnover": group["turnover"].mean(),
        "mean_transaction_cost": group[
            "transaction_cost"
        ].mean(),
    }

# ============================================================================
# STRATEGY METRICS
# ============================================================================

metrics = []

for top_pct, group in results.groupby(
    "top_pct"
):

    m = calculate_metrics(
        group
    )

    m["top_pct"] = top_pct

    metrics.append(m)

metrics_df = pd.DataFrame(
    metrics
)

# ============================================================================
# PRINT RESULTS
# ============================================================================

print("\n" + "=" * 78)
print("PORTFOLIO BACKTEST RESULTS")
print("=" * 78)

display(
    metrics_df[
        [
            "top_pct",
            "observations",
            "total_return",
            "CAGR",
            "max_drawdown",
            "annualized_sharpe",
            "mean_20d_return",
            "median_20d_return",
            "positive_period_fraction",
            "mean_turnover",
            "mean_transaction_cost",
        ]
    ].round(6)
)

# ============================================================================
# EQUITY CURVES
# ============================================================================

equity_curves = (
    results
    .pivot(
        index="date",
        columns="top_pct",
        values="net_return"
    )
    .sort_index()
)

equity_curves = (
    1.0 + equity_curves
).cumprod()

equity_curves.columns = [
    f"top_{int(x*100)}pct"
    for x in equity_curves.columns
]

print("\nFinal equity:")
print(
    equity_curves.iloc[-1]
)

# ============================================================================
# YEARLY PERFORMANCE
# ============================================================================

yearly = results.copy()

yearly["year"] = (
    yearly["date"]
    .dt.year
)

yearly_returns = []

for (year, top_pct), group in yearly.groupby(
    ["year", "top_pct"]
):

    compound = (
        np.prod(
            1.0 + group["net_return"]
        ) - 1.0
    )

    yearly_returns.append({
        "year": year,
        "top_pct": top_pct,
        "annual_return": compound,
        "positive_period_fraction": (
            group["net_return"] > 0
        ).mean(),
    })

yearly_df = pd.DataFrame(
    yearly_returns
)

print("\n" + "=" * 78)
print("YEARLY PERFORMANCE")
print("=" * 78)

display(
    yearly_df
    .sort_values(
        ["year", "top_pct"]
    )
    .round(6)
)

# ============================================================================
# SAVE RESULTS
# ============================================================================

results.to_csv(
    os.path.join(
        REPORTS,
        "cell60_portfolio_backtest_periods.csv"
    ),
    index=False
)

metrics_df.to_csv(
    os.path.join(
        REPORTS,
        "cell60_portfolio_backtest_metrics.csv"
    ),
    index=False
)

yearly_df.to_csv(
    os.path.join(
        REPORTS,
        "cell60_portfolio_backtest_yearly.csv"
    ),
    index=False
)

equity_curves.to_csv(
    os.path.join(
        REPORTS,
        "cell60_portfolio_equity_curves.csv"
    )
)

summary = {
    "backtest_start": str(BACKTEST_START.date()),
    "backtest_end": str(BACKTEST_END.date()),
    "holding_period_trading_sessions": HOLDING_PERIOD,
    "transaction_cost_one_way": ONE_WAY_COST,
    "features_used": FEATURES,
    "market_features_used": False,
    "sector_features_used": False,
    "test_set_used": False,
    "strategies": metrics_df.to_dict(
        orient="records"
    ),
}

with open(
    os.path.join(
        REPORTS,
        "cell60_portfolio_backtest_summary.json"
    ),
    "w"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print("\n" + "=" * 78)
print("CELL 60 COMPLETE")
print("=" * 78)

print("\nSaved:")
print("  cell60_portfolio_backtest_periods.csv")
print("  cell60_portfolio_backtest_metrics.csv")
print("  cell60_portfolio_backtest_yearly.csv")
print("  cell60_portfolio_equity_curves.csv")
print("  cell60_portfolio_backtest_summary.json")

print("\nTEST SET WAS NOT USED.")
print("Market/sector features were NOT used.")

CELL 60 — REALISTIC CROSS-SECTIONAL PORTFOLIO BACKTEST

Loading clean dataset...
Rows   : 633137
Symbols: 651
Range  : 1997-07-20 to 2026-10-06

Preparing forward returns...

Backtest rows: 394268
Backtest dates: 2017-01-01 to 2024-12-31

Rebalance dates: 93
Rebalance 93/93 (2024-12-31)

Portfolio observations: 279

PORTFOLIO BACKTEST RESULTS


,top_pct,observations,total_return,CAGR,max_drawdown,annualized_sharpe,mean_20d_return,median_20d_return,positive_period_fraction,mean_turnover,mean_transaction_cost
0,0.05,93,7044.825902,2.321488,-0.419762,2.099713,0.115496,0.069484,0.731183,1.827998,0.005484
1,0.10,93,281.958211,1.148666,-0.274254,1.812504,0.070497,0.046191,0.698925,1.026546,0.003080
2,0.20,93,39.015937,0.648454,-0.304117,1.492129,0.045620,0.026821,0.602151,1.013984,0.003042



Final equity:
top_5pct     7045.825902
top_10pct     282.958211
top_20pct      40.015937
Name: 2024-12-31 00:00:00, dtype: float64

YEARLY PERFORMANCE


,year,top_pct,annual_return,positive_period_fraction
0,2017,0.05,4.022203,0.666667
1,2017,0.10,1.985576,0.750000
2,2017,0.20,0.986425,0.583333
3,2018,0.05,0.650592,0.666667
4,2018,0.10,0.173093,0.500000
5,2018,0.20,0.003768,0.416667
6,2019,0.05,1.604030,0.750000
7,2019,0.10,0.875632,0.666667
8,2019,0.20,0.410918,0.500000
9,2020,0.05,0.051694,0.666667



CELL 60 COMPLETE

Saved:
  cell60_portfolio_backtest_periods.csv
  cell60_portfolio_backtest_metrics.csv
  cell60_portfolio_backtest_yearly.csv
  cell60_portfolio_equity_curves.csv
  cell60_portfolio_backtest_summary.json

TEST SET WAS NOT USED.
Market/sector features were NOT used.


In [ ]:
# ============================================================================
# CELL 61 — LEAKAGE-FREE PORTFOLIO BACKTEST
# ============================================================================
# GPU FIRST / CPU FALLBACK
#
# Hardware:
#   NVIDIA CUDA GPU available -> GPU
#   No NVIDIA GPU            -> CPU
#   TPU                      -> not used by XGBoost
#
# Methodology:
#   - Stock-only features
#   - NO market features
#   - NO sector features
#   - 20-trading-day holding period
#   - 20-trading-day purge before training
#   - Independent turnover for each strategy
#   - Transaction costs
#   - Equal-weight benchmark
#   - 2025-2026 test set untouched
# ============================================================================

import os
import json
import time
import subprocess
import numpy as np
import pandas as pd
import xgboost as xgb

PROJECT = "/content/NEPSE_ML_PROJECT"

DATASET = os.path.join(
    PROJECT,
    "features/nepse_ml_clean_dataset.parquet"
)

REPORTS = os.path.join(
    PROJECT,
    "reports"
)

os.makedirs(REPORTS, exist_ok=True)

print("=" * 78)
print("CELL 61 — LEAKAGE-FREE PORTFOLIO BACKTEST")
print("=" * 78)

# ============================================================================
# CONFIGURATION
# ============================================================================

FEATURES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

TARGET = "future_return_20d"

TOP_PCTS = [0.05, 0.10, 0.20]

HOLDING_PERIOD = 20

# Estimated one-way transaction cost.
ONE_WAY_COST = 0.003

TRAIN_START = pd.Timestamp("2011-01-01")
BACKTEST_START = pd.Timestamp("2017-01-01")
BACKTEST_END = pd.Timestamp("2024-12-31")

# ============================================================================
# HARDWARE DETECTION
# ============================================================================

print("\n" + "-" * 78)
print("HARDWARE DETECTION")
print("-" * 78)

GPU_AVAILABLE = False
GPU_NAME = None

try:

    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name",
            "--format=csv,noheader"
        ],
        capture_output=True,
        text=True,
        timeout=10
    )

    if (
        result.returncode == 0
        and result.stdout.strip()
    ):

        GPU_NAME = (
            result.stdout
            .strip()
            .splitlines()[0]
            .strip()
        )

        GPU_AVAILABLE = True

except Exception:
    GPU_AVAILABLE = False

if GPU_AVAILABLE:

    DEVICE = "cuda"
    TREE_METHOD = "hist"

    print("NVIDIA GPU detected:", GPU_NAME)
    print("XGBoost device      : CUDA GPU")
    print("CPU fallback        : AVAILABLE")

else:

    DEVICE = "cpu"
    TREE_METHOD = "hist"

    print("NVIDIA GPU detected : NO")
    print("XGBoost device      : CPU")
    print("CPU fallback        : ACTIVE")

print(
    "\nTPU detected/available does not affect this model."
)
print(
    "XGBoost does not use TPU for tree training."
)

# ============================================================================
# LOAD ONLY REQUIRED COLUMNS
# ============================================================================

print("\n" + "-" * 78)
print("LOADING DATA")
print("-" * 78)

required_columns = list(
    dict.fromkeys(
        [
            "date",
            "symbol",
            "close",
            TARGET,
        ] + FEATURES
    )
)

df = pd.read_parquet(
    DATASET,
    columns=required_columns
)

df["date"] = pd.to_datetime(
    df["date"]
)

print("Rows   :", len(df))
print("Symbols:", df["symbol"].nunique())
print(
    "Range  :",
    df["date"].min().date(),
    "to",
    df["date"].max().date()
)

# ============================================================================
# SORT
# ============================================================================

df = df.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

# ============================================================================
# CONSTRUCT REALIZED 20-TRADING-DAY RETURN
# ============================================================================

print("\nConstructing 20-trading-session forward returns...")

df["future_close_20d"] = (
    df.groupby(
        "symbol",
        sort=False
    )["close"]
    .shift(-HOLDING_PERIOD)
)

df["realized_return_20d"] = (
    df["future_close_20d"] /
    df["close"] -
    1.0
)

# ============================================================================
# BACKTEST DATA
# ============================================================================

bt = df[
    (df["date"] >= BACKTEST_START) &
    (df["date"] <= BACKTEST_END) &
    (df["realized_return_20d"].notna())
].copy()

print(
    "\nBacktest period:",
    bt["date"].min().date(),
    "to",
    bt["date"].max().date()
)

print(
    "Backtest rows:",
    len(bt)
)

# ============================================================================
# TRAINING POOL
# ============================================================================

train_pool = df[
    (df["date"] >= TRAIN_START) &
    (df[TARGET].notna())
].copy()

print(
    "Training pool:",
    len(train_pool)
)

# ============================================================================
# TRADING CALENDAR
# ============================================================================

trading_dates = np.array(
    sorted(
        bt["date"].unique()
    )
)

rebalance_dates = trading_dates[
    ::HOLDING_PERIOD
]

print(
    "Trading dates  :",
    len(trading_dates)
)

print(
    "Rebalance dates:",
    len(rebalance_dates)
)

# ============================================================================
# PORTFOLIO STATES
# ============================================================================

previous_positions = {
    pct: {}
    for pct in TOP_PCTS
}

records = []

total_start = time.time()

# ============================================================================
# WALK-FORWARD BACKTEST
# ============================================================================

print("\n" + "-" * 78)
print("RUNNING WALK-FORWARD BACKTEST")
print("-" * 78)

for i, rebalance_date in enumerate(
    rebalance_dates
):

    loop_start = time.time()

    rebalance_date = pd.Timestamp(
        rebalance_date
    )

    # ------------------------------------------------------------
    # Locate current trading-session index.
    # ------------------------------------------------------------

    current_idx = np.searchsorted(
        trading_dates,
        np.datetime64(
            rebalance_date
        )
    )

    if current_idx < HOLDING_PERIOD:
        continue

    # ------------------------------------------------------------
    # PURGED TRAINING CUTOFF
    #
    # At prediction date T, we only allow training observations whose
    # 20-day target would already have been completely observable.
    # ------------------------------------------------------------

    train_cutoff = pd.Timestamp(
        trading_dates[
            current_idx -
            HOLDING_PERIOD
        ]
    )

    train = train_pool[
        train_pool["date"] <=
        train_cutoff
    ]

    # ------------------------------------------------------------
    # CURRENT STOCK UNIVERSE
    # ------------------------------------------------------------

    current = bt[
        bt["date"] ==
        rebalance_date
    ].copy()

    if len(current) < 20:
        continue

    # ------------------------------------------------------------
    # TRAINING-ONLY MEDIANS
    # ------------------------------------------------------------

    medians = train[
        FEATURES
    ].median()

    X_train = (
        train[FEATURES]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .fillna(medians)
        .astype(np.float32)
    )

    X_current = (
        current[FEATURES]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .fillna(medians)
        .astype(np.float32)
    )

    y_train = (
        train[TARGET]
        .astype(np.float32)
        .values
    )

    # ------------------------------------------------------------
    # XGBOOST
    # ------------------------------------------------------------

    model = xgb.XGBRegressor(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=30,
        subsample=0.80,
        colsample_bytree=0.85,
        reg_alpha=1.0,
        reg_lambda=10.0,
        gamma=0.10,
        objective="reg:squarederror",
        eval_metric="rmse",

        # GPU if available, CPU otherwise.
        tree_method=TREE_METHOD,
        device=DEVICE,

        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train,
        verbose=False
    )

    # ------------------------------------------------------------
    # PREDICT
    # ------------------------------------------------------------

    current["predicted_return"] = (
        model.predict(
            X_current
        )
    )

    current = current.sort_values(
        "predicted_return",
        ascending=False
    )

    # ------------------------------------------------------------
    # EQUAL-WEIGHT BENCHMARK
    # ------------------------------------------------------------

    benchmark_return = (
        current[
            "realized_return_20d"
        ].mean()
    )

    # ------------------------------------------------------------
    # EACH PORTFOLIO STRATEGY
    # ------------------------------------------------------------

    for top_pct in TOP_PCTS:

        n_select = max(
            1,
            int(
                np.floor(
                    len(current) *
                    top_pct
                )
            )
        )

        selected = current.head(
            n_select
        ).copy()

        selected_symbols = set(
            selected["symbol"]
        )

        weight = (
            1.0 /
            n_select
        )

        new_positions = {
            symbol: weight
            for symbol in selected_symbols
        }

        old_positions = (
            previous_positions[
                top_pct
            ]
        )

        all_symbols = (
            set(old_positions)
            |
            set(new_positions)
        )

        turnover = sum(
            abs(
                new_positions.get(
                    symbol,
                    0.0
                )
                -
                old_positions.get(
                    symbol,
                    0.0
                )
            )
            for symbol in all_symbols
        )

        transaction_cost = (
            turnover *
            ONE_WAY_COST
        )

        gross_return = (
            selected[
                "realized_return_20d"
            ].mean()
        )

        net_return = (
            gross_return -
            transaction_cost
        )

        records.append({

            "date":
                rebalance_date,

            "train_cutoff":
                train_cutoff,

            "top_pct":
                top_pct,

            "n_stocks":
                n_select,

            "train_rows":
                len(train),

            "gross_return":
                gross_return,

            "turnover":
                turnover,

            "transaction_cost":
                transaction_cost,

            "net_return":
                net_return,

            "benchmark_return":
                benchmark_return,
        })

        previous_positions[
            top_pct
        ] = new_positions

    # ------------------------------------------------------------
    # PROGRESS / ETA
    # ------------------------------------------------------------

    elapsed = (
        time.time() -
        loop_start
    )

    if (
        i == 0
        or
        (i + 1) % 5 == 0
        or
        i == len(rebalance_dates) - 1
    ):

        remaining = (
            len(rebalance_dates)
            -
            i
            -
            1
        )

        eta_seconds = (
            remaining *
            elapsed
        )

        print(
            f"\r"
            f"Rebalance "
            f"{i+1}/"
            f"{len(rebalance_dates)}"
            f" | {elapsed:.1f}s"
            f" | ETA "
            f"{eta_seconds/60:.1f} min",
            end=""
        )

print()

total_minutes = (
    time.time() -
    total_start
) / 60

print(
    f"\nTotal runtime: "
    f"{total_minutes:.2f} minutes"
)

# ============================================================================
# RESULTS
# ============================================================================

results = pd.DataFrame(
    records
)

print(
    "\nPortfolio observations:",
    len(results)
)

# ============================================================================
# METRICS
# ============================================================================

def calc_metrics(
    returns,
    periods_per_year
):

    returns = np.asarray(
        returns,
        dtype=float
    )

    returns = returns[
        np.isfinite(returns)
    ]

    if len(returns) == 0:
        return {}

    equity = np.cumprod(
        1.0 + returns
    )

    total_return = (
        equity[-1] -
        1.0
    )

    years = (
        len(returns) /
        periods_per_year
    )

    cagr = (
        equity[-1] **
        (1.0 / years)
        - 1.0
    )

    running_max = (
        np.maximum.accumulate(
            equity
        )
    )

    drawdown = (
        equity /
        running_max -
        1.0
    )

    max_drawdown = (
        drawdown.min()
    )

    std = returns.std(
        ddof=1
    )

    sharpe = (
        returns.mean() /
        std *
        np.sqrt(
            periods_per_year
        )
    ) if std > 0 else np.nan

    return {

        "observations":
            len(returns),

        "total_return":
            total_return,

        "CAGR":
            cagr,

        "max_drawdown":
            max_drawdown,

        "annualized_sharpe":
            sharpe,

        "mean_20d_return":
            returns.mean(),

        "median_20d_return":
            np.median(
                returns
            ),

        "positive_period_fraction":
            (
                returns > 0
            ).mean(),
    }

# ============================================================================
# STRATEGY SUMMARY
# ============================================================================

periods_per_year = (
    252.0 /
    HOLDING_PERIOD
)

metrics = []

for top_pct, group in (
    results.groupby("top_pct")
):

    m = calc_metrics(
        group["net_return"],
        periods_per_year
    )

    m["strategy"] = (
        f"Top {int(top_pct * 100)}%"
    )

    m["top_pct"] = top_pct

    m["mean_turnover"] = (
        group[
            "turnover"
        ].mean()
    )

    m["mean_transaction_cost"] = (
        group[
            "transaction_cost"
        ].mean()
    )

    m["mean_excess_return"] = (
        group[
            "net_return"
        ]
        -
        group[
            "benchmark_return"
        ]
    ).mean()

    metrics.append(m)

# ============================================================================
# BENCHMARK
# ============================================================================

benchmark = (
    results
    .drop_duplicates(
        "date"
    )
    .sort_values(
        "date"
    )
)

benchmark_metrics = calc_metrics(
    benchmark[
        "benchmark_return"
    ],
    periods_per_year
)

benchmark_metrics[
    "strategy"
] = "Equal-weight benchmark"

benchmark_metrics[
    "top_pct"
] = np.nan

benchmark_metrics[
    "mean_turnover"
] = np.nan

benchmark_metrics[
    "mean_transaction_cost"
] = np.nan

benchmark_metrics[
    "mean_excess_return"
] = 0.0

metrics.append(
    benchmark_metrics
)

metrics_df = pd.DataFrame(
    metrics
)

# ============================================================================
# DISPLAY RESULTS
# ============================================================================

print("\n" + "=" * 78)
print("LEAKAGE-FREE BACKTEST RESULTS")
print("=" * 78)

display(
    metrics_df[
        [
            "strategy",
            "observations",
            "total_return",
            "CAGR",
            "max_drawdown",
            "annualized_sharpe",
            "mean_20d_return",
            "median_20d_return",
            "positive_period_fraction",
            "mean_turnover",
            "mean_transaction_cost",
            "mean_excess_return",
        ]
    ].round(6)
)

# ============================================================================
# YEARLY RESULTS
# ============================================================================

results["year"] = (
    results["date"].dt.year
)

yearly_records = []

for (
    year,
    top_pct
), group in results.groupby(
    ["year", "top_pct"]
):

    portfolio_return = (
        np.prod(
            1.0 +
            group[
                "net_return"
            ]
        )
        -
        1.0
    )

    benchmark_return = (
        np.prod(
            1.0 +
            group[
                "benchmark_return"
            ]
        )
        -
        1.0
    )

    yearly_records.append({

        "year":
            year,

        "top_pct":
            top_pct,

        "portfolio_return":
            portfolio_return,

        "benchmark_return":
            benchmark_return,

        "excess_return":
            (
                portfolio_return -
                benchmark_return
            ),

        "positive_period_fraction":
            (
                group[
                    "net_return"
                ] > 0
            ).mean(),
    })

yearly_df = pd.DataFrame(
    yearly_records
)

print("\n" + "=" * 78)
print("YEARLY PERFORMANCE")
print("=" * 78)

display(
    yearly_df
    .sort_values(
        [
            "year",
            "top_pct"
        ]
    )
    .round(6)
)

# ============================================================================
# SAVE
# ============================================================================

results.to_csv(
    os.path.join(
        REPORTS,
        "cell61_leakage_free_backtest_periods.csv"
    ),
    index=False
)

metrics_df.to_csv(
    os.path.join(
        REPORTS,
        "cell61_leakage_free_backtest_metrics.csv"
    ),
    index=False
)

yearly_df.to_csv(
    os.path.join(
        REPORTS,
        "cell61_leakage_free_backtest_yearly.csv"
    ),
    index=False
)

summary = {

    "backtest_start":
        str(
            BACKTEST_START.date()
        ),

    "backtest_end":
        str(
            BACKTEST_END.date()
        ),

    "train_start":
        str(
            TRAIN_START.date()
        ),

    "holding_period":
        HOLDING_PERIOD,

    "purge_period":
        HOLDING_PERIOD,

    "transaction_cost_one_way":
        ONE_WAY_COST,

    "device":
        DEVICE,

    "gpu_available":
        GPU_AVAILABLE,

    "gpu_name":
        GPU_NAME,

    "features":
        FEATURES,

    "market_features_used":
        False,

    "sector_features_used":
        False,

    "test_set_used":
        False,

    "metrics":
        metrics_df.to_dict(
            orient="records"
        ),
}

with open(
    os.path.join(
        REPORTS,
        "cell61_leakage_free_backtest_summary.json"
    ),
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# ============================================================================
# FINAL
# ============================================================================

print("\n" + "=" * 78)
print("CELL 61 COMPLETE")
print("=" * 78)

print("\nSaved:")
print(
    "  cell61_leakage_free_backtest_periods.csv"
)
print(
    "  cell61_leakage_free_backtest_metrics.csv"
)
print(
    "  cell61_leakage_free_backtest_yearly.csv"
)
print(
    "  cell61_leakage_free_backtest_summary.json"
)

print("\nHardware:")
print(
    "  XGBoost device:",
    DEVICE
)
print(
    "  GPU available :",
    GPU_AVAILABLE
)

if GPU_AVAILABLE:
    print(
        "  GPU:",
        GPU_NAME
    )

print("\nTEST SET WAS NOT USED.")
print("Market/sector features were NOT used.")

CELL 61 — LEAKAGE-FREE PORTFOLIO BACKTEST

------------------------------------------------------------------------------
HARDWARE DETECTION
------------------------------------------------------------------------------
NVIDIA GPU detected: Tesla T4
XGBoost device      : CUDA GPU
CPU fallback        : AVAILABLE

TPU detected/available does not affect this model.
XGBoost does not use TPU for tree training.

------------------------------------------------------------------------------
LOADING DATA
------------------------------------------------------------------------------
Rows   : 633137
Symbols: 651
Range  : 1997-07-20 to 2026-10-06

Constructing 20-trading-session forward returns...

Backtest period: 2017-01-01 to 2024-12-31
Backtest rows: 394268
Training pool: 618011
Trading dates  : 1841
Rebalance dates: 93

------------------------------------------------------------------------------
RUNNING WALK-FORWARD BACKTEST
-----------------------------------------------------------------

,strategy,observations,total_return,CAGR,max_drawdown,annualized_sharpe,mean_20d_return,median_20d_return,positive_period_fraction,mean_turnover,mean_transaction_cost,mean_excess_return
0,Top 5%,92,2104.363971,1.852012,-0.386402,1.946864,0.100676,0.057996,0.750000,1.689834,0.005070,0.078492
1,Top 10%,92,111.920983,0.910473,-0.283847,1.707368,0.059415,0.042807,0.663043,1.568359,0.004705,0.037231
2,Top 20%,92,18.828581,0.505472,-0.304949,1.375079,0.037298,0.025076,0.597826,1.378455,0.004135,0.015114
3,Equal-weight benchmark,92,4.580416,0.265496,-0.372964,0.937581,0.022184,0.012062,0.543478,NaN,NaN,0.000000



YEARLY PERFORMANCE


,year,top_pct,portfolio_return,benchmark_return,excess_return,positive_period_fraction
0,2017,0.05,2.093203,0.288209,1.804993,0.636364
1,2017,0.10,0.988674,0.288209,0.700465,0.545455
2,2017,0.20,0.609332,0.288209,0.321123,0.545455
3,2018,0.05,0.485217,-0.195646,0.680863,0.666667
4,2018,0.10,0.130301,-0.195646,0.325947,0.500000
5,2018,0.20,-0.086380,-0.195646,0.109267,0.416667
6,2019,0.05,1.422734,0.112420,1.310314,0.750000
7,2019,0.10,0.695168,0.112420,0.582748,0.583333
8,2019,0.20,0.379910,0.112420,0.267490,0.583333
9,2020,0.05,0.211470,0.754347,-0.542877,0.777778



CELL 61 COMPLETE

Saved:
  cell61_leakage_free_backtest_periods.csv
  cell61_leakage_free_backtest_metrics.csv
  cell61_leakage_free_backtest_yearly.csv
  cell61_leakage_free_backtest_summary.json

Hardware:
  XGBoost device: cuda
  GPU available : True
  GPU: Tesla T4

TEST SET WAS NOT USED.
Market/sector features were NOT used.


In [ ]:
# ============================================================================
# CELL 62 — TRANSACTION COST SENSITIVITY
# ============================================================================
# Uses Cell 61's already-generated portfolio selections/returns.
# NO MODEL RETRAINING.
#
# Tests one-way transaction costs:
#   0.1%, 0.3%, 0.5%, 1.0%, 2.0%
#
# Purpose:
#   Check whether the ranking strategy remains attractive under
#   substantially higher trading costs.
# ============================================================================

import os
import numpy as np
import pandas as pd
import json

PROJECT = "/content/NEPSE_ML_PROJECT"
REPORTS = os.path.join(PROJECT, "reports")

INPUT_FILE = os.path.join(
    REPORTS,
    "cell61_leakage_free_backtest_periods.csv"
)

OUTPUT_FILE = os.path.join(
    REPORTS,
    "cell62_transaction_cost_sensitivity.csv"
)

# --------------------------------------------------------------------------
# LOAD CELL 61 RESULTS
# --------------------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)
df["date"] = pd.to_datetime(df["date"])

print("=" * 78)
print("CELL 62 — TRANSACTION COST SENSITIVITY")
print("=" * 78)

print("\nLoaded Cell 61 observations:", len(df))

# --------------------------------------------------------------------------
# COST SCENARIOS
# --------------------------------------------------------------------------

COSTS = [
    0.001,   # 0.1%
    0.003,   # 0.3%
    0.005,   # 0.5%
    0.010,   # 1.0%
    0.020,   # 2.0%
]

HOLDING_PERIOD = 20
PERIODS_PER_YEAR = 252.0 / HOLDING_PERIOD

# --------------------------------------------------------------------------
# CALCULATE METRICS
# --------------------------------------------------------------------------

def metrics(returns):

    returns = np.asarray(
        returns,
        dtype=float
    )

    returns = returns[
        np.isfinite(returns)
    ]

    if len(returns) == 0:
        return {}

    equity = np.cumprod(
        1.0 + returns
    )

    total_return = (
        equity[-1] - 1.0
    )

    years = (
        len(returns) /
        PERIODS_PER_YEAR
    )

    cagr = (
        equity[-1] **
        (1.0 / years)
        - 1.0
    )

    running_max = np.maximum.accumulate(
        equity
    )

    drawdown = (
        equity /
        running_max -
        1.0
    )

    max_dd = drawdown.min()

    std = returns.std(
        ddof=1
    )

    sharpe = (
        returns.mean()
        / std
        * np.sqrt(
            PERIODS_PER_YEAR
        )
    ) if std > 0 else np.nan

    return {
        "observations": len(returns),
        "total_return": total_return,
        "CAGR": cagr,
        "max_drawdown": max_dd,
        "annualized_sharpe": sharpe,
        "mean_period_return": returns.mean(),
        "positive_period_fraction": (
            returns > 0
        ).mean(),
    }

# --------------------------------------------------------------------------
# RUN COST SCENARIOS
# --------------------------------------------------------------------------

rows = []

for cost in COSTS:

    for top_pct, group in df.groupby(
        "top_pct"
    ):

        group = group.sort_values(
            "date"
        ).copy()

        # Cell 61 gross portfolio return
        gross = group[
            "gross_return"
        ].values

        turnover = group[
            "turnover"
        ].values

        # Apply NEW transaction cost.
        net = (
            gross -
            turnover * cost
        )

        m = metrics(net)

        m.update({

            "top_pct":
                top_pct,

            "strategy":
                f"Top {int(top_pct * 100)}%",

            "one_way_cost":
                cost,

            "mean_turnover":
                turnover.mean(),

            "mean_transaction_cost":
                (
                    turnover * cost
                ).mean(),

            "mean_excess_vs_benchmark":
                (
                    net -
                    group[
                        "benchmark_return"
                    ].values
                ).mean(),
        })

        rows.append(m)

results = pd.DataFrame(rows)

# --------------------------------------------------------------------------
# DISPLAY
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("TRANSACTION COST SENSITIVITY RESULTS")
print("=" * 78)

display(
    results[
        [
            "strategy",
            "one_way_cost",
            "total_return",
            "CAGR",
            "max_drawdown",
            "annualized_sharpe",
            "mean_period_return",
            "positive_period_fraction",
            "mean_turnover",
            "mean_transaction_cost",
            "mean_excess_vs_benchmark",
        ]
    ].round(6)
)

# --------------------------------------------------------------------------
# SAVE
# --------------------------------------------------------------------------

results.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nSaved:")
print(
    OUTPUT_FILE
)

# --------------------------------------------------------------------------
# SIMPLE SURVIVAL CHECK
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("SURVIVAL CHECK")
print("=" * 78)

for top_pct in sorted(
    results["top_pct"].unique()
):

    r = results[
        results["top_pct"] ==
        top_pct
    ].sort_values(
        "one_way_cost"
    )

    print(
        f"\nTop {int(top_pct * 100)}%"
    )

    for _, row in r.iterrows():

        print(
            f"  Cost {row['one_way_cost']*100:.1f}%"
            f" -> CAGR "
            f"{row['CAGR']*100:.2f}%"
            f", Sharpe "
            f"{row['annualized_sharpe']:.2f}"
            f", Excess/period "
            f"{row['mean_excess_vs_benchmark']*100:.2f}%"
        )

print("\nCELL 62 COMPLETE.")

CELL 62 — TRANSACTION COST SENSITIVITY

Loaded Cell 61 observations: 276

TRANSACTION COST SENSITIVITY RESULTS


,strategy,one_way_cost,total_return,CAGR,max_drawdown,annualized_sharpe,mean_period_return,positive_period_fraction,mean_turnover,mean_transaction_cost,mean_excess_vs_benchmark
0,Top 5%,0.001,2811.010214,1.967327,-0.382802,2.012273,0.104056,0.760870,1.689834,0.001690,0.081872
1,Top 10%,0.001,147.678497,0.983826,-0.266323,1.796914,0.062552,0.663043,1.568359,0.001568,0.040367
2,Top 20%,0.001,24.364950,0.557109,-0.289736,1.476786,0.040055,0.597826,1.378455,0.001378,0.017871
3,Top 5%,0.003,2104.363971,1.852012,-0.386402,1.946864,0.100676,0.750000,1.689834,0.005070,0.078492
4,Top 10%,0.003,111.920983,0.910473,-0.283847,1.707368,0.059415,0.663043,1.568359,0.004705,0.037231
5,Top 20%,0.003,18.828581,0.505472,-0.304949,1.375079,0.037298,0.597826,1.378455,0.004135,0.015114
6,Top 5%,0.005,1573.774423,1.740816,-0.390002,1.881440,0.097297,0.739130,1.689834,0.008449,0.075112
7,Top 10%,0.005,84.689894,0.839617,-0.301013,1.617735,0.056278,0.652174,1.568359,0.007842,0.034094
8,Top 20%,0.005,14.489973,0.455409,-0.319883,1.273346,0.034541,0.576087,1.378455,0.006892,0.012357
9,Top 5%,0.010,757.740251,1.479976,-0.399002,1.717830,0.088848,0.706522,1.689834,0.016898,0.066663



Saved:
/content/NEPSE_ML_PROJECT/reports/cell62_transaction_cost_sensitivity.csv

SURVIVAL CHECK

Top 5%
  Cost 0.1% -> CAGR 196.73%, Sharpe 2.01, Excess/period 8.19%
  Cost 0.3% -> CAGR 185.20%, Sharpe 1.95, Excess/period 7.85%
  Cost 0.5% -> CAGR 174.08%, Sharpe 1.88, Excess/period 7.51%
  Cost 1.0% -> CAGR 148.00%, Sharpe 1.72, Excess/period 6.67%
  Cost 2.0% -> CAGR 102.52%, Sharpe 1.39, Excess/period 4.98%

Top 10%
  Cost 0.1% -> CAGR 98.38%, Sharpe 1.80, Excess/period 4.04%
  Cost 0.3% -> CAGR 91.05%, Sharpe 1.71, Excess/period 3.72%
  Cost 0.5% -> CAGR 83.96%, Sharpe 1.62, Excess/period 3.41%
  Cost 1.0% -> CAGR 67.29%, Sharpe 1.39, Excess/period 2.63%
  Cost 2.0% -> CAGR 38.03%, Sharpe 0.94, Excess/period 1.06%

Top 20%
  Cost 0.1% -> CAGR 55.71%, Sharpe 1.48, Excess/period 1.79%
  Cost 0.3% -> CAGR 50.55%, Sharpe 1.38, Excess/period 1.51%
  Cost 0.5% -> CAGR 45.54%, Sharpe 1.27, Excess/period 1.24%
  Cost 1.0% -> CAGR 33.69%, Sharpe 1.02, Excess/period 0.55%
  Cost 2.0% -> CA

In [ ]:
# ============================================================================
# CELL 63 — HISTORICAL UNIVERSE / SURVIVORSHIP-BIAS AUDIT
# ============================================================================
#
# Purpose:
#   Determine whether the Cell 61 backtest is using a survivorship-biased
#   universe.
#
# We examine:
#   1. Number of securities available by year
#   2. New securities appearing over time
#   3. Securities disappearing before the end of the dataset
#   4. Historical start/end dates by symbol
#   5. Whether the 2017-2024 backtest contains securities that later disappear
#   6. Whether the universe is simply today's surviving securities
#
# NO MODEL TRAINING.
# ============================================================================

import os
import numpy as np
import pandas as pd

PROJECT = "/content/NEPSE_ML_PROJECT"
REPORTS = os.path.join(PROJECT, "reports")

DATASET = os.path.join(
    PROJECT,
    "features/nepse_ml_clean_dataset.parquet"
)

OUTPUT_SUMMARY = os.path.join(
    REPORTS,
    "cell63_survivorship_audit_summary.csv"
)

OUTPUT_SYMBOLS = os.path.join(
    REPORTS,
    "cell63_symbol_history_audit.csv"
)

print("=" * 78)
print("CELL 63 — HISTORICAL UNIVERSE / SURVIVORSHIP-BIAS AUDIT")
print("=" * 78)

# ============================================================================
# LOAD ONLY REQUIRED COLUMNS
# ============================================================================

df = pd.read_parquet(
    DATASET,
    columns=[
        "date",
        "symbol",
        "close"
    ]
)

df["date"] = pd.to_datetime(
    df["date"]
)

df = df.sort_values(
    ["symbol", "date"]
)

print("\nRows:", len(df))
print("Symbols:", df["symbol"].nunique())
print(
    "Date range:",
    df["date"].min().date(),
    "to",
    df["date"].max().date()
)

# ============================================================================
# SYMBOL-LEVEL HISTORY
# ============================================================================

symbol_history = (
    df.groupby("symbol")
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        observations=("date", "size"),
        first_close=("close", "first"),
        last_close=("close", "last")
    )
    .reset_index()
)

# Duration in calendar years
symbol_history["history_years"] = (
    (
        symbol_history["last_date"] -
        symbol_history["first_date"]
    ).dt.days / 365.25
)

# ============================================================================
# CLASSIFY SYMBOLS
# ============================================================================

DATA_END = df["date"].max()

symbol_history["still_active_at_data_end"] = (
    symbol_history["last_date"] >=
    DATA_END - pd.Timedelta(days=90)
)

symbol_history["ended_before_2025"] = (
    symbol_history["last_date"] <
    pd.Timestamp("2025-01-01")
)

symbol_history["ended_before_2024"] = (
    symbol_history["last_date"] <
    pd.Timestamp("2024-01-01")
)

symbol_history["started_after_2017"] = (
    symbol_history["first_date"] >
    pd.Timestamp("2017-01-01")
)

symbol_history["present_during_backtest"] = (
    (
        symbol_history["first_date"] <=
        pd.Timestamp("2024-12-31")
    )
    &
    (
        symbol_history["last_date"] >=
        pd.Timestamp("2017-01-01")
    )
)

# ============================================================================
# BASIC SURVIVORSHIP STATISTICS
# ============================================================================

total_symbols = len(
    symbol_history
)

ended_before_2025 = (
    symbol_history[
        "ended_before_2025"
    ].sum()
)

ended_before_2024 = (
    symbol_history[
        "ended_before_2024"
    ].sum()
)

still_active = (
    symbol_history[
        "still_active_at_data_end"
    ].sum()
)

backtest_symbols = (
    symbol_history[
        "present_during_backtest"
    ].sum()
)

print("\n" + "-" * 78)
print("SYMBOL HISTORY SUMMARY")
print("-" * 78)

print(
    f"Total symbols with history       : "
    f"{total_symbols:,}"
)

print(
    f"Still active near data end       : "
    f"{still_active:,}"
)

print(
    f"Ended before 2025                : "
    f"{ended_before_2025:,}"
)

print(
    f"Ended before 2024                : "
    f"{ended_before_2024:,}"
)

print(
    f"Present at some point 2017-2024  : "
    f"{backtest_symbols:,}"
)

print(
    f"Historical/non-survivor symbols  : "
    f"{ended_before_2025:,}"
)

# ============================================================================
# ANNUAL UNIVERSE
# ============================================================================

print("\n" + "-" * 78)
print("ANNUAL HISTORICAL UNIVERSE")
print("-" * 78)

df["year"] = df["date"].dt.year

annual_universe = (
    df[
        (df["year"] >= 2011) &
        (df["year"] <= 2026)
    ]
    .groupby("year")
    .agg(
        securities=("symbol", "nunique"),
        observations=("symbol", "size")
    )
    .reset_index()
)

annual_universe["new_symbols"] = 0
annual_universe["ended_symbols"] = 0

# ============================================================================
# SYMBOL ENTRY / EXIT BY YEAR
# ============================================================================

symbol_history["first_year"] = (
    symbol_history["first_date"]
    .dt.year
)

symbol_history["last_year"] = (
    symbol_history["last_date"]
    .dt.year
)

entry_counts = (
    symbol_history
    .groupby("first_year")
    .size()
)

exit_counts = (
    symbol_history
    .groupby("last_year")
    .size()
)

annual_universe["new_symbols"] = (
    annual_universe["year"]
    .map(entry_counts)
    .fillna(0)
    .astype(int)
)

annual_universe["ended_symbols"] = (
    annual_universe["year"]
    .map(exit_counts)
    .fillna(0)
    .astype(int)
)

display(
    annual_universe.round(2)
)

# ============================================================================
# BACKTEST-YEAR UNIVERSE
# ============================================================================

print("\n" + "-" * 78)
print("BACKTEST PERIOD UNIVERSE")
print("-" * 78)

backtest_annual = (
    df[
        (df["date"] >= "2017-01-01") &
        (df["date"] <= "2024-12-31")
    ]
    .groupby(df["date"].dt.year)
    .agg(
        securities=("symbol", "nunique"),
        observations=("symbol", "size")
    )
    .reset_index(names="year")
)

display(
    backtest_annual
)

# ============================================================================
# NON-SURVIVOR PARTICIPATION IN BACKTEST
# ============================================================================

backtest_df = df[
    (df["date"] >= "2017-01-01") &
    (df["date"] <= "2024-12-31")
].copy()

backtest_symbols_set = set(
    backtest_df["symbol"].unique()
)

symbol_history["in_backtest"] = (
    symbol_history["symbol"].isin(
        backtest_symbols_set
    )
)

# A symbol is particularly important for survivorship audit if it was
# available during the backtest but its history ended before 2025.
historical_only = symbol_history[
    symbol_history["in_backtest"] &
    symbol_history["ended_before_2025"]
].copy()

print(
    "\nSymbols that participated in 2017-2024 "
    "but disappeared before 2025:",
    len(historical_only)
)

# ============================================================================
# SYMBOLS THAT DISAPPEARED DURING BACKTEST
# ============================================================================

disappeared_during_backtest = symbol_history[
    (
        symbol_history["first_date"] <=
        pd.Timestamp("2024-12-31")
    )
    &
    (
        symbol_history["last_date"] <
        pd.Timestamp("2024-12-31")
    )
    &
    (
        symbol_history["last_date"] >=
        pd.Timestamp("2017-01-01")
    )
].copy()

print(
    "Symbols whose history ended during 2017-2024:",
    len(disappeared_during_backtest)
)

# ============================================================================
# LONG-LIVED SURVIVORS
# ============================================================================

long_lived_survivors = symbol_history[
    (
        symbol_history["first_date"] <=
        pd.Timestamp("2017-01-01")
    )
    &
    (
        symbol_history["last_date"] >=
        pd.Timestamp("2024-12-31")
    )
].copy()

print(
    "Symbols spanning entire 2017-2024 period:",
    len(long_lived_survivors)
)

# ============================================================================
# HISTORY DURATION DISTRIBUTION
# ============================================================================

print("\n" + "-" * 78)
print("HISTORY DURATION")
print("-" * 78)

duration_summary = (
    symbol_history[
        "history_years"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

display(
    duration_summary.to_frame(
        "history_years"
    ).round(2)
)

# ============================================================================
# CHECK FOR PURE SURVIVOR UNIVERSE
# ============================================================================

survivors_at_end = set(
    symbol_history[
        symbol_history[
            "still_active_at_data_end"
        ]
    ]["symbol"]
)

backtest_all_symbols = set(
    backtest_df["symbol"].unique()
)

historical_non_survivors = (
    backtest_all_symbols -
    survivors_at_end
)

survivor_only_fraction = (
    len(
        backtest_all_symbols &
        survivors_at_end
    )
    /
    len(backtest_all_symbols)
)

print("\n" + "-" * 78)
print("SURVIVORSHIP CHECK")
print("-" * 78)

print(
    f"Backtest symbols                  : "
    f"{len(backtest_all_symbols):,}"
)

print(
    f"Backtest symbols still active now : "
    f"{len(backtest_all_symbols & survivors_at_end):,}"
)

print(
    f"Backtest symbols no longer active : "
    f"{len(historical_non_survivors):,}"
)

print(
    f"Fraction that are current survivors: "
    f"{survivor_only_fraction:.2%}"
)

if len(historical_non_survivors) > 0:

    print(
        "\nGOOD SIGN: historical symbols that are no longer "
        "active are present in the backtest universe."
    )

else:

    print(
        "\nWARNING: no historical/non-surviving symbols "
        "were found in the backtest universe."
    )

# ============================================================================
# SAVE DETAILED SYMBOL AUDIT
# ============================================================================

symbol_history.to_csv(
    OUTPUT_SYMBOLS,
    index=False
)

annual_universe.to_csv(
    OUTPUT_SUMMARY,
    index=False
)

print("\nSaved:")
print(
    OUTPUT_SYMBOLS
)
print(
    OUTPUT_SUMMARY
)

print("\n" + "=" * 78)
print("CELL 63 COMPLETE")
print("=" * 78)

CELL 63 — HISTORICAL UNIVERSE / SURVIVORSHIP-BIAS AUDIT

Rows: 633137
Symbols: 651
Date range: 1997-07-20 to 2026-10-06

------------------------------------------------------------------------------
SYMBOL HISTORY SUMMARY
------------------------------------------------------------------------------
Total symbols with history       : 651
Still active near data end       : 474
Ended before 2025                : 139
Ended before 2024                : 122
Present at some point 2017-2024  : 565
Historical/non-survivor symbols  : 139

------------------------------------------------------------------------------
ANNUAL HISTORICAL UNIVERSE
------------------------------------------------------------------------------


,year,securities,observations,new_symbols,ended_symbols
0,2011,10,1572,9,0
1,2012,109,12293,99,0
2,2013,128,14906,25,1
3,2014,153,19870,27,1
4,2015,172,19686,25,1
5,2016,188,24713,23,1
6,2017,232,32318,48,4
7,2018,255,37710,25,8
8,2019,284,42649,35,24
9,2020,287,35985,33,13



------------------------------------------------------------------------------
BACKTEST PERIOD UNIVERSE
------------------------------------------------------------------------------


,year,securities,observations
0,2017,232,32318
1,2018,255,37710
2,2019,284,42649
3,2020,287,35985
4,2021,317,53988
5,2022,351,57570
6,2023,392,62912
7,2024,439,73775



Symbols that participated in 2017-2024 but disappeared before 2025: 135
Symbols whose history ended during 2017-2024: 135
Symbols spanning entire 2017-2024 period: 128

------------------------------------------------------------------------------
HISTORY DURATION
------------------------------------------------------------------------------


,history_years
count,651.00
mean,6.07
std,4.68
min,0.00
10%,0.92
25%,2.18
50%,4.78
75%,9.36
90%,14.11
max,29.21



------------------------------------------------------------------------------
SURVIVORSHIP CHECK
------------------------------------------------------------------------------
Backtest symbols                  : 565
Backtest symbols still active now : 397
Backtest symbols no longer active : 168
Fraction that are current survivors: 70.27%

GOOD SIGN: historical symbols that are no longer active are present in the backtest universe.

Saved:
/content/NEPSE_ML_PROJECT/reports/cell63_symbol_history_audit.csv
/content/NEPSE_ML_PROJECT/reports/cell63_survivorship_audit_summary.csv

CELL 63 COMPLETE


In [ ]:
# ============================================================================
# CELL 64 — SHUFFLED-TARGET PLACEBO TEST
# ============================================================================
#
# PURPOSE:
#   Destroy the relationship between FEATURES and TARGET while keeping:
#     - same dates
#     - same stocks
#     - same training sizes
#     - same XGBoost architecture
#     - same rebalance dates
#     - same transaction-cost framework
#
# EXPECTATION:
#   A genuine model should lose most/all of its ranking advantage.
#
# If shuffled-target performance remains spectacular, there is likely
# a methodological problem in the backtest.
#
# GPU FIRST / CPU FALLBACK
# ============================================================================

import os
import time
import subprocess
import numpy as np
import pandas as pd
import xgboost as xgb

PROJECT = "/content/NEPSE_ML_PROJECT"

DATASET = os.path.join(
    PROJECT,
    "features/nepse_ml_clean_dataset.parquet"
)

REPORTS = os.path.join(
    PROJECT,
    "reports"
)

os.makedirs(
    REPORTS,
    exist_ok=True
)

print("=" * 78)
print("CELL 64 — SHUFFLED-TARGET PLACEBO TEST")
print("=" * 78)

# ============================================================================
# CONFIG
# ============================================================================

FEATURES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

TARGET = "future_return_20d"

TOP_PCTS = [
    0.05,
    0.10,
    0.20
]

HOLDING_PERIOD = 20

ONE_WAY_COST = 0.003

TRAIN_START = pd.Timestamp(
    "2011-01-01"
)

BACKTEST_START = pd.Timestamp(
    "2017-01-01"
)

BACKTEST_END = pd.Timestamp(
    "2024-12-31"
)

SEED = 20261006

# ============================================================================
# HARDWARE
# ============================================================================

print("\n" + "-" * 78)
print("HARDWARE")
print("-" * 78)

GPU_AVAILABLE = False
GPU_NAME = None

try:

    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name",
            "--format=csv,noheader"
        ],
        capture_output=True,
        text=True,
        timeout=10
    )

    if (
        result.returncode == 0
        and result.stdout.strip()
    ):

        GPU_NAME = (
            result.stdout
            .strip()
            .splitlines()[0]
            .strip()
        )

        GPU_AVAILABLE = True

except Exception:
    pass

if GPU_AVAILABLE:

    DEVICE = "cuda"

    print(
        "GPU:",
        GPU_NAME
    )

    print(
        "XGBoost device: CUDA"
    )

else:

    DEVICE = "cpu"

    print(
        "GPU: not available"
    )

    print(
        "XGBoost device: CPU"
    )

# ============================================================================
# LOAD DATA
# ============================================================================

print("\n" + "-" * 78)
print("LOADING DATA")
print("-" * 78)

columns = list(
    dict.fromkeys(
        [
            "date",
            "symbol",
            "close",
            TARGET
        ] + FEATURES
    )
)

df = pd.read_parquet(
    DATASET,
    columns=columns
)

df["date"] = pd.to_datetime(
    df["date"]
)

df = df.sort_values(
    [
        "symbol",
        "date"
    ]
).reset_index(
    drop=True
)

print(
    "Rows:",
    len(df)
)

print(
    "Symbols:",
    df["symbol"].nunique()
)

# ============================================================================
# REALIZED FUTURE RETURN
# ============================================================================

df["future_close_20d"] = (
    df.groupby(
        "symbol",
        sort=False
    )["close"]
    .shift(
        -HOLDING_PERIOD
    )
)

df["realized_return_20d"] = (
    df["future_close_20d"]
    /
    df["close"]
    -
    1.0
)

# ============================================================================
# BACKTEST
# ============================================================================

bt = df[
    (df["date"] >= BACKTEST_START)
    &
    (df["date"] <= BACKTEST_END)
    &
    (df["realized_return_20d"].notna())
].copy()

train_pool = df[
    (df["date"] >= TRAIN_START)
    &
    (df[TARGET].notna())
].copy()

trading_dates = np.array(
    sorted(
        bt["date"].unique()
    )
)

rebalance_dates = (
    trading_dates[
        ::HOLDING_PERIOD
    ]
)

print(
    "\nBacktest rows:",
    len(bt)
)

print(
    "Training rows:",
    len(train_pool)
)

print(
    "Rebalance dates:",
    len(rebalance_dates)
)

# ============================================================================
# IMPORTANT:
# SHUFFLE THE TARGET ONCE PER TRAINING WINDOW
#
# The features remain exactly as they are.
# The target values are randomly permuted.
# ============================================================================

rng = np.random.default_rng(
    SEED
)

records = []

start_time = time.time()

print("\n" + "-" * 78)
print("RUNNING SHUFFLED-TARGET PLACEBO")
print("-" * 78)

for i, rebalance_date in enumerate(
    rebalance_dates
):

    loop_start = time.time()

    rebalance_date = pd.Timestamp(
        rebalance_date
    )

    current_idx = np.searchsorted(
        trading_dates,
        np.datetime64(
            rebalance_date
        )
    )

    if current_idx < HOLDING_PERIOD:
        continue

    # ------------------------------------------------------------------------
    # SAME PURGE AS CELL 61
    # ------------------------------------------------------------------------

    train_cutoff = pd.Timestamp(
        trading_dates[
            current_idx -
            HOLDING_PERIOD
        ]
    )

    train = train_pool[
        train_pool["date"] <=
        train_cutoff
    ].copy()

    current = bt[
        bt["date"] ==
        rebalance_date
    ].copy()

    if len(current) < 20:
        continue

    # ------------------------------------------------------------------------
    # TRAINING-ONLY IMPUTATION
    # ------------------------------------------------------------------------

    medians = train[
        FEATURES
    ].median()

    X_train = (
        train[
            FEATURES
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .fillna(medians)
        .astype(np.float32)
    )

    X_current = (
        current[
            FEATURES
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .fillna(medians)
        .astype(np.float32)
    )

    # ------------------------------------------------------------------------
    # REAL TARGET
    # ------------------------------------------------------------------------

    y_real = (
        train[TARGET]
        .astype(np.float32)
        .values
    )

    # ------------------------------------------------------------------------
    # DESTROY FEATURE/TARGET RELATIONSHIP
    # ------------------------------------------------------------------------

    y_shuffled = (
        rng.permutation(
            y_real
        )
    )

    # ------------------------------------------------------------------------
    # MODEL
    # ------------------------------------------------------------------------

    model = xgb.XGBRegressor(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=30,
        subsample=0.80,
        colsample_bytree=0.85,
        reg_alpha=1.0,
        reg_lambda=10.0,
        gamma=0.10,
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        device=DEVICE,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_shuffled,
        verbose=False
    )

    # ------------------------------------------------------------------------
    # PREDICTION
    # ------------------------------------------------------------------------

    current[
        "predicted_return"
    ] = model.predict(
        X_current
    )

    current = current.sort_values(
        "predicted_return",
        ascending=False
    )

    # ------------------------------------------------------------------------
    # BENCHMARK
    # ------------------------------------------------------------------------

    benchmark_return = (
        current[
            "realized_return_20d"
        ].mean()
    )

    # ------------------------------------------------------------------------
    # PORTFOLIOS
    # ------------------------------------------------------------------------

    for top_pct in TOP_PCTS:

        n_select = max(
            1,
            int(
                np.floor(
                    len(current)
                    *
                    top_pct
                )
            )
        )

        selected = (
            current
            .head(
                n_select
            )
        )

        gross_return = (
            selected[
                "realized_return_20d"
            ].mean()
        )

        # --------------------------------------------------------------------
        # IMPORTANT PLACEBO APPROXIMATION:
        # Use the same average turnover observed by Cell 61 for each strategy.
        #
        # This isolates the effect of the predictive ranking while avoiding
        # another stateful portfolio-selection calculation.
        # --------------------------------------------------------------------

        records.append({

            "date":
                rebalance_date,

            "top_pct":
                top_pct,

            "gross_return":
                gross_return,

            "benchmark_return":
                benchmark_return,

            "n_stocks":
                n_select,

            "train_rows":
                len(train)
        })

    elapsed = (
        time.time()
        -
        loop_start
    )

    if (
        i == 0
        or
        (i + 1) % 5 == 0
        or
        i == len(rebalance_dates) - 1
    ):

        remaining = (
            len(rebalance_dates)
            -
            i
            -
            1
        )

        eta = (
            remaining *
            elapsed
        )

        print(
            f"\r"
            f"Rebalance "
            f"{i+1}/"
            f"{len(rebalance_dates)}"
            f" | {elapsed:.1f}s"
            f" | ETA "
            f"{eta/60:.1f} min",
            end=""
        )

print()

# ============================================================================
# RESULTS
# ============================================================================

placebo = pd.DataFrame(
    records
)

print(
    "\nPortfolio observations:",
    len(placebo)
)

# ============================================================================
# USE CELL 61 ACTUAL TURNOVER
# ============================================================================
#
# We already measured turnover in Cell 61.
# Merge the actual turnover by date/top_pct so the placebo has realistic
# transaction costs rather than assuming zero turnover.
# ============================================================================

cell61_file = os.path.join(
    REPORTS,
    "cell61_leakage_free_backtest_periods.csv"
)

cell61 = pd.read_csv(
    cell61_file
)

cell61["date"] = pd.to_datetime(
    cell61["date"]
)

turnover_lookup = cell61[
    [
        "date",
        "top_pct",
        "turnover"
    ]
].drop_duplicates(
    [
        "date",
        "top_pct"
    ]
)

placebo = placebo.merge(
    turnover_lookup,
    on=[
        "date",
        "top_pct"
    ],
    how="left"
)

placebo["transaction_cost"] = (
    placebo["turnover"]
    *
    ONE_WAY_COST
)

placebo["net_return"] = (
    placebo["gross_return"]
    -
    placebo["transaction_cost"]
)

# ============================================================================
# METRICS
# ============================================================================

PERIODS_PER_YEAR = (
    252.0 /
    HOLDING_PERIOD
)

def calculate_metrics(
    returns
):

    returns = np.asarray(
        returns,
        dtype=float
    )

    returns = returns[
        np.isfinite(returns)
    ]

    equity = np.cumprod(
        1.0 + returns
    )

    total_return = (
        equity[-1] - 1.0
    )

    years = (
        len(returns)
        /
        PERIODS_PER_YEAR
    )

    cagr = (
        equity[-1]
        **
        (1.0 / years)
        -
        1.0
    )

    running_max = (
        np.maximum.accumulate(
            equity
        )
    )

    drawdown = (
        equity /
        running_max
        -
        1.0
    )

    max_dd = drawdown.min()

    std = returns.std(
        ddof=1
    )

    sharpe = (
        returns.mean()
        /
        std
        *
        np.sqrt(
            PERIODS_PER_YEAR
        )
    ) if std > 0 else np.nan

    return {
        "observations":
            len(returns),

        "total_return":
            total_return,

        "CAGR":
            cagr,

        "max_drawdown":
            max_dd,

        "annualized_sharpe":
            sharpe,

        "mean_period_return":
            returns.mean(),

        "positive_fraction":
            (
                returns > 0
            ).mean()
    }

metrics = []

for top_pct, group in placebo.groupby(
    "top_pct"
):

    m = calculate_metrics(
        group[
            "net_return"
        ]
    )

    m[
        "strategy"
    ] = (
        f"Placebo Top {int(top_pct * 100)}%"
    )

    m[
        "top_pct"
    ] = top_pct

    m[
        "mean_excess_vs_benchmark"
    ] = (
        group[
            "net_return"
        ]
        -
        group[
            "benchmark_return"
        ]
    ).mean()

    metrics.append(
        m
    )

# Benchmark

benchmark_group = (
    placebo
    .drop_duplicates("date")
)

bm = calculate_metrics(
    benchmark_group[
        "benchmark_return"
    ]
)

bm[
    "strategy"
] = "Benchmark"

bm[
    "top_pct"
] = np.nan

bm[
    "mean_excess_vs_benchmark"
] = 0.0

metrics.append(
    bm
)

metrics_df = pd.DataFrame(
    metrics
)

# ============================================================================
# DISPLAY
# ============================================================================

print("\n" + "=" * 78)
print("SHUFFLED-TARGET PLACEBO RESULTS")
print("=" * 78)

display(
    metrics_df[
        [
            "strategy",
            "observations",
            "total_return",
            "CAGR",
            "max_drawdown",
            "annualized_sharpe",
            "mean_period_return",
            "positive_fraction",
            "mean_excess_vs_benchmark"
        ]
    ].round(6)
)

# ============================================================================
# COMPARE AGAINST REAL CELL 61
# ============================================================================

real_file = os.path.join(
    REPORTS,
    "cell61_leakage_free_backtest_metrics.csv"
)

real = pd.read_csv(
    real_file
)

print("\n" + "=" * 78)
print("REAL MODEL VS PLACEBO")
print("=" * 78)

comparison = []

for top_pct in TOP_PCTS:

    real_row = real[
        real["top_pct"] ==
        top_pct
    ].iloc[0]

    placebo_row = metrics_df[
        metrics_df["top_pct"] ==
        top_pct
    ].iloc[0]

    comparison.append({

        "strategy":
            f"Top {int(top_pct * 100)}%",

        "real_CAGR":
            real_row["CAGR"],

        "placebo_CAGR":
            placebo_row["CAGR"],

        "real_Sharpe":
            real_row[
                "annualized_sharpe"
            ],

        "placebo_Sharpe":
            placebo_row[
                "annualized_sharpe"
            ],

        "real_excess":
            real_row[
                "mean_excess_return"
            ],

        "placebo_excess":
            placebo_row[
                "mean_excess_vs_benchmark"
            ]
    })

comparison_df = pd.DataFrame(
    comparison
)

display(
    comparison_df.round(6)
)

# ============================================================================
# SAVE
# ============================================================================

placebo_file = os.path.join(
    REPORTS,
    "cell64_shuffled_target_placebo_periods.csv"
)

metrics_file = os.path.join(
    REPORTS,
    "cell64_shuffled_target_placebo_metrics.csv"
)

comparison_file = os.path.join(
    REPORTS,
    "cell64_real_vs_placebo_comparison.csv"
)

placebo.to_csv(
    placebo_file,
    index=False
)

metrics_df.to_csv(
    metrics_file,
    index=False
)

comparison_df.to_csv(
    comparison_file,
    index=False
)

print("\nSaved:")
print(placebo_file)
print(metrics_file)
print(comparison_file)

print("\n" + "=" * 78)
print("CELL 64 COMPLETE")
print("=" * 78)

print(
    "\nGPU:",
    GPU_NAME if GPU_AVAILABLE else "CPU"
)

print(
    "Real model remains untouched."
)

print(
    "2025-2026 test set was NOT used."
)

CELL 64 — SHUFFLED-TARGET PLACEBO TEST

------------------------------------------------------------------------------
HARDWARE
------------------------------------------------------------------------------
GPU: Tesla T4
XGBoost device: CUDA

------------------------------------------------------------------------------
LOADING DATA
------------------------------------------------------------------------------
Rows: 633137
Symbols: 651

Backtest rows: 394268
Training rows: 618011
Rebalance dates: 93

------------------------------------------------------------------------------
RUNNING SHUFFLED-TARGET PLACEBO
------------------------------------------------------------------------------
Rebalance 93/93 | 2.0s | ETA 0.0 min

Portfolio observations: 276

SHUFFLED-TARGET PLACEBO RESULTS


,strategy,observations,total_return,CAGR,max_drawdown,annualized_sharpe,mean_period_return,positive_fraction,mean_excess_vs_benchmark
0,Placebo Top 5%,92,10.835790,0.402754,-0.527472,0.933638,0.034411,0.543478,0.012226
1,Placebo Top 10%,92,6.577000,0.319632,-0.486131,0.908012,0.027515,0.576087,0.005331
2,Placebo Top 20%,92,3.838677,0.241017,-0.502026,0.803727,0.021428,0.543478,-0.000756
3,Benchmark,92,4.580416,0.265496,-0.372964,0.937581,0.022184,0.543478,0.000000



REAL MODEL VS PLACEBO


,strategy,real_CAGR,placebo_CAGR,real_Sharpe,placebo_Sharpe,real_excess,placebo_excess
0,Top 5%,1.852012,0.402754,1.946864,0.933638,0.078492,0.012226
1,Top 10%,0.910473,0.319632,1.707368,0.908012,0.037231,0.005331
2,Top 20%,0.505472,0.241017,1.375079,0.803727,0.015114,-0.000756



Saved:
/content/NEPSE_ML_PROJECT/reports/cell64_shuffled_target_placebo_periods.csv
/content/NEPSE_ML_PROJECT/reports/cell64_shuffled_target_placebo_metrics.csv
/content/NEPSE_ML_PROJECT/reports/cell64_real_vs_placebo_comparison.csv

CELL 64 COMPLETE

GPU: Tesla T4
Real model remains untouched.
2025-2026 test set was NOT used.


In [ ]:
# ==============================================================================
# CELL 65 — NEXT-DAY EXECUTION / STRICT CAUSALITY BACKTEST
# ==============================================================================
#
# Purpose:
#   Re-run the stock-ranking strategy under a stricter execution assumption:
#
#   Signal:
#       information available through day T close
#
#   Entry:
#       day T+1 close
#
#   Exit:
#       day T+21 close
#
#   Therefore the realized holding period is exactly 20 trading sessions,
#   starting AFTER the signal day.
#
#   This checks that the Cell 61 result is not dependent on same-close
#   execution assumptions.
#
# Rules:
#   - stock-only features
#   - NO market features
#   - NO sector features
#   - NO relative features
#   - 20-session purge
#   - 20-session holding period
#   - 0.3% one-way transaction cost
#   - train start: 2011
#   - backtest: 2017-2024
#   - 2025-2026 untouched
# ==============================================================================

import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb

warnings.filterwarnings("ignore")

PROJECT = "/content/NEPSE_ML_PROJECT"
DATA_PATH = f"{PROJECT}/features/nepse_ml_clean_dataset.parquet"

REPORT_DIR = f"{PROJECT}/reports"
MODEL_DIR = f"{PROJECT}/models"
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ------------------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------------------

TARGET = "future_return_20d"

FEATURES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

TOP_PCTS = [0.05, 0.10, 0.20]

TRAIN_START = "2011-01-01"
BACKTEST_START = "2017-01-01"
BACKTEST_END = "2024-12-31"

HOLDING_SESSIONS = 20
PURGE_SESSIONS = 20

ONE_WAY_COST = 0.003

RANDOM_STATE = 42

# ------------------------------------------------------------------------------
# HARDWARE
# ------------------------------------------------------------------------------

try:
    gpu_info = xgb.build_info()

    if "USE_CUDA" in gpu_info.get("USE_CUDA", ""):
        pass

    # Actual runtime test
    test_model = xgb.XGBRegressor(
        n_estimators=1,
        max_depth=2,
        tree_method="hist",
        device="cuda",
        random_state=RANDOM_STATE,
    )

    DEVICE = "cuda"

    print("GPU test: CUDA requested")

except Exception:
    DEVICE = "cpu"

print()
print("=" * 78)
print("CELL 65 — NEXT-DAY EXECUTION BACKTEST")
print("=" * 78)
print()
print("XGBoost device:", DEVICE)

# ------------------------------------------------------------------------------
# LOAD
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("LOADING DATA")
print("-" * 78)

df = pd.read_parquet(DATA_PATH)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

print("Rows:", len(df))
print("Symbols:", df["symbol"].nunique())
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

# ------------------------------------------------------------------------------
# ONLY REQUIRED COLUMNS
# ------------------------------------------------------------------------------

required = ["date", "symbol", "close", TARGET] + FEATURES

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

data = df[required].copy()

# ------------------------------------------------------------------------------
# NEXT-DAY ENTRY / 20-DAY HOLDING
# ------------------------------------------------------------------------------
#
# At signal day T:
#   signal_close = close[T]
#
# Entry:
#   close[T+1]
#
# Exit:
#   close[T+21]
#
# Realized return:
#   close[T+21] / close[T+1] - 1
#
# This deliberately does NOT use close[T] as the entry price.
# ------------------------------------------------------------------------------

g = data.groupby("symbol", sort=False)

data["entry_close_next_day"] = g["close"].shift(-1)
data["exit_close_20d_after_entry"] = g["close"].shift(-(HOLDING_SESSIONS + 1))

data["strict_realized_return"] = (
    data["exit_close_20d_after_entry"]
    / data["entry_close_next_day"]
    - 1.0
)

# Valid realized observations
data = data.replace([np.inf, -np.inf], np.nan)

# ------------------------------------------------------------------------------
# BACKTEST DATES
# ------------------------------------------------------------------------------

bt = data[
    (data["date"] >= BACKTEST_START) &
    (data["date"] <= BACKTEST_END)
].copy()

print()
print("Backtest rows:", len(bt))

# ------------------------------------------------------------------------------
# REBALANCE DATES
# ------------------------------------------------------------------------------
#
# Use the SAME rebalance dates as Cell 61 so this is an apples-to-apples
# execution-timing comparison.
# ------------------------------------------------------------------------------

cell61_periods_path = f"{REPORT_DIR}/cell61_leakage_free_backtest_periods.csv"

if not os.path.exists(cell61_periods_path):
    raise FileNotFoundError(
        "Cell 61 periods file not found:\n"
        + cell61_periods_path
    )

cell61_periods = pd.read_csv(cell61_periods_path)
cell61_periods["date"] = pd.to_datetime(cell61_periods["date"])

rebalance_dates = sorted(cell61_periods["date"].unique())

rebalance_dates = [
    d for d in rebalance_dates
    if pd.Timestamp(BACKTEST_START)
    <= d
    <= pd.Timestamp(BACKTEST_END)
]

print("Rebalance dates:", len(rebalance_dates))

# ------------------------------------------------------------------------------
# TRAINING MODEL
# ------------------------------------------------------------------------------

def make_model():

    return xgb.XGBRegressor(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=30,
        subsample=0.80,
        colsample_bytree=0.85,
        reg_alpha=1.0,
        reg_lambda=10.0,
        gamma=0.10,
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        device=DEVICE,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

# ------------------------------------------------------------------------------
# TRAINING MEDIANS
# ------------------------------------------------------------------------------

train_all = data[
    (data["date"] >= TRAIN_START) &
    (data["date"] < BACKTEST_START)
].copy()

# We will calculate medians independently inside each training window.
# This is intentionally not fit using future data.
# ------------------------------------------------------------------------------

results = []

start_time = time.time()

print()
print("-" * 78)
print("RUNNING STRICT NEXT-DAY EXECUTION BACKTEST")
print("-" * 78)

for i, rebalance_date in enumerate(rebalance_dates, 1):

    rebalance_date = pd.Timestamp(rebalance_date)

    # --------------------------------------------------------------------------
    # TRAINING END
    # --------------------------------------------------------------------------
    #
    # Exclude the most recent 20 sessions before the signal date.
    # --------------------------------------------------------------------------

    dates_before = np.sort(
        train_all.loc[
            train_all["date"] < rebalance_date,
            "date"
        ].unique()
    )

    if len(dates_before) <= PURGE_SESSIONS:
        continue

    training_end_date = dates_before[-PURGE_SESSIONS]

    train = train_all[
        train_all["date"] < training_end_date
    ].copy()

    # --------------------------------------------------------------------------
    # CURRENT CROSS-SECTION
    # --------------------------------------------------------------------------

    current = data[
        data["date"] == rebalance_date
    ].copy()

    if current.empty:
        continue

    # Need strict next-day entry and 20-session exit.
    current = current[
        current["strict_realized_return"].notna()
    ].copy()

    if len(current) < 10:
        continue

    # --------------------------------------------------------------------------
    # MEDIAN IMPUTATION — TRAINING ONLY
    # --------------------------------------------------------------------------

    medians = train[FEATURES].median()

    X_train = train[FEATURES].fillna(medians).astype(np.float32)
    y_train = train[TARGET].astype(np.float32)

    X_current = current[FEATURES].fillna(medians).astype(np.float32)

    # Remove invalid target rows from training
    valid_train = np.isfinite(y_train.values)

    X_train = X_train.loc[valid_train]
    y_train = y_train.loc[valid_train]

    if len(X_train) < 10000:
        continue

    # --------------------------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------------------------

    model = make_model()

    model.fit(
        X_train,
        y_train,
        verbose=False
    )

    # --------------------------------------------------------------------------
    # PREDICT
    # --------------------------------------------------------------------------

    current["predicted_return"] = model.predict(X_current)

    current = current.sort_values(
        "predicted_return",
        ascending=False
    )

    # --------------------------------------------------------------------------
    # PORTFOLIOS
    # --------------------------------------------------------------------------

    for top_pct in TOP_PCTS:

        n_select = max(
            1,
            int(np.floor(len(current) * top_pct))
        )

        selected = current.head(n_select)

        gross_return = selected[
            "strict_realized_return"
        ].mean()

        if not np.isfinite(gross_return):
            continue

        results.append({
            "date": rebalance_date,
            "top_pct": top_pct,
            "n_stocks": len(selected),
            "gross_return": float(gross_return),
        })

    elapsed = time.time() - start_time
    per_iter = elapsed / i
    eta = per_iter * (len(rebalance_dates) - i)

    print(
        f"\rRebalance {i}/{len(rebalance_dates)} "
        f"| {elapsed:.1f}s "
        f"| ETA {eta/60:.1f} min",
        end=""
    )

print("\n")

# ------------------------------------------------------------------------------
# RESULTS DATAFRAME
# ------------------------------------------------------------------------------

periods = pd.DataFrame(results)

if periods.empty:
    raise RuntimeError("No backtest observations were generated.")

periods["date"] = pd.to_datetime(periods["date"])

print("Portfolio observations:", len(periods))

# ------------------------------------------------------------------------------
# BENCHMARK
# ------------------------------------------------------------------------------
#
# Equal-weight universe on the same signal dates, using the SAME strict
# next-day realization.
# ------------------------------------------------------------------------------

benchmark_rows = []

for d in sorted(periods["date"].unique()):

    d = pd.Timestamp(d)

    current = data[
        data["date"] == d
    ]

    current = current[
        current["strict_realized_return"].notna()
    ]

    if current.empty:
        continue

    benchmark_rows.append({
        "date": d,
        "benchmark_return": current["strict_realized_return"].mean(),
    })

benchmark = pd.DataFrame(benchmark_rows)

periods = periods.merge(
    benchmark,
    on="date",
    how="left"
)

# ------------------------------------------------------------------------------
# TURNOVER
# ------------------------------------------------------------------------------
#
# We calculate turnover independently for each strategy.
# Portfolio holdings are the selected symbols on each rebalance date.
#
# Because this cell reconstructs rankings, we need to store selected symbols.
# For this strict execution test, use the period-level approximation from
# realized ranking overlap by rerunning the selection information is expensive.
#
# Therefore we use Cell 61 turnover only as a conservative comparable
# transaction-cost estimate.
# ------------------------------------------------------------------------------

cell61_turnover = cell61_periods[
    ["date", "top_pct", "turnover"]
].drop_duplicates(
    ["date", "top_pct"]
)

periods = periods.merge(
    cell61_turnover,
    on=["date", "top_pct"],
    how="left"
)

periods["turnover"] = periods["turnover"].fillna(0.0)

periods["transaction_cost"] = (
    periods["turnover"] * ONE_WAY_COST
)

periods["net_return"] = (
    periods["gross_return"]
    - periods["transaction_cost"]
)

periods["excess_return"] = (
    periods["net_return"]
    - periods["benchmark_return"]
)

# ------------------------------------------------------------------------------
# METRICS
# ------------------------------------------------------------------------------

def calculate_metrics(x):

    x = x.sort_values("date").copy()

    returns = x["net_return"].astype(float).values

    equity = np.cumprod(1.0 + returns)

    total_return = equity[-1] - 1.0

    years = (
        (x["date"].iloc[-1] - x["date"].iloc[0]).days
        / 365.25
    )

    cagr = (
        equity[-1] ** (1.0 / years) - 1.0
        if years > 0
        else np.nan
    )

    running_max = np.maximum.accumulate(equity)

    drawdown = equity / running_max - 1.0

    max_drawdown = drawdown.min()

    mean_return = returns.mean()
    std_return = returns.std(ddof=1)

    # Approximately 13 twenty-session periods per year.
    annualized_sharpe = (
        mean_return / std_return * np.sqrt(13)
        if std_return > 0
        else np.nan
    )

    return {
        "observations": len(x),
        "total_return": total_return,
        "CAGR": cagr,
        "max_drawdown": max_drawdown,
        "annualized_sharpe": annualized_sharpe,
        "mean_period_return": mean_return,
        "median_period_return": np.median(returns),
        "positive_fraction": np.mean(returns > 0),
        "mean_excess_vs_benchmark": x[
            "excess_return"
        ].mean(),
        "mean_turnover": x[
            "turnover"
        ].mean(),
        "mean_transaction_cost": x[
            "transaction_cost"
        ].mean(),
    }

# ------------------------------------------------------------------------------
# STRATEGY METRICS
# ------------------------------------------------------------------------------

metric_rows = []

for top_pct in TOP_PCTS:

    x = periods[
        periods["top_pct"] == top_pct
    ].copy()

    m = calculate_metrics(x)

    m["strategy"] = f"Next-Day Top {int(top_pct*100)}%"

    metric_rows.append(m)

# ------------------------------------------------------------------------------
# STRICT BENCHMARK METRICS
# ------------------------------------------------------------------------------

benchmark_metrics = benchmark.copy()

benchmark_returns = benchmark_metrics[
    "benchmark_return"
].astype(float).values

benchmark_equity = np.cumprod(
    1.0 + benchmark_returns
)

benchmark_total = benchmark_equity[-1] - 1.0

benchmark_years = (
    (
        benchmark_metrics["date"].iloc[-1]
        - benchmark_metrics["date"].iloc[0]
    ).days
    / 365.25
)

benchmark_cagr = (
    benchmark_equity[-1] ** (1.0 / benchmark_years) - 1.0
)

benchmark_running_max = np.maximum.accumulate(
    benchmark_equity
)

benchmark_dd = (
    benchmark_equity
    / benchmark_running_max
    - 1.0
)

benchmark_std = benchmark_returns.std(ddof=1)

benchmark_sharpe = (
    benchmark_returns.mean()
    / benchmark_std
    * np.sqrt(13)
)

metric_rows.append({
    "strategy": "Benchmark",
    "observations": len(benchmark_returns),
    "total_return": benchmark_total,
    "CAGR": benchmark_cagr,
    "max_drawdown": benchmark_dd.min(),
    "annualized_sharpe": benchmark_sharpe,
    "mean_period_return": benchmark_returns.mean(),
    "median_period_return": np.median(benchmark_returns),
    "positive_fraction": np.mean(benchmark_returns > 0),
    "mean_excess_vs_benchmark": 0.0,
    "mean_turnover": np.nan,
    "mean_transaction_cost": 0.0,
})

metrics = pd.DataFrame(metric_rows)

# ------------------------------------------------------------------------------
# PRINT
# ------------------------------------------------------------------------------

print("=" * 78)
print("STRICT NEXT-DAY EXECUTION RESULTS")
print("=" * 78)

print(
    metrics[
        [
            "strategy",
            "observations",
            "total_return",
            "CAGR",
            "max_drawdown",
            "annualized_sharpe",
            "mean_period_return",
            "positive_fraction",
            "mean_excess_vs_benchmark",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------------------------
# SAVE
# ------------------------------------------------------------------------------

periods_path = (
    f"{REPORT_DIR}/cell65_next_day_execution_periods.csv"
)

metrics_path = (
    f"{REPORT_DIR}/cell65_next_day_execution_metrics.csv"
)

periods.to_csv(periods_path, index=False)
metrics.to_csv(metrics_path, index=False)

# ------------------------------------------------------------------------------
# REAL VS CELL 61
# ------------------------------------------------------------------------------

cell61_metrics_path = (
    f"{REPORT_DIR}/cell61_leakage_free_backtest_metrics.csv"
)

if os.path.exists(cell61_metrics_path):

    cell61_metrics = pd.read_csv(
        cell61_metrics_path
    )

    compare_rows = []

    for top_pct in TOP_PCTS:

        strategy_name = f"Next-Day Top {int(top_pct*100)}%"

        a = metrics[
            metrics["strategy"] == strategy_name
        ]

        if a.empty:
            continue

        a = a.iloc[0]

        # Find Cell 61 equivalent
        possible = cell61_metrics[
            cell61_metrics["strategy"].astype(str).str.contains(
                f"Top {int(top_pct*100)}",
                regex=False
            )
        ]

        if possible.empty:
            continue

        b = possible.iloc[0]

        compare_rows.append({
            "top_pct": top_pct,
            "cell61_CAGR": b["CAGR"],
            "next_day_CAGR": a["CAGR"],
            "cell61_Sharpe": b["annualized_sharpe"],
            "next_day_Sharpe": a["annualized_sharpe"],
            "cell61_excess": b[
                "mean_excess_vs_benchmark"
            ],
            "next_day_excess": a[
                "mean_excess_vs_benchmark"
            ],
        })

    comparison = pd.DataFrame(compare_rows)

    comparison_path = (
        f"{REPORT_DIR}/cell65_vs_cell61_comparison.csv"
    )

    comparison.to_csv(
        comparison_path,
        index=False
    )

    print()
    print("=" * 78)
    print("CELL 61 VS STRICT NEXT-DAY EXECUTION")
    print("=" * 78)

    print(
        comparison.to_string(index=False)
    )

# ------------------------------------------------------------------------------
# DONE
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("CELL 65 COMPLETE")
print("=" * 78)

print()
print("Saved:")
print(periods_path)
print(metrics_path)

if os.path.exists(
    f"{REPORT_DIR}/cell65_vs_cell61_comparison.csv"
):
    print(
        f"{REPORT_DIR}/cell65_vs_cell61_comparison.csv"
    )

print()
print("2025-2026 test set was NOT used.")


CELL 65 — NEXT-DAY EXECUTION BACKTEST

XGBoost device: cpu

------------------------------------------------------------------------------
LOADING DATA
------------------------------------------------------------------------------
Rows: 633137
Symbols: 651
Date range: 1997-07-20 to 2026-10-06

Backtest rows: 396907
Rebalance dates: 92

------------------------------------------------------------------------------
RUNNING STRICT NEXT-DAY EXECUTION BACKTEST
------------------------------------------------------------------------------
Rebalance 92/92 | 224.2s | ETA 0.0 min

Portfolio observations: 276
STRICT NEXT-DAY EXECUTION RESULTS
        strategy  observations  total_return     CAGR  max_drawdown  annualized_sharpe  mean_period_return  positive_fraction  mean_excess_vs_benchmark
 Next-Day Top 5%            92    813.978818 1.331676     -0.369738           1.967600            0.086292           0.673913                  0.063167
Next-Day Top 10%            92     76.794212 0.733087 

KeyError: 'mean_excess_vs_benchmark'

In [ ]:
# ==============================================================================
# CELL 65A — FIX CELL 65 VS CELL 61 COMPARISON
# ==============================================================================

import os
import pandas as pd

PROJECT = "/content/NEPSE_ML_PROJECT"
REPORT_DIR = f"{PROJECT}/reports"

cell61_path = f"{REPORT_DIR}/cell61_leakage_free_backtest_metrics.csv"
cell65_path = f"{REPORT_DIR}/cell65_next_day_execution_metrics.csv"

cell61 = pd.read_csv(cell61_path)
cell65 = pd.read_csv(cell65_path)

print("=" * 78)
print("CELL 61 VS CELL 65 — EXECUTION TIMING COMPARISON")
print("=" * 78)

print("\nCell 61 columns:")
print(list(cell61.columns))

print("\nCell 65 columns:")
print(list(cell65.columns))

# ------------------------------------------------------------------------------
# Find strategy rows robustly
# ------------------------------------------------------------------------------

comparison_rows = []

for top_pct in [0.05, 0.10, 0.20]:

    pct = int(top_pct * 100)

    # Cell 61
    c61 = cell61[
        cell61["strategy"].astype(str).str.contains(
            f"Top {pct}%",
            regex=False
        )
    ]

    # Cell 65
    c65 = cell65[
        cell65["strategy"].astype(str).str.contains(
            f"Top {pct}%",
            regex=False
        )
    ]

    if c61.empty or c65.empty:
        print(f"WARNING: Could not find Top {pct}% in both files")
        continue

    c61 = c61.iloc[0]
    c65 = c65.iloc[0]

    # ----------------------------------------------------------------------------
    # Cell 61 excess column may have a different name.
    # Find it safely.
    # ----------------------------------------------------------------------------

    excess_candidates_61 = [
        "mean_excess_vs_benchmark",
        "mean_excess_return",
        "mean_excess",
        "excess_return",
    ]

    excess_col_61 = next(
        (
            c for c in excess_candidates_61
            if c in cell61.columns
        ),
        None
    )

    if excess_col_61 is None:
        excess_61 = float("nan")
    else:
        excess_61 = c61[excess_col_61]

    # Cell 65 uses mean_excess_vs_benchmark
    excess_65 = c65.get(
        "mean_excess_vs_benchmark",
        float("nan")
    )

    comparison_rows.append({
        "strategy": f"Top {pct}%",
        "cell61_CAGR": c61["CAGR"],
        "cell65_next_day_CAGR": c65["CAGR"],
        "cell61_Sharpe": c61["annualized_sharpe"],
        "cell65_next_day_Sharpe": c65["annualized_sharpe"],
        "cell61_excess": excess_61,
        "cell65_next_day_excess": excess_65,
        "cell61_max_drawdown": c61["max_drawdown"],
        "cell65_next_day_max_drawdown": c65["max_drawdown"],
    })

comparison = pd.DataFrame(comparison_rows)

print()
print(comparison.to_string(index=False))

# ------------------------------------------------------------------------------
# Calculate retention ratios
# ------------------------------------------------------------------------------

comparison["CAGR_retention"] = (
    comparison["cell65_next_day_CAGR"]
    / comparison["cell61_CAGR"]
)

comparison["Sharpe_retention"] = (
    comparison["cell65_next_day_Sharpe"]
    / comparison["cell61_Sharpe"]
)

comparison["excess_retention"] = (
    comparison["cell65_next_day_excess"]
    / comparison["cell61_excess"]
)

print()
print("=" * 78)
print("ROBUSTNESS RETENTION")
print("=" * 78)

print(
    comparison[
        [
            "strategy",
            "CAGR_retention",
            "Sharpe_retention",
            "excess_retention",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------------------------
# Save
# ------------------------------------------------------------------------------

output_path = (
    f"{REPORT_DIR}/cell65_vs_cell61_comparison.csv"
)

comparison.to_csv(output_path, index=False)

print()
print("Saved:")
print(output_path)

print()
print("=" * 78)
print("CELL 65A COMPLETE")
print("=" * 78)

CELL 61 VS CELL 65 — EXECUTION TIMING COMPARISON

Cell 61 columns:
['observations', 'total_return', 'CAGR', 'max_drawdown', 'annualized_sharpe', 'mean_20d_return', 'median_20d_return', 'positive_period_fraction', 'strategy', 'top_pct', 'mean_turnover', 'mean_transaction_cost', 'mean_excess_return']

Cell 65 columns:
['observations', 'total_return', 'CAGR', 'max_drawdown', 'annualized_sharpe', 'mean_period_return', 'median_period_return', 'positive_fraction', 'mean_excess_vs_benchmark', 'mean_turnover', 'mean_transaction_cost', 'strategy']

strategy  cell61_CAGR  cell65_next_day_CAGR  cell61_Sharpe  cell65_next_day_Sharpe  cell61_excess  cell65_next_day_excess  cell61_max_drawdown  cell65_next_day_max_drawdown
  Top 5%     1.852012              1.331676       1.946864                1.967600       0.078492                0.063167            -0.386402                     -0.369738
 Top 10%     0.910473              0.733087       1.707368                1.677405       0.037231           

In [ ]:
# ==============================================================================
# CELL 66 — BLOCK BOOTSTRAP SIGNIFICANCE TEST
# ==============================================================================
#
# Tests whether the strategy's excess return over the benchmark is
# statistically distinguishable from zero.
#
# Uses Cell 65 strict next-day execution results.
#
# Block bootstrap preserves short-term temporal dependence by resampling
# consecutive blocks of portfolio periods instead of individual observations.
#
# 2025-2026 test data is NOT used.
# ==============================================================================

import os
import numpy as np
import pandas as pd

PROJECT = "/content/NEPSE_ML_PROJECT"
REPORT_DIR = f"{PROJECT}/reports"

PERIODS_PATH = (
    f"{REPORT_DIR}/cell65_next_day_execution_periods.csv"
)

N_BOOTSTRAPS = 10000
BLOCK_LENGTH = 5
RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)

# ------------------------------------------------------------------------------
# LOAD
# ------------------------------------------------------------------------------

periods = pd.read_csv(PERIODS_PATH)
periods["date"] = pd.to_datetime(periods["date"])

periods = periods.sort_values(
    ["top_pct", "date"]
).reset_index(drop=True)

print("=" * 78)
print("CELL 66 — BLOCK BOOTSTRAP SIGNIFICANCE TEST")
print("=" * 78)

print()
print("Observations:", len(periods))
print("Bootstrap samples:", N_BOOTSTRAPS)
print("Block length:", BLOCK_LENGTH)

# ------------------------------------------------------------------------------
# BOOTSTRAP FUNCTION
# ------------------------------------------------------------------------------

def block_bootstrap_mean(
    values,
    block_length=5,
    n_bootstrap=10000,
    rng=None
):
    """
    Circular block bootstrap for the mean.

    Returns:
        observed mean
        bootstrap distribution
        95% CI
        two-sided p-value for mean = 0
    """

    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    n = len(values)

    if n < block_length:
        raise ValueError(
            f"Not enough observations ({n}) "
            f"for block length {block_length}"
        )

    observed = values.mean()

    # --------------------------------------------------------------------------
    # Circular blocks
    # --------------------------------------------------------------------------

    n_blocks = int(np.ceil(n / block_length))

    # Pre-build circular blocks
    blocks = np.empty(
        (n, block_length),
        dtype=np.float64
    )

    for i in range(n):
        idx = (
            np.arange(i, i + block_length)
            % n
        )
        blocks[i] = values[idx]

    bootstrap_means = np.empty(
        n_bootstrap,
        dtype=np.float64
    )

    for b in range(n_bootstrap):

        starts = rng.integers(
            0,
            n,
            size=n_blocks
        )

        sample = blocks[
            starts
        ].reshape(-1)[:n]

        bootstrap_means[b] = sample.mean()

    # --------------------------------------------------------------------------
    # Confidence interval
    # --------------------------------------------------------------------------

    ci_low, ci_high = np.percentile(
        bootstrap_means,
        [2.5, 97.5]
    )

    # --------------------------------------------------------------------------
    # Two-sided bootstrap p-value around zero.
    #
    # If observed > 0:
    #   fraction of bootstrap estimates <= 0
    #
    # If observed < 0:
    #   fraction >= 0
    # --------------------------------------------------------------------------

    if observed >= 0:
        p_value = (
            2.0
            * np.mean(
                bootstrap_means <= 0
            )
        )
    else:
        p_value = (
            2.0
            * np.mean(
                bootstrap_means >= 0
            )
        )

    p_value = min(1.0, p_value)

    return (
        observed,
        bootstrap_means,
        ci_low,
        ci_high,
        p_value
    )

# ------------------------------------------------------------------------------
# RUN FOR EACH STRATEGY
# ------------------------------------------------------------------------------

results = []

bootstrap_distributions = {}

for top_pct in [0.05, 0.10, 0.20]:

    x = periods[
        periods["top_pct"] == top_pct
    ].copy()

    x = x.sort_values("date")

    # --------------------------------------------------------------------------
    # Excess return is already:
    #
    # net strategy return - benchmark return
    # --------------------------------------------------------------------------

    excess = (
        x["excess_return"]
        .astype(float)
        .values
    )

    (
        observed,
        boot,
        ci_low,
        ci_high,
        p_value
    ) = block_bootstrap_mean(
        excess,
        block_length=BLOCK_LENGTH,
        n_bootstrap=N_BOOTSTRAPS,
        rng=rng
    )

    bootstrap_distributions[top_pct] = boot

    # --------------------------------------------------------------------------
    # Additional statistics
    # --------------------------------------------------------------------------

    standard_error = (
        np.std(
            boot,
            ddof=1
        )
    )

    z_score = (
        observed / standard_error
        if standard_error > 0
        else np.nan
    )

    results.append({
        "strategy": f"Top {int(top_pct*100)}%",
        "observations": len(excess),
        "observed_mean_excess": observed,
        "bootstrap_std_error": standard_error,
        "bootstrap_95ci_low": ci_low,
        "bootstrap_95ci_high": ci_high,
        "bootstrap_z": z_score,
        "bootstrap_p_value": p_value,
        "significant_5pct": p_value < 0.05,
    })

# ------------------------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------------------------

results_df = pd.DataFrame(results)

print()
print("=" * 78)
print("BLOCK BOOTSTRAP RESULTS")
print("=" * 78)

print(
    results_df.to_string(index=False)
)

# ------------------------------------------------------------------------------
# YEAR-BLOCK CHECK
# ------------------------------------------------------------------------------
#
# Also calculate the sign of annual excess returns.
# This is not the significance test; it is a robustness diagnostic.
# ------------------------------------------------------------------------------

annual_rows = []

for top_pct in [0.05, 0.10, 0.20]:

    x = periods[
        periods["top_pct"] == top_pct
    ].copy()

    x["year"] = x["date"].dt.year

    yearly = (
        x.groupby("year")["excess_return"]
        .mean()
        .reset_index()
    )

    yearly["top_pct"] = top_pct

    annual_rows.append(yearly)

annual = pd.concat(
    annual_rows,
    ignore_index=True
)

print()
print("=" * 78)
print("ANNUAL MEAN EXCESS RETURNS")
print("=" * 78)

annual_display = annual.copy()

annual_display["strategy"] = (
    "Top "
    + (annual_display["top_pct"] * 100)
    .astype(int)
    .astype(str)
    + "%"
)

print(
    annual_display[
        ["year", "strategy", "excess_return"]
    ].to_string(index=False)
)

# ------------------------------------------------------------------------------
# COUNT POSITIVE YEARS
# ------------------------------------------------------------------------------

year_summary = (
    annual
    .groupby("top_pct")
    .agg(
        years=("year", "count"),
        positive_years=(
            "excess_return",
            lambda x: int((x > 0).sum())
        ),
        negative_years=(
            "excess_return",
            lambda x: int((x < 0).sum())
        ),
        mean_annual_excess=(
            "excess_return",
            "mean"
        ),
        median_annual_excess=(
            "excess_return",
            "median"
        ),
    )
    .reset_index()
)

year_summary["strategy"] = (
    "Top "
    + (year_summary["top_pct"] * 100)
    .astype(int)
    .astype(str)
    + "%"
)

print()
print("=" * 78)
print("YEARLY ROBUSTNESS SUMMARY")
print("=" * 78)

print(
    year_summary[
        [
            "strategy",
            "years",
            "positive_years",
            "negative_years",
            "mean_annual_excess",
            "median_annual_excess",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------------------------
# SAVE
# ------------------------------------------------------------------------------

results_path = (
    f"{REPORT_DIR}/cell66_block_bootstrap_results.csv"
)

annual_path = (
    f"{REPORT_DIR}/cell66_annual_excess_returns.csv"
)

year_summary_path = (
    f"{REPORT_DIR}/cell66_yearly_robustness_summary.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

annual_display.to_csv(
    annual_path,
    index=False
)

year_summary.to_csv(
    year_summary_path,
    index=False
)

print()
print("=" * 78)
print("CELL 66 COMPLETE")
print("=" * 78)

print()
print("Saved:")
print(results_path)
print(annual_path)
print(year_summary_path)

print()
print("2025-2026 test set was NOT used.")

CELL 66 — BLOCK BOOTSTRAP SIGNIFICANCE TEST

Observations: 276
Bootstrap samples: 10000
Block length: 5

BLOCK BOOTSTRAP RESULTS
strategy  observations  observed_mean_excess  bootstrap_std_error  bootstrap_95ci_low  bootstrap_95ci_high  bootstrap_z  bootstrap_p_value  significant_5pct
  Top 5%            92              0.063167             0.015940            0.030339             0.092808     3.962874             0.0002              True
 Top 10%            92              0.031298             0.007871            0.015713             0.046458     3.976515             0.0000              True
 Top 20%            92              0.016474             0.003851            0.008868             0.023956     4.277272             0.0002              True

ANNUAL MEAN EXCESS RETURNS
 year strategy  excess_return
 2017   Top 5%       0.083937
 2018   Top 5%       0.058227
 2019   Top 5%       0.067614
 2020   Top 5%      -0.051570
 2021   Top 5%       0.083780
 2022   Top 5%       0.129068
 2023

In [ ]:
# ==============================================================================
# CELL 67A — FINAL TEST SET INTEGRITY AUDIT
# ==============================================================================
#
# This cell does NOT train a model.
# It verifies that the 2025-2026 test period remains completely untouched.
# ==============================================================================

import os
import pandas as pd

PROJECT = "/content/NEPSE_ML_PROJECT"
DATA_PATH = f"{PROJECT}/features/nepse_ml_clean_dataset.parquet"

df = pd.read_parquet(DATA_PATH)

df["date"] = pd.to_datetime(df["date"])

print("=" * 78)
print("CELL 67A — FINAL TEST SET INTEGRITY AUDIT")
print("=" * 78)

# ------------------------------------------------------------------------------
# Define protected test period
# ------------------------------------------------------------------------------

TEST_START = pd.Timestamp("2025-01-01")
TEST_END = pd.Timestamp("2026-12-31")

test = df[
    (df["date"] >= TEST_START) &
    (df["date"] <= TEST_END)
].copy()

pre_test = df[
    df["date"] < TEST_START
].copy()

print()
print("Total dataset rows:", len(df))
print("Pre-test rows:", len(pre_test))
print("Protected test rows:", len(test))

print()
print("Protected test date range:")
print(test["date"].min().date(), "to", test["date"].max().date())

print()
print("Protected test symbols:", test["symbol"].nunique())

# ------------------------------------------------------------------------------
# Check that no previous backtest report contains test dates
# ------------------------------------------------------------------------------

reports_to_check = [
    "cell61_leakage_free_backtest_periods.csv",
    "cell61_leakage_free_backtest_metrics.csv",
    "cell62_transaction_cost_sensitivity.csv",
    "cell63_symbol_history_audit.csv",
    "cell63_survivorship_audit_summary.csv",
    "cell64_shuffled_target_placebo_periods.csv",
    "cell64_shuffled_target_placebo_metrics.csv",
    "cell65_next_day_execution_periods.csv",
    "cell65_next_day_execution_metrics.csv",
    "cell65_vs_cell61_comparison.csv",
    "cell66_block_bootstrap_results.csv",
    "cell66_annual_excess_returns.csv",
    "cell66_yearly_robustness_summary.csv",
]

print()
print("-" * 78)
print("CHECKING PREVIOUS BACKTEST REPORTS")
print("-" * 78)

violations = []

for filename in reports_to_check:

    path = os.path.join(
        PROJECT,
        "reports",
        filename
    )

    if not os.path.exists(path):
        print(f"NOT FOUND: {filename}")
        continue

    try:

        x = pd.read_csv(path)

        date_columns = [
            c for c in x.columns
            if c.lower() in [
                "date",
                "datetime",
                "timestamp",
            ]
        ]

        found_test_dates = False

        for col in date_columns:

            parsed = pd.to_datetime(
                x[col],
                errors="coerce"
            )

            if (
                parsed.notna()
                & (parsed >= TEST_START)
                & (parsed <= TEST_END)
            ).any():

                found_test_dates = True

                violations.append(
                    (filename, col)
                )

        if found_test_dates:
            print(
                f"WARNING — TEST DATES FOUND: {filename}"
            )
        else:
            print(
                f"OK — no 2025-2026 dates: {filename}"
            )

    except Exception as e:

        print(
            f"Could not inspect {filename}: {e}"
        )

# ------------------------------------------------------------------------------
# Explicit chronological boundary
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("CHRONOLOGICAL BOUNDARY")
print("-" * 78)

print("Training / validation / backtest must end before:")
print(TEST_START.date())

print("Final test begins:")
print(TEST_START.date())

print("Final test ends:")
print(TEST_END.date())

# ------------------------------------------------------------------------------
# Result
# ------------------------------------------------------------------------------

print()
print("=" * 78)

if violations:

    print("WARNING: POTENTIAL TEST-SET CONTAMINATION")
    print()

    for item in violations:
        print("  ", item)

    print()
    print("DO NOT RUN FINAL TEST YET.")

else:

    print("FINAL TEST SET INTEGRITY: PASS")
    print()
    print("2025-2026 contains no dates in previous backtest reports.")
    print("The final test period is ready for ONE-TIME evaluation.")

print("=" * 78)

CELL 67A — FINAL TEST SET INTEGRITY AUDIT

Total dataset rows: 633137
Pre-test rows: 492946
Protected test rows: 140191

Protected test date range:
2025-01-01 to 2026-10-06

Protected test symbols: 512

------------------------------------------------------------------------------
CHECKING PREVIOUS BACKTEST REPORTS
------------------------------------------------------------------------------
OK — no 2025-2026 dates: cell61_leakage_free_backtest_periods.csv
OK — no 2025-2026 dates: cell61_leakage_free_backtest_metrics.csv
OK — no 2025-2026 dates: cell62_transaction_cost_sensitivity.csv
OK — no 2025-2026 dates: cell63_symbol_history_audit.csv
OK — no 2025-2026 dates: cell63_survivorship_audit_summary.csv
OK — no 2025-2026 dates: cell64_shuffled_target_placebo_periods.csv
OK — no 2025-2026 dates: cell64_shuffled_target_placebo_metrics.csv
OK — no 2025-2026 dates: cell65_next_day_execution_periods.csv
OK — no 2025-2026 dates: cell65_next_day_execution_metrics.csv
OK — no 2025-2026 dates: 

In [ ]:
# ==============================================================================
# CELL 67B — FINAL 2025-2026 OUT-OF-SAMPLE TEST
# ==============================================================================
#
# FINAL EVALUATION — DO NOT MODIFY MODEL AFTER SEEING RESULTS
#
# Frozen model:
#   13 stock-only technical features
#   XGBRegressor
#
# NO:
#   market features
#   sector features
#   relative features
#   event timing features
#
# Training:
#   expanding window
#   starts 2011
#   20-session purge
#
# Final test:
#   2025-01-01 through latest available date
#
# Execution:
#   next trading-day close
#
# Holding:
#   20 trading sessions
#
# Cost:
#   0.3% one-way
#
# IMPORTANT:
#   2025-2026 data is used ONLY for final evaluation.
# ==============================================================================

import os
import json
import time
import warnings

import numpy as np
import pandas as pd
import xgboost as xgb

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------

PROJECT = "/content/NEPSE_ML_PROJECT"

DATA_PATH = (
    f"{PROJECT}/features/nepse_ml_clean_dataset.parquet"
)

REPORT_DIR = f"{PROJECT}/reports"
MODEL_DIR = f"{PROJECT}/models"

os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ------------------------------------------------------------------------------
# FROZEN CONFIGURATION
# ------------------------------------------------------------------------------

TARGET = "future_return_20d"

FEATURES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

TOP_PCTS = [0.05, 0.10, 0.20]

TRAIN_START = pd.Timestamp("2011-01-01")
TEST_START = pd.Timestamp("2025-01-01")

HOLDING_SESSIONS = 20
PURGE_SESSIONS = 20

ONE_WAY_COST = 0.003

RANDOM_STATE = 42

# ------------------------------------------------------------------------------
# HARDWARE
# ------------------------------------------------------------------------------

DEVICE = "cpu"

try:

    probe = xgb.XGBRegressor(
        n_estimators=1,
        max_depth=2,
        tree_method="hist",
        device="cuda",
        random_state=RANDOM_STATE,
    )

    # Check CUDA build/runtime
    probe.fit(
        np.array([[0.0], [1.0]], dtype=np.float32),
        np.array([0.0, 1.0], dtype=np.float32),
        verbose=False,
    )

    DEVICE = "cuda"

except Exception:

    DEVICE = "cpu"

print("=" * 78)
print("CELL 67B — FINAL 2025-2026 OUT-OF-SAMPLE TEST")
print("=" * 78)

print()
print("XGBoost device:", DEVICE)

# ------------------------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------------------------

print()
print("-" * 78)
print("LOADING DATA")
print("-" * 78)

df = pd.read_parquet(DATA_PATH)

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

print("Rows:", len(df))
print("Symbols:", df["symbol"].nunique())
print(
    "Date range:",
    df["date"].min().date(),
    "to",
    df["date"].max().date()
)

# ------------------------------------------------------------------------------
# VERIFY TEST BOUNDARY
# ------------------------------------------------------------------------------

test = df[
    df["date"] >= TEST_START
].copy()

pre_test = df[
    df["date"] < TEST_START
].copy()

print()
print("Pre-test rows:", len(pre_test))
print("Final test rows:", len(test))
print(
    "Final test range:",
    test["date"].min().date(),
    "to",
    test["date"].max().date()
)

if test.empty:
    raise RuntimeError("Final test dataset is empty.")

# ------------------------------------------------------------------------------
# REQUIRED COLUMNS
# ------------------------------------------------------------------------------

required = [
    "date",
    "symbol",
    "close",
    TARGET,
] + FEATURES

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

data = df[required].copy()

# ------------------------------------------------------------------------------
# STRICT NEXT-DAY EXECUTION
# ------------------------------------------------------------------------------
#
# Signal at T
# Entry at T+1 close
# Exit at T+21 close
#
# Therefore:
#
# realized_return =
#       close[T+21] / close[T+1] - 1
#
# ------------------------------------------------------------------------------
g = data.groupby(
    "symbol",
    sort=False
)

data["entry_close_next_day"] = (
    g["close"].shift(-1)
)

data["exit_close_20d_after_entry"] = (
    g["close"].shift(
        -(HOLDING_SESSIONS + 1)
    )
)

data["strict_realized_return"] = (
    data["exit_close_20d_after_entry"]
    / data["entry_close_next_day"]
    - 1.0
)

data = data.replace(
    [np.inf, -np.inf],
    np.nan
)

# ------------------------------------------------------------------------------
# TEST REBALANCE DATES
# ------------------------------------------------------------------------------
#
# Use every 20th available test trading date.
#
# This creates a non-overlapping 20-session portfolio cycle.
# ------------------------------------------------------------------------------

test_dates = np.sort(
    test["date"].unique()
)

# Start from the first available test date.
# Then advance approximately every 20 trading sessions.

rebalance_dates = list(
    test_dates[::HOLDING_SESSIONS]
)

print()
print("Final test rebalance dates:", len(rebalance_dates))

print(
    "First:",
    pd.Timestamp(rebalance_dates[0]).date()
)

print(
    "Last:",
    pd.Timestamp(rebalance_dates[-1]).date()
)

# ------------------------------------------------------------------------------
# MODEL
# ------------------------------------------------------------------------------

def make_model():

    return xgb.XGBRegressor(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=30,
        subsample=0.80,
        colsample_bytree=0.85,
        reg_alpha=1.0,
        reg_lambda=10.0,
        gamma=0.10,
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        device=DEVICE,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

# ------------------------------------------------------------------------------
# BACKTEST
# ------------------------------------------------------------------------------

results = []

start_time = time.time()

print()
print("-" * 78)
print("RUNNING FINAL OUT-OF-SAMPLE TEST")
print("-" * 78)

for i, rebalance_date in enumerate(
    rebalance_dates,
    1
):

    rebalance_date = pd.Timestamp(
        rebalance_date
    )

    # --------------------------------------------------------------------------
    # TRAINING DATA
    # --------------------------------------------------------------------------
    #
    # Only dates BEFORE the test date.
    #
    # Purge the most recent 20 available sessions.
    # --------------------------------------------------------------------------

    dates_before = np.sort(
        pre_test.loc[
            pre_test["date"] < rebalance_date,
            "date"
        ].unique()
    )

    if len(dates_before) <= PURGE_SESSIONS:
        continue

    training_end_date = (
        dates_before[-PURGE_SESSIONS]
    )

    train = pre_test[
        pre_test["date"] < training_end_date
    ].copy()

    if train.empty:
        continue

    # --------------------------------------------------------------------------
    # CURRENT TEST CROSS-SECTION
    # --------------------------------------------------------------------------

    current = data[
        data["date"] == rebalance_date
    ].copy()

    if current.empty:
        continue

    # Require valid future realization.
    current = current[
        current["strict_realized_return"].notna()
    ].copy()

    if len(current) < 10:
        continue

    # --------------------------------------------------------------------------
    # TRAINING-ONLY MEDIAN IMPUTATION
    # --------------------------------------------------------------------------

    medians = train[
        FEATURES
    ].median()

    X_train = (
        train[FEATURES]
        .fillna(medians)
        .astype(np.float32)
    )

    y_train = (
        train[TARGET]
        .astype(np.float32)
    )

    valid_train = np.isfinite(
        y_train.values
    )

    X_train = X_train.loc[
        valid_train
    ]

    y_train = y_train.loc[
        valid_train
    ]

    X_current = (
        current[FEATURES]
        .fillna(medians)
        .astype(np.float32)
    )

    if len(X_train) < 10000:
        continue

    # --------------------------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------------------------

    model = make_model()

    model.fit(
        X_train,
        y_train,
        verbose=False
    )

    # --------------------------------------------------------------------------
    # PREDICT
    # --------------------------------------------------------------------------

    current[
        "predicted_return"
    ] = model.predict(
        X_current
    )

    current = current.sort_values(
        "predicted_return",
        ascending=False
    )

    # --------------------------------------------------------------------------
    # PORTFOLIOS
    # --------------------------------------------------------------------------

    for top_pct in TOP_PCTS:

        n_select = max(
            1,
            int(
                np.floor(
                    len(current) * top_pct
                )
            )
        )

        selected = current.head(
            n_select
        )

        gross_return = (
            selected[
                "strict_realized_return"
            ]
            .mean()
        )

        if not np.isfinite(
            gross_return
        ):
            continue

        results.append({
            "date": rebalance_date,
            "top_pct": top_pct,
            "n_stocks": len(selected),
            "gross_return": float(
                gross_return
            ),
        })

    elapsed = time.time() - start_time

    per_iter = (
        elapsed / i
    )

    eta = (
        per_iter
        * (len(rebalance_dates) - i)
    )

    print(
        f"\rRebalance {i}/{len(rebalance_dates)} "
        f"| {elapsed:.1f}s "
        f"| ETA {eta/60:.1f} min",
        end=""
    )

print("\n")

periods = pd.DataFrame(
    results
)

if periods.empty:
    raise RuntimeError(
        "No final test observations generated."
    )

periods["date"] = pd.to_datetime(
    periods["date"]
)

print(
    "Portfolio observations:",
    len(periods)
)

# ------------------------------------------------------------------------------
# BENCHMARK
# ------------------------------------------------------------------------------

benchmark_rows = []

for d in sorted(
    periods["date"].unique()
):

    d = pd.Timestamp(d)

    current = data[
        data["date"] == d
    ].copy()

    current = current[
        current[
            "strict_realized_return"
        ].notna()
    ]

    if current.empty:
        continue

    benchmark_rows.append({
        "date": d,
        "benchmark_return": float(
            current[
                "strict_realized_return"
            ].mean()
        ),
    })

benchmark = pd.DataFrame(
    benchmark_rows
)

periods = periods.merge(
    benchmark,
    on="date",
    how="left"
)

# ------------------------------------------------------------------------------
# CALCULATE TURNOVER FROM ACTUAL TEST HOLDINGS
# ------------------------------------------------------------------------------
#
# For the final test we calculate turnover independently.
# We reconstruct rankings once more and save selected symbols.
#
# To avoid retraining, the simplest robust approach is to calculate turnover
# from the actual selected-stock lists during a second pass would be expensive.
#
# Therefore, for this final evaluation we use a conservative fixed turnover
# assumption of 100% one-way portfolio replacement.
#
# This avoids borrowing turnover information from the training/backtest period.
# ------------------------------------------------------------------------------

# 100% turnover = full replacement of portfolio.
# This is intentionally conservative.
periods["turnover"] = 1.0

periods["transaction_cost"] = (
    periods["turnover"]
    * ONE_WAY_COST
)

periods["net_return"] = (
    periods["gross_return"]
    - periods["transaction_cost"]
)

periods["excess_return"] = (
    periods["net_return"]
    - periods["benchmark_return"]
)

# ------------------------------------------------------------------------------
# METRICS
# ------------------------------------------------------------------------------

def calculate_metrics(x):

    x = x.sort_values(
        "date"
    ).copy()

    returns = (
        x["net_return"]
        .astype(float)
        .values
    )

    equity = np.cumprod(
        1.0 + returns
    )

    total_return = (
        equity[-1] - 1.0
    )

    years = (
        (
            x["date"].iloc[-1]
            - x["date"].iloc[0]
        ).days
        / 365.25
    )

    cagr = (
        equity[-1]
        ** (1.0 / years)
        - 1.0
        if years > 0
        else np.nan
    )

    running_max = np.maximum.accumulate(
        equity
    )

    drawdown = (
        equity
        / running_max
        - 1.0
    )

    max_drawdown = drawdown.min()

    mean_return = returns.mean()

    std_return = returns.std(
        ddof=1
    )

    sharpe = (
        mean_return
        / std_return
        * np.sqrt(13)
        if std_return > 0
        else np.nan
    )

    return {
        "observations": len(x),
        "total_return": total_return,
        "CAGR": cagr,
        "max_drawdown": max_drawdown,
        "annualized_sharpe": sharpe,
        "mean_period_return": mean_return,
        "median_period_return": np.median(
            returns
        ),
        "positive_fraction": np.mean(
            returns > 0
        ),
        "mean_excess_vs_benchmark": x[
            "excess_return"
        ].mean(),
        "mean_turnover": x[
            "turnover"
        ].mean(),
        "mean_transaction_cost": x[
            "transaction_cost"
        ].mean(),
    }

# ------------------------------------------------------------------------------
# STRATEGY RESULTS
# ------------------------------------------------------------------------------

metric_rows = []

for top_pct in TOP_PCTS:

    x = periods[
        periods["top_pct"] == top_pct
    ].copy()

    m = calculate_metrics(x)

    m["strategy"] = (
        f"FINAL Top {int(top_pct*100)}%"
    )

    metric_rows.append(m)

# ------------------------------------------------------------------------------
# BENCHMARK RESULTS
# ------------------------------------------------------------------------------

benchmark_returns = (
    benchmark[
        "benchmark_return"
    ]
    .astype(float)
    .values
)

benchmark_equity = np.cumprod(
    1.0 + benchmark_returns
)

benchmark_total = (
    benchmark_equity[-1] - 1.0
)

benchmark_years = (
    (
        benchmark["date"].iloc[-1]
        - benchmark["date"].iloc[0]
    ).days
    / 365.25
)

benchmark_cagr = (
    benchmark_equity[-1]
    ** (1.0 / benchmark_years)
    - 1.0
)

benchmark_running_max = (
    np.maximum.accumulate(
        benchmark_equity
    )
)

benchmark_drawdown = (
    benchmark_equity
    / benchmark_running_max
    - 1.0
)

benchmark_std = (
    benchmark_returns.std(
        ddof=1
    )
)

benchmark_sharpe = (
    benchmark_returns.mean()
    / benchmark_std
    * np.sqrt(13)
)

metric_rows.append({
    "strategy": "FINAL Benchmark",
    "observations": len(
        benchmark_returns
    ),
    "total_return": benchmark_total,
    "CAGR": benchmark_cagr,
    "max_drawdown": benchmark_drawdown.min(),
    "annualized_sharpe": benchmark_sharpe,
    "mean_period_return": benchmark_returns.mean(),
    "median_period_return": np.median(
        benchmark_returns
    ),
    "positive_fraction": np.mean(
        benchmark_returns > 0
    ),
    "mean_excess_vs_benchmark": 0.0,
    "mean_turnover": np.nan,
    "mean_transaction_cost": 0.0,
})

metrics = pd.DataFrame(
    metric_rows
)

# ------------------------------------------------------------------------------
# PRINT FINAL RESULTS
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("FINAL 2025-2026 OUT-OF-SAMPLE RESULTS")
print("=" * 78)

print(
    metrics[
        [
            "strategy",
            "observations",
            "total_return",
            "CAGR",
            "max_drawdown",
            "annualized_sharpe",
            "mean_period_return",
            "positive_fraction",
            "mean_excess_vs_benchmark",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------------------------
# YEARLY FINAL TEST RESULTS
# ------------------------------------------------------------------------------

periods["year"] = (
    periods["date"].dt.year
)

yearly = (
    periods
    .groupby(
        ["year", "top_pct"],
        as_index=False
    )
    .agg(
        mean_gross_return=(
            "gross_return",
            "mean"
        ),
        mean_net_return=(
            "net_return",
            "mean"
        ),
        mean_excess_return=(
            "excess_return",
            "mean"
        ),
        positive_fraction=(
            "net_return",
            lambda x: np.mean(x > 0)
        ),
        observations=(
            "net_return",
            "count"
        ),
    )
)

yearly["strategy"] = (
    "Top "
    + (
        yearly["top_pct"] * 100
    ).astype(int).astype(str)
    + "%"
)

print()
print("=" * 78)
print("FINAL TEST YEARLY RESULTS")
print("=" * 78)

print(
    yearly[
        [
            "year",
            "strategy",
            "observations",
            "mean_gross_return",
            "mean_net_return",
            "mean_excess_return",
            "positive_fraction",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------------------------
# SAVE
# ------------------------------------------------------------------------------

periods_path = (
    f"{REPORT_DIR}/cell67_final_test_periods.csv"
)

metrics_path = (
    f"{REPORT_DIR}/cell67_final_test_metrics.csv"
)

yearly_path = (
    f"{REPORT_DIR}/cell67_final_test_yearly.csv"
)

periods.to_csv(
    periods_path,
    index=False
)

metrics.to_csv(
    metrics_path,
    index=False
)

yearly.to_csv(
    yearly_path,
    index=False
)

# ------------------------------------------------------------------------------
# FINAL SUMMARY JSON
# ------------------------------------------------------------------------------

summary = {
    "test_start": str(
        TEST_START.date()
    ),
    "test_end": str(
        test["date"].max().date()
    ),
    "rows": int(len(test)),
    "symbols": int(
        test["symbol"].nunique()
    ),
    "rebalance_dates": int(
        len(rebalance_dates)
    ),
    "holding_sessions": HOLDING_SESSIONS,
    "purge_sessions": PURGE_SESSIONS,
    "one_way_transaction_cost": ONE_WAY_COST,
    "features": FEATURES,
    "market_features_used": False,
    "sector_features_used": False,
    "relative_features_used": False,
    "test_used_for_training": False,
    "device": DEVICE,
}

summary_path = (
    f"{REPORT_DIR}/cell67_final_test_summary.json"
)

with open(
    summary_path,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# ------------------------------------------------------------------------------
# DONE
# ------------------------------------------------------------------------------

print()
print("=" * 78)
print("CELL 67B COMPLETE")
print("=" * 78)

print()
print("Saved:")
print(periods_path)
print(metrics_path)
print(yearly_path)
print(summary_path)

print()
print("IMPORTANT:")
print("2025-2026 was used ONLY for final evaluation.")
print("Do NOT retrain or tune the model based on these results.")

CELL 67B — FINAL 2025-2026 OUT-OF-SAMPLE TEST

XGBoost device: cuda

------------------------------------------------------------------------------
LOADING DATA
------------------------------------------------------------------------------
Rows: 633137
Symbols: 651
Date range: 1997-07-20 to 2026-10-06

Pre-test rows: 492946
Final test rows: 140191
Final test range: 2025-01-01 to 2026-10-06

Final test rebalance dates: 21
First: 2025-01-01
Last: 2026-10-01

------------------------------------------------------------------------------
RUNNING FINAL OUT-OF-SAMPLE TEST
------------------------------------------------------------------------------
Rebalance 20/21 | 71.3s | ETA 0.1 min

Portfolio observations: 60

FINAL 2025-2026 OUT-OF-SAMPLE RESULTS
       strategy  observations  total_return      CAGR  max_drawdown  annualized_sharpe  mean_period_return  positive_fraction  mean_excess_vs_benchmark
   FINAL Top 5%            20      1.132467  0.582017     -0.134723           1.410707     

In [ ]:
# ==============================================================================
# SAVE CURRENT NEPSE ML PROJECT TO GOOGLE DRIVE
# ==============================================================================

import os
import shutil
from datetime import datetime

PROJECT = "/content/NEPSE_ML_PROJECT"
DRIVE_ROOT = "/content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP"

# Make sure Drive is mounted
from google.colab import drive
drive.mount("/content/drive")

print("Saving current project to:")
print(DRIVE_ROOT)

# Copy/update the important folders
for folder in ["reports", "models", "features"]:
    src = os.path.join(PROJECT, folder)
    dst = os.path.join(DRIVE_ROOT, folder)

    if os.path.exists(src):
        shutil.copytree(
            src,
            dst,
            dirs_exist_ok=True
        )

print()
print("=" * 70)
print("BACKUP COMPLETE")
print("=" * 70)

# Show important latest reports
reports_dir = os.path.join(DRIVE_ROOT, "reports")

for f in sorted(os.listdir(reports_dir)):
    if f.startswith(("cell61", "cell62", "cell63", "cell64", "cell65", "cell66")):
        print(f)

print()
print("Google Drive backup is up to date.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving current project to:
/content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP

BACKUP COMPLETE
cell61_leakage_free_backtest_metrics.csv
cell61_leakage_free_backtest_periods.csv
cell61_leakage_free_backtest_summary.json
cell61_leakage_free_backtest_yearly.csv
cell62_transaction_cost_sensitivity.csv
cell63_survivorship_audit_summary.csv
cell63_symbol_history_audit.csv
cell64_real_vs_placebo_comparison.csv
cell64_shuffled_target_placebo_metrics.csv
cell64_shuffled_target_placebo_periods.csv
cell65_next_day_execution_metrics.csv
cell65_next_day_execution_periods.csv
cell65_vs_cell61_comparison.csv
cell66_annual_excess_returns.csv
cell66_block_bootstrap_results.csv
cell66_yearly_robustness_summary.csv

Google Drive backup is up to date.


In [2]:
# ==============================================================================
# CELL 68A — CHECK PROJECT / DRIVE MODEL FILES
# ==============================================================================

import os

PROJECT = "/content/NEPSE_ML_PROJECT"
DRIVE_BACKUP = "/content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP"

print("=" * 78)
print("CHECKING FINAL MODEL AND PROJECT BACKUP")
print("=" * 78)

# ------------------------------------------------------------------
# Current runtime
# ------------------------------------------------------------------

print("\nCURRENT RUNTIME")
print("-" * 78)

for path in [
    f"{PROJECT}/models",
    f"{PROJECT}/reports",
    f"{PROJECT}/features/model_ready",
]:
    print(f"\n{path}")
    if os.path.exists(path):
        files = sorted(os.listdir(path))
        print(f"Files: {len(files)}")
        for f in files:
            print(" ", f)
    else:
        print("NOT FOUND")

# ------------------------------------------------------------------
# Google Drive backup
# ------------------------------------------------------------------

print("\n\nGOOGLE DRIVE BACKUP")
print("-" * 78)

for path in [
    f"{DRIVE_BACKUP}/models",
    f"{DRIVE_BACKUP}/reports",
]:
    print(f"\n{path}")
    if os.path.exists(path):
        files = sorted(os.listdir(path))
        print(f"Files: {len(files)}")
        for f in files:
            print(" ", f)
    else:
        print("NOT FOUND")

# ------------------------------------------------------------------
# Specifically search for Cell 58 model anywhere in project/backup
# ------------------------------------------------------------------

TARGET = "cell58_stock_only_cross_sectional_xgboost.json"

print("\n\nSEARCHING FOR FINAL CELL 58 MODEL")
print("-" * 78)

matches = []

for root in [PROJECT, DRIVE_BACKUP]:
    if not os.path.exists(root):
        continue

    for dirpath, dirnames, filenames in os.walk(root):
        if TARGET in filenames:
            matches.append(os.path.join(dirpath, TARGET))

if matches:
    print("FOUND:")
    for m in matches:
        print(" ", m)
else:
    print("NOT FOUND ANYWHERE")

print("\n" + "=" * 78)
print("DIAGNOSTIC COMPLETE")
print("=" * 78)

CHECKING FINAL MODEL AND PROJECT BACKUP

CURRENT RUNTIME
------------------------------------------------------------------------------

/content/NEPSE_ML_PROJECT/models
NOT FOUND

/content/NEPSE_ML_PROJECT/reports
NOT FOUND

/content/NEPSE_ML_PROJECT/features/model_ready
NOT FOUND


GOOGLE DRIVE BACKUP
------------------------------------------------------------------------------

/content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP/models
NOT FOUND

/content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP/reports
NOT FOUND


SEARCHING FOR FINAL CELL 58 MODEL
------------------------------------------------------------------------------
NOT FOUND ANYWHERE

DIAGNOSTIC COMPLETE


In [3]:
# ==============================================================================
# CELL 68B — FIND AND RESTORE FULL PROJECT BACKUP
# ==============================================================================

import os
import shutil
import zipfile

DRIVE_ROOT = "/content/drive/MyDrive"
PROJECT = "/content/NEPSE_ML_PROJECT"

TARGET_ZIP = "NEPSE_ML_PROJECT_BACKUP_20261006_165417.zip"

print("=" * 78)
print("SEARCHING GOOGLE DRIVE FOR FULL PROJECT BACKUP")
print("=" * 78)

matches = []

for root, dirs, files in os.walk(DRIVE_ROOT):
    if TARGET_ZIP in files:
        matches.append(os.path.join(root, TARGET_ZIP))

if not matches:
    print("\nFULL ZIP NOT FOUND:")
    print(TARGET_ZIP)
    print("\nSearching for any NEPSE backup ZIP files...\n")

    for root, dirs, files in os.walk(DRIVE_ROOT):
        for f in files:
            if f.lower().endswith(".zip") and "NEPSE" in f.upper():
                matches.append(os.path.join(root, f))

if not matches:
    print("NO NEPSE BACKUP ZIP FOUND ON GOOGLE DRIVE.")
    print("\nSTOP HERE — DO NOT RETRAIN.")
else:
    print("\nBACKUP FILE(S) FOUND:")
    for i, m in enumerate(matches, 1):
        size_mb = os.path.getsize(m) / (1024 * 1024)
        print(f"{i}. {m}")
        print(f"   Size: {size_mb:.1f} MB")

    # Prefer the exact known backup
    exact = [
        m for m in matches
        if os.path.basename(m) == TARGET_ZIP
    ]

    backup_zip = exact[0] if exact else matches[0]

    print("\nUsing:")
    print(backup_zip)

    # ------------------------------------------------------------------
    # Remove incomplete runtime project if it exists
    # ------------------------------------------------------------------

    if os.path.exists(PROJECT):
        print("\nRemoving incomplete runtime project...")
        shutil.rmtree(PROJECT)

    os.makedirs(PROJECT, exist_ok=True)

    # ------------------------------------------------------------------
    # Extract
    # ------------------------------------------------------------------

    print("\nExtracting full project...")

    with zipfile.ZipFile(backup_zip, "r") as z:
        z.extractall("/content")

    print("Extraction complete.")

    # ------------------------------------------------------------------
    # Locate restored project
    # ------------------------------------------------------------------

    candidates = [
        "/content/NEPSE_ML_PROJECT",
        "/content/NEPSE_ML_PROJECT_BACKUP",
    ]

    restored = None

    for c in candidates:
        if os.path.exists(os.path.join(c, "models")):
            restored = c
            break

    if restored is None:
        print("\nWARNING: Expected project structure not found.")
        print("Top-level /content items:")
        for x in sorted(os.listdir("/content")):
            if "NEPSE" in x.upper():
                print(" ", x)
    else:
        # If extraction produced a different root, normalize it.
        if restored != PROJECT:
            print(f"\nMoving restored project:")
            print(restored)
            print("->")
            print(PROJECT)

            if os.path.exists(PROJECT):
                shutil.rmtree(PROJECT)

            shutil.move(restored, PROJECT)

        print("\n" + "=" * 78)
        print("PROJECT RESTORED")
        print("=" * 78)

        # ------------------------------------------------------------------
        # Verify important files
        # ------------------------------------------------------------------

        checks = [
            "models/cell58_stock_only_cross_sectional_xgboost.json",

            "reports/cell61_leakage_free_backtest_metrics.csv",
            "reports/cell62_transaction_cost_sensitivity.csv",
            "reports/cell63_survivorship_audit_summary.csv",
            "reports/cell64_real_vs_placebo_comparison.csv",
            "reports/cell65_next_day_execution_metrics.csv",
            "reports/cell66_block_bootstrap_results.csv",
            "reports/cell67_final_test_metrics.csv",
            "reports/cell67_final_test_yearly.csv",
            "reports/cell67_final_test_summary.json",

            "features/nepse_ml_clean_dataset.parquet",
        ]

        print("\nIMPORTANT FILE CHECK")
        print("-" * 78)

        missing = []

        for rel in checks:
            path = os.path.join(PROJECT, rel)

            if os.path.exists(path):
                size_mb = os.path.getsize(path) / (1024 * 1024)
                print(f"OK      {rel}  ({size_mb:.1f} MB)")
            else:
                print(f"MISSING {rel}")
                missing.append(rel)

        print("\n" + "=" * 78)

        if not missing:
            print("RESTORE SUCCESSFUL — ALL IMPORTANT FILES PRESENT")
        else:
            print(f"RESTORE PARTIAL — {len(missing)} IMPORTANT FILE(S) MISSING")

        print("=" * 78)

SEARCHING GOOGLE DRIVE FOR FULL PROJECT BACKUP

FULL ZIP NOT FOUND:
NEPSE_ML_PROJECT_BACKUP_20261006_165417.zip

Searching for any NEPSE backup ZIP files...

NO NEPSE BACKUP ZIP FOUND ON GOOGLE DRIVE.

STOP HERE — DO NOT RETRAIN.


In [4]:
# ==============================================================================
# CELL 68C — BROAD GOOGLE DRIVE BACKUP SEARCH
# ==============================================================================

import os

DRIVE = "/content/drive"

print("=" * 78)
print("BROAD GOOGLE DRIVE SEARCH")
print("=" * 78)

# ------------------------------------------------------------------------------
# 1. Show mounted Drive structure
# ------------------------------------------------------------------------------

print("\nMOUNTED DRIVE")
print("-" * 78)

for root in [
    "/content/drive",
    "/content/drive/MyDrive",
]:
    print(f"\n{root}")

    if os.path.exists(root):
        try:
            items = sorted(os.listdir(root))
            print(f"Items: {len(items)}")

            for item in items[:100]:
                path = os.path.join(root, item)

                if os.path.isdir(path):
                    print("  [DIR] ", item)
                else:
                    size_mb = os.path.getsize(path) / (1024 * 1024)
                    print(f"  [FILE] {item} ({size_mb:.1f} MB)")

        except Exception as e:
            print("ERROR:", e)
    else:
        print("NOT FOUND")

# ------------------------------------------------------------------------------
# 2. Search entire mounted Drive for anything NEPSE-related
# ------------------------------------------------------------------------------

print("\n\nSEARCHING ENTIRE /content/drive")
print("-" * 78)

matches = []

for root, dirs, files in os.walk(DRIVE):

    # Ignore hidden/system directories
    dirs[:] = [
        d for d in dirs
        if not d.startswith(".")
        and d not in ["lost+found"]
    ]

    for name in files:
        if "nepse" in name.lower():
            path = os.path.join(root, name)

            try:
                size_mb = os.path.getsize(path) / (1024 * 1024)
            except:
                size_mb = 0

            matches.append((path, size_mb))

print(f"NEPSE-related files found: {len(matches)}")

for path, size_mb in matches[:200]:
    print(f"{size_mb:10.1f} MB   {path}")

# ------------------------------------------------------------------------------
# 3. Search for backup-looking ZIP files
# ------------------------------------------------------------------------------

print("\n\nSEARCHING FOR BACKUP ZIP FILES")
print("-" * 78)

zips = []

for root, dirs, files in os.walk(DRIVE):

    dirs[:] = [
        d for d in dirs
        if not d.startswith(".")
    ]

    for name in files:
        if name.lower().endswith(".zip"):
            path = os.path.join(root, name)

            try:
                size_mb = os.path.getsize(path) / (1024 * 1024)
            except:
                size_mb = 0

            zips.append((path, size_mb))

print(f"ZIP files found: {len(zips)}")

for path, size_mb in zips[:200]:
    print(f"{size_mb:10.1f} MB   {path}")

print("\n" + "=" * 78)
print("BROAD SEARCH COMPLETE")
print("=" * 78)

BROAD GOOGLE DRIVE SEARCH

MOUNTED DRIVE
------------------------------------------------------------------------------

/content/drive
NOT FOUND

/content/drive/MyDrive
NOT FOUND


SEARCHING ENTIRE /content/drive
------------------------------------------------------------------------------
NEPSE-related files found: 0


SEARCHING FOR BACKUP ZIP FILES
------------------------------------------------------------------------------
ZIP files found: 0

BROAD SEARCH COMPLETE


In [5]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
# ==============================================================================
# CELL 68D — SEARCH MOUNTED GOOGLE DRIVE FOR NEPSE PROJECT
# ==============================================================================

import os

DRIVE_ROOT = "/content/drive/MyDrive"

print("=" * 78)
print("SEARCHING MOUNTED GOOGLE DRIVE")
print("=" * 78)

# ------------------------------------------------------------------------------
# 1. Check expected locations
# ------------------------------------------------------------------------------

expected = [
    "/content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP",
    "/content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP_20261006_165417.zip",
    "/content/drive/MyDrive/NEPSE_ML_PROJECT",
]

print("\nEXPECTED LOCATIONS")
print("-" * 78)

for path in expected:
    if os.path.exists(path):
        if os.path.isdir(path):
            print("FOUND DIRECTORY:", path)
            try:
                items = os.listdir(path)
                print("  Items:", len(items))
            except:
                pass
        else:
            size_mb = os.path.getsize(path) / (1024 * 1024)
            print(f"FOUND FILE: {path} ({size_mb:.1f} MB)")
    else:
        print("NOT FOUND:", path)

# ------------------------------------------------------------------------------
# 2. Search entire MyDrive
# ------------------------------------------------------------------------------

print("\n\nSEARCHING ENTIRE MYDRIVE")
print("-" * 78)

nepse_matches = []
zip_matches = []

for root, dirs, files in os.walk(DRIVE_ROOT):

    # Skip hidden/system directories
    dirs[:] = [
        d for d in dirs
        if not d.startswith(".")
    ]

    # NEPSE files/directories
    for name in files:
        if "nepse" in name.lower():
            path = os.path.join(root, name)

            try:
                size_mb = os.path.getsize(path) / (1024 * 1024)
            except:
                size_mb = 0

            nepse_matches.append((path, size_mb))

    # Also check directories
    for name in dirs:
        if "nepse" in name.lower():
            path = os.path.join(root, name)
            nepse_matches.append((path, -1))

    # ZIP files
    for name in files:
        if name.lower().endswith(".zip"):
            path = os.path.join(root, name)

            try:
                size_mb = os.path.getsize(path) / (1024 * 1024)
            except:
                size_mb = 0

            zip_matches.append((path, size_mb))

# ------------------------------------------------------------------------------
# 3. Print NEPSE matches
# ------------------------------------------------------------------------------

print(f"\nNEPSE-related items found: {len(nepse_matches)}")

for path, size_mb in nepse_matches[:300]:
    if size_mb >= 0:
        print(f"[FILE] {size_mb:10.1f} MB  {path}")
    else:
        print(f"[DIR]              {path}")

# ------------------------------------------------------------------------------
# 4. Print ZIP matches
# ------------------------------------------------------------------------------

print(f"\nZIP files found: {len(zip_matches)}")

for path, size_mb in zip_matches[:300]:
    print(f"{size_mb:10.1f} MB  {path}")

print("\n" + "=" * 78)
print("DRIVE SEARCH COMPLETE")
print("=" * 78)

SEARCHING MOUNTED GOOGLE DRIVE

EXPECTED LOCATIONS
------------------------------------------------------------------------------
FOUND DIRECTORY: /content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP
  Items: 6
FOUND FILE: /content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP_20261006_165417.zip (80.9 MB)
NOT FOUND: /content/drive/MyDrive/NEPSE_ML_PROJECT


SEARCHING ENTIRE MYDRIVE
------------------------------------------------------------------------------

NEPSE-related items found: 19
[FILE]       80.9 MB  /content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP_20261006_165417.zip
[DIR]              /content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP
[FILE]        1.1 MB  /content/drive/MyDrive/Colab Notebooks/01_NEPSE_RAW_DATA_VALIDATION.ipynb
[FILE]       12.5 MB  /content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP/raw/nepse_daily_ohlcv_unadjusted.json.gz
[FILE]       13.1 MB  /content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP/raw/nepse_daily_ohlcv.json.gz
[FILE]        0.2 MB  /content/drive/MyDrive/NEPSE_ML_PROJ

In [7]:
# ==============================================================================
# CELL 68E — RESTORE COMPLETE NEPSE ML PROJECT FROM GOOGLE DRIVE ZIP
# ==============================================================================

import os
import zipfile
import shutil
import time

ZIP_PATH = "/content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP_20261006_165417.zip"
PROJECT_DIR = "/content/NEPSE_ML_PROJECT"

print("=" * 78)
print("RESTORING COMPLETE NEPSE ML PROJECT")
print("=" * 78)

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f"Backup ZIP not found: {ZIP_PATH}")

zip_size_mb = os.path.getsize(ZIP_PATH) / (1024**2)
print(f"Backup ZIP: {ZIP_PATH}")
print(f"ZIP size: {zip_size_mb:.1f} MB")

# Remove incomplete runtime copy if one exists
if os.path.exists(PROJECT_DIR):
    print("\nRemoving existing incomplete project...")
    shutil.rmtree(PROJECT_DIR)

os.makedirs(PROJECT_DIR, exist_ok=True)

start = time.time()

print("\nExtracting backup...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    members = z.namelist()

    print(f"Files in ZIP: {len(members)}")

    # Show top-level structure
    top_levels = sorted(set(
        m.split("/")[0]
        for m in members
        if m.strip()
    ))
    print("Top-level entries:", top_levels)

    z.extractall("/content")

elapsed = time.time() - start

print(f"\nExtraction completed in {elapsed:.1f} seconds.")

# Some ZIPs may contain NEPSE_ML_PROJECT directly.
# Verify expected project directory.
if not os.path.isdir(PROJECT_DIR):
    # Search /content for a restored project directory
    candidates = []

    for name in os.listdir("/content"):
        path = os.path.join("/content", name)
        if os.path.isdir(path) and "NEPSE_ML_PROJECT" in name:
            candidates.append(path)

    if candidates:
        print("\nFound restored project candidate(s):")
        for c in candidates:
            print(" ", c)

        source = candidates[0]

        if source != PROJECT_DIR:
            if os.path.exists(PROJECT_DIR):
                shutil.rmtree(PROJECT_DIR)
            shutil.move(source, PROJECT_DIR)

# Final verification
print("\n" + "=" * 78)
print("RESTORE VERIFICATION")
print("=" * 78)

required_dirs = [
    "raw",
    "processed",
    "features",
    "models",
    "reports",
]

for d in required_dirs:
    path = os.path.join(PROJECT_DIR, d)
    print(f"{'OK' if os.path.isdir(path) else 'MISSING':8s} {path}")

required_files = [
    "features/nepse_ml_clean_dataset.parquet",
    "models/cell58_stock_only_cross_sectional_xgboost.json",
    "reports/cell61_leakage_free_backtest_metrics.csv",
    "reports/cell62_transaction_cost_sensitivity.csv",
    "reports/cell63_survivorship_audit_summary.csv",
    "reports/cell64_real_vs_placebo_comparison.csv",
    "reports/cell65_next_day_execution_metrics.csv",
    "reports/cell66_block_bootstrap_results.csv",
]

print("\nCritical files:")

all_ok = True

for rel in required_files:
    path = os.path.join(PROJECT_DIR, rel)
    exists = os.path.isfile(path)

    if exists:
        size_mb = os.path.getsize(path) / (1024**2)
        print(f"OK       {rel}  ({size_mb:.1f} MB)")
    else:
        print(f"MISSING  {rel}")
        all_ok = False

print("\n" + "=" * 78)

if all_ok:
    print("RESTORE SUCCESSFUL")
    print("The complete NEPSE ML project is back in /content.")
else:
    print("RESTORE INCOMPLETE")
    print("Do NOT retrain yet. We need to inspect the missing files.")

print("=" * 78)

RESTORING COMPLETE NEPSE ML PROJECT
Backup ZIP: /content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP_20261006_165417.zip
ZIP size: 80.9 MB

Extracting backup...
Files in ZIP: 33
Top-level entries: ['NEPSE_ML_PROJECT']

Extraction completed in 6.9 seconds.

RESTORE VERIFICATION
OK       /content/NEPSE_ML_PROJECT/raw
OK       /content/NEPSE_ML_PROJECT/processed
OK       /content/NEPSE_ML_PROJECT/features
MISSING  /content/NEPSE_ML_PROJECT/models
MISSING  /content/NEPSE_ML_PROJECT/reports

Critical files:
MISSING  features/nepse_ml_clean_dataset.parquet
MISSING  models/cell58_stock_only_cross_sectional_xgboost.json
MISSING  reports/cell61_leakage_free_backtest_metrics.csv
MISSING  reports/cell62_transaction_cost_sensitivity.csv
MISSING  reports/cell63_survivorship_audit_summary.csv
MISSING  reports/cell64_real_vs_placebo_comparison.csv
MISSING  reports/cell65_next_day_execution_metrics.csv
MISSING  reports/cell66_block_bootstrap_results.csv

RESTORE INCOMPLETE
Do NOT retrain yet. We need to ins

In [8]:
# ==============================================================================
# CELL 68F — AUDIT COMPLETE GOOGLE DRIVE BACKUP FOLDER
# ==============================================================================

import os

BACKUP_DIR = "/content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP"

print("=" * 78)
print("AUDITING GOOGLE DRIVE NEPSE BACKUP FOLDER")
print("=" * 78)

if not os.path.isdir(BACKUP_DIR):
    raise FileNotFoundError(BACKUP_DIR)

files = []
dirs = []

for root, subdirs, filenames in os.walk(BACKUP_DIR):
    for d in subdirs:
        dirs.append(os.path.join(root, d))

    for f in filenames:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / (1024**2)
        files.append((path, size_mb))

print(f"\nDirectories: {len(dirs)}")
print(f"Files:       {len(files)}")
print(f"Total size:  {sum(x[1] for x in files):.1f} MB")

print("\n" + "=" * 78)
print("ALL BACKUP FILES")
print("=" * 78)

for path, size_mb in sorted(files):
    rel = os.path.relpath(path, BACKUP_DIR)
    print(f"{size_mb:8.1f} MB   {rel}")

print("\n" + "=" * 78)
print("IMPORTANT FILE CHECK")
print("=" * 78)

important_patterns = [
    "nepse_ml_clean_dataset.parquet",
    "cell58_stock_only_cross_sectional_xgboost.json",
    "cell61_",
    "cell62_",
    "cell63_",
    "cell64_",
    "cell65_",
    "cell66_",
    "cell67_",
    "cell68_",
]

for pattern in important_patterns:
    matches = [
        (p, s) for p, s in files
        if pattern.lower() in os.path.basename(p).lower()
        or pattern.lower() in p.lower()
    ]

    print(f"\n[{pattern}]")
    if matches:
        for p, s in matches:
            print(f"  FOUND   {s:.1f} MB   {os.path.relpath(p, BACKUP_DIR)}")
    else:
        print("  NOT FOUND")

print("\n" + "=" * 78)
print("BACKUP AUDIT COMPLETE")
print("=" * 78)

AUDITING GOOGLE DRIVE NEPSE BACKUP FOLDER

Directories: 7
Files:       120
Total size:  2023.3 MB

ALL BACKUP FILES
    94.8 MB   features/event_feature_daily_cell31_checkpoint.parquet
   130.0 MB   features/event_feature_daily_cell32_checkpoint.parquet
   166.8 MB   features/event_feature_daily_cell33_checkpoint.parquet
   285.6 MB   features/event_feature_daily_cell34_checkpoint.parquet
   318.8 MB   features/event_feature_daily_cell35_targets_checkpoint.parquet
   318.8 MB   features/event_feature_daily_cell36_audited_checkpoint.parquet
     3.1 MB   features/event_feature_daily_checkpoint.parquet
    56.1 MB   features/model_ready/X_test.parquet
   131.4 MB   features/model_ready/X_train.parquet
    86.5 MB   features/model_ready/X_validation.parquet
     0.0 MB   features/model_ready/feature_medians.csv
     0.0 MB   features/model_ready/model_ready_manifest.json
     0.1 MB   features/model_ready/test_metadata.parquet
     0.1 MB   features/model_ready/train_metadata.parquet
    

In [9]:
# ==============================================================================
# CELL 68G — RESTORE COMPLETE PROJECT FROM AUTHORITATIVE GOOGLE DRIVE BACKUP
# ==============================================================================

import os
import shutil
import time

SOURCE = "/content/drive/MyDrive/NEPSE_ML_PROJECT_BACKUP"
DEST = "/content/NEPSE_ML_PROJECT"

print("=" * 78)
print("RESTORING COMPLETE NEPSE ML PROJECT FROM GOOGLE DRIVE")
print("=" * 78)

if not os.path.isdir(SOURCE):
    raise FileNotFoundError(f"Drive backup not found: {SOURCE}")

# Remove incomplete runtime copy
if os.path.exists(DEST):
    print("Removing incomplete runtime project...")
    shutil.rmtree(DEST)

start = time.time()

print("\nCopying backup...")
shutil.copytree(SOURCE, DEST)

elapsed = time.time() - start

print(f"Copy completed in {elapsed:.1f} seconds.")

# --------------------------------------------------------------------------
# VERIFY
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("RESTORE VERIFICATION")
print("=" * 78)

required_dirs = [
    "raw",
    "processed",
    "validation",
    "features",
    "features/model_ready",
    "models",
    "reports",
]

required_files = [
    "features/nepse_ml_clean_dataset.parquet",

    "features/model_ready/X_train.parquet",
    "features/model_ready/X_validation.parquet",
    "features/model_ready/X_test.parquet",

    "models/cell58_stock_only_cross_sectional_xgboost.json",

    "reports/cell61_leakage_free_backtest_metrics.csv",
    "reports/cell62_transaction_cost_sensitivity.csv",
    "reports/cell63_survivorship_audit_summary.csv",
    "reports/cell64_real_vs_placebo_comparison.csv",
    "reports/cell65_next_day_execution_metrics.csv",
    "reports/cell66_block_bootstrap_results.csv",

    "reports/cell67_final_test_metrics.csv",
    "reports/cell67_final_test_periods.csv",
    "reports/cell67_final_test_yearly.csv",
    "reports/cell67_final_test_summary.json",
]

all_ok = True

print("\nDIRECTORIES:")

for rel in required_dirs:
    path = os.path.join(DEST, rel)
    ok = os.path.isdir(path)
    print(f"{'OK' if ok else 'MISSING':8s} {rel}")
    if not ok:
        all_ok = False

print("\nCRITICAL FILES:")

for rel in required_files:
    path = os.path.join(DEST, rel)

    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024**2)
        print(f"OK       {rel} ({size_mb:.1f} MB)")
    else:
        print(f"MISSING  {rel}")
        all_ok = False

# --------------------------------------------------------------------------
# COUNT EVERYTHING
# --------------------------------------------------------------------------

file_count = 0
total_bytes = 0

for root, _, filenames in os.walk(DEST):
    for filename in filenames:
        path = os.path.join(root, filename)
        file_count += 1
        total_bytes += os.path.getsize(path)

print("\n" + "=" * 78)
print("FINAL RESTORE STATUS")
print("=" * 78)

print(f"Files restored : {file_count}")
print(f"Total size     : {total_bytes / (1024**3):.2f} GB")

if all_ok:
    print("\nPROJECT RESTORE: SUCCESS")
    print("All critical model, dataset, validation, backtest, and final-test")
    print("artifacts are restored.")
    print("\nNO RETRAINING IS REQUIRED.")
else:
    print("\nPROJECT RESTORE: INCOMPLETE")
    print("Do NOT retrain. Missing artifacts need investigation.")

print("=" * 78)

RESTORING COMPLETE NEPSE ML PROJECT FROM GOOGLE DRIVE
Removing incomplete runtime project...

Copying backup...
Copy completed in 123.7 seconds.

RESTORE VERIFICATION

DIRECTORIES:
OK       raw
OK       processed
OK       validation
OK       features
OK       features/model_ready
OK       models
OK       reports

CRITICAL FILES:
OK       features/nepse_ml_clean_dataset.parquet (318.0 MB)
OK       features/model_ready/X_train.parquet (131.4 MB)
OK       features/model_ready/X_validation.parquet (86.5 MB)
OK       features/model_ready/X_test.parquet (56.1 MB)
OK       models/cell58_stock_only_cross_sectional_xgboost.json (0.6 MB)
OK       reports/cell61_leakage_free_backtest_metrics.csv (0.0 MB)
OK       reports/cell62_transaction_cost_sensitivity.csv (0.0 MB)
OK       reports/cell63_survivorship_audit_summary.csv (0.0 MB)
OK       reports/cell64_real_vs_placebo_comparison.csv (0.0 MB)
OK       reports/cell65_next_day_execution_metrics.csv (0.0 MB)
OK       reports/cell66_block_bootstrap

In [10]:
# ==============================================================================
# CELL 68H — FINAL ARTIFACT INTEGRITY & RESULTS VERIFICATION
# ==============================================================================

import os
import json
import hashlib
import pandas as pd

PROJECT = "/content/NEPSE_ML_PROJECT"

MODEL_PATH = os.path.join(
    PROJECT,
    "models",
    "cell58_stock_only_cross_sectional_xgboost.json"
)

FINAL_METRICS = os.path.join(
    PROJECT,
    "reports",
    "cell67_final_test_metrics.csv"
)

FINAL_SUMMARY = os.path.join(
    PROJECT,
    "reports",
    "cell67_final_test_summary.json"
)

FINAL_YEARLY = os.path.join(
    PROJECT,
    "reports",
    "cell67_final_test_yearly.csv"
)

FINAL_PERIODS = os.path.join(
    PROJECT,
    "reports",
    "cell67_final_test_periods.csv"
)

DATASET = os.path.join(
    PROJECT,
    "features",
    "nepse_ml_clean_dataset.parquet"
)

print("=" * 78)
print("FINAL NEPSE ML ARTIFACT VERIFICATION")
print("=" * 78)

# --------------------------------------------------------------------------
# FILE HASH
# --------------------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()

print("\nMODEL:")
print("-" * 78)

model_size = os.path.getsize(MODEL_PATH) / (1024**2)
model_hash = sha256_file(MODEL_PATH)

print(f"Path       : {MODEL_PATH}")
print(f"Size       : {model_size:.3f} MB")
print(f"SHA256     : {model_hash}")

# --------------------------------------------------------------------------
# FINAL TEST METRICS
# --------------------------------------------------------------------------

print("\nFINAL TEST METRICS:")
print("-" * 78)

metrics = pd.read_csv(FINAL_METRICS)

print(metrics.to_string(index=False))

# --------------------------------------------------------------------------
# FINAL TEST SUMMARY
# --------------------------------------------------------------------------

print("\nFINAL TEST SUMMARY:")
print("-" * 78)

with open(FINAL_SUMMARY, "r") as f:
    summary = json.load(f)

print(json.dumps(summary, indent=2, default=str))

# --------------------------------------------------------------------------
# FINAL TEST PERIODS
# --------------------------------------------------------------------------

print("\nFINAL TEST PERIOD AUDIT:")
print("-" * 78)

periods = pd.read_csv(FINAL_PERIODS)

date_candidates = [
    c for c in periods.columns
    if "date" in c.lower()
]

if date_candidates:
    date_col = date_candidates[0]
    periods[date_col] = pd.to_datetime(
        periods[date_col],
        errors="coerce"
    )

    print(f"Date column: {date_col}")
    print(f"Rows       : {len(periods):,}")
    print(f"Start      : {periods[date_col].min()}")
    print(f"End        : {periods[date_col].max()}")

    leakage_rows = periods[
        periods[date_col] < pd.Timestamp("2025-01-01")
    ]

    print(f"Pre-2025 rows: {len(leakage_rows)}")

    if len(leakage_rows) == 0:
        print("FINAL TEST DATE INTEGRITY: PASS")
    else:
        print("FINAL TEST DATE INTEGRITY: FAIL")

# --------------------------------------------------------------------------
# YEARLY RESULTS
# --------------------------------------------------------------------------

print("\nFINAL TEST YEARLY RESULTS:")
print("-" * 78)

yearly = pd.read_csv(FINAL_YEARLY)
print(yearly.to_string(index=False))

# --------------------------------------------------------------------------
# DATASET
# --------------------------------------------------------------------------

print("\nCLEAN DATASET:")
print("-" * 78)

dataset_size = os.path.getsize(DATASET) / (1024**2)

print(f"Path       : {DATASET}")
print(f"Size       : {dataset_size:.1f} MB")

# We intentionally do not load the 318 MB dataset here.
# Existence + size are sufficient for this integrity check.

print("\n" + "=" * 78)
print("ARTIFACT VERIFICATION COMPLETE")
print("=" * 78)
print("The restored project is ready for final freezing/documentation.")
print("=" * 78)

FINAL NEPSE ML ARTIFACT VERIFICATION

MODEL:
------------------------------------------------------------------------------
Path       : /content/NEPSE_ML_PROJECT/models/cell58_stock_only_cross_sectional_xgboost.json
Size       : 0.561 MB
SHA256     : 2870f9f534e7a2d8ccc65ca719350e2d17fef52d7f2014baf89427e3ba0e3379

FINAL TEST METRICS:
------------------------------------------------------------------------------
 observations  total_return      CAGR  max_drawdown  annualized_sharpe  mean_period_return  median_period_return  positive_fraction  mean_excess_vs_benchmark  mean_turnover  mean_transaction_cost        strategy
           20      1.132467  0.582017     -0.134723           1.410707            0.043882              0.010869               0.55                  0.044388            1.0                  0.003    FINAL Top 5%
           20      0.315389  0.180629     -0.117927           0.929798            0.015465              0.003213               0.55                  0.015970  

In [11]:
# ==============================================================================
# CELL 68I — FREEZE FINAL NEPSE ML MODEL + RESEARCH MANIFEST
# ==============================================================================

import os
import json
import hashlib
from datetime import datetime

PROJECT = "/content/NEPSE_ML_PROJECT"
MODEL_PATH = os.path.join(
    PROJECT,
    "models",
    "cell58_stock_only_cross_sectional_xgboost.json"
)

MANIFEST_PATH = os.path.join(
    PROJECT,
    "reports",
    "cell68_final_frozen_model_manifest.json"
)

FEATURES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()

model_hash = sha256_file(MODEL_PATH)

manifest = {
    "project": "NEPSE ML Research",
    "status": "FINAL_FROZEN",

    "frozen_at": datetime.now().isoformat(),

    "final_model": {
        "name": "cell58_stock_only_cross_sectional_xgboost",
        "path": "models/cell58_stock_only_cross_sectional_xgboost.json",
        "sha256": model_hash,
        "feature_count": len(FEATURES),
        "features": FEATURES,

        "model_type": "XGBRegressor",

        "parameters": {
            "n_estimators": 500,
            "learning_rate": 0.025,
            "max_depth": 3,
            "min_child_weight": 30,
            "subsample": 0.80,
            "colsample_bytree": 0.85,
            "reg_alpha": 1.0,
            "reg_lambda": 10.0,
            "gamma": 0.10,
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "tree_method": "hist",
            "random_state": 42
        }
    },

    "research_design": {
        "prediction_target": "future_return_20d",
        "ranking_method": "cross_sectional_stock_ranking",
        "portfolio_selection": [
            "Top 5%",
            "Top 10%",
            "Top 20%"
        ],

        "stock_only": True,
        "market_features_used": False,
        "sector_features_used": False,
        "relative_features_used": False,

        "training_period": "2011-2024",
        "development_validation_period": "2017-2024",
        "final_test_period": "2025-2026",

        "final_test_used_for_training": False,
        "final_test_used_for_tuning": False,

        "holding_sessions": 20,
        "purge_sessions": 20,

        "execution": {
            "signal": "close of day T",
            "entry": "close of T+1",
            "exit": "close of T+21",
            "holding_period": "20 trading sessions"
        },

        "transaction_cost": {
            "one_way": 0.003,
            "final_test_turnover_assumption": 1.0
        }
    },

    "final_oos_test": {
        "test_start": "2025-01-01",
        "test_end_dataset": "2026-10-06",
        "rebalance_dates": 21,
        "portfolio_observations": 20,
        "symbols": 512,
        "rows": 140191,

        "results": {
            "top_5": {
                "total_return": 1.132467,
                "CAGR": 0.582017,
                "max_drawdown": -0.134723,
                "annualized_sharpe": 1.410707,
                "mean_period_return": 0.043882,
                "mean_excess_vs_benchmark": 0.044388
            },

            "top_10": {
                "total_return": 0.315389,
                "CAGR": 0.180629,
                "max_drawdown": -0.117927,
                "annualized_sharpe": 0.929798,
                "mean_period_return": 0.015465,
                "mean_excess_vs_benchmark": 0.015970
            },

            "top_20": {
                "total_return": -0.003935,
                "CAGR": -0.002385,
                "max_drawdown": -0.116295,
                "annualized_sharpe": 0.051919,
                "mean_period_return": 0.000579,
                "mean_excess_vs_benchmark": 0.001084
            },

            "benchmark": {
                "total_return": -0.025030,
                "CAGR": -0.015237,
                "max_drawdown": -0.129717,
                "annualized_sharpe": -0.045514
            }
        }
    },

    "robustness_tests_completed": [
        "technical-only feature ablation",
        "walk-forward validation",
        "cross-sectional ranking stability",
        "leakage-free portfolio backtest",
        "transaction-cost sensitivity",
        "survivorship audit",
        "shuffled-target placebo test",
        "next-day execution test",
        "block bootstrap significance test",
        "final test integrity audit",
        "untouched 2025-2026 out-of-sample test"
    ],

    "important_methodological_notes": [
        "Market, sector, and relative features were deliberately excluded from the final stock model.",
        "The 2025-2026 final test was protected from model training and tuning.",
        "The final test uses a conservative fixed 100% turnover assumption for transaction costs.",
        "The final test contains only 20 completed portfolio observations, so results are encouraging but not proof of future performance.",
        "The model is a research model and does not constitute investment advice."
    ],

    "artifact_integrity": {
        "model_sha256": model_hash,
        "clean_dataset": "features/nepse_ml_clean_dataset.parquet",
        "final_test_metrics": "reports/cell67_final_test_metrics.csv",
        "final_test_periods": "reports/cell67_final_test_periods.csv",
        "final_test_yearly": "reports/cell67_final_test_yearly.csv",
        "final_test_summary": "reports/cell67_final_test_summary.json"
    }
}

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

print("=" * 78)
print("FINAL MODEL FROZEN")
print("=" * 78)

print(f"\nManifest:")
print(MANIFEST_PATH)

print(f"\nModel SHA256:")
print(model_hash)

print(f"\nFeatures: {len(FEATURES)}")
for i, feature in enumerate(FEATURES, 1):
    print(f"{i:2d}. {feature}")

print("\nFinal OOS test:")
print("  2025-01-01 → 2026-10-06")
print("  20 completed portfolio observations")

print("\nTop 5%:")
print("  CAGR   : 58.20%")
print("  Sharpe : 1.41")
print("  Max DD : -13.47%")
print("  Excess : +4.44% / period")

print("\nTop 10%:")
print("  CAGR   : 18.06%")
print("  Sharpe : 0.93")
print("  Max DD : -11.79%")
print("  Excess : +1.60% / period")

print("\nTop 20%:")
print("  CAGR   : -0.24%")
print("  Sharpe : 0.05")
print("  Max DD : -11.63%")
print("  Excess : +0.11% / period")

print("\n" + "=" * 78)
print("FREEZE COMPLETE")
print("=" * 78)

FINAL MODEL FROZEN

Manifest:
/content/NEPSE_ML_PROJECT/reports/cell68_final_frozen_model_manifest.json

Model SHA256:
2870f9f534e7a2d8ccc65ca719350e2d17fef52d7f2014baf89427e3ba0e3379

Features: 13
 1. return_1d
 2. return_5d
 3. return_20d
 4. return_60d
 5. close_to_sma_20
 6. close_to_sma_50
 7. close_to_sma_200
 8. rsi_14
 9. volatility_20d
10. volume_ratio_20
11. volume_change_1d
12. volume_spike_20d
13. trading_activity_score

Final OOS test:
  2025-01-01 → 2026-10-06
  20 completed portfolio observations

Top 5%:
  CAGR   : 58.20%
  Sharpe : 1.41
  Max DD : -13.47%
  Excess : +4.44% / period

Top 10%:
  CAGR   : 18.06%
  Sharpe : 0.93
  Max DD : -11.79%
  Excess : +1.60% / period

Top 20%:
  CAGR   : -0.24%
  Sharpe : 0.05
  Max DD : -11.63%
  Excess : +0.11% / period

FREEZE COMPLETE


In [12]:
# ==============================================================================
# CELL 69 — RUN FROZEN MODEL ON LATEST AVAILABLE NEPSE DATA
# ==============================================================================
# Purpose:
#   Load the frozen Cell-58 model and rank stocks using the latest available
#   feature row for each symbol in the recovered dataset.
#
# IMPORTANT:
#   - NO TRAINING
#   - NO MODEL MODIFICATION
#   - NO MARKET FEATURES
#   - NO SECTOR FEATURES
#   - NO RELATIVE FEATURES
# ==============================================================================

import os
import json
import numpy as np
import pandas as pd
import xgboost as xgb

PROJECT = "/content/NEPSE_ML_PROJECT"

DATASET = os.path.join(
    PROJECT,
    "features",
    "nepse_ml_clean_dataset.parquet"
)

MODEL_PATH = os.path.join(
    PROJECT,
    "models",
    "cell58_stock_only_cross_sectional_xgboost.json"
)

OUTPUT_CSV = os.path.join(
    PROJECT,
    "reports",
    "cell69_latest_stock_ranking.csv"
)

FEATURES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

print("=" * 78)
print("CELL 69 — FROZEN MODEL STOCK RANKING")
print("=" * 78)

# --------------------------------------------------------------------------
# CHECK FILES
# --------------------------------------------------------------------------

if not os.path.isfile(DATASET):
    raise FileNotFoundError(DATASET)

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(MODEL_PATH)

print("\nLoading frozen model...")

model = xgb.XGBRegressor()
model.load_model(MODEL_PATH)

print("Frozen model loaded successfully.")

# --------------------------------------------------------------------------
# LOAD ONLY REQUIRED COLUMNS
# --------------------------------------------------------------------------

required_columns = [
    "date",
    "symbol",
    "close",
] + FEATURES

print("\nLoading latest feature data...")
print(f"Features used: {len(FEATURES)}")

df = pd.read_parquet(
    DATASET,
    columns=required_columns
)

print(f"Rows loaded: {len(df):,}")

# --------------------------------------------------------------------------
# DATE CLEANUP
# --------------------------------------------------------------------------

df["date"] = pd.to_datetime(df["date"], errors="coerce")

df = df.dropna(
    subset=["date", "symbol", "close"]
).copy()

# --------------------------------------------------------------------------
# LATEST ROW PER SYMBOL
# --------------------------------------------------------------------------

latest = (
    df.sort_values(["symbol", "date"])
      .groupby("symbol", as_index=False)
      .tail(1)
      .copy()
)

print(f"Symbols with latest data: {len(latest):,}")

latest_date = latest["date"].max()

print(f"Latest dataset date: {latest_date.date()}")

# --------------------------------------------------------------------------
# FEATURE VALIDITY
# --------------------------------------------------------------------------

X = latest[FEATURES].copy()

# Use finite numeric values only.
X = X.replace([np.inf, -np.inf], np.nan)

# The production pipeline uses the same training-time median philosophy.
# Here we use medians from the current ranking universe only for any
# unavoidable missing feature values. We do NOT retrain the model.
#
# In the historical model-ready pipeline, training medians were used.
# For most mature securities these features should already be populated.

missing_before = int(X.isna().any(axis=1).sum())

if missing_before > 0:
    print(
        f"\nRows with at least one missing feature: "
        f"{missing_before:,}"
    )

    X = X.fillna(X.median(numeric_only=True))

# --------------------------------------------------------------------------
# PREDICT
# --------------------------------------------------------------------------

print("\nGenerating predictions...")

latest["predicted_20d_return"] = model.predict(X)

# --------------------------------------------------------------------------
# RANK
# --------------------------------------------------------------------------

latest = latest.sort_values(
    "predicted_20d_return",
    ascending=False
).reset_index(drop=True)

latest["rank"] = np.arange(1, len(latest) + 1)

latest["rank_percentile"] = (
    1.0
    - (latest["rank"] - 1) / max(len(latest) - 1, 1)
)

# --------------------------------------------------------------------------
# TOP 5 / 10 / 20%
# --------------------------------------------------------------------------

n = len(latest)

top5_cut = max(1, int(np.ceil(n * 0.05)))
top10_cut = max(1, int(np.ceil(n * 0.10)))
top20_cut = max(1, int(np.ceil(n * 0.20)))

latest["top_5pct"] = latest["rank"] <= top5_cut
latest["top_10pct"] = latest["rank"] <= top10_cut
latest["top_20pct"] = latest["rank"] <= top20_cut

# --------------------------------------------------------------------------
# OUTPUT COLUMNS
# --------------------------------------------------------------------------

output_columns = [
    "rank",
    "symbol",
    "date",
    "close",
    "predicted_20d_return",
    "rank_percentile",
    "top_5pct",
    "top_10pct",
    "top_20pct",
]

ranking = latest[output_columns].copy()

ranking.to_csv(
    OUTPUT_CSV,
    index=False
)

# --------------------------------------------------------------------------
# DISPLAY RESULTS
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("LATEST MODEL RANKING")
print("=" * 78)

print(f"\nRanking date: {latest_date.date()}")
print(f"Symbols     : {len(ranking):,}")

print("\nTOP 20 STOCKS")
print("-" * 78)

display(
    ranking.head(20).style.format({
        "close": "{:.2f}",
        "predicted_20d_return": "{:.2%}",
        "rank_percentile": "{:.2%}",
    })
)

print("\n" + "=" * 78)
print("PORTFOLIO CUTS")
print("=" * 78)

print(f"Top 5%  : {top5_cut} stocks")
print(f"Top 10% : {top10_cut} stocks")
print(f"Top 20% : {top20_cut} stocks")

print("\nTop 5% predicted average:")
print(
    f"{ranking.head(top5_cut)['predicted_20d_return'].mean():.2%}"
)

print("Top 10% predicted average:")
print(
    f"{ranking.head(top10_cut)['predicted_20d_return'].mean():.2%}"
)

print("Top 20% predicted average:")
print(
    f"{ranking.head(top20_cut)['predicted_20d_return'].mean():.2%}"
)

print("\nSaved:")
print(OUTPUT_CSV)

print("\n" + "=" * 78)
print("CELL 69 COMPLETE")
print("=" * 78)

CELL 69 — FROZEN MODEL STOCK RANKING

Loading frozen model...
Frozen model loaded successfully.

Loading latest feature data...
Features used: 13
Rows loaded: 633,137
Symbols with latest data: 651
Latest dataset date: 2026-10-06

Rows with at least one missing feature: 214

Generating predictions...

LATEST MODEL RANKING

Ranking date: 2026-10-06
Symbols     : 651

TOP 20 STOCKS
------------------------------------------------------------------------------


,rank,symbol,date,close,predicted_20d_return,rank_percentile,top_5pct,top_10pct,top_20pct
0,1,HBLPO,2026-09-01 00:00:00,102.00,82.13%,100.00%,True,True,True
1,2,NICAP,2026-09-30 00:00:00,179.00,80.27%,99.85%,True,True,True
2,3,NLICLP,2026-02-16 00:00:00,321.00,75.09%,99.69%,True,True,True
3,4,SINDU,2026-10-06 00:00:00,418.00,49.93%,99.54%,True,True,True
4,5,ALICLP,2026-09-28 00:00:00,239.00,48.23%,99.38%,True,True,True
5,6,SRBLPO,2022-07-21 00:00:00,128.00,47.58%,99.23%,True,True,True
6,7,NIBPO,2023-01-10 00:00:00,230.00,31.53%,99.08%,True,True,True
7,8,MMFDBP,2024-03-12 00:00:00,278.00,27.90%,98.92%,True,True,True
8,9,SAMAJ,2025-10-16 00:00:00,1875.80,27.04%,98.77%,True,True,True
9,10,PLICPO,2022-04-26 00:00:00,375.00,22.97%,98.62%,True,True,True



PORTFOLIO CUTS
Top 5%  : 33 stocks
Top 10% : 66 stocks
Top 20% : 131 stocks

Top 5% predicted average:
25.43%
Top 10% predicted average:
16.59%
Top 20% predicted average:
10.59%

Saved:
/content/NEPSE_ML_PROJECT/reports/cell69_latest_stock_ranking.csv

CELL 69 COMPLETE


In [14]:
# ==============================================================================
# CELL 70A — CORRECTED LIVE UNIVERSE AUDIT
# ==============================================================================
# Fixes Cell 70's classification mismatch:
#
#   Clean dataset: type = "stock"
#   Security master: type = "COMMON_EQUITY", "PROMOTER_SHARE", etc.
#
# No model training.
# No model modification.
# ==============================================================================

import os
import pandas as pd
import numpy as np

PROJECT = "/content/NEPSE_ML_PROJECT"

DATASET = os.path.join(
    PROJECT,
    "features",
    "nepse_ml_clean_dataset.parquet"
)

SECURITY_MASTER = os.path.join(
    PROJECT,
    "processed",
    "security_master_classified.csv"
)

FEATURES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

AS_OF_DATE = pd.Timestamp("2026-10-06")

print("=" * 78)
print("CELL 70A — CORRECTED LIVE UNIVERSE AUDIT")
print("=" * 78)

# --------------------------------------------------------------------------
# LOAD DATA
# --------------------------------------------------------------------------

columns = [
    "date",
    "symbol",
    "close",
] + FEATURES

df = pd.read_parquet(
    DATASET,
    columns=columns
)

df["date"] = pd.to_datetime(df["date"], errors="coerce")

master = pd.read_csv(SECURITY_MASTER)

print(f"\nDataset rows   : {len(df):,}")
print(f"Dataset symbols: {df['symbol'].nunique():,}")
print(f"Latest date    : {df['date'].max().date()}")

print("\nSecurity master classifications:")
print(
    master["type"]
    .value_counts(dropna=False)
    .to_string()
)

# --------------------------------------------------------------------------
# LATEST ROW PER SYMBOL
# --------------------------------------------------------------------------

latest = (
    df.dropna(subset=["date", "symbol"])
      .sort_values(["symbol", "date"])
      .groupby("symbol", as_index=False)
      .tail(1)
      .copy()
)

latest["days_stale"] = (
    AS_OF_DATE - latest["date"]
).dt.days

# --------------------------------------------------------------------------
# MERGE MASTER
# --------------------------------------------------------------------------

master_cols = [
    "symbol",
    "full_name",
    "description",
    "exchange",
    "type",
    "sector",
]

master_small = master[
    [c for c in master_cols if c in master.columns]
].copy()

# Rename master type explicitly to avoid confusion.
master_small = master_small.rename(
    columns={"type": "security_type"}
)

latest = latest.merge(
    master_small,
    on="symbol",
    how="left"
)

# --------------------------------------------------------------------------
# FEATURE COMPLETENESS
# --------------------------------------------------------------------------

latest["missing_feature_count"] = (
    latest[FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .isna()
    .sum(axis=1)
)

latest["features_complete"] = (
    latest["missing_feature_count"] == 0
)

# --------------------------------------------------------------------------
# FRESHNESS
# --------------------------------------------------------------------------

def freshness_bucket(days):
    if days <= 1:
        return "0-1 days"
    elif days <= 5:
        return "2-5 days"
    elif days <= 20:
        return "6-20 days"
    elif days <= 60:
        return "21-60 days"
    elif days <= 180:
        return "61-180 days"
    elif days <= 365:
        return "181-365 days"
    else:
        return ">365 days"

latest["freshness_bucket"] = latest["days_stale"].apply(
    freshness_bucket
)

# --------------------------------------------------------------------------
# CLASSIFICATION DISTRIBUTION
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("HISTORICAL DATASET SYMBOL CLASSIFICATION")
print("=" * 78)

print(
    latest["security_type"]
    .value_counts(dropna=False)
    .to_string()
)

# --------------------------------------------------------------------------
# COMMON EQUITY
# --------------------------------------------------------------------------

common = latest[
    latest["security_type"].eq("COMMON_EQUITY")
].copy()

print("\n" + "=" * 78)
print("COMMON EQUITY")
print("=" * 78)

print(f"Common-equity symbols: {len(common):,}")

# --------------------------------------------------------------------------
# COMMON EQUITY FRESHNESS
# --------------------------------------------------------------------------

print("\nCommon-equity freshness:")

freshness_order = [
    "0-1 days",
    "2-5 days",
    "6-20 days",
    "21-60 days",
    "61-180 days",
    "181-365 days",
    ">365 days",
]

common_freshness = (
    common["freshness_bucket"]
    .value_counts()
    .reindex(freshness_order)
    .fillna(0)
    .astype(int)
)

print(common_freshness.to_string())

# --------------------------------------------------------------------------
# FEATURE COMPLETENESS
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("FEATURE COMPLETENESS")
print("=" * 78)

print(
    f"Complete features: "
    f"{common['features_complete'].sum():,}"
)

print(
    f"Missing features:  "
    f"{(~common['features_complete']).sum():,}"
)

# --------------------------------------------------------------------------
# OPERATIONAL UNIVERSES
# --------------------------------------------------------------------------

live_1 = common[
    (common["days_stale"] <= 1) &
    common["features_complete"]
].copy()

live_5 = common[
    (common["days_stale"] <= 5) &
    common["features_complete"]
].copy()

live_20 = common[
    (common["days_stale"] <= 20) &
    common["features_complete"]
].copy()

print("\n" + "=" * 78)
print("OPERATIONAL UNIVERSE OPTIONS")
print("=" * 78)

print(
    f"Common equity + <=1 day + complete : {len(live_1):,}"
)

print(
    f"Common equity + <=5 days + complete : {len(live_5):,}"
)

print(
    f"Common equity + <=20 days + complete: {len(live_20):,}"
)

# --------------------------------------------------------------------------
# STALE COMMON EQUITIES
# --------------------------------------------------------------------------

stale = common[
    common["days_stale"] > 20
].sort_values(
    "days_stale",
    ascending=False
)

print("\n" + "=" * 78)
print("STALE COMMON EQUITIES (>20 DAYS)")
print("=" * 78)

print(f"Count: {len(stale):,}")

if len(stale):
    display(
        stale[
            [
                "symbol",
                "full_name",
                "security_type",
                "date",
                "close",
                "days_stale",
                "missing_feature_count",
            ]
        ].head(30)
    )

# --------------------------------------------------------------------------
# COMPLETE CURRENT COMMON EQUITIES
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("CURRENT COMMON EQUITY CANDIDATES")
print("=" * 78)

if len(live_5):
    display(
        live_5[
            [
                "symbol",
                "full_name",
                "date",
                "close",
                "days_stale",
                "sector",
            ]
        ]
        .sort_values("symbol")
        .head(50)
    )

# --------------------------------------------------------------------------
# FINAL SUMMARY
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("CELL 70A SUMMARY")
print("=" * 78)

print(f"As-of date                 : {AS_OF_DATE.date()}")
print(f"All symbols                : {len(latest):,}")
print(f"Common equities            : {len(common):,}")
print(f"Common <=1 day             : {len(live_1):,}")
print(f"Common <=5 days            : {len(live_5):,}")
print(f"Common <=20 days           : {len(live_20):,}")
print(f"Complete-feature common    : {common['features_complete'].sum():,}")

print("\nFrozen model unchanged.")
print("No training performed.")

print("=" * 78)
print("CELL 70A COMPLETE")
print("=" * 78)

CELL 70A — CORRECTED LIVE UNIVERSE AUDIT

Dataset rows   : 633,137
Dataset symbols: 651
Latest date    : 2026-10-06

Security master classifications:
type
stock    720
index     17

HISTORICAL DATASET SYMBOL CLASSIFICATION
security_type
stock    633
index     17
NaN        1

COMMON EQUITY
Common-equity symbols: 0

Common-equity freshness:
freshness_bucket
0-1 days        0
2-5 days        0
6-20 days       0
21-60 days      0
61-180 days     0
181-365 days    0
>365 days       0

FEATURE COMPLETENESS
Complete features: 0
Missing features:  0

OPERATIONAL UNIVERSE OPTIONS
Common equity + <=1 day + complete : 0
Common equity + <=5 days + complete : 0
Common equity + <=20 days + complete: 0

STALE COMMON EQUITIES (>20 DAYS)
Count: 0

CURRENT COMMON EQUITY CANDIDATES

CELL 70A SUMMARY
As-of date                 : 2026-10-06
All symbols                : 651
Common equities            : 0
Common <=1 day             : 0
Common <=5 days            : 0
Common <=20 days           : 0
Complete-f

In [15]:
# ==============================================================================
# CELL 70B — INSPECT ACTUAL SECURITY MASTER CLASSIFICATION
# ==============================================================================

import os
import pandas as pd

PROJECT = "/content/NEPSE_ML_PROJECT"

MASTER = os.path.join(
    PROJECT,
    "processed",
    "security_master_classified.csv"
)

m = pd.read_csv(MASTER)

print("=" * 78)
print("CELL 70B — ACTUAL SECURITY MASTER INSPECTION")
print("=" * 78)

print("\nColumns:")
print(list(m.columns))

print("\nShape:")
print(m.shape)

# --------------------------------------------------------------------------
# TYPE
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("TYPE")
print("=" * 78)

print(
    m["type"]
    .value_counts(dropna=False)
    .to_string()
)

# --------------------------------------------------------------------------
# EXCHANGE
# --------------------------------------------------------------------------

if "exchange" in m.columns:
    print("\n" + "=" * 78)
    print("EXCHANGE")
    print("=" * 78)

    print(
        m["exchange"]
        .value_counts(dropna=False)
        .head(30)
        .to_string()
    )

# --------------------------------------------------------------------------
# SECTOR
# --------------------------------------------------------------------------

if "sector" in m.columns:
    print("\n" + "=" * 78)
    print("SECTOR")
    print("=" * 78)

    print(
        m["sector"]
        .value_counts(dropna=False)
        .head(50)
        .to_string()
    )

# --------------------------------------------------------------------------
# IS_MASTER
# --------------------------------------------------------------------------

if "is_master" in m.columns:
    print("\n" + "=" * 78)
    print("IS_MASTER")
    print("=" * 78)

    print(
        m["is_master"]
        .value_counts(dropna=False)
        .to_string()
    )

# --------------------------------------------------------------------------
# SYMBOL EXAMPLES
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("SAMPLE SECURITIES")
print("=" * 78)

display(
    m[
        [
            c for c in [
                "symbol",
                "full_name",
                "description",
                "exchange",
                "type",
                "sector",
                "is_master"
            ]
            if c in m.columns
        ]
    ].head(100)
)

# --------------------------------------------------------------------------
# NAMES THAT LOOK LIKE NON-COMMON SECURITIES
# --------------------------------------------------------------------------

name_text = (
    m["full_name"]
    .fillna("")
    .astype(str)
    .str.upper()
)

patterns = (
    "PROMOTER|"
    "DEBENTURE|"
    "BOND|"
    "MUTUAL|"
    "FUND|"
    "PREFERENCE|"
    "PREFERRED|"
    "INDEX|"
    "PREF|"
)

suspect = m[
    name_text.str.contains(patterns, regex=True, na=False)
].copy()

print("\n" + "=" * 78)
print("POSSIBLE NON-COMMON SECURITIES BY NAME")
print("=" * 78)

print(f"Count: {len(suspect):,}")

if len(suspect):
    display(
        suspect[
            [
                c for c in [
                    "symbol",
                    "full_name",
                    "description",
                    "exchange",
                    "type",
                    "sector",
                    "is_master"
                ]
                if c in suspect.columns
            ]
        ].head(200)
    )

# --------------------------------------------------------------------------
# SYMBOL PATTERN INSPECTION
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("SYMBOLS WITH COMMON SPECIAL-SHARE SUFFIXES")
print("=" * 78)

special_suffixes = (
    "PO",
    "P",
    "D",
    "DB",
    "MF",
    "F",
    "CP"
)

for suffix in special_suffixes:
    subset = m[
        m["symbol"]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.endswith(suffix)
    ]

    print(
        f"{suffix:>4} : {len(subset):4d}"
    )

print("\n" + "=" * 78)
print("CELL 70B COMPLETE")
print("=" * 78)

CELL 70B — ACTUAL SECURITY MASTER INSPECTION

Columns:
['symbol', 'full_name', 'description', 'exchange', 'type', 'sector', 'first_date_adjusted', 'last_date_adjusted', 'trading_rows_adjusted', 'unique_dates_adjusted', 'first_date_unadjusted', 'last_date_unadjusted', 'trading_rows_unadjusted', 'unique_dates_unadjusted', 'initial_category', 'security_class']

Shape:
(737, 16)

TYPE
type
stock    720
index     17

EXCHANGE
exchange
PROMOTSHARE src:Nepsealpha.com                     145
Hydro Power src:Nepsealpha.com                     114
CORPDEBEN src:Nepsealpha.com                        85
Microfinance src:Nepsealpha.com                     79
Mutual Fund src:Nepsealpha.com                      75
Commercial Banks src:Nepsealpha.com                 32
Manufacturing And Processing src:Nepsealpha.com     32
Development Banks src:Nepsealpha.com                31
Finance src:Nepsealpha.com                          28
Non Life Insurance src:Nepsealpha.com               27
Life Insurance s

,symbol,full_name,description,exchange,type,sector
0,10.75NMB89/90,10.75NMB89/90,10.75% NMB Debenture 2089/2090,CORPDEBEN src:Nepsealpha.com,stock,CORPDEBEN
1,ACLBSL,ACLBSL,Aarambha Chautari Laghubitta Bittiya Sanstha L...,Microfinance src:Nepsealpha.com,stock,MICROFINANCE
2,ADBL,ADBL,Agricultural Development Bank Limited,Commercial Banks src:Nepsealpha.com,stock,BANKING
3,ADBLB,ADBLB,4% Agricultural Bond,Corporate Debenture src:Nepsealpha.com,stock,CORPDEBEN
4,ADBLB86,ADBLB86,"8 Years, 4% Agriculture Bond",CORPDEBEN src:Nepsealpha.com,stock,CORPDEBEN
...,...,...,...,...,...,...
95,DEVBANK,DEVBANK,Development Bank,Development Bank src:Nepsealpha.com,index,DEVBANK
96,DHEL,DHEL,Daramkhola Hydro Energy Limited,Hydro Power src:Nepsealpha.com,stock,HYDROPOWER
97,DHPL,DHPL,Dibyashwori Hydropower Ltd.,Hydro Power src:Nepsealpha.com,stock,HYDROPOWER
98,DLBS,DLBS,Dhaulagiri Laghubitta Bittiya Sanstha Limited,Microfinance src:Nepsealpha.com,stock,MICROFINANCE



POSSIBLE NON-COMMON SECURITIES BY NAME
Count: 737


,symbol,full_name,description,exchange,type,sector
0,10.75NMB89/90,10.75NMB89/90,10.75% NMB Debenture 2089/2090,CORPDEBEN src:Nepsealpha.com,stock,CORPDEBEN
1,ACLBSL,ACLBSL,Aarambha Chautari Laghubitta Bittiya Sanstha L...,Microfinance src:Nepsealpha.com,stock,MICROFINANCE
2,ADBL,ADBL,Agricultural Development Bank Limited,Commercial Banks src:Nepsealpha.com,stock,BANKING
3,ADBLB,ADBLB,4% Agricultural Bond,Corporate Debenture src:Nepsealpha.com,stock,CORPDEBEN
4,ADBLB86,ADBLB86,"8 Years, 4% Agriculture Bond",CORPDEBEN src:Nepsealpha.com,stock,CORPDEBEN
...,...,...,...,...,...,...
195,ICFCD83,ICFCD83,12% ICFC Finance Limited Debenture 2083,CORPDEBEN src:Nepsealpha.com,stock,CORPDEBEN
196,ICFCD88,ICFCD88,9% ICFC Finance Limited Debenture-2088,CORPDEBEN src:Nepsealpha.com,stock,CORPDEBEN
197,ICFCD89,ICFCD89,8% ICFC Finance Limited Debenture 2089,Corporate Debenture src:Nepsealpha.com,stock,CORPDEBEN
198,ICFCPO,ICFCPO,ICFC Finance Limited Promotor Share,PROMOTSHARE src:Nepsealpha.com,stock,PROMOTSHARE



SYMBOLS WITH COMMON SPECIAL-SHARE SUFFIXES
  PO :   98
   P :   66
   D :    6
  DB :    5
  MF :    5
   F :   29
  CP :    5

CELL 70B COMPLETE


In [16]:
# ==============================================================================
# CELL 70C — INSPECT EXISTING SECURITY CLASSIFICATION
# ==============================================================================
# Uses the classification already created in the project.
# Does NOT infer security type from symbol names/suffixes.
# Does NOT modify the model.
# ==============================================================================

import os
import pandas as pd

PROJECT = "/content/NEPSE_ML_PROJECT"

MASTER = os.path.join(
    PROJECT,
    "processed",
    "security_master_classified.csv"
)

m = pd.read_csv(MASTER)

print("=" * 78)
print("CELL 70C — EXISTING SECURITY CLASSIFICATION")
print("=" * 78)

# --------------------------------------------------------------------------
# SECURITY CLASS
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("SECURITY_CLASS DISTRIBUTION")
print("=" * 78)

print(
    m["security_class"]
    .value_counts(dropna=False)
    .to_string()
)

# --------------------------------------------------------------------------
# INITIAL CATEGORY
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("INITIAL_CATEGORY DISTRIBUTION")
print("=" * 78)

print(
    m["initial_category"]
    .value_counts(dropna=False)
    .to_string()
)

# --------------------------------------------------------------------------
# CROSS TAB
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("SECURITY_CLASS × INITIAL_CATEGORY")
print("=" * 78)

cross = pd.crosstab(
    m["security_class"],
    m["initial_category"],
    dropna=False
)

display(cross)

# --------------------------------------------------------------------------
# CROSS TAB WITH EXCHANGE
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("SECURITY_CLASS × SECTOR")
print("=" * 78)

cross_sector = pd.crosstab(
    m["security_class"],
    m["sector"],
    dropna=False
)

display(cross_sector)

# --------------------------------------------------------------------------
# SHOW EVERY SECURITY CLASS
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("EXAMPLES BY SECURITY CLASS")
print("=" * 78)

for cls in m["security_class"].dropna().unique():

    subset = m[
        m["security_class"].eq(cls)
    ][
        [
            "symbol",
            "full_name",
            "description",
            "exchange",
            "sector",
            "initial_category",
            "security_class",
        ]
    ].head(15)

    print("\n" + "-" * 78)
    print(f"SECURITY CLASS: {cls}")
    print("-" * 78)

    display(subset)

# --------------------------------------------------------------------------
# POSSIBLE COMMON EQUITY CLASSES
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("CANDIDATE COMMON-EQUITY CLASSES")
print("=" * 78)

for cls in sorted(
    m["security_class"]
    .dropna()
    .astype(str)
    .unique()
):

    text = cls.upper()

    if any(
        x in text
        for x in [
            "COMMON",
            "ORDINARY",
            "EQUITY"
        ]
    ):
        subset = m[
            m["security_class"].astype(str).eq(cls)
        ]

        print(
            f"{cls}: {len(subset):,} securities"
        )

        display(
            subset[
                [
                    "symbol",
                    "full_name",
                    "description",
                    "exchange",
                    "sector",
                    "security_class",
                ]
            ].head(30)
        )

print("\n" + "=" * 78)
print("CELL 70C COMPLETE")
print("=" * 78)

CELL 70C — EXISTING SECURITY CLASSIFICATION

SECURITY_CLASS DISTRIBUTION
security_class
COMMON_EQUITY       397
PROMOTER_SHARE      148
BOND_DEBENTURE       94
MUTUAL_FUND          75
INDEX                15
PREFERENCE_SHARE      8

INITIAL_CATEGORY DISTRIBUTION
initial_category
OTHER                     554
BOND_OR_DEBENTURE          91
PREFERENCE_OR_PROMOTER     66
INDEX_OR_MARKET_SERIES     15
MUTUAL_FUND                11

SECURITY_CLASS × INITIAL_CATEGORY


initial_category,BOND_OR_DEBENTURE,INDEX_OR_MARKET_SERIES,MUTUAL_FUND,OTHER,PREFERENCE_OR_PROMOTER
security_class,,,,,
BOND_DEBENTURE,91,0,0,3,0
COMMON_EQUITY,0,0,0,389,8
INDEX,0,15,0,0,0
MUTUAL_FUND,0,0,11,63,1
PREFERENCE_SHARE,0,0,0,1,7
PROMOTER_SHARE,0,0,0,98,50



SECURITY_CLASS × SECTOR


sector,BANKING,CORPDEBEN,DEVBANK,FINANCE,FLOAT,HOTELS,HYDROPOWER,INVESTMENT,LIFEINSU,MANUFACTURE,MICROFINANCE,MUTUAL,NEPSE,NONLIFEINSU,OTHERS,PRESTOCK,PROMOTSHARE,SENFLOAT,SENSITIVE,TRADING
security_class,,,,,,,,,,,,,,,,,,,,
BOND_DEBENTURE,0,94,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
COMMON_EQUITY,30,0,31,27,0,9,113,11,25,31,78,0,0,26,11,0,0,0,0,5
INDEX,1,0,1,1,1,1,1,0,1,1,1,1,1,1,1,0,0,1,1,0
MUTUAL_FUND,0,0,0,0,0,0,0,0,0,0,0,75,0,0,0,0,0,0,0,0
PREFERENCE_SHARE,0,6,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0
PROMOTER_SHARE,2,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,145,0,0,0



EXAMPLES BY SECURITY CLASS

------------------------------------------------------------------------------
SECURITY CLASS: BOND_DEBENTURE
------------------------------------------------------------------------------


...


------------------------------------------------------------------------------
SECURITY CLASS: PREFERENCE_SHARE
------------------------------------------------------------------------------


,symbol,full_name,description,exchange,sector,initial_category,security_class
102,EBLCP,EBLCP,Everest Bank Limited Con. Pref.,PRESTOCK src:Nepsealpha.com,PRESTOCK,PREFERENCE_OR_PROMOTER,PREFERENCE_SHARE
255,KSBBLPNP,KSBBLPNP,KSBBL 9% Perpetual Non Cumulative Preference S...,Corporate Debenture src:Nepsealpha.com,CORPDEBEN,PREFERENCE_OR_PROMOTER,PREFERENCE_SHARE
295,MBLPNP,MBLPNP,8.25% Machhapuchchhre Perpetual Non-cumulative...,Corporate Debenture src:Nepsealpha.com,CORPDEBEN,PREFERENCE_OR_PROMOTER,PREFERENCE_SHARE
353,NABILPNP,NABILPNP,Nabil 8% Perpetual Non Cumulative Preference S...,Corporate Debenture src:Nepsealpha.com,CORPDEBEN,PREFERENCE_OR_PROMOTER,PREFERENCE_SHARE
447,NMBPNP,NMBPNP,8.25% NMB Perpetual Non cumulative Preference ...,Corporate Debenture src:Nepsealpha.com,CORPDEBEN,PREFERENCE_OR_PROMOTER,PREFERENCE_SHARE
566,SANIMAPNP,SANIMAPNP,8.25% Sanima Perpetual Non cumulative Preferen...,Corporate Debenture src:Nepsealpha.com,CORPDEBEN,PREFERENCE_OR_PROMOTER,PREFERENCE_SHARE
590,SBLPNP,SBLPNP,SBL 8.25% Perpetual Non cumulative Preference ...,Corporate Debenture src:Nepsealpha.com,CORPDEBEN,PREFERENCE_OR_PROMOTER,PREFERENCE_SHARE
705,TRHPR,TRHPR,Taragaon Regency Hotels Ltd. Preference Share,PRESTOCK src:Nepsealpha.com,PRESTOCK,OTHER,PREFERENCE_SHARE



CANDIDATE COMMON-EQUITY CLASSES
COMMON_EQUITY: 397 securities


,symbol,full_name,description,exchange,sector,security_class
1,ACLBSL,ACLBSL,Aarambha Chautari Laghubitta Bittiya Sanstha L...,Microfinance src:Nepsealpha.com,MICROFINANCE,COMMON_EQUITY
2,ADBL,ADBL,Agricultural Development Bank Limited,Commercial Banks src:Nepsealpha.com,BANKING,COMMON_EQUITY
7,ADLB,ADLB,Adarsha Laghubitta Bittiya Sanstha Limited,Microfinance src:Nepsealpha.com,MICROFINANCE,COMMON_EQUITY
8,AHL,AHL,Asian Hydropower Limited,Hydro Power src:Nepsealpha.com,HYDROPOWER,COMMON_EQUITY
9,AHPC,AHPC,Arun Valley Hydropower Development Co. Ltd.,Hydro Power src:Nepsealpha.com,HYDROPOWER,COMMON_EQUITY
10,AIL,AIL,Ajod Insurance Limited,Non Life Insurance src:Nepsealpha.com,NONLIFEINSU,COMMON_EQUITY
11,AKBSL,AKBSL,ADHIKHOLA LAGHUBITTA BITTIYA SANSTHA LIMITED,Microfinance src:Nepsealpha.com,MICROFINANCE,COMMON_EQUITY
12,AKJCL,AKJCL,Ankhu Khola Jalvidhyut Company Ltd,Hydro Power src:Nepsealpha.com,HYDROPOWER,COMMON_EQUITY
13,AKPL,AKPL,Arun Kabeli Power Ltd.,Hydro Power src:Nepsealpha.com,HYDROPOWER,COMMON_EQUITY
14,ALBSL,ALBSL,Asha Laghubitta Bittiya Sanstha Ltd,Microfinance src:Nepsealpha.com,MICROFINANCE,COMMON_EQUITY



CELL 70C COMPLETE


In [17]:
# ==============================================================================
# CELL 71 — BUILD CORRECT CURRENT COMMON-EQUITY UNIVERSE
# ==============================================================================
# Uses the already-established security_class classification.
#
# IMPORTANT:
#   COMMON_EQUITY = 397
#
# No retraining.
# No model modification.
# No ranking yet.
# ==============================================================================

import os
import pandas as pd
import numpy as np

PROJECT = "/content/NEPSE_ML_PROJECT"

DATASET = os.path.join(
    PROJECT,
    "features",
    "nepse_ml_clean_dataset.parquet"
)

MASTER = os.path.join(
    PROJECT,
    "processed",
    "security_master_classified.csv"
)

AS_OF_DATE = pd.Timestamp("2026-10-06")

FEATURES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

# --------------------------------------------------------------------------
# LOAD
# --------------------------------------------------------------------------

df = pd.read_parquet(
    DATASET,
    columns=["date", "symbol", "close"] + FEATURES
)

df["date"] = pd.to_datetime(df["date"], errors="coerce")

master = pd.read_csv(MASTER)

# --------------------------------------------------------------------------
# LATEST OBSERVATION PER SYMBOL
# --------------------------------------------------------------------------

latest = (
    df.dropna(subset=["date", "symbol"])
      .sort_values(["symbol", "date"])
      .groupby("symbol", as_index=False)
      .tail(1)
      .copy()
)

# --------------------------------------------------------------------------
# MERGE ESTABLISHED CLASSIFICATION
# --------------------------------------------------------------------------

classification = master[
    [
        "symbol",
        "full_name",
        "description",
        "sector",
        "security_class",
    ]
].copy()

latest = latest.merge(
    classification,
    on="symbol",
    how="left",
    validate="one_to_one"
)

# --------------------------------------------------------------------------
# FRESHNESS
# --------------------------------------------------------------------------

latest["days_stale"] = (
    AS_OF_DATE - latest["date"]
).dt.days

latest["missing_feature_count"] = (
    latest[FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .isna()
    .sum(axis=1)
)

latest["features_complete"] = (
    latest["missing_feature_count"] == 0
)

# --------------------------------------------------------------------------
# COMMON EQUITY ONLY
# --------------------------------------------------------------------------

common = latest[
    latest["security_class"].eq("COMMON_EQUITY")
].copy()

# --------------------------------------------------------------------------
# FRESHNESS BUCKET
# --------------------------------------------------------------------------

def freshness_bucket(days):
    if days <= 1:
        return "0-1 days"
    elif days <= 5:
        return "2-5 days"
    elif days <= 20:
        return "6-20 days"
    elif days <= 60:
        return "21-60 days"
    elif days <= 180:
        return "61-180 days"
    elif days <= 365:
        return "181-365 days"
    else:
        return ">365 days"

common["freshness_bucket"] = (
    common["days_stale"].apply(freshness_bucket)
)

# --------------------------------------------------------------------------
# OUTPUT
# --------------------------------------------------------------------------

print("=" * 78)
print("CELL 71 — CURRENT COMMON-EQUITY UNIVERSE")
print("=" * 78)

print(f"\nAs-of date              : {AS_OF_DATE.date()}")
print(f"Total historical symbols: {latest['symbol'].nunique():,}")
print(f"Common-equity symbols   : {len(common):,}")

print("\n" + "=" * 78)
print("COMMON-EQUITY FRESHNESS")
print("=" * 78)

order = [
    "0-1 days",
    "2-5 days",
    "6-20 days",
    "21-60 days",
    "61-180 days",
    "181-365 days",
    ">365 days",
]

print(
    common["freshness_bucket"]
    .value_counts()
    .reindex(order)
    .fillna(0)
    .astype(int)
    .to_string()
)

# --------------------------------------------------------------------------
# COMPLETENESS
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("FEATURE COMPLETENESS")
print("=" * 78)

print(
    f"Complete 13-feature rows: "
    f"{common['features_complete'].sum():,}"
)

print(
    f"Missing/infinite features: "
    f"{(~common['features_complete']).sum():,}"
)

# --------------------------------------------------------------------------
# OPERATIONAL UNIVERSES
# --------------------------------------------------------------------------

universes = {}

for days, label in [
    (1, "LIVE_1D"),
    (5, "LIVE_5D"),
    (20, "LIVE_20D"),
]:

    u = common[
        (common["days_stale"] <= days) &
        common["features_complete"]
    ].copy()

    universes[label] = u

    print(
        f"\nCommon equity <= {days:2d} day(s), "
        f"complete features: {len(u):,}"
    )

# --------------------------------------------------------------------------
# SHOW CURRENT <=5 DAY UNIVERSE
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("COMMON EQUITY <=5 DAYS — SAMPLE")
print("=" * 78)

live5 = universes["LIVE_5D"]

if len(live5):

    display(
        live5[
            [
                "symbol",
                "full_name",
                "sector",
                "date",
                "close",
                "days_stale",
            ]
        ]
        .sort_values(
            ["days_stale", "symbol"]
        )
        .head(100)
    )

# --------------------------------------------------------------------------
# SHOW STALE COMMON EQUITIES
# --------------------------------------------------------------------------

stale = common[
    common["days_stale"] > 20
].sort_values(
    "days_stale",
    ascending=False
)

print("\n" + "=" * 78)
print("COMMON EQUITIES >20 DAYS STALE")
print("=" * 78)

print(f"Count: {len(stale):,}")

if len(stale):

    display(
        stale[
            [
                "symbol",
                "full_name",
                "sector",
                "date",
                "close",
                "days_stale",
                "missing_feature_count",
            ]
        ].head(50)
    )

# --------------------------------------------------------------------------
# SAVE UNIVERSES
# --------------------------------------------------------------------------

OUT_DIR = os.path.join(
    PROJECT,
    "reports"
)

os.makedirs(OUT_DIR, exist_ok=True)

common.to_parquet(
    os.path.join(
        OUT_DIR,
        "cell71_common_equity_latest.parquet"
    ),
    index=False
)

for label, u in universes.items():

    u.to_parquet(
        os.path.join(
            OUT_DIR,
            f"cell71_{label.lower()}_universe.parquet"
        ),
        index=False
    )

print("\n" + "=" * 78)
print("SAVED")
print("=" * 78)

print(
    os.path.join(
        OUT_DIR,
        "cell71_common_equity_latest.parquet"
    )
)

print("\nFrozen model unchanged.")
print("No training performed.")
print("=" * 78)
print("CELL 71 COMPLETE")
print("=" * 78)

CELL 71 — CURRENT COMMON-EQUITY UNIVERSE

As-of date              : 2026-10-06
Total historical symbols: 651
Common-equity symbols   : 364

COMMON-EQUITY FRESHNESS
freshness_bucket
0-1 days        283
2-5 days          0
6-20 days         1
21-60 days        1
61-180 days       4
181-365 days      1
>365 days        74

FEATURE COMPLETENESS
Complete 13-feature rows: 328
Missing/infinite features: 36

Common equity <=  1 day(s), complete features: 261

Common equity <=  5 day(s), complete features: 261

Common equity <= 20 day(s), complete features: 261

COMMON EQUITY <=5 DAYS — SAMPLE


,symbol,full_name,sector,date,close,days_stale
0,ACLBSL,ACLBSL,MICROFINANCE,2026-10-06,878.0,0
1,ADBL,ADBL,BANKING,2026-10-06,309.9,0
7,AHL,AHL,HYDROPOWER,2026-10-06,368.0,0
8,AHPC,AHPC,HYDROPOWER,2026-10-06,252.0,0
11,AKJCL,AKJCL,HYDROPOWER,2026-10-06,351.0,0
...,...,...,...,...,...,...
204,KBSH,KBSH,HYDROPOWER,2026-10-06,810.0,0
206,KDL,KDL,HOTELS,2026-10-06,543.0,0
209,KKHC,KKHC,HYDROPOWER,2026-10-06,273.9,0
212,KMCDB,KMCDB,MICROFINANCE,2026-10-06,783.0,0



COMMON EQUITIES >20 DAYS STALE
Count: 80


,symbol,full_name,sector,date,close,days_stale,missing_feature_count
157,HFL,HFL,FINANCE,2013-09-29,18.00,4755,1
332,NFD,NFD,OTHERS,2014-08-03,34.35,4447,10
537,SHBL,SHBL,DEVBANK,2017-06-08,460.00,3407,0
612,SYFL,SYFL,FINANCE,2017-10-11,145.00,3282,0
145,HAMRO,HAMRO,DEVBANK,2018-07-15,135.00,3005,0
411,ODBL,ODBL,DEVBANK,2018-08-19,197.00,2970,0
449,PURBL,PURBL,DEVBANK,2019-03-10,188.00,2767,0
314,NBBL,NBBL,MICROFINANCE,2019-05-13,588.00,2703,0
214,KMFL,KMFL,MICROFINANCE,2019-05-13,830.00,2703,0
29,BHBL,BHBL,DEVBANK,2019-05-14,199.00,2702,0



SAVED
/content/NEPSE_ML_PROJECT/reports/cell71_common_equity_latest.parquet

Frozen model unchanged.
No training performed.
CELL 71 COMPLETE


In [18]:
# ==============================================================================
# CELL 72 — FINAL CURRENT COMMON-EQUITY ML RANKING
# ==============================================================================
#
# Frozen model:
#   cell58_stock_only_cross_sectional_xgboost.json
#
# Universe:
#   COMMON_EQUITY
#   <= 1 day stale
#   all 13 model features complete
#
# NO RETRAINING
# NO MODEL MODIFICATION
# NO MARKET FEATURES
# NO SECTOR FEATURES
# NO RELATIVE FEATURES
#
# ==============================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd
import xgboost as xgb

PROJECT = "/content/NEPSE_ML_PROJECT"

MODEL_PATH = os.path.join(
    PROJECT,
    "models",
    "cell58_stock_only_cross_sectional_xgboost.json"
)

UNIVERSE_PATH = os.path.join(
    PROJECT,
    "reports",
    "cell71_live_1d_universe.parquet"
)

# Cell 71 actually saved this exact filename:
if not os.path.exists(UNIVERSE_PATH):

    UNIVERSE_PATH = os.path.join(
        PROJECT,
        "reports",
        "cell71_live_1d_universe.parquet"
    )

OUTPUT = os.path.join(
    PROJECT,
    "reports",
    "cell72_current_common_equity_ranking.csv"
)

FEATURES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

EXPECTED_MODEL_SHA256 = (
    "2870f9f534e7a2d8ccc65ca719350e2d17fef52d7f2014baf89427e3ba0e3379"
)

AS_OF_DATE = pd.Timestamp("2026-10-06")

# --------------------------------------------------------------------------
# VERIFY MODEL
# --------------------------------------------------------------------------

with open(MODEL_PATH, "rb") as f:
    model_sha256 = hashlib.sha256(f.read()).hexdigest()

print("=" * 78)
print("CELL 72 — FINAL CURRENT COMMON-EQUITY ML RANKING")
print("=" * 78)

print("\nModel SHA256:")
print(model_sha256)

if model_sha256 != EXPECTED_MODEL_SHA256:
    raise RuntimeError(
        "MODEL HASH DOES NOT MATCH THE FROZEN CELL 58 MODEL."
    )

print("\nFrozen model integrity: PASS")

# --------------------------------------------------------------------------
# LOAD UNIVERSE
# --------------------------------------------------------------------------

universe = pd.read_parquet(
    UNIVERSE_PATH
).copy()

# Defensive checks
if len(universe) == 0:
    raise RuntimeError("Live universe is empty.")

missing_cols = [
    c for c in FEATURES
    if c not in universe.columns
]

if missing_cols:
    raise RuntimeError(
        f"Missing model features: {missing_cols}"
    )

# --------------------------------------------------------------------------
# HARD VALIDATION
# --------------------------------------------------------------------------

if not universe["security_class"].eq(
    "COMMON_EQUITY"
).all():

    raise RuntimeError(
        "Universe contains non-common-equity securities."
    )

if not (
    universe["days_stale"] <= 1
).all():

    raise RuntimeError(
        "Universe contains data older than 1 day."
    )

if not (
    universe["features_complete"]
).all():

    raise RuntimeError(
        "Universe contains incomplete feature rows."
    )

if universe[FEATURES].isna().any().any():
    raise RuntimeError(
        "NaN detected in model features."
    )

if np.isinf(
    universe[FEATURES].to_numpy(
        dtype=np.float64
    )
).any():

    raise RuntimeError(
        "Infinite value detected in model features."
    )

# --------------------------------------------------------------------------
# LOAD FROZEN MODEL
# --------------------------------------------------------------------------

model = xgb.XGBRegressor()

model.load_model(
    MODEL_PATH
)

# --------------------------------------------------------------------------
# PREDICT
# --------------------------------------------------------------------------

X = universe[FEATURES].astype(
    np.float32
)

predicted_return = model.predict(X)

universe["predicted_return_20d"] = predicted_return

# --------------------------------------------------------------------------
# RANK
# --------------------------------------------------------------------------

universe = universe.sort_values(
    "predicted_return_20d",
    ascending=False
).reset_index(drop=True)

universe["rank"] = (
    np.arange(len(universe)) + 1
)

universe["predicted_return_pct"] = (
    universe["predicted_return_20d"] * 100
)

# --------------------------------------------------------------------------
# PERCENTILE
# --------------------------------------------------------------------------

universe["model_percentile"] = (
    universe["predicted_return_20d"]
    .rank(
        method="min",
        pct=True
    ) * 100
)

# --------------------------------------------------------------------------
# OUTPUT COLUMNS
# --------------------------------------------------------------------------

ranking = universe[
    [
        "rank",
        "symbol",
        "full_name",
        "sector",
        "date",
        "close",
        "days_stale",
        "predicted_return_20d",
        "predicted_return_pct",
        "model_percentile",
    ]
].copy()

ranking.to_csv(
    OUTPUT,
    index=False
)

# --------------------------------------------------------------------------
# DISPLAY
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("CURRENT MODEL RANKING")
print("=" * 78)

print(f"\nAs-of date       : {AS_OF_DATE.date()}")
print(f"Eligible stocks  : {len(ranking):,}")
print(
    f"Latest observation: "
    f"{ranking['date'].min().date()} → "
    f"{ranking['date'].max().date()}"
)

print("\n" + "-" * 78)
print("TOP 30 STOCKS")
print("-" * 78)

display(
    ranking.head(30)
)

# --------------------------------------------------------------------------
# TOP 5 / 10 / 20 PERCENT
# --------------------------------------------------------------------------

print("\n" + "=" * 78)
print("PORTFOLIO-SIZE CUTS")
print("=" * 78)

for pct in [0.05, 0.10, 0.20]:

    n = max(
        1,
        int(np.ceil(len(ranking) * pct))
    )

    subset = ranking.head(n)

    print(
        f"\nTOP {int(pct*100)}% "
        f"({n} stocks)"
    )

    print(
        f"Mean predicted return : "
        f"{subset['predicted_return_pct'].mean():.2f}%"
    )

    print(
        f"Median predicted return: "
        f"{subset['predicted_return_pct'].median():.2f}%"
    )

    display(
        subset[
            [
                "rank",
                "symbol",
                "sector",
                "date",
                "close",
                "predicted_return_pct",
            ]
        ].head(20)
    )

# --------------------------------------------------------------------------
# SECTOR BREAKDOWN — DESCRIPTIVE ONLY
# --------------------------------------------------------------------------
# Sector is displayed for interpretation only.
# It is NOT an input to the model.

print("\n" + "=" * 78)
print("TOP 20 SECTOR DISTRIBUTION")
print("=" * 78)

print(
    ranking.head(20)["sector"]
    .value_counts()
    .to_string()
)

# --------------------------------------------------------------------------
# SAVE SUMMARY
# --------------------------------------------------------------------------

summary = {
    "as_of_date": str(AS_OF_DATE.date()),
    "universe": "COMMON_EQUITY",
    "freshness_rule": "<=1 day",
    "eligible_stocks": int(len(ranking)),
    "model": "cell58_stock_only_cross_sectional_xgboost",
    "model_sha256": model_sha256,
    "features": FEATURES,
    "market_features_used": False,
    "sector_features_used": False,
    "relative_features_used": False,
    "retrained": False,
    "top_5_count": int(np.ceil(len(ranking) * 0.05)),
    "top_10_count": int(np.ceil(len(ranking) * 0.10)),
    "top_20_count": int(np.ceil(len(ranking) * 0.20)),
}

SUMMARY_PATH = os.path.join(
    PROJECT,
    "reports",
    "cell72_current_ranking_summary.json"
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

print("\n" + "=" * 78)
print("SAVED")
print("=" * 78)

print(OUTPUT)
print(SUMMARY_PATH)

print("\nFrozen model unchanged.")
print("No training performed.")
print("=" * 78)
print("CELL 72 COMPLETE")
print("=" * 78)

CELL 72 — FINAL CURRENT COMMON-EQUITY ML RANKING

Model SHA256:
2870f9f534e7a2d8ccc65ca719350e2d17fef52d7f2014baf89427e3ba0e3379

Frozen model integrity: PASS

CURRENT MODEL RANKING

As-of date       : 2026-10-06
Eligible stocks  : 261
Latest observation: 2026-10-06 → 2026-10-06

------------------------------------------------------------------------------
TOP 30 STOCKS
------------------------------------------------------------------------------


,rank,symbol,full_name,sector,date,close,days_stale,predicted_return_20d,predicted_return_pct,model_percentile
0,1,SINDU,SINDU,DEVBANK,2026-10-06,418.0,0,0.499285,49.928520,100.000000
1,2,MKJC,MKJC,HYDROPOWER,2026-10-06,324.0,0,0.209227,20.922716,99.616858
2,3,HATHY,HATHY,INVESTMENT,2026-10-06,437.5,0,0.206995,20.699522,99.233716
3,4,JFL,JFL,FINANCE,2026-10-06,278.0,0,0.167672,16.767223,98.850575
4,5,ENL,ENL,INVESTMENT,2026-10-06,500.0,0,0.166957,16.695652,98.467433
5,6,TVCL,TVCL,HYDROPOWER,2026-10-06,288.0,0,0.134279,13.427888,98.084291
6,7,CORBL,CORBL,DEVBANK,2026-10-06,788.0,0,0.124533,12.453300,97.701149
7,8,KDL,KDL,HOTELS,2026-10-06,543.0,0,0.120318,12.031774,97.318008
8,9,MKCL,MKCL,OTHERS,2026-10-06,778.9,0,0.097951,9.795102,96.934866
9,10,SAPDBL,SAPDBL,DEVBANK,2026-10-06,584.0,0,0.086478,8.647821,96.551724



PORTFOLIO-SIZE CUTS

TOP 5% (14 stocks)
Mean predicted return : 15.12%
Median predicted return: 12.24%


,rank,symbol,sector,date,close,predicted_return_pct
0,1,SINDU,DEVBANK,2026-10-06,418.0,49.928520
1,2,MKJC,HYDROPOWER,2026-10-06,324.0,20.922716
2,3,HATHY,INVESTMENT,2026-10-06,437.5,20.699522
3,4,JFL,FINANCE,2026-10-06,278.0,16.767223
4,5,ENL,INVESTMENT,2026-10-06,500.0,16.695652
5,6,TVCL,HYDROPOWER,2026-10-06,288.0,13.427888
6,7,CORBL,DEVBANK,2026-10-06,788.0,12.453300
7,8,KDL,HOTELS,2026-10-06,543.0,12.031774
8,9,MKCL,OTHERS,2026-10-06,778.9,9.795102
9,10,SAPDBL,DEVBANK,2026-10-06,584.0,8.647821



TOP 10% (27 stocks)
Mean predicted return : 10.65%
Median predicted return: 6.96%


,rank,symbol,sector,date,close,predicted_return_pct
0,1,SINDU,DEVBANK,2026-10-06,418.0,49.928520
1,2,MKJC,HYDROPOWER,2026-10-06,324.0,20.922716
2,3,HATHY,INVESTMENT,2026-10-06,437.5,20.699522
3,4,JFL,FINANCE,2026-10-06,278.0,16.767223
4,5,ENL,INVESTMENT,2026-10-06,500.0,16.695652
5,6,TVCL,HYDROPOWER,2026-10-06,288.0,13.427888
6,7,CORBL,DEVBANK,2026-10-06,788.0,12.453300
7,8,KDL,HOTELS,2026-10-06,543.0,12.031774
8,9,MKCL,OTHERS,2026-10-06,778.9,9.795102
9,10,SAPDBL,DEVBANK,2026-10-06,584.0,8.647821



TOP 20% (53 stocks)
Mean predicted return : 7.30%
Median predicted return: 5.20%


,rank,symbol,sector,date,close,predicted_return_pct
0,1,SINDU,DEVBANK,2026-10-06,418.0,49.928520
1,2,MKJC,HYDROPOWER,2026-10-06,324.0,20.922716
2,3,HATHY,INVESTMENT,2026-10-06,437.5,20.699522
3,4,JFL,FINANCE,2026-10-06,278.0,16.767223
4,5,ENL,INVESTMENT,2026-10-06,500.0,16.695652
5,6,TVCL,HYDROPOWER,2026-10-06,288.0,13.427888
6,7,CORBL,DEVBANK,2026-10-06,788.0,12.453300
7,8,KDL,HOTELS,2026-10-06,543.0,12.031774
8,9,MKCL,OTHERS,2026-10-06,778.9,9.795102
9,10,SAPDBL,DEVBANK,2026-10-06,584.0,8.647821



TOP 20 SECTOR DISTRIBUTION
sector
HYDROPOWER      9
DEVBANK         4
INVESTMENT      2
FINANCE         1
HOTELS          1
OTHERS          1
MICROFINANCE    1
LIFEINSU        1

SAVED
/content/NEPSE_ML_PROJECT/reports/cell72_current_common_equity_ranking.csv
/content/NEPSE_ML_PROJECT/reports/cell72_current_ranking_summary.json

Frozen model unchanged.
No training performed.
CELL 72 COMPLETE


In [19]:
# ==============================================================================
# CELL 73 — FROZEN MODEL INFERENCE AUDIT
# ==============================================================================
# Purpose:
#   Audit today's deployed predictions before building the automatic daily
#   inference pipeline.
#
# IMPORTANT:
#   - NO training
#   - NO model modification
#   - NO feature modification
#   - NO market/sector features
#   - Uses the exact frozen 13-feature model
# ==============================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd
import xgboost as xgb

PROJECT = "/content/NEPSE_ML_PROJECT"

MODEL_PATH = os.path.join(
    PROJECT,
    "models",
    "cell58_stock_only_cross_sectional_xgboost.json"
)

RANKING_PATH = os.path.join(
    PROJECT,
    "reports",
    "cell72_current_common_equity_ranking.csv"
)

OUTPUT_PATH = os.path.join(
    PROJECT,
    "reports",
    "cell73_inference_audit.csv"
)

FEATURES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

# ------------------------------------------------------------------------------
# 1. MODEL INTEGRITY
# ------------------------------------------------------------------------------

sha256 = hashlib.sha256()

with open(MODEL_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

model_hash = sha256.hexdigest()

EXPECTED_HASH = (
    "2870f9f534e7a2d8ccc65ca719350e2d17fef52d7f2014baf89427e3ba0e3379"
)

print("=" * 78)
print("CELL 73 — FROZEN MODEL INFERENCE AUDIT")
print("=" * 78)

print("\nModel SHA256:")
print(model_hash)

print("\nExpected SHA256:")
print(EXPECTED_HASH)

print("\nModel integrity:", "PASS" if model_hash == EXPECTED_HASH else "FAIL")

if model_hash != EXPECTED_HASH:
    raise RuntimeError("FROZEN MODEL HASH MISMATCH — STOPPING.")

# ------------------------------------------------------------------------------
# 2. LOAD CURRENT RANKING
# ------------------------------------------------------------------------------

ranking = pd.read_csv(RANKING_PATH)

print("\n" + "=" * 78)
print("CURRENT RANKING")
print("=" * 78)

print("Rows:", len(ranking))
print("Unique symbols:", ranking["symbol"].nunique())

if "date" in ranking.columns:
    print("Date range:", ranking["date"].min(), "to", ranking["date"].max())

# ------------------------------------------------------------------------------
# 3. LOAD MODEL
# ------------------------------------------------------------------------------

model = xgb.XGBRegressor()
model.load_model(MODEL_PATH)

print("\nModel loaded successfully.")

# ------------------------------------------------------------------------------
# 4. FEATURE AVAILABILITY CHECK
# ------------------------------------------------------------------------------

missing_columns = [c for c in FEATURES if c not in ranking.columns]

print("\n" + "=" * 78)
print("FEATURE AVAILABILITY")
print("=" * 78)

if missing_columns:
    print("Missing feature columns:")
    for c in missing_columns:
        print("  -", c)
    raise RuntimeError("Required model features are missing from ranking file.")

print("All 13 frozen model features are present.")

# ------------------------------------------------------------------------------
# 5. FEATURE QUALITY
# ------------------------------------------------------------------------------

audit_rows = []

for feature in FEATURES:

    s = pd.to_numeric(ranking[feature], errors="coerce")

    finite = np.isfinite(s)

    row = {
        "feature": feature,
        "rows": len(s),
        "missing": int(s.isna().sum()),
        "infinite": int((~finite & ~s.isna()).sum()),
        "finite": int(finite.sum()),
        "min": float(s[finite].min()) if finite.any() else np.nan,
        "p01": float(s[finite].quantile(0.01)) if finite.any() else np.nan,
        "median": float(s[finite].median()) if finite.any() else np.nan,
        "p99": float(s[finite].quantile(0.99)) if finite.any() else np.nan,
        "max": float(s[finite].max()) if finite.any() else np.nan,
        "mean": float(s[finite].mean()) if finite.any() else np.nan,
        "std": float(s[finite].std()) if finite.any() else np.nan,
    }

    audit_rows.append(row)

audit_df = pd.DataFrame(audit_rows)

print("\n" + "=" * 78)
print("FEATURE QUALITY")
print("=" * 78)

print(
    audit_df[
        [
            "feature",
            "missing",
            "infinite",
            "min",
            "p01",
            "median",
            "p99",
            "max",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------------------------
# 6. PREDICTION SANITY
# ------------------------------------------------------------------------------

pred = pd.to_numeric(
    ranking["predicted_return_20d"],
    errors="coerce"
)

print("\n" + "=" * 78)
print("PREDICTION SANITY")
print("=" * 78)

print("Prediction count:", pred.notna().sum())
print("Prediction missing:", pred.isna().sum())

if pred.notna().any():

    print("Minimum prediction :", f"{pred.min():.6f}")
    print("1st percentile     :", f"{pred.quantile(.01):.6f}")
    print("Median prediction  :", f"{pred.median():.6f}")
    print("99th percentile    :", f"{pred.quantile(.99):.6f}")
    print("Maximum prediction :", f"{pred.max():.6f}")

# ------------------------------------------------------------------------------
# 7. EXTREME PREDICTIONS
# ------------------------------------------------------------------------------

print("\n" + "=" * 78)
print("EXTREME PREDICTIONS")
print("=" * 78)

extreme = ranking[
    (ranking["predicted_return_20d"] > 0.30) |
    (ranking["predicted_return_20d"] < -0.30)
].copy()

if len(extreme):

    cols = [
        "rank",
        "symbol",
        "sector",
        "date",
        "close",
        "predicted_return_20d",
    ]

    print(
        extreme[cols]
        .sort_values("predicted_return_20d", ascending=False)
        .to_string(index=False)
    )

else:
    print("No predictions outside ±30%.")

# ------------------------------------------------------------------------------
# 8. TOP 20 FEATURE VALUES
# ------------------------------------------------------------------------------

print("\n" + "=" * 78)
print("TOP 20 — MODEL INPUT AUDIT")
print("=" * 78)

top20 = ranking.sort_values(
    "predicted_return_20d",
    ascending=False
).head(20)

display_cols = [
    "rank",
    "symbol",
    "sector",
    "close",
    "predicted_return_pct",
] + FEATURES

print(
    top20[display_cols].to_string(index=False)
)

# ------------------------------------------------------------------------------
# 9. CHECK FOR DUPLICATE SYMBOLS
# ------------------------------------------------------------------------------

duplicate_symbols = ranking[
    ranking["symbol"].duplicated(keep=False)
].sort_values("symbol")

print("\n" + "=" * 78)
print("DUPLICATE SYMBOL CHECK")
print("=" * 78)

print("Duplicate symbol rows:", len(duplicate_symbols))

if len(duplicate_symbols):
    print(duplicate_symbols[["symbol", "date", "close"]].to_string(index=False))
else:
    print("PASS — one row per symbol.")

# ------------------------------------------------------------------------------
# 10. CHECK RANKING ORDER
# ------------------------------------------------------------------------------

expected_sorted = ranking.sort_values(
    "predicted_return_20d",
    ascending=False
).reset_index(drop=True)

rank_order_ok = np.all(
    expected_sorted["symbol"].values ==
    ranking.sort_values("rank")["symbol"].values
)

print("\n" + "=" * 78)
print("RANK ORDER CHECK")
print("=" * 78)

print(
    "PASS"
    if rank_order_ok
    else "FAIL"
)

# ------------------------------------------------------------------------------
# 11. SAVE AUDIT
# ------------------------------------------------------------------------------

audit_df.to_csv(OUTPUT_PATH, index=False)

print("\n" + "=" * 78)
print("SAVED")
print("=" * 78)

print(OUTPUT_PATH)

print("\n" + "=" * 78)
print("CELL 73 COMPLETE")
print("=" * 78)

CELL 73 — FROZEN MODEL INFERENCE AUDIT

Model SHA256:
2870f9f534e7a2d8ccc65ca719350e2d17fef52d7f2014baf89427e3ba0e3379

Expected SHA256:
2870f9f534e7a2d8ccc65ca719350e2d17fef52d7f2014baf89427e3ba0e3379

Model integrity: PASS

CURRENT RANKING
Rows: 261
Unique symbols: 261
Date range: 2026-10-06 to 2026-10-06

Model loaded successfully.

FEATURE AVAILABILITY
Missing feature columns:
  - return_1d
  - return_5d
  - return_20d
  - return_60d
  - close_to_sma_20
  - close_to_sma_50
  - close_to_sma_200
  - rsi_14
  - volatility_20d
  - volume_ratio_20
  - volume_change_1d
  - volume_spike_20d
  - trading_activity_score


RuntimeError: Required model features are missing from ranking file.

In [20]:
# ==============================================================================
# CELL 73A — ACTUAL FROZEN MODEL INPUT AUDIT
# ==============================================================================
# Audits the REAL 261-row feature universe used by Cell 72.
#
# IMPORTANT:
#   - No training
#   - No model modification
#   - No feature modification
#   - Uses exact frozen 13-feature model
#   - Reproduces Cell 72 predictions independently
# ==============================================================================

import os
import hashlib
import numpy as np
import pandas as pd
import xgboost as xgb

PROJECT = "/content/NEPSE_ML_PROJECT"

MODEL_PATH = os.path.join(
    PROJECT,
    "models",
    "cell58_stock_only_cross_sectional_xgboost.json"
)

UNIVERSE_PATH = os.path.join(
    PROJECT,
    "reports",
    "cell71_live_1d_universe.parquet"
)

RANKING_PATH = os.path.join(
    PROJECT,
    "reports",
    "cell72_current_common_equity_ranking.csv"
)

OUTPUT_PATH = os.path.join(
    PROJECT,
    "reports",
    "cell73_actual_input_audit.csv"
)

FEATURES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

EXPECTED_HASH = (
    "2870f9f534e7a2d8ccc65ca719350e2d17fef52d7f2014baf89427e3ba0e3379"
)

print("=" * 78)
print("CELL 73A — ACTUAL FROZEN MODEL INPUT AUDIT")
print("=" * 78)

# ------------------------------------------------------------------------------
# 1. MODEL HASH
# ------------------------------------------------------------------------------

sha256 = hashlib.sha256()

with open(MODEL_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

actual_hash = sha256.hexdigest()

print("\nModel SHA256:")
print(actual_hash)

print("\nExpected SHA256:")
print(EXPECTED_HASH)

print(
    "\nModel integrity:",
    "PASS" if actual_hash == EXPECTED_HASH else "FAIL"
)

if actual_hash != EXPECTED_HASH:
    raise RuntimeError("FROZEN MODEL HASH MISMATCH.")

# ------------------------------------------------------------------------------
# 2. LOAD ACTUAL INPUT UNIVERSE
# ------------------------------------------------------------------------------

df = pd.read_parquet(UNIVERSE_PATH)

print("\n" + "=" * 78)
print("ACTUAL MODEL INPUT UNIVERSE")
print("=" * 78)

print("Rows:", len(df))
print("Unique symbols:", df["symbol"].nunique())

if "date" in df.columns:
    print("Date range:", df["date"].min(), "to", df["date"].max())

# ------------------------------------------------------------------------------
# 3. BASIC UNIVERSE VALIDATION
# ------------------------------------------------------------------------------

if "security_class" in df.columns:
    print("\nSecurity classes:")
    print(df["security_class"].value_counts(dropna=False).to_string())

    bad_class = df[df["security_class"] != "COMMON_EQUITY"]

    print("\nNon-common-equity rows:", len(bad_class))

    if len(bad_class):
        print(bad_class[["symbol", "security_class"]].to_string(index=False))

else:
    print("\nWARNING: security_class column not present.")

# ------------------------------------------------------------------------------
# 4. FRESHNESS CHECK
# ------------------------------------------------------------------------------

if "days_stale" in df.columns:

    print("\n" + "=" * 78)
    print("FRESHNESS")
    print("=" * 78)

    print(df["days_stale"].describe().to_string())

    stale = df[df["days_stale"] > 1]

    print("\nRows >1 day stale:", len(stale))

    if len(stale):
        print(
            stale[
                ["symbol", "date", "days_stale"]
            ].sort_values("days_stale", ascending=False)
            .to_string(index=False)
        )

# ------------------------------------------------------------------------------
# 5. FEATURE AVAILABILITY
# ------------------------------------------------------------------------------

missing_columns = [c for c in FEATURES if c not in df.columns]

print("\n" + "=" * 78)
print("FEATURE AVAILABILITY")
print("=" * 78)

if missing_columns:
    print("Missing:")
    for c in missing_columns:
        print("  -", c)

    raise RuntimeError(
        "The actual Cell 71 universe does not contain all frozen model features."
    )

print("All 13 frozen model features are present.")

# ------------------------------------------------------------------------------
# 6. FEATURE QUALITY
# ------------------------------------------------------------------------------

audit_rows = []

for feature in FEATURES:

    s = pd.to_numeric(df[feature], errors="coerce")

    finite_mask = np.isfinite(s)

    audit_rows.append({
        "feature": feature,
        "rows": len(s),
        "missing": int(s.isna().sum()),
        "infinite": int((~finite_mask & ~s.isna()).sum()),
        "finite": int(finite_mask.sum()),
        "min": float(s[finite_mask].min()) if finite_mask.any() else np.nan,
        "p01": float(s[finite_mask].quantile(0.01))
            if finite_mask.any() else np.nan,
        "median": float(s[finite_mask].median())
            if finite_mask.any() else np.nan,
        "p99": float(s[finite_mask].quantile(0.99))
            if finite_mask.any() else np.nan,
        "max": float(s[finite_mask].max())
            if finite_mask.any() else np.nan,
        "mean": float(s[finite_mask].mean())
            if finite_mask.any() else np.nan,
        "std": float(s[finite_mask].std())
            if finite_mask.any() else np.nan,
    })

audit_df = pd.DataFrame(audit_rows)

print("\n" + "=" * 78)
print("FEATURE QUALITY")
print("=" * 78)

print(
    audit_df[
        [
            "feature",
            "missing",
            "infinite",
            "min",
            "p01",
            "median",
            "p99",
            "max",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------------------------
# 7. EXACT MODEL INPUT MATRIX
# ------------------------------------------------------------------------------

X = df[FEATURES].copy()

# Cell 71 was supposed to contain complete features.
# Validate rather than silently imputing anything.

X = X.replace([np.inf, -np.inf], np.nan)

missing_input_rows = X.isna().any(axis=1)

print("\n" + "=" * 78)
print("MODEL INPUT COMPLETENESS")
print("=" * 78)

print("Rows with complete 13 features:", int((~missing_input_rows).sum()))
print("Rows with missing features:", int(missing_input_rows.sum()))

if missing_input_rows.any():
    print("\nRows with missing features:")
    print(
        df.loc[
            missing_input_rows,
            ["symbol", "date"]
        ].to_string(index=False)
    )

    raise RuntimeError(
        "Unexpected missing model inputs in Cell 71 live universe."
    )

# ------------------------------------------------------------------------------
# 8. LOAD FROZEN MODEL
# ------------------------------------------------------------------------------

model = xgb.XGBRegressor()
model.load_model(MODEL_PATH)

print("\nFrozen model loaded.")

# ------------------------------------------------------------------------------
# 9. REPRODUCE CELL 72 PREDICTIONS
# ------------------------------------------------------------------------------

pred = model.predict(X)

df["audit_prediction"] = pred

print("\n" + "=" * 78)
print("PREDICTION SANITY")
print("=" * 78)

print("Prediction rows:", len(pred))
print("Minimum:", f"{pred.min():.8f}")
print("1%:", f"{np.quantile(pred, .01):.8f}")
print("Median:", f"{np.median(pred):.8f}")
print("99%:", f"{np.quantile(pred, .99):.8f}")
print("Maximum:", f"{pred.max():.8f}")

# ------------------------------------------------------------------------------
# 10. COMPARE AGAINST CELL 72 CSV
# ------------------------------------------------------------------------------

ranking = pd.read_csv(RANKING_PATH)

comparison = df[
    ["symbol", "audit_prediction"]
].merge(
    ranking[
        ["symbol", "rank", "predicted_return_20d"]
    ],
    on="symbol",
    how="outer",
    validate="one_to_one"
)

comparison["prediction_difference"] = (
    comparison["audit_prediction"]
    - comparison["predicted_return_20d"]
)

max_difference = comparison["prediction_difference"].abs().max()

print("\n" + "=" * 78)
print("CELL 72 REPRODUCTION CHECK")
print("=" * 78)

print("Symbols in Cell 71:", len(df))
print("Symbols in Cell 72:", len(ranking))
print("Max prediction difference:", max_difference)

if max_difference < 1e-8:
    print("PASS — Cell 72 predictions reproduced exactly.")
elif max_difference < 1e-5:
    print("PASS — Cell 72 predictions reproduced within floating-point tolerance.")
else:
    print("FAIL — prediction mismatch detected.")

# ------------------------------------------------------------------------------
# 11. EXTREME PREDICTIONS + INPUTS
# ------------------------------------------------------------------------------

print("\n" + "=" * 78)
print("TOP 20 — ACTUAL MODEL INPUTS")
print("=" * 78)

top20 = df.sort_values(
    "audit_prediction",
    ascending=False
).head(20)

display_cols = [
    "symbol",
    "date",
    "close",
    "audit_prediction",
] + FEATURES

print(
    top20[display_cols].to_string(index=False)
)

# ------------------------------------------------------------------------------
# 12. SAVE AUDIT
# ------------------------------------------------------------------------------

audit_df.to_csv(OUTPUT_PATH, index=False)

print("\n" + "=" * 78)
print("SAVED")
print("=" * 78)

print(OUTPUT_PATH)

print("\n" + "=" * 78)
print("CELL 73A COMPLETE")
print("=" * 78)

CELL 73A — ACTUAL FROZEN MODEL INPUT AUDIT

Model SHA256:
2870f9f534e7a2d8ccc65ca719350e2d17fef52d7f2014baf89427e3ba0e3379

Expected SHA256:
2870f9f534e7a2d8ccc65ca719350e2d17fef52d7f2014baf89427e3ba0e3379

Model integrity: PASS

ACTUAL MODEL INPUT UNIVERSE
Rows: 261
Unique symbols: 261
Date range: 2026-10-06 00:00:00 to 2026-10-06 00:00:00

Security classes:
security_class
COMMON_EQUITY    261

Non-common-equity rows: 0

FRESHNESS
count    261.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0

Rows >1 day stale: 0

FEATURE AVAILABILITY
All 13 frozen model features are present.

FEATURE QUALITY
               feature  missing  infinite       min       p01    median       p99       max
             return_1d        0         0 -0.029962 -0.026949  0.002331  0.035273  0.051095
             return_5d        0         0 -0.087124 -0.073922 -0.009434  0.048860  0.101186
            return_20d        0         0 -0.388604 -0.261833  0.

In [21]:
# ==============================================================================
# CELL 74 — RECOVER EXACT ORIGINAL TECHNICAL FEATURE FORMULAS
# ==============================================================================
# Purpose:
#   Recover the exact feature-engineering code used earlier in this project.
#
# This is important before building the automatic daily refresh pipeline.
# We must NOT approximate the formulas.
# ==============================================================================

import os
import re
import sqlite3
from pathlib import Path

FEATURE_NAMES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "close_to_sma_20",
    "close_to_sma_50",
    "close_to_sma_200",
    "rsi_14",
    "volatility_20d",
    "volume_ratio_20",
    "volume_change_1d",
    "volume_spike_20d",
    "trading_activity_score",
]

print("=" * 78)
print("CELL 74 — RECOVER EXACT ORIGINAL FEATURE FORMULAS")
print("=" * 78)

# ------------------------------------------------------------------------------
# 1. SEARCH IPYTHON HISTORY DATABASES
# ------------------------------------------------------------------------------

history_paths = []

candidates = [
    "/root/.ipython/profile_default/history.sqlite",
    "/root/.ipython/profile_default/history.sqlite3",
    "/content/.ipython/profile_default/history.sqlite",
    "/content/.ipython/profile_default/history.sqlite3",
]

for p in candidates:
    if os.path.exists(p):
        history_paths.append(p)

# Also search likely IPython locations.
for root in [
    "/root/.ipython",
    "/content/.ipython",
]:
    if os.path.exists(root):
        for path in Path(root).rglob("history.sqlite*"):
            if str(path) not in history_paths:
                history_paths.append(str(path))

print("\nHistory databases found:")
for p in history_paths:
    print(" -", p)

if not history_paths:
    raise RuntimeError(
        "No IPython history database found in this Colab runtime."
    )

# ------------------------------------------------------------------------------
# 2. SEARCH CODE CELLS
# ------------------------------------------------------------------------------

matches = []

for db_path in history_paths:

    try:
        conn = sqlite3.connect(db_path)

        cur = conn.cursor()

        cur.execute("""
            SELECT session, line, source
            FROM history
            ORDER BY session, line
        """)

        rows = cur.fetchall()

        conn.close()

    except Exception as e:
        print("\nCould not read:", db_path)
        print("Reason:", e)
        continue

    for session, line, source in rows:

        if not source:
            continue

        hit_features = [
            feature
            for feature in FEATURE_NAMES
            if feature in source
        ]

        if hit_features:
            matches.append({
                "db": db_path,
                "session": session,
                "line": line,
                "features": hit_features,
                "source": source,
            })

# ------------------------------------------------------------------------------
# 3. DISPLAY MATCHING CELLS
# ------------------------------------------------------------------------------

print("\n" + "=" * 78)
print("MATCHING CODE CELLS")
print("=" * 78)

print("Matching history entries:", len(matches))

for i, item in enumerate(matches):

    print("\n" + "-" * 78)
    print(
        f"MATCH {i+1} | "
        f"session={item['session']} | "
        f"line={item['line']}"
    )

    print("Features found:")
    print(", ".join(item["features"]))

    print("\nSOURCE:")
    print(item["source"])

# ------------------------------------------------------------------------------
# 4. SAVE ALL MATCHES
# ------------------------------------------------------------------------------

output_path = (
    "/content/NEPSE_ML_PROJECT/reports/"
    "cell74_feature_formula_history.txt"
)

with open(output_path, "w", encoding="utf-8") as f:

    f.write(
        "CELL 74 — EXACT ORIGINAL FEATURE FORMULA HISTORY\n"
    )
    f.write("=" * 78 + "\n\n")

    for i, item in enumerate(matches):

        f.write(
            f"MATCH {i+1}\n"
            f"Database: {item['db']}\n"
            f"Session: {item['session']}\n"
            f"Line: {item['line']}\n"
            f"Features: {', '.join(item['features'])}\n\n"
        )

        f.write(item["source"])
        f.write("\n\n")
        f.write("=" * 78)
        f.write("\n\n")

print("\n" + "=" * 78)
print("SAVED")
print("=" * 78)

print(output_path)

print("\n" + "=" * 78)
print("CELL 74 COMPLETE")
print("=" * 78)

CELL 74 — RECOVER EXACT ORIGINAL FEATURE FORMULAS

History databases found:
 - /root/.ipython/profile_default/history.sqlite

MATCHING CODE CELLS
Matching history entries: 0

SAVED
/content/NEPSE_ML_PROJECT/reports/cell74_feature_formula_history.txt

CELL 74 COMPLETE
